# Figure 5, S10, S23, S23bis, S24 — growth model (ML chain)

Original 2FAST_ML pipeline code, vendored verbatim in `ml/` (only paths and imports changed); driver `ml/run.py` runs exactly the notebook cells the paper depends on (original notebooks 30, 31b, 33, 36 as a script, 40, 43, 46).

* Dataset: 189 selected → **151** (`hardcut`); the t_conv model is trained on the 156-run `clean` particle set (original choice, `step36.load_conv`).
* Growth time = the pipeline's own plateau rule (`step36.convergence_time_v2`), not the platform rule.
* **Fig 5 top and S23bis colour maps are regenerated with the current growth model** (decision (a), 2026-10-12); the manuscript version came from a stale cache (errata).

Data products → `data/processed/ml/`, figures → `figures/ml/` (original file names). From raw/ the run takes ≈ 30 min; caches already in `data/processed/ml/` are reused.

## Analysis code
All the code of this figure is in this notebook.

### `paths`

In [ ]:
import sys
import types

from IPython.core.magic import register_cell_magic

MODULE_SOURCE = {}


@register_cell_magic
def module(line, cell):
    """`%%module a.b`: run this cell as the Python module a.b, so that the original code keeps its own namespaces."""
    name = line.strip()
    parts = name.split(".")
    for i in range(1, len(parts)):                              # parent packages
        pkg = ".".join(parts[:i])
        if pkg not in sys.modules:
            sys.modules[pkg] = types.ModuleType(pkg)
            sys.modules[pkg].__path__ = []
            if i > 1:
                setattr(sys.modules[".".join(parts[:i - 1])], parts[i - 1], sys.modules[pkg])
    mod = types.ModuleType(name)
    mod.__package__ = name.rpartition(".")[0]
    sys.modules[name] = mod
    setattr(sys.modules[mod.__package__], parts[-1], mod)
    MODULE_SOURCE[name] = cell
    exec(compile(cell, f"<{name}>", "exec"), mod.__dict__)


In [ ]:
%%module toofast_analysis.paths
"""Repository paths."""
import os
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment.yml").exists())   # notebook: no __file__
# raw/ can live elsewhere (e.g. an extracted Zenodo archive): set TOOFAST_RAW
RAW = Path(os.environ.get("TOOFAST_RAW", ROOT / "raw"))
PROCESSED = ROOT / "data" / "processed"
INPUTS = ROOT / "data" / "inputs"      # frozen artefacts used by the platform during the runs (versioned, read-only)
RESOURCE_DIR = INPUTS
FIGURES = ROOT / "figures"


In [ ]:
from toofast_analysis.paths import *

### `ml.catalog_2fast`
*Original analysis code.*

In [ ]:
%%module toofast_analysis.ml.catalog_2fast
# [vendored] verbatim copy of 2FAST_ML/catalog_2fast.py (md5 75e3fabcda385493168709da6d85972d), generated by tools/vendor_ml.py — do not edit by hand.
"""
catalog_2fast.py
================
Cataloguing pipeline for 2FAST H5 experimental files.

Scans a root directory tree, parses each H5 file (SD and RG families),
classifies experiments, and exports a curated CSV catalogue ready for
GP pre-training and inter-session drift diagnostics.

Usage
-----
    python catalog_2fast.py --root /path/to/data --output catalogue.csv

Classification logic
--------------------
Filename pattern      | Flow condition                    | exp_type
----------------------|-----------------------------------|------------------
*_sample_*            | all flows == 2.8 µL/s (±tol)     | STANDARD
*_sample_*            | flows not all equal               | LHS
*_init_*              | any                               | BO_INIT
*_iter_*              | any                               | BO_ITER

n_pumps_free : number of pumps NOT fixed at 2.8 µL/s (relevant for BO sub-space)

Quality flags
-------------
- flag_low_abs   : final abs_max < ABS_MIN_THRESHOLD (likely failed experiment)
- flag_invalid   : parsing error or missing critical field
"""

import h5py
import numpy as np
import pandas as pd
import re
import argparse
import logging
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional

# ── Configuration ────────────────────────────────────────────────────────────

STANDARD_FLOW     = 2.8          # µL/s — nominal standard flow rate
STANDARD_TOL      = 0.15         # µL/s — tolerance for standard detection (±5%)
ABS_MIN_THRESHOLD = 0.05         # below this abs_max → flag_low_abs
N_PUMPS           = 5

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger(__name__)


# ── Data structure ────────────────────────────────────────────────────────────

@dataclass
class ExperimentRecord:
    # File identity
    filepath:       str  = ""
    filename:       str  = ""
    h5_family:      str  = ""       # file format
    # Experimental hierarchy
    session:        str  = ""       # e.g. S00 / S12
    manip:          str  = ""       # e.g. S00 / S12-M04
    sample_id:      str  = ""       # e.g. iter_2 / sample_3
    timestamp:      str  = ""
    # Classification
    exp_type:       str  = ""       # STANDARD / LHS / BO_INIT / BO_ITER
    n_pumps_free:   int  = -1       # 0-5
    # Flow rates (set points)
    Q1:             float = np.nan
    Q2:             float = np.nan
    Q3:             float = np.nan
    Q4:             float = np.nan
    Q5:             float = np.nan
    Q_total_set:    float = np.nan
    Q_total_meas:   float = np.nan
    # Spectral output
    lambda_max:     float = np.nan  # final λmax (nm)
    abs_max:        float = np.nan  # final absorbance at λmax
    # Kinetic features (from time-series)
    lmax_initial:   float = np.nan  # λmax at t=0 (first spectrum)
    lmax_final:     float = np.nan  # λmax at convergence (last spectrum)
    abs_initial:    float = np.nan  # abs_max at t=0
    abs_final:      float = np.nan  # abs_max at convergence
    n_spectra:      int   = -1      # number of spectra in time series
    # Quality
    flag_low_abs:   bool  = False
    flag_invalid:   bool  = False
    parse_error:    str   = ""


# ── H5 parsers ───────────────────────────────────────────────────────────────

def parse_sample(f: h5py.File, rec: ExperimentRecord) -> None:
    """Parse a sample H5 (format 2fast-sample 2.0)."""

    # Flow rates: commanded, fallback to measured
    if 'conditions/flow_set' in f:
        flows = f['conditions/flow_set'][()]
    elif 'conditions/flow_measured' in f:
        flows = f['conditions/flow_measured'][()]
    else:
        raise KeyError("No flow rate dataset found")

    rec.Q1, rec.Q2, rec.Q3, rec.Q4, rec.Q5 = flows
    rec.Q_total_set  = float(np.sum(flows))
    if 'conditions/flow_measured' in f:
        rec.Q_total_meas = float(np.sum(f['conditions/flow_measured'][()]))

    # fixed/varying pump masks
    if 'conditions/varying_pumps' in f:
        rec.n_pumps_free = int(np.sum(f['conditions/varying_pumps'][()]))

    # Metadata
    rec.timestamp = str(f['sample'].attrs.get('timestamp', ''))
    rec.session   = str(f['sample'].attrs.get('run', ''))

    # Spectral output
    rec.lambda_max = float(f['result'].attrs['lambda_max_nm'])
    rec.abs_max    = float(f['result'].attrs['extinction_max'])

    # Kinetic time series
    if 'spectra/lambda_max' in f:
        lmax_ts = f['spectra/lambda_max'][()]
        abs_ts  = f['spectra/extinction_max'][()]
        rec.lmax_initial = float(lmax_ts[0])
        rec.lmax_final   = float(lmax_ts[-1])
        rec.abs_initial  = float(abs_ts[0])
        rec.abs_final    = float(abs_ts[-1])
        rec.n_spectra    = len(lmax_ts)


# ── Classification helpers ────────────────────────────────────────────────────

def classify_experiment(filename: str, flows: np.ndarray) -> tuple[str, int]:
    """
    Returns (exp_type, n_pumps_free).

    exp_type  : STANDARD | LHS | BO_INIT | BO_ITER
    n_pumps_free : number of pumps not fixed at STANDARD_FLOW
    """
    stem = filename.lower()

    # Identify pumps at standard flow
    at_standard = np.abs(flows - STANDARD_FLOW) <= STANDARD_TOL
    n_at_std    = int(np.sum(at_standard))
    n_free      = N_PUMPS - n_at_std

    if '_iter_' in stem:
        exp_type = 'BO_ITER'
    elif '_init_' in stem:
        exp_type = 'BO_INIT'
    elif '_sample_' in stem:
        all_standard = np.all(at_standard)
        exp_type = 'STANDARD' if all_standard else 'LHS'
    else:
        exp_type = 'UNKNOWN'

    return exp_type, n_free


# Intermediate directory names that are structural (not manip identifiers)
_STRUCTURAL_DIRS = {"samples", "previous_samples", "data", "results", "output", "exports", "documents"}

def parse_session_manip(filepath: Path, metadata_session: str, family: str) -> tuple[str, str]:
    """
    Infer (session, manip) from directory structure and metadata.

    Handles two layouts:
      Standard layout : .../SESSION/MANIP/filename.h5
        e.g.  S00/S00/S00.h5
              S12/S12-M04/S12-M04_sample_3.h5

      SD standard layout (with intermediate structural subfolder):
        .../SESSION/MANIP/samples/filename.h5
        e.g.  S00/S00/samples/S00.h5

    Any directory name in _STRUCTURAL_DIRS is skipped when walking up
    the path to find the manip and session directories.
    """
    parts = filepath.parts

    # Keep only experiment-like directory names (starting with SD or RG).
    # This filters out structural dirs AND Windows system folders alike.
    _EXP_PREFIX = re.compile(r'^S\d', re.IGNORECASE)  # must start with SD/RG + digit
    candidates = [
        p for p in parts[:-1]
        if _EXP_PREFIX.match(p) and p.lower() not in _STRUCTURAL_DIRS
    ]

    if len(candidates) >= 2:
        manip   = candidates[-1]   # nearest non-structural parent
        session = candidates[-2]   # grandparent (skipping structural dirs)
    elif len(candidates) == 1:
        # Manip directly under root (no session folder) — infer session from manip name
        # e.g. S00 → S00  |  S00 → S00
        manip = candidates[0]
        match = re.match(r'^(S\d+)', manip)  # S00 or S00, not S00
        session = match.group(1) if match else manip
    elif len(parts) >= 3:
        manip   = parts[-2]
        session = parts[-3]
    else:
        # Last resort: parse from metadata experiment_name
        manip   = metadata_session
        match   = re.match(r'^(.+?)(?:_\d+)?$', metadata_session)
        session = match.group(1) if match else metadata_session

    return session, manip


# ── Main parser ───────────────────────────────────────────────────────────────

def parse_h5_file(filepath: Path) -> ExperimentRecord:
    """Parse a single H5 file and return an ExperimentRecord."""
    rec = ExperimentRecord()
    rec.filepath = str(filepath)
    rec.filename = filepath.stem

    try:
        rec.h5_family = "2.0"

        with h5py.File(filepath, 'r') as f:
            parse_sample(f, rec)

            rec.sample_id = str(f['sample'].attrs.get('sample_id', ''))

        # Classification
        flows = np.array([rec.Q1, rec.Q2, rec.Q3, rec.Q4, rec.Q5])
        rec.exp_type, n_free = classify_experiment(rec.filename, flows)

        # n_pumps_free: use explicit mask from RG if available, else infer
        if rec.n_pumps_free < 0:
            rec.n_pumps_free = n_free

        # Session / manip hierarchy
        rec.session, rec.manip = parse_session_manip(filepath, rec.session, rec.h5_family)

        # Quality flags
        rec.flag_low_abs = (rec.abs_max < ABS_MIN_THRESHOLD)

    except Exception as e:
        rec.flag_invalid = True
        rec.parse_error  = str(e)
        log.warning(f"Parse error on {filepath.name}: {e}")

    return rec


# ── Directory scanner ─────────────────────────────────────────────────────────

def scan_directory(root: Path) -> pd.DataFrame:
    """Recursively scan root for H5 files and build catalogue DataFrame."""
    h5_files = (list(root) if isinstance(root, (list, tuple))   # [vendored] explicit ordered list
                else sorted(root.rglob("*.h5")))
    # Exclude files inside previous_samples/ — they are copies of older
    # experiments reused as BO init points and would create duplicates
    h5_files = [f for f in h5_files
                if "previous_samples" not in f.parts]
    log.info(f"Found {len(h5_files)} H5 files under "   # [vendored] do not print the whole list
             f"{'the explicit list' if isinstance(root, (list, tuple)) else root} "
             f"(previous_samples/ excluded)")

    records = []
    for i, fp in enumerate(h5_files):
        if i % 50 == 0:
            log.info(f"  Parsing {i}/{len(h5_files)} ...")
        rec = parse_h5_file(fp)
        records.append(asdict(rec))

    df = pd.DataFrame(records)

    # Summary statistics
    log.info("\n=== Catalogue summary ===")
    log.info(f"Total files   : {len(df)}")
    log.info(f"Invalid files : {df['flag_invalid'].sum()}")
    log.info(f"Low abs files : {df['flag_low_abs'].sum()}")
    log.info("\nBy exp_type:")
    log.info(df['exp_type'].value_counts().to_string())
    log.info("\nBy h5_family:")
    log.info(df['h5_family'].value_counts().to_string())
    log.info("\nBy session (top 10):")
    log.info(df['session'].value_counts().head(10).to_string())

    return df


# ── CLI ───────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="Catalogue 2FAST H5 experimental files.")
    parser.add_argument('--root',   required=True, help="Root directory containing H5 files")
    parser.add_argument('--output', default='catalogue.csv', help="Output CSV path")
    parser.add_argument('--debug',  action='store_true', help="Verbose debug logging")
    args = parser.parse_args()

    if args.debug:
        logging.getLogger().setLevel(logging.DEBUG)

    root = Path(args.root)
    if not root.exists():
        log.error(f"Root directory not found: {root}")
        return

    df = scan_directory(root)
    df.to_csv(args.output, index=False)
    log.info(f"\nCatalogue saved → {args.output}")

    # Quick validity report
    valid = df[~df['flag_invalid']]
    log.info(f"\nValid records  : {len(valid)}")
    log.info(f"λmax range     : {valid['lambda_max'].min():.1f} – {valid['lambda_max'].max():.1f} nm")
    log.info(f"abs_max range  : {valid['abs_max'].min():.4f} – {valid['abs_max'].max():.4f}")


if __name__ == '__main__':
    main()


### `ml.anchor_2fast`
*Original analysis code.*

In [ ]:
%%module toofast_analysis.ml.anchor_2fast
# [vendored] verbatim copy of 2FAST_ML/anchor_2fast.py (md5 5709b756c3d2207c2fe78a4389dad982), generated by tools/vendor_ml.py — do not edit by hand.
"""
anchor_2fast.py
===============
Standard qualification and temporal anchoring for 2FAST dataset.

Two-step pipeline
-----------------
Step 1 — Standard group qualification
    For each manip sub-directory containing STANDARD points, compute the
    intra-group spread (max - min of λmax). If spread ≤ STABILITY_THRESHOLD,
    the group is declared STABLE and its mean λmax is the anchor value δ_manip.
    Otherwise the group is UNSTABLE (warm-up / tuning run) and is excluded
    from training.

Step 2 — Temporal anchoring of LHS / BO manips
    Each non-STANDARD *manip* is anchored to the STANDARD manip that immediately
    precedes it in time within the same session (e.g. BO manip S00 →
    standard manip S00). All samples of the manip share that anchor. If no
    standard precedes it, fall back to the temporally closest standard manip and
    flag anchor_is_preceding=False.

    The anchor value is the mean λmax of the anchor standard manip's samples.
    STANDARD manips are the references and are NOT corrected — their anchor_*
    columns stay empty / NaN.

    The bias / drift estimate is the deviation of the anchor standard from the
    dataset-wide standard reference:

        anchor_bias = anchor_lambda_mean − global_standard_mean

    This is the additive correction subtracted from λmax. It is NOT computed
    from the LHS/BO point itself (that would mix the genuine chemistry of the
    explored flow condition with the session bias).

    Manips with no standard at all in the same session are flagged
    flag_no_anchor=True and kept but marked for downstream handling.

Output columns added to the catalogue
--------------------------------------
std_group_stable    : bool   — True if the manip's standard group is stable
                               (STANDARD rows only)
anchor_manip        : str    — manip name of the anchor standard
anchor_lambda_mean  : float  — mean λmax of that anchor manip's samples (nm)
anchor_bias         : float  — anchor_lambda_mean − global_standard_mean (nm)
                               → the additive drift/correction term
anchor_dt_min       : float  — time distance to the anchor manip (minutes)
anchor_is_preceding : bool   — True if the anchor precedes the manip in time
anchor_stable       : bool   — True if the chosen anchor standard was stable
flag_no_anchor      : bool   — True if no standard found in session
                               (STANDARD rows have all anchor_* empty too)

A module-level attribute GLOBAL_STANDARD_MEAN is also set after the pipeline
runs, holding the dataset-wide reference used for anchor_bias.
"""

import numpy as np
import pandas as pd
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────────
STABILITY_THRESHOLD = 10.0   # nm — max(λmax) - min(λmax) within a standard group

# Set after run_anchoring_pipeline(): dataset-wide reference λmax of all stable
# STANDARD points. Exposed so notebooks can report / reuse it.
GLOBAL_STANDARD_MEAN = np.nan


# ── Step 1 : Standard group qualification ────────────────────────────────────

def qualify_standard_groups(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each manip sub-directory that contains STANDARD points, compute
    intra-group spread and flag as stable / unstable.

    Adds columns:
        std_group_n        : int   — number of STANDARD points in the manip
        std_group_spread   : float — max(λmax) - min(λmax) within the group
        std_group_mean     : float — mean λmax of the group
        std_group_stable   : bool  — spread ≤ STABILITY_THRESHOLD
    """
    df = df.copy()
    df["std_group_n"]      = np.nan
    df["std_group_spread"] = np.nan
    df["std_group_mean"]   = np.nan
    df["std_group_stable"] = False

    std_mask = df["exp_type"] == "STANDARD"
    std_df   = df[std_mask]

    for manip, grp in std_df.groupby("manip"):
        idx    = grp.index
        lmax   = grp["lambda_max"].values
        n      = len(lmax)
        spread = float(lmax.max() - lmax.min()) if n > 1 else 0.0
        mean   = float(lmax.mean())
        stable = spread <= STABILITY_THRESHOLD

        df.loc[idx, "std_group_n"]      = n
        df.loc[idx, "std_group_spread"] = spread
        df.loc[idx, "std_group_mean"]   = mean
        df.loc[idx, "std_group_stable"] = stable

    return df


def build_anchor_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a table of candidate anchor manips: one row per STANDARD manip
    (stable OR not), with the mean λmax of its samples and its start time.

    The anchor value is the *mean λmax of the standard manip's samples* — e.g. a
    standard manip with 3 samples contributes the average of the three λmax.

    Stability is reported (anchor_stable) but NOT used to filter candidates: a
    measurement should be anchored to the standard manip that actually precedes
    it in time, even if that standard was flagged unstable. Downstream tiering
    can downgrade points whose chosen anchor is unstable.

    Columns: session, manip, anchor_lambda_mean, anchor_t0, anchor_n, anchor_stable
    """
    std = df[df["exp_type"] == "STANDARD"].copy()
    std["_ts"] = pd.to_datetime(std["timestamp"], errors="coerce")

    anchors = (
        std
        .groupby(["session", "manip"])
        .agg(
            anchor_lambda_mean=("lambda_max", "mean"),
            anchor_t0=("_ts", "min"),
            anchor_n=("lambda_max", "count"),
            anchor_stable=("std_group_stable", "first"),
        )
        .reset_index()
    )
    return anchors


def compute_global_standard_mean(df: pd.DataFrame) -> float:
    """
    Dataset-wide reference λmax: mean of *all* stable STANDARD points.

    Per the chosen convention this is a point-level mean (each stable standard
    measurement counts once), not a mean of per-manip means.
    """
    std_stable = df[(df["exp_type"] == "STANDARD") & (df["std_group_stable"] == True)]
    if std_stable.empty:
        return float("nan")
    return float(std_stable["lambda_max"].mean())


# ── Step 2 : Temporal anchoring ───────────────────────────────────────────────

def assign_temporal_anchors(df: pd.DataFrame) -> pd.DataFrame:
    """
    Anchor each non-STANDARD *manip* to the STANDARD manip that immediately
    precedes it in time within the same session. If no standard precedes it,
    fall back to the temporally closest standard manip (anchor_is_preceding=False).

    All samples of a non-standard manip share the same anchor (the previous
    standard manip), so e.g. BO manip S00 is anchored to standard manip
    S00. The anchor value is the mean λmax of that standard manip's samples.

    STANDARD manips are the anchors themselves and are NOT corrected: their
    anchor_* columns are left empty / NaN.

    The correction term is derived from the STANDARDS ONLY:
        anchor_bias = anchor_lambda_mean − global_standard_mean

    Adds columns: anchor_manip, anchor_lambda_mean, anchor_bias, anchor_dt_min,
    anchor_is_preceding, anchor_stable, flag_no_anchor.
    """
    global GLOBAL_STANDARD_MEAN
    df = df.copy()

    global_mean = compute_global_standard_mean(df)
    GLOBAL_STANDARD_MEAN = global_mean

    # Initialise output columns (STANDARD rows keep these empty — never corrected)
    df["anchor_manip"]        = ""
    df["anchor_lambda_mean"]  = np.nan
    df["anchor_bias"]         = np.nan
    df["anchor_dt_min"]       = np.nan
    df["anchor_is_preceding"] = False
    df["anchor_stable"]       = False
    df["flag_no_anchor"]      = False

    df["_ts"] = pd.to_datetime(df["timestamp"], errors="coerce")

    anchors = build_anchor_table(df)

    # Representative time of each non-standard manip = its first sample's timestamp
    nonstd_mask = df["exp_type"] != "STANDARD"
    manip_t0 = (
        df[nonstd_mask]
        .groupby(["session", "manip"])["_ts"].min()
        .reset_index().rename(columns={"_ts": "manip_t0"})
    )

    for _, m in manip_t0.iterrows():
        session, manip, t0 = m["session"], m["manip"], m["manip_t0"]
        # only the non-STANDARD rows of this manip get an anchor/correction;
        # any STANDARD samples in the same manip stay references (anchor_* empty)
        rows = df.index[(df["session"] == session) & (df["manip"] == manip) &
                        (df["exp_type"] != "STANDARD")]
        sess_anchors = anchors[(anchors["session"] == session) &
                               (anchors["manip"] != manip)]

        if sess_anchors.empty or pd.isna(t0):
            df.loc[rows, "flag_no_anchor"] = True
            continue

        gap = (t0 - sess_anchors["anchor_t0"]).dt.total_seconds() / 60.0  # >0 → precedes
        preceding = gap[gap >= 0]
        if not preceding.empty:
            best_idx     = preceding.idxmin()   # most recent standard before the manip
            is_preceding = True
        else:
            best_idx     = gap.idxmax()         # gap<0 everywhere; closest following
            is_preceding = False

        best = sess_anchors.loc[best_idx]
        df.loc[rows, "anchor_manip"]        = best["manip"]
        df.loc[rows, "anchor_lambda_mean"]  = best["anchor_lambda_mean"]
        df.loc[rows, "anchor_bias"]         = best["anchor_lambda_mean"] - global_mean
        df.loc[rows, "anchor_dt_min"]       = float(abs(gap[best_idx]))
        df.loc[rows, "anchor_is_preceding"] = is_preceding
        df.loc[rows, "anchor_stable"]       = bool(best["anchor_stable"])

    df = df.drop(columns=["_ts"])
    return df


# ── Top-level pipeline ────────────────────────────────────────────────────────

def run_anchoring_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    """
    Full pipeline: qualify standards → assign anchors.
    Returns enriched DataFrame.
    """
    df = qualify_standard_groups(df)
    df = assign_temporal_anchors(df)
    return df


# ── Summary helpers ───────────────────────────────────────────────────────────

def print_anchor_summary(df: pd.DataFrame) -> None:
    std = df[df["exp_type"] == "STANDARD"]
    n_stable   = (std["std_group_stable"] == True).sum()
    n_unstable = (std["std_group_stable"] == False).sum()

    non_std = df[df["exp_type"] != "STANDARD"]
    n_anchored   = (~non_std["flag_no_anchor"]).sum()
    n_unanchored = non_std["flag_no_anchor"].sum()

    print("=== Anchoring summary ===")
    print(f"Standard groups   : {std['manip'].nunique()} manips")
    print(f"  stable          : {std[std['std_group_stable']==True]['manip'].nunique()}")
    print(f"  unstable        : {std[std['std_group_stable']==False]['manip'].nunique()}")
    print(f"  STANDARD points : {n_stable} stable / {n_unstable} unstable")
    print()
    print(f"Non-standard points : {len(non_std)}")
    print(f"  anchored        : {n_anchored}")
    print(f"    of which preceding-anchor : "
          f"{(non_std['anchor_is_preceding'] & ~non_std['flag_no_anchor']).sum()}")
    print(f"  flag_no_anchor  : {n_unanchored}")
    print()
    print(f"Global standard reference (stable STDs) : {GLOBAL_STANDARD_MEAN:.2f} nm")
    if n_anchored > 0:
        anc = ~df["flag_no_anchor"] & (df["exp_type"] != "STANDARD")
        bias = df.loc[anc, "anchor_bias"]
        print(f"Anchor bias stats (non-std, anchored) — the correction term:")
        print(f"  mean  = {bias.mean():+.2f} nm")
        print(f"  std   = {bias.std():.2f} nm")
        print(f"  range = [{bias.min():+.1f}, {bias.max():+.1f}] nm")


### `ml.eda_rg024`
*Original analysis code.*

In [ ]:
%%module toofast_analysis.ml.eda_rg024
# [vendored] verbatim copy of 2FAST_ML/eda_rg024.py (md5 eb3b3486db20c4f4a51cbb7b4c1e19c2), generated by tools/vendor_ml.py — do not edit by hand.
"""
eda_rg024.py
============
Exploratory data analysis (pre-ML) for the curated 2FAST training set
(`csv_romain`): 100 composition-varying (LHS) silver-nanoparticle syntheses
drawn from 6 manips (S12-M04, S13-M09, S14-M14, S15-M05, S15-M08, S19-M01).

Microfluidic reactor; target = surface-plasmon-resonance wavelength
(lambda_max) of the in-line UV-Vis spectrum. Five reagent pumps (Q1..Q5) are
varied on a fixed-total-flow simplex (sum Q = 14 uL/s):

    Q1 = AgNO3        (silver precursor)
    Q2 = PVP          (capping / shape & size control)
    Q3 = acetic acid  (pH / reduction kinetics)
    Q4 = ascorbic acid(mild reductant -> growth)
    Q5 = NaBH4        (strong reductant -> nucleation / seeds)

This script loads every H5 file of the set, then produces publication-format
figures (English labels, 300 dpi PNG + vector PDF) in ./figures_eda/.

Run:  py eda_rg024.py
The notebook 10_eda_S14.ipynb mirrors this code cell-by-cell.
"""

from pathlib import Path
import re
import numpy as np
import pandas as pd
import h5py
from scipy.signal import savgol_filter
import matplotlib as mpl
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------------
# 0. Configuration
# ----------------------------------------------------------------------------
DATASET    = "csv_romain"   # curated training set (label for figure titles)
NOTEBOOK   = "10_eda_S14.ipynb"   # provenance tag stamped on every saved figure
from .. import paths as _paths
DATA_ROOT  = None   # [vendored] csv_romain flat copy: not used by the ML chain (Fig 3 = toofast_analysis.lhs100)
FIG_DIR    = _paths.FIGURES / "ml"   # [vendored] was ./figures_eda
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Reagent identity and stock concentrations (from metadata.txt)
REAGENTS = ["AgNO3", "PVP", "AcOH", "AscAcid", "NaBH4"]
STOCK    = np.array([1.8, 1.2, 1256.0, 1.8, 0.036])   # pump 1..5 stock conc.
QCOLS    = ["Q1", "Q2", "Q3", "Q4", "Q5"]
STANDARD_FLOW, STANDARD_TOL = 2.8, 0.15

# Convergence / quality thresholds (match the instrument's termination_* settings)
AMP_THR, CORR_THR, MIN_SPECTRA = 0.02, 0.998, 10   # Δabs_max<2%, Pearson>0.998, ≥10 spectra
FLAT_STD_THR = 0.03                                # flat spectrum: std(absorbance) below this

# Exploration: 4 representative samples whose full spectra + time evolution to plot.
# Fill with filenames (e.g. "S14-M14_sample_12"); empty -> auto-pick by λmax spread.
REPRESENTATIVE_SAMPLES = ["S19-M01_sample_17", "S19-M01_sample_1",
                          "S19-M01_sample_12", "S12-M04_sample_12"]
EVOL_WL_RANGE = (315, 950)      # wavelength window for the evolution panels (nm)
SG_WINDOW, SG_POLY = 15, 3      # Savitzky-Golay smoothing (window points, poly order)

# Colour-blind-friendly palette assigned to manips (filled in after loading)
PALETTE = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9",
           "#F0E442", "#999999"]

# fig8/fig9 (scatter_amax_lmax): growth-time colour scale -- tweak freely to try
# other palettes/scales (any matplotlib colormap name works for GROWTH_CMAP).
GROWTH_CMAP       = "plasma"
GROWTH_VMAX       = 200      # s -- colour scale upper bound
GROWTH_SPLIT      = 20       # s -- boundary between the expanded/compressed zones
GROWTH_SPLIT_FRAC = 0.75     # fraction of the colourbar given to [0, GROWTH_SPLIT]

# fig8/fig9 annotation regimes (data-driven, see the diagnostic in-chat):
FLAT_GROWN_EXTMAX = 0.15     # Ext_max (a.u.) threshold for the flat|grown line
SPHERE_LMAX       = (375, 415)     # nm -- 395 +/- 20 (widened from +/-10)
DISK_LMAX         = (420, 750)     # nm -- widened + extended to 750 nm (was 420-600,
                                   # a narrower fit-optimised band); now a qualitative
                                   # "disk region" box rather than a precise fit -> no
                                   # r annotation shown (see DISK_SHOW_R below)
DISK_BAND_K       = 1.3      # shaded dispersion band half-width, in residual std
DISK_SHOW_R       = False    # annotate "disks (r = ..)" on the band? off by request

# ----------------------------------------------------------------------------
# 1. Publication matplotlib style
# ----------------------------------------------------------------------------
def set_pub_style():
    mpl.rcParams.update({
        "figure.dpi":        120,
        "savefig.dpi":       300,
        "savefig.bbox":      "tight",
        "font.family":       "sans-serif",
        "font.sans-serif":   ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size":         11,
        "axes.titlesize":    12,
        "axes.labelsize":    11,
        "axes.linewidth":    0.9,
        "axes.spines.top":   False,
        "axes.spines.right": False,
        "xtick.direction":   "out",
        "ytick.direction":   "out",
        "xtick.labelsize":   10,
        "ytick.labelsize":   10,
        "legend.fontsize":   9,
        "legend.frameon":    False,
        "lines.linewidth":   1.4,
        "figure.facecolor":  "white",
        "axes.facecolor":    "white",
    })

set_pub_style()

def savefig(fig, name):
    """Save a figure as PNG + PDF in FIG_DIR, stamped with the source notebook."""
    fig.text(0.995, 0.003, f"source: {NOTEBOOK}", ha="right", va="bottom",
             fontsize=6, color="0.55")
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}")
    print(f"  saved {name}.png / .pdf")

def conv_norm(vmin=0, vmax=GROWTH_VMAX, split=GROWTH_SPLIT, split_frac=GROWTH_SPLIT_FRAC):
    """Piecewise colour normalisation: max colour spread over [0, split] s
    (split_frac of the bar) and compressed over [split, vmax] s. Defaults come
    from the GROWTH_* config constants -- tweak those to try other scales."""
    def fwd(x):
        x = np.asarray(x, float)
        return np.where(x <= split,
                        split_frac * (x - vmin) / (split - vmin),
                        split_frac + (1 - split_frac) * (x - split) / (vmax - split))
    def inv(y):
        y = np.asarray(y, float)
        return np.where(y <= split_frac,
                        vmin + y / split_frac * (split - vmin),
                        split + (y - split_frac) / (1 - split_frac) * (vmax - split))
    return mpl.colors.FuncNorm((fwd, inv), vmin=vmin, vmax=vmax)

# ----------------------------------------------------------------------------
# 2. Load all H5 files of the set
# ----------------------------------------------------------------------------
def classify(filename, flows):
    stem = filename.lower()
    at_std = np.abs(np.asarray(flows) - STANDARD_FLOW) <= STANDARD_TOL
    if "_iter_" in stem:
        return "BO_ITER"
    if "_init_" in stem:
        return "BO_INIT"
    if "_sample_" in stem:
        return "STANDARD" if at_std.all() else "LHS"
    return "UNKNOWN"

def manip_of(filename):
    """S12-M04_sample_3 -> S12-M04 (works for flat layout)."""
    return re.split(r"_(?:sample|init|iter)_", filename)[0]

def load_session(root):
    """Return (df_scalar, spectra_dict). Each spectrum entry keyed by filename."""
    files = sorted(root.rglob("*.h5"))
    files = [f for f in files if "previous_samples" not in f.parts]
    rows, spectra = [], {}
    for fp in files:
        try:
            with h5py.File(fp, "r") as f:
                flows = f["conditions/flow_set"][()]
                lam   = f["spectra/lambda_max"][()]
                amx   = f["spectra/extinction_max"][()]
                ts    = [t.decode() if isinstance(t, bytes) else t
                         for t in f["spectra/timestamp"][()]]
                wl    = f["spectra/wavelength"][()]
                absb  = f["spectra/extinction"][()]
                meta  = dict(f["sample"].attrs)
                stat  = dict(f["result"].attrs)
            name  = fp.stem
            etype = classify(name, flows)
            try:
                tsec = (pd.to_datetime(ts) - pd.to_datetime(ts[0])).total_seconds().values
            except Exception:
                tsec = np.arange(len(lam), dtype=float)
            rows.append({
                "filename":   name,
                "manip":      manip_of(name),
                "sample_id":  meta.get("sample_id", ""),
                "timestamp":  meta.get("timestamp", ""),
                "exp_type":   etype,
                "Q1": flows[0], "Q2": flows[1], "Q3": flows[2],
                "Q4": flows[3], "Q5": flows[4],
                "Q_total":    float(np.sum(flows)),
                "lambda_max": float(stat.get("lambda_max_nm", lam[-1])),
                "abs_max":    float(stat.get("extinction_max", amx[-1])),
                "lmax_initial": float(lam[0]),
                "abs_initial":  float(amx[0]),
                "n_spectra":  len(lam),
                "growth_time_s": float(tsec[-1]),
            })
            spectra[name] = {"wl": wl, "abs": absb, "lam_ts": lam,
                             "abs_ts": amx, "t": tsec}
        except Exception as e:
            print(f"  skip {fp.name}: {e}")
    df = pd.DataFrame(rows)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.sort_values(["manip", "timestamp"]).reset_index(drop=True)
    df["seq"] = np.arange(len(df))
    return df, spectra

def manip_colors(df):
    manips = sorted(df.manip.unique())
    return {m: PALETTE[i % len(PALETTE)] for i, m in enumerate(manips)}

# molar amounts proportional to Q_i * stock_i -> mechanistic ratios
def add_chem_features(df):
    mol = df[QCOLS].values * STOCK
    ag  = mol[:, 0]
    df["reductant_to_Ag"]      = (mol[:, 3] + mol[:, 4]) / ag
    df["NaBH4_to_Ag"]          = mol[:, 4] / ag
    df["nucleation_vs_growth"] = mol[:, 4] / np.where(mol[:, 3] == 0, np.nan, mol[:, 3])
    df["PVP_to_Ag"]            = mol[:, 1] / ag
    df["acid_to_Ag"]           = mol[:, 2] / ag
    return df

# Convergence time recomputed from the spectral time series, using the
# instrument's own termination criteria between consecutive spectra.
def convergence_time(sp, amp=AMP_THR, corr=CORR_THR, kmin=MIN_SPECTRA):
    """First elapsed time where consecutive spectra satisfy both criteria.

    Returns (time_s, index, status) with status in {'ok','never','too_short'}.
    """
    A   = np.asarray(sp["abs"])      # (n, n_wl) full spectra over time
    amx = np.asarray(sp["abs_ts"])   # (n,) abs_max over time
    t   = np.asarray(sp["t"])        # (n,) elapsed seconds
    n   = len(t)
    if n < kmin:
        return np.nan, n - 1, "too_short"
    for i in range(1, n):
        if (i + 1) < kmin:           # need >= kmin spectra acquired first
            continue
        d_amp = abs(amx[i] - amx[i - 1]) / max(abs(amx[i - 1]), 1e-9)
        r = np.corrcoef(A[i], A[i - 1])[0, 1]
        if d_amp < amp and r > corr:
            return float(t[i]), i, "ok"
    return np.nan, n - 1, "never"

def flat_std(sp):
    """Std of the final absorbance spectrum across wavelengths (flatness)."""
    return float(np.asarray(sp["abs"][-1]).std())

def enrich_convergence(df, spectra):
    """Add recomputed convergence time and the flat-spectrum flag to df."""
    ct, idx, st = zip(*(convergence_time(spectra[f]) for f in df.filename))
    df["conv_time_s"] = ct
    df["conv_index"]  = idx
    df["conv_status"] = st
    df["abs_std"]     = [flat_std(spectra[f]) for f in df.filename]
    df["is_flat"]     = df.abs_std < FLAT_STD_THR
    return df

# ----------------------------------------------------------------------------
# 3. Figures
# ----------------------------------------------------------------------------
def fig1_lmax_amax(df, cmap_manip):
    """Distributions of lambda_max & A_max, and their joint relationship."""
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
    manips = sorted(df.manip.unique())

    bins = np.linspace(df.lambda_max.min() - 5, df.lambda_max.max() + 5, 28)
    ax[0].hist([df.loc[df.manip == m, "lambda_max"] for m in manips],
               bins=bins, stacked=True,
               color=[cmap_manip[m] for m in manips], label=manips)
    ax[0].set_xlabel(r"$\lambda_{\max}$ (nm)"); ax[0].set_ylabel("Count")
    ax[0].set_title("(a) Plasmon peak position")
    ax[0].legend(title="Manip", fontsize=7)

    bins2 = np.linspace(df.abs_max.min(), df.abs_max.max(), 28)
    ax[1].hist([df.loc[df.manip == m, "abs_max"] for m in manips],
               bins=bins2, stacked=True, color=[cmap_manip[m] for m in manips])
    ax[1].set_xlabel(r"$A_{\max}$ (a.u.)"); ax[1].set_ylabel("Count")
    ax[1].set_title("(b) Peak absorbance")

    for m in manips:
        s = df[df.manip == m]
        ax[2].scatter(s.lambda_max, s.abs_max, s=26, alpha=0.85,
                      color=cmap_manip[m], edgecolor="white", linewidth=0.4)
    ax[2].set_xlabel(r"$\lambda_{\max}$ (nm)"); ax[2].set_ylabel(r"$A_{\max}$ (a.u.)")
    ax[2].set_title("(c) Peak position vs. intensity")

    fig.suptitle(f"{DATASET}: final spectral descriptors  (N = {len(df)})",
                 y=1.04, fontsize=13, fontweight="bold")
    fig.tight_layout()
    return fig

def fig2_spectra(df, spectra, cmap_manip):
    """All final UV-Vis spectra coloured by lambda_max + mean spectrum per manip."""
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))

    norm = mpl.colors.Normalize(df.lambda_max.min(), df.lambda_max.max())
    cmap = mpl.cm.viridis
    for _, r in df.iterrows():
        sp = spectra[r.filename]
        ax[0].plot(sp["wl"], sp["abs"][-1], color=cmap(norm(r.lambda_max)),
                   lw=0.8, alpha=0.7)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cb = fig.colorbar(sm, ax=ax[0]); cb.set_label(r"$\lambda_{\max}$ (nm)")
    ax[0].set_xlabel("Wavelength (nm)"); ax[0].set_ylabel("Absorbance (a.u.)")
    ax[0].set_title(f"(a) All final spectra (N = {len(df)})")

    # mean spectrum per manip on a common wavelength grid
    wl0 = spectra[df.iloc[0].filename]["wl"]
    for m in sorted(df.manip.unique()):
        sub = df[df.manip == m]
        stack = np.vstack([np.interp(wl0, spectra[r.filename]["wl"],
                                     spectra[r.filename]["abs"][-1])
                           for _, r in sub.iterrows()])
        mean, std = stack.mean(0), stack.std(0)
        ax[1].plot(wl0, mean, color=cmap_manip[m], lw=1.3, label=f"{m} (n={len(sub)})")
        ax[1].fill_between(wl0, mean - std, mean + std, color=cmap_manip[m], alpha=0.12)
    ax[1].set_xlabel("Wavelength (nm)"); ax[1].set_ylabel("Absorbance (a.u.)")
    ax[1].set_title("(b) Mean spectrum per manip ($\\pm 1\\sigma$)")
    ax[1].legend(fontsize=7)

    fig.suptitle(f"{DATASET}: in-line UV-Vis spectra", y=1.02,
                 fontsize=13, fontweight="bold")
    fig.tight_layout()
    return fig

def fig3_kinetics(df, spectra):
    """Growth kinetics: lambda_max(t) / A_max(t) and growth-time distribution."""
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
    norm = mpl.colors.Normalize(df.lambda_max.min(), df.lambda_max.max())
    cmap = mpl.cm.viridis
    for _, r in df.iterrows():
        sp = spectra[r.filename]; c = cmap(norm(r.lambda_max))
        ax[0].plot(sp["t"], sp["lam_ts"], color=c, lw=0.7, alpha=0.6)
        ax[1].plot(sp["t"], sp["abs_ts"], color=c, lw=0.7, alpha=0.6)
    ax[0].set_xlabel("Elapsed time (s)"); ax[0].set_ylabel(r"$\lambda_{\max}$ (nm)")
    ax[0].set_title("(a) Peak-position kinetics")
    ax[1].set_xlabel("Elapsed time (s)"); ax[1].set_ylabel(r"$A_{\max}$ (a.u.)")
    ax[1].set_title("(b) Absorbance kinetics")
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cb = fig.colorbar(sm, ax=ax[1]); cb.set_label(r"final $\lambda_{\max}$ (nm)")

    ct = df.conv_time_s.dropna()
    ax[2].hist(ct, bins=20, color="#0072B2", alpha=0.85)
    ax[2].set_xlabel("Convergence time (s)"); ax[2].set_ylabel("Count")
    ax[2].set_title("(c) Time to convergence")
    ax[2].axvline(ct.median(), color="k", ls="--", lw=1)
    ax[2].text(0.97, 0.95,
               f"median = {ct.median():.0f} s\n(criteria-based, "
               f"n={len(ct)} converged)",
               transform=ax[2].transAxes, ha="right", va="top", fontsize=9)

    fig.suptitle(f"{DATASET}: nanoparticle growth kinetics", y=1.04,
                 fontsize=13, fontweight="bold")
    fig.tight_layout()
    return fig

def fig4_manip_variation(df, cmap_manip):
    """Per-manip distribution of lambda_max and A_max (inter-manip variability)."""
    manips = sorted(df.manip.unique())
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    for axis, col, lab in ((ax[0], "lambda_max", r"$\lambda_{\max}$ (nm)"),
                           (ax[1], "abs_max", r"$A_{\max}$ (a.u.)")):
        data = [df.loc[df.manip == m, col].values for m in manips]
        bp = axis.boxplot(data, vert=True, patch_artist=True, widths=0.6,
                          showfliers=False, medianprops=dict(color="k"))
        for patch, m in zip(bp["boxes"], manips):
            patch.set_facecolor(cmap_manip[m]); patch.set_alpha(0.5)
        for i, m in enumerate(manips):
            y = df.loc[df.manip == m, col].values
            x = np.random.normal(i + 1, 0.06, len(y))
            axis.scatter(x, y, s=14, color=cmap_manip[m], edgecolor="white",
                         linewidth=0.3, zorder=3)
        axis.set_xticks(range(1, len(manips) + 1))
        axis.set_xticklabels(manips, rotation=30, ha="right", fontsize=8)
        axis.set_ylabel(lab)
    ax[0].set_title("(a) $\\lambda_{\\max}$ by manip")
    ax[1].set_title("(b) $A_{\\max}$ by manip")
    fig.suptitle(f"{DATASET}: variability across manips", y=1.03,
                 fontsize=13, fontweight="bold")
    fig.tight_layout()
    return fig

def fig5_coverage(df):
    """Composition coverage: pump pair scatter coloured by lambda_max."""
    fig, axes = plt.subplots(5, 5, figsize=(11, 11))
    norm = mpl.colors.Normalize(df.lambda_max.min(), df.lambda_max.max())
    cmap = mpl.cm.viridis
    for i in range(5):
        for j in range(5):
            ax = axes[i, j]
            if i == j:
                ax.hist(df[QCOLS[i]], bins=14, color="#888888"); ax.set_yticks([])
            else:
                ax.scatter(df[QCOLS[j]], df[QCOLS[i]], c=df.lambda_max,
                           cmap=cmap, norm=norm, s=14, alpha=0.8,
                           edgecolor="white", linewidth=0.2)
            if i == 4: ax.set_xlabel(f"{QCOLS[j]}\n{REAGENTS[j]}", fontsize=9)
            else:      ax.set_xticklabels([])
            if j == 0: ax.set_ylabel(f"{QCOLS[i]}\n{REAGENTS[i]}", fontsize=9)
            else:      ax.set_yticklabels([])
            ax.tick_params(labelsize=7)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cb = fig.colorbar(sm, ax=axes, shrink=0.6, pad=0.02)
    cb.set_label(r"$\lambda_{\max}$ (nm)")
    fig.suptitle(f"{DATASET}: simplex coverage of the 5-pump space "
                 f"(sum Q = 14 uL/s)", y=0.92, fontsize=13, fontweight="bold")
    return fig

def fig6_variogram(df):
    """lambda_max smoothness: |delta lambda| vs composition distance."""
    X = df[QCOLS].values; y = df.lambda_max.values; n = len(df)
    di, dl = [], []
    for a in range(n):
        for b in range(a + 1, n):
            di.append(np.linalg.norm(X[a] - X[b])); dl.append(abs(y[a] - y[b]))
    di, dl = np.array(di), np.array(dl)

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    ax[0].scatter(di, dl, s=8, alpha=0.25, color="#0072B2", edgecolor="none")
    edges = np.linspace(0, di.max(), 13); cen = 0.5 * (edges[:-1] + edges[1:])
    means = [dl[(di >= edges[k]) & (di < edges[k + 1])].mean()
             if ((di >= edges[k]) & (di < edges[k + 1])).any() else np.nan
             for k in range(len(cen))]
    ax[0].plot(cen, means, "o-", color="#D55E00", lw=1.6, label="binned mean")
    ax[0].set_xlabel("Composition distance  ||$\\Delta Q$|| (uL/s)")
    ax[0].set_ylabel(r"$|\Delta \lambda_{\max}|$ (nm)")
    ax[0].set_title("(a) Empirical variogram"); ax[0].legend()

    feats = ["reductant_to_Ag", "NaBH4_to_Ag", "nucleation_vs_growth",
             "PVP_to_Ag", "acid_to_Ag"]
    labels = ["reductant/Ag", "NaBH4/Ag", "NaBH4/AscAcid", "PVP/Ag", "acid/Ag"]
    cors = [np.corrcoef(df[f], df.lambda_max)[0, 1] for f in feats]
    order = np.argsort(np.abs(cors))
    ax[1].barh([labels[k] for k in order], [cors[k] for k in order],
               color=["#D55E00" if cors[k] < 0 else "#0072B2" for k in order])
    ax[1].axvline(0, color="k", lw=0.8)
    ax[1].set_xlabel(r"Pearson $r$ with $\lambda_{\max}$")
    ax[1].set_title("(b) Mechanistic feature correlations"); ax[1].set_xlim(-1, 1)

    fig.suptitle(f"{DATASET}: composition -> $\\lambda_{{\\max}}$ structure",
                 y=1.03, fontsize=13, fontweight="bold")
    fig.tight_layout()
    return fig

def fig7_knn_local(df, k=5):
    """Local smoothness via k-nearest-neighbours in (standardised) composition.

    For each recipe we look at its k nearest neighbours in pump space and ask
    how much lambda_max varies among them. This isolates 'nearby points' and is
    not drowned by the dominant lambda_max mode (unlike the pairwise variogram).
    """
    Q = df[QCOLS].values
    Qz = (Q - Q.mean(0)) / Q.std(0)                 # standardise pumps
    y = df.lambda_max.values
    n = len(df)
    # pairwise distance matrix
    D = np.sqrt(((Qz[:, None, :] - Qz[None, :, :]) ** 2).sum(-1))
    np.fill_diagonal(D, np.inf)
    nn = np.argsort(D, axis=1)[:, :k]               # k nearest neighbours
    nn_mean = y[nn].mean(1)                          # mean lambda of neighbours
    local_std = y[nn].std(1)                         # local dispersion (nm)

    fig, ax = plt.subplots(1, 2, figsize=(11, 4.0))

    # (a) self vs neighbour-mean lambda_max, coloured by local dispersion
    sc = ax[0].scatter(y, nn_mean, c=local_std, cmap="magma_r", s=30,
                       edgecolor="white", linewidth=0.4)
    lims = [min(y.min(), nn_mean.min()) - 20, max(y.max(), nn_mean.max()) + 20]
    ax[0].plot(lims, lims, "k--", lw=1, label="y = x (perfectly smooth)")
    ax[0].set_xlim(lims); ax[0].set_ylim(lims)
    ax[0].set_xlabel(r"$\lambda_{\max}$ of the recipe (nm)")
    ax[0].set_ylabel(rf"mean $\lambda_{{\max}}$ of its {k} neighbours (nm)")
    ax[0].set_title("(a) Neighbour consistency")
    ax[0].legend(loc="upper left")
    cb = fig.colorbar(sc, ax=ax[0]); cb.set_label("local dispersion (nm)")

    # (b) distribution of local dispersion
    ax[1].hist(local_std, bins=20, color="#0072B2", alpha=0.85)
    ax[1].axvline(np.median(local_std), color="k", ls="--", lw=1)
    ax[1].text(0.97, 0.95,
               f"median = {np.median(local_std):.0f} nm\n"
               f"(noise floor among\nthe {k} closest recipes)",
               transform=ax[1].transAxes, ha="right", va="top", fontsize=9)
    ax[1].set_xlabel(rf"local $\lambda_{{\max}}$ dispersion among {k}-NN (nm)")
    ax[1].set_ylabel("Count")
    ax[1].set_title("(b) How much nearby recipes differ")

    fig.suptitle(f"{DATASET}: local smoothness of $\\lambda_{{\\max}}$ "
                 f"({k}-nearest-neighbour view)", y=1.03,
                 fontsize=13, fontweight="bold")
    fig.tight_layout()
    return fig

def compute_flat_boundary(df):
    """Midpoint between the highest-A_max flat spectrum and the lowest-A_max
    non-flat spectrum -- the empirical Ext_max threshold separating flats."""
    flat_max = df.loc[df.is_flat, "abs_max"].max()
    clean_min = df.loc[~df.is_flat, "abs_max"].min()
    return 0.5 * (flat_max + clean_min)

def compute_regimes(df, sphere_lmax=SPHERE_LMAX, disk_lmax=DISK_LMAX):
    """Data-driven annotation regimes for the Ext_max-vs-lambda_max scatter.

    'spheres': the dense low-lambda cluster (roughly monodisperse Ag nanospheres,
    single dipole plasmon ~400 nm) spans the full Ext_max range -> a vertical
    band. 'disks': the red-shifted population (anisotropic growth), where Ext_max
    tends to rise with lambda_max -> a shaded dispersion band (+-DISK_BAND_K
    residual std) along a linear fit. DISK_LMAX is intentionally a wide,
    qualitative "disk region" box (420-750 nm) rather than a narrow fit-optimised
    window -- by request, no r/slope annotation is drawn (DISK_SHOW_R=False);
    a fit is still computed internally to place/orient the band.
    Bounds/fit computed once on the non-flat population so both fig8 (with
    flats) and fig9 (flats removed) share identical, comparable annotations.
    """
    clean = df[~df.is_flat]
    dsk = clean[(clean.lambda_max >= disk_lmax[0]) & (clean.lambda_max <= disk_lmax[1])]
    slope, intercept = np.polyfit(dsk.lambda_max, dsk.abs_max, 1)
    r = np.corrcoef(dsk.lambda_max, dsk.abs_max)[0, 1]
    resid_std = float((dsk.abs_max - (slope * dsk.lambda_max + intercept)).std())
    return dict(sphere_lmax=sphere_lmax, disk_lmax=disk_lmax, disk_slope=slope,
               disk_intercept=intercept, disk_r=r, disk_resid_std=resid_std)

def scatter_amax_lmax(d, flat_boundary=None, regimes=None):
    """Ext_max vs lambda_max coloured by growth time (colour bar on the side).

    No figure title (publication style). y starts at 0. Growth-time colour scale
    is expanded over 0-20 s (most recipes) and compressed over 20-200 s. Optional
    annotations (from compute_flat_boundary / compute_regimes): a horizontal line
    separating flat spectra from the rest, a vertical band over the 'spheres'
    cluster, and a diagonal fit over the red-shifted 'disks' regime.
    """
    fig, ax = plt.subplots(figsize=(6.6, 4.9))
    ok = d[d.conv_status == "ok"]
    na = d[d.conv_status != "ok"]

    if regimes is not None:
        lo, hi = regimes["sphere_lmax"]
        ax.axvspan(lo, hi, color="#0072B2", alpha=0.08, zorder=0)
        ax.axvline(lo, color="#0072B2", lw=1, ls=":", alpha=0.6, zorder=1)
        ax.axvline(hi, color="#0072B2", lw=1, ls=":", alpha=0.6, zorder=1)
        ax.text((lo + hi) / 2, 0.97, "spheres", transform=ax.get_xaxis_transform(),
                ha="center", va="top", fontsize=9, color="#0072B2", fontweight="bold")
        dlo, dhi = regimes["disk_lmax"]
        xs = np.linspace(dlo, dhi, 20)
        ys = regimes["disk_slope"] * xs + regimes["disk_intercept"]
        band = DISK_BAND_K * regimes["disk_resid_std"]
        # shaded dispersion band (a 'rectangle' along the trend) instead of a bare
        # line -- a thin line overstates the precision of a r~0.6 relationship
        ax.fill_between(xs, ys - band, ys + band, color="#D55E00", alpha=0.15,
                        zorder=1, edgecolor="none")
        ax.plot(xs, ys, "--", color="#D55E00", lw=1.6, zorder=4)
        if DISK_SHOW_R:
            ax.annotate(f"disks (r = {regimes['disk_r']:.2f})", (xs[-1], ys[-1] + band),
                        xytext=(6, 2), textcoords="offset points", ha="left",
                        fontsize=9, color="#D55E00", fontweight="bold")
        else:
            ax.annotate("disks", (xs[-1], ys[-1] + band),
                        xytext=(6, 2), textcoords="offset points", ha="left",
                        fontsize=9, color="#D55E00", fontweight="bold")

    sc = ax.scatter(ok.lambda_max, ok.abs_max, c=ok.conv_time_s, cmap=GROWTH_CMAP,
                    norm=conv_norm(), s=46, edgecolor="white", linewidth=0.5, zorder=3)
    if len(na):
        ax.scatter(na.lambda_max, na.abs_max, facecolor="0.8", edgecolor="0.5",
                   s=46, linewidth=0.5, zorder=2,
                   label=f"not converged (n = {len(na)})")
        ax.legend(loc="upper right", fontsize=8)

    if flat_boundary is not None:
        ax.axhline(flat_boundary, color="k", lw=1, ls="--", zorder=1)
        ax.text(0.99, flat_boundary, "flat  |  grown", transform=ax.get_yaxis_transform(),
                ha="right", va="bottom", fontsize=7.5, color="0.35")

    cb = fig.colorbar(sc, ax=ax, ticks=[0, 5, 10, 15, 20, 60, 100, 150, 200])
    cb.set_label("growth time (s)")
    ax.set_xlabel(r"$\lambda_{\max}$ (nm)"); ax.set_ylabel(r"$A_{\max}$ (a.u.)")
    top = float(d.abs_max.max())
    ax.set_ylim(0, top * 1.12)             # headroom so the 'spheres' label clears points
    fig.tight_layout()
    return fig

def fig_flat(df, spectra):
    """Show the flat spectra and where they sit in the A_max-lambda_max plane."""
    flat = df[df.is_flat]; keep = df[~df.is_flat]
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.3))
    for _, r in flat.iterrows():
        sp = spectra[r.filename]
        ax[0].plot(sp["wl"], sp["abs"][-1], lw=0.8, alpha=0.8)
    ax[0].axhline(0, color="k", lw=0.6, ls=":")
    ax[0].set_xlabel("Wavelength (nm)"); ax[0].set_ylabel("Absorbance (a.u.)")
    ax[0].set_title(f"(a) Flat spectra (std < {FLAT_STD_THR}, n = {len(flat)})")
    ax[1].scatter(keep.lambda_max, keep.abs_max, s=36, color="#0072B2",
                  alpha=0.7, edgecolor="white", linewidth=0.4, label="kept")
    ax[1].scatter(flat.lambda_max, flat.abs_max, s=60, facecolor="none",
                  edgecolor="#D55E00", linewidth=1.6, label="flat (flagged)")
    ax[1].set_xlabel(r"$\lambda_{\max}$ (nm)"); ax[1].set_ylabel(r"$A_{\max}$ (a.u.)")
    ax[1].set_title("(b) Flat samples in the descriptor plane"); ax[1].legend(fontsize=8)
    fig.suptitle(f"{DATASET}: flat-spectrum flagging", y=1.02,
                 fontsize=13, fontweight="bold")
    fig.tight_layout()
    return fig

# ── Exploration aids ──────────────────────────────────────────────────────────
def short_name(fn):
    """S14-M14_sample_12 -> S14-M14·s12 (compact label for annotations)."""
    return (fn.replace("_sample_", "·s").replace("_init_", "·i")
              .replace("_iter_", "·it"))

def scatter_amax_lmax_labelled(d, highlight=None):
    """Exploration copy of fig8 (A_max vs λmax, coloured by convergence time) with
    every sample annotated by name. `highlight` = filenames to ring (e.g. the
    representative samples of fig8c). Large + tiny font since ~100 labels overlap."""
    highlight = set(highlight or [])
    fig, ax = plt.subplots(figsize=(15, 10))
    ok = d[d.conv_status == "ok"]; na = d[d.conv_status != "ok"]
    sc = ax.scatter(ok.lambda_max, ok.abs_max, c=ok.conv_time_s, cmap="plasma",
                    norm=conv_norm(), s=44, edgecolor="white", linewidth=0.5, zorder=3)
    handles = []
    if len(na):
        handles.append(ax.scatter(na.lambda_max, na.abs_max, facecolor="0.8",
                       edgecolor="0.5", s=44, linewidth=0.5, zorder=2,
                       label=f"not converged (n = {len(na)})"))
    if highlight:
        h = d[d.filename.isin(highlight)]
        handles.append(ax.scatter(h.lambda_max, h.abs_max, s=320, facecolor="none",
                       edgecolor="black", linewidth=1.8, zorder=5,
                       label="representative (fig8c)"))
    if handles:
        ax.legend(handles=handles, loc="upper right", fontsize=10)
    cb = fig.colorbar(sc, ax=ax, ticks=[0, 5, 10, 15, 20, 60, 100, 150, 200])
    cb.set_label("growth time (s)")
    for _, r in d.iterrows():
        hi = r.filename in highlight
        ax.annotate(short_name(r.filename), (r.lambda_max, r.abs_max),
                    xytext=(3, 3), textcoords="offset points",
                    fontsize=(8 if hi else 4.5), color=("black" if hi else "0.25"),
                    fontweight=("bold" if hi else "normal"), zorder=6)
    ax.set_xlabel(r"$\lambda_{\max}$ (nm)"); ax.set_ylabel(r"$A_{\max}$ (a.u.)")
    fig.tight_layout(); return fig

def fig_sample_evolution(spectra, names, df, wl_range=EVOL_WL_RANGE,
                         sg_window=SG_WINDOW, sg_poly=SG_POLY):
    """### PLACEHOLDER ### Full spectra + time evolution of chosen representative
    samples. Set REPRESENTATIVE_SAMPLES to 4 filenames; each panel overlays every
    time-point spectrum (Savitzky-Golay smoothed) coloured from t=0 (dark) to
    convergence (yellow). x = wl_range; y = [0, max absorbance]. A legend box lists
    the pump flows (µL/s) used to generate the sample, read from the H5 via df."""
    names = [n for n in names if n in spectra]
    flows = df.set_index("filename")[QCOLS]
    ncol = 2; nrow = int(np.ceil(len(names) / ncol)) or 1
    fig, ax = plt.subplots(nrow, ncol, figsize=(5.2*ncol, 3.4*nrow), squeeze=False)
    ax = ax.ravel()
    for a, name in zip(ax, names):
        sp = spectra[name]; A = np.asarray(sp["abs"]); wl = sp["wl"]; t = np.asarray(sp["t"])
        m = (wl >= wl_range[0]) & (wl <= wl_range[1]); wlm = wl[m]
        win = min(sg_window, len(wlm) - (len(wlm) + 1) % 2)   # odd, <= len
        if win <= sg_poly: win = sg_poly + 1 + (sg_poly % 2)
        norm = mpl.colors.Normalize(t.min(), t.max()); ymax = 0.0
        for j in range(len(A)):
            s = savgol_filter(A[j][m], win, sg_poly)
            a.plot(wlm, s, color=mpl.cm.viridis(norm(t[j])), lw=0.6, alpha=0.85)
            ymax = max(ymax, s.max())
        a.set_xlim(*wl_range); a.set_ylim(0, ymax)
        a.set_xlabel("Wavelength (nm)"); a.set_ylabel("Absorbance (a.u.)")
        a.set_title(short_name(name), fontsize=10)
        # pump-flow legend box (µL/s), read from the H5; placed opposite the peak
        # so it does not mask the spectrum (left if the peak is red-shifted)
        env = savgol_filter(A[:, m].max(0), win, sg_poly)
        peak_wl = wlm[np.argmax(env)]
        left = peak_wl > 0.5 * (wl_range[0] + wl_range[1])
        q = flows.loc[name]
        txt = "\n".join(f"{REAGENTS[k]:>7s} {q[QCOLS[k]]:4.2f}" for k in range(5)) + "\n  (µL/s)"
        a.text(0.03 if left else 0.97, 0.96, txt, transform=a.transAxes,
               ha="left" if left else "right", va="top",
               fontsize=6.5, family="monospace",
               bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.85))
        sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis"); sm.set_array([])
        cb = fig.colorbar(sm, ax=a); cb.set_label("time (s)", fontsize=8)
    for a in ax[len(names):]: a.axis("off")
    fig.tight_layout(); return fig

def choose_representatives(df):
    """REPRESENTATIVE_SAMPLES if set, else 4 clean samples spanning the λmax range."""
    if REPRESENTATIVE_SAMPLES:
        return REPRESENTATIVE_SAMPLES
    clean = df[~df.is_flat].sort_values("lambda_max")
    return [clean.iloc[int(f*(len(clean)-1))].filename for f in (0.1, 0.4, 0.6, 0.9)]

# ----------------------------------------------------------------------------
# 4. Run
# ----------------------------------------------------------------------------
def main():
    print(f"Loading {DATASET} from {DATA_ROOT} ...")
    df, spectra = load_session(DATA_ROOT)
    df = add_chem_features(df)
    df = enrich_convergence(df, spectra)
    cmap_manip = manip_colors(df)
    print(f"Loaded {len(df)} samples:")
    print(df.manip.value_counts().to_string())
    print("\nConvergence status:")
    print(df.conv_status.value_counts().to_string())
    okc = df.loc[df.conv_status == "ok", "conv_time_s"]
    print(f"Convergence time (converged): median {okc.median():.1f} s "
          f"({okc.min():.1f}-{okc.max():.1f} s)")
    df.to_csv(FIG_DIR / f"{DATASET}_eda_table.csv", index=False)

    # flat-spectrum flagging + cleaned ML set
    print(f"\nFlat spectra (abs std < {FLAT_STD_THR}): {df.is_flat.sum()} / {len(df)}")
    flat_list = df.loc[df.is_flat, ["filename", "manip", "lambda_max", "abs_max",
                                    "abs_std", "conv_status"]].sort_values("abs_std")
    print(flat_list.to_string(index=False))
    flat_list.to_csv(FIG_DIR / f"{DATASET}_flat_spectra.csv", index=False)
    ml = df[~df.is_flat].reset_index(drop=True)
    ml.to_csv(FIG_DIR / f"{DATASET}_ml_set.csv", index=False)
    print(f"\nML set (flat removed): {len(ml)} samples "
          f"-> {DATASET}_ml_set.csv")

    print("\nGenerating figures ...")
    savefig(fig1_lmax_amax(df, cmap_manip),         "fig1_lmax_amax_distributions")
    savefig(fig2_spectra(df, spectra, cmap_manip),  "fig2_uvvis_spectra")
    savefig(fig3_kinetics(df, spectra),             "fig3_growth_kinetics")
    savefig(fig4_manip_variation(df, cmap_manip),   "fig4_manip_variability")
    savefig(fig5_coverage(df),                      "fig5_simplex_coverage")
    savefig(fig6_variogram(df),                     "fig6_composition_structure")
    savefig(fig7_knn_local(df),                     "fig7_knn_local_smoothness")
    flat_boundary = FLAT_GROWN_EXTMAX
    regimes = compute_regimes(df)
    print(f"\nFlat/grown Ext_max boundary: {flat_boundary:.3f} "
          f"(fixed; data-driven midpoint would be {compute_flat_boundary(df):.3f})")
    print(f"Disk regime {regimes['disk_lmax']} nm: r = {regimes['disk_r']:.2f} "
          f"(slope {regimes['disk_slope']:.4f} /nm, resid_std {regimes['disk_resid_std']:.3f})")
    savefig(scatter_amax_lmax(df, flat_boundary, regimes), "fig8_amax_lmax_convtime")
    reps = choose_representatives(df)
    print(f"Representative samples: {reps}")
    savefig(scatter_amax_lmax_labelled(df, reps),   "fig8b_amax_lmax_labelled")
    savefig(fig_sample_evolution(spectra, reps, df), "fig8c_sample_evolution")
    savefig(fig_flat(df, spectra),                  "fig_flat_spectra")
    savefig(scatter_amax_lmax(ml, None, regimes),   "fig9_amax_lmax_convtime_noflat")
    print(f"Done -> {FIG_DIR.resolve()}")

if __name__ == "__main__":
    main()


### `ml.spectral_pipeline`
*Original analysis code.*

In [ ]:
%%module toofast_analysis.ml.spectral_pipeline
# [vendored] verbatim copy of 2FAST_ML/spectral_pipeline.py (md5 93a0bbed16a247b416dfe09fc105cdcb), generated by tools/vendor_ml.py — do not edit by hand.
"""
spectral_pipeline.py
====================
Shared engine for the spectral pipeline (notebooks 22-24). Generalised from the
notebook-13 module: normalisation, PCA, feature engineering, the model zoo
(isotropic Matérn-5/2 GP matching the BO prior), grouped cross-validation,
spectrum reconstruction, perceived-colour rendering and figures.

Everything is parameterised by data passed in (S, Q, groups, wl) — no hard-coded
paths — so it runs on any selected multi-session dataset (`spectral_selected.csv`
+ `spectra_matrix.npz`). Requires scikit-learn (run in the 3.11 kernel).
"""
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from sklearn.model_selection import cross_val_predict, KFold, GroupKFold, LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge, BayesianRidge
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor)
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
try:
    import xgboost as xgb; XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

from .eda_rg024 import set_pub_style, FIG_DIR
set_pub_style()

FEATURES = ["Q1", "Q2", "Q3", "Q4", "Q5"]
Q_TOTAL, SEED, EPS, N_PC = 14.0, 1, 0.01, 6
N_JOBS = 2          # bounded parallelism — n_jobs=-1 over-subscribes and crashes
                    # loky workers when called hundreds of times (greedy search)
np.random.seed(SEED)

def savefig(fig, name, notebook):
    fig.text(0.995, 0.003, f"source: {notebook}", ha="right", va="bottom",
             fontsize=6, color="0.55")
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}")
    print(f"  saved {name}.png / .pdf")

# ── Normalisation + PCA ───────────────────────────────────────────────────────
def normalise(S, mode="max"):
    """Return (S_norm, amplitude). mode: 'max' | 'area' | 'l2' | None."""
    if mode == "max":
        amp = S.max(1, keepdims=True)
    elif mode == "area":
        amp = np.trapz(np.clip(S, 0, None), axis=1)[:, None]
    elif mode == "l2":
        amp = np.linalg.norm(S, axis=1, keepdims=True)
    else:
        amp = np.ones((len(S), 1))
    amp[amp == 0] = 1
    return S / amp, amp.ravel()

def fit_pca(S, n_pc=N_PC):
    mu = S.mean(0); Sc = S - mu
    U, sv, Vt = np.linalg.svd(Sc, full_matrices=False)
    return mu, Vt[:n_pc], (sv**2 / (sv**2).sum())[:n_pc]

def reconstruct(mu, comps, scores):
    return mu + scores @ comps

# ── Feature engineering (Q -> X) ──────────────────────────────────────────────
def f_raw(Q):       return Q.astype(float)
def f_fractions(Q):
    Q = Q.astype(float); tot = Q.sum(1, keepdims=True); return np.hstack([Q / tot, tot])
def f_clr(Q):
    Q = Q.astype(float) + EPS; lg = np.log(Q); return lg - lg.mean(1, keepdims=True)
def f_chemistry(Q):
    Q = Q.astype(float); Ag = Q[:, 0] + EPS
    return np.column_stack([(Q[:,3]+Q[:,4])/Ag, Q[:,4]/Ag, Q[:,4]/(Q[:,3]+EPS),
                            Q[:,1]/Ag, Q[:,2]/Ag, Q.sum(1)])
def f_chem_v2(Q):
    Q = Q.astype(float); Ag, PVP, Ace, Asc, BH4 = Q[:,0], Q[:,1], Q[:,2], Q[:,3], Q[:,4]
    def sr(n, d, lo, hi): return np.clip(n / (d + EPS), lo, hi)
    feats = [Ag, PVP, Ace, Asc, BH4, Ag/Q_TOTAL, PVP/Q_TOTAL, Ace/Q_TOTAL, Asc/Q_TOTAL,
             BH4/Q_TOTAL, sr(Ag,PVP,-50,50), sr(BH4,Asc,-50,50), sr(Asc+BH4,PVP,-50,50)]
    for i in range(5):
        for j in range(i+1, 5): feats.append(sr(Q[:,i], Q[:,j], -20, 20))
    for i in range(5):
        for j in range(i+1, 5): feats.append(Q[:,i] * Q[:,j])
    feats += [np.log1p(sr(Ag,PVP,0,50)), np.log1p(sr(BH4,Asc,0,50)),
              BH4/(Asc+BH4+EPS), Ace/Q_TOTAL]
    Xf = np.column_stack(feats)
    Xp = PolynomialFeatures(2, include_bias=False).fit_transform(Q)
    return np.hstack([Xf, Xp[:, Q.shape[1]:]])
FEATURE_SETS = {"raw": f_raw, "fractions": f_fractions, "clr": f_clr,
                "chemistry": f_chemistry, "chem_v2": f_chem_v2}

# ── Model zoo (isotropic Matérn-5/2 GP = BO prior) ────────────────────────────
def make_kernel():
    return (C(1.0, (1e-2, 1e4)) * Matern(1.0, (1e-2, 1e3), nu=2.5)
            + WhiteKernel(0.1, (1e-6, 1e1)))
def gp_factory():
    return make_pipeline(StandardScaler(),
                         GaussianProcessRegressor(kernel=make_kernel(), normalize_y=True,
                                                  alpha=1e-8, n_restarts_optimizer=4,
                                                  random_state=SEED))
MODELS = {
    "Ridge":            lambda: make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "BayesianRidge":    lambda: make_pipeline(StandardScaler(), BayesianRidge()),
    "RandomForest":     lambda: RandomForestRegressor(200, max_depth=15,
                                    min_samples_split=3, random_state=SEED, n_jobs=N_JOBS),
    "ExtraTrees":       lambda: ExtraTreesRegressor(200, max_depth=15,
                                    min_samples_split=3, random_state=SEED, n_jobs=N_JOBS),
    "GradientBoosting": lambda: GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                    learning_rate=0.05, subsample=0.8, random_state=SEED),
    "GP-Matern52":      lambda: gp_factory(),
    "MLP":              lambda: make_pipeline(StandardScaler(), MLPRegressor(
                                    (64,32,16), max_iter=2000, early_stopping=True,
                                    random_state=SEED)),
}
if XGBOOST_AVAILABLE:
    MODELS["XGBoost"] = lambda: xgb.XGBRegressor(n_estimators=200, max_depth=5,
                                learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                                random_state=SEED, n_jobs=N_JOBS)

class SpectralModel:
    """One regressor per PCA component, on features(Q)."""
    def __init__(self, n, feat_fn, factory):
        self.feat = feat_fn; self.models = [factory() for _ in range(n)]
    def fit(self, Q, Y):
        X = self.feat(Q)
        for k, m in enumerate(self.models): m.fit(X, Y[:, k])
        return self
    def predict(self, Q):
        X = self.feat(Q); return np.column_stack([m.predict(X) for m in self.models])
    def predict_std(self, Q):
        X = self.feat(Q)
        try:
            return np.column_stack([m.predict(X, return_std=True)[1] for m in self.models])
        except TypeError:
            return None

# ── Cross-validation (grouped by session) ─────────────────────────────────────
def make_cv(groups, k=6):
    ng = len(np.unique(groups))
    return (LeaveOneGroupOut() if k is None else GroupKFold(n_splits=min(k, ng)))

def cv_scores(Q, scores, feat_fn, factory, groups, k=6):
    X = feat_fn(Q); cv = make_cv(groups, k)
    return np.column_stack([cross_val_predict(factory(), X, scores[:, kk], cv=cv, groups=groups)
                            for kk in range(scores.shape[1])])

def derived_lmax(wl, spec, lo=350, hi=900):
    w = (wl >= lo) & (wl <= hi); return wl[w][np.argmax(spec[:, w], axis=1)]

def benchmark(S, Q, scores, mu, comps, wl, groups, lam, k=6,
              feature_sets=None, models=None):
    feature_sets = feature_sets or FEATURE_SETS
    models = models or MODELS
    rows = []
    for fname, ffn in feature_sets.items():
        for mname, fac in models.items():
            try:
                sp = cv_scores(Q, scores, ffn, fac, groups, k)
            except Exception as e:
                print(f"  {fname:10s}|{mname:14s} FAILED: {e}"); continue
            Sp = mu + sp @ comps
            rows.append(dict(feature=fname, model=mname,
                             spec_rmse=np.sqrt(np.mean((S - Sp)**2)),
                             lmax_rmse=np.sqrt(np.nanmean((lam - derived_lmax(wl, Sp))**2))))
            print(f"  {fname:10s}|{mname:14s} spec={rows[-1]['spec_rmse']:.4f} "
                  f"lmax={rows[-1]['lmax_rmse']:5.1f}")
    return pd.DataFrame(rows)

# ── Colour (spectrum -> sRGB) + colour-map slice ──────────────────────────────
def _g(x, m, s1, s2): return np.exp(-0.5 * ((x - m) / np.where(x < m, s1, s2))**2)
def _cmf(l):
    x = 1.056*_g(l,599.8,37.9,31.0)+.362*_g(l,442.0,16.0,26.7)-.065*_g(l,501.1,20.4,26.2)
    y = .821*_g(l,568.8,46.9,40.5)+.286*_g(l,530.9,16.3,31.1)
    z = 1.217*_g(l,437.0,11.8,36.0)+.681*_g(l,459.0,26.0,13.8)
    return x, y, z
_M = np.array([[3.2406,-1.5372,-.4986],[-.9689,1.8758,.0415],[.0557,-.2040,1.0570]])
def make_colouriser(wl):
    vis = (wl >= 380) & (wl <= 780); xb, yb, zb = _cmf(wl[vis]); Yn = yb.sum()
    def rgb(a):
        T = np.clip(10**(-np.clip(a[vis], 0, None)), 0, 1)
        XYZ = np.array([(T*xb).sum(), (T*yb).sum(), (T*zb).sum()]) / Yn
        c = np.clip(_M @ XYZ, 0, 1)
        return np.clip(np.where(c <= .0031308, 12.92*c, 1.055*c**(1/2.4)-.055), 0, 1)
    return rgb

def colour_slice(ax, Q, model, mu, comps, colour, qa, qb, grid_n=44, hatch=True):
    ia, ib = FEATURES.index(qa), FEATURES.index(qb)
    others = [k for k in range(5) if k not in (ia, ib)]
    med = np.median(Q, 0); prop = med[others] / med[others].sum()
    ga = np.linspace(np.percentile(Q[:,ia],5), np.percentile(Q[:,ia],95), grid_n)
    gb = np.linspace(np.percentile(Q[:,ib],5), np.percentile(Q[:,ib],95), grid_n)
    grid = np.full((grid_n, grid_n, 5), np.nan); mask = np.zeros((grid_n, grid_n), bool)
    for iy, vb in enumerate(gb):
        for ix, va in enumerate(ga):
            rem = Q_TOTAL - va - vb
            if rem <= 0: continue
            x = np.empty(5); x[ia]=va; x[ib]=vb; x[others]=rem*prop
            grid[iy,ix]=x; mask[iy,ix]=True
    X = grid[mask]; sc = model.predict(X)
    img = np.ones((grid_n, grid_n, 3)); img[mask] = np.array([colour(mu+s@comps) for s in sc])
    ax.imshow(img, origin="lower", aspect="auto", extent=[ga[0],ga[-1],gb[0],gb[-1]])
    if hatch:
        sd = model.predict_std(X)
        if sd is not None:
            u = np.full((grid_n, grid_n), np.nan); u[mask] = sd.mean(1)
            ax.contourf(ga, gb, u, levels=[np.nanpercentile(u,80), np.nanmax(u)],
                        colors="none", hatches=["xx"], alpha=0)
    ax.scatter(Q[:,ia], Q[:,ib], s=10, facecolor="none", edgecolor="k", lw=.4, alpha=.4)
    ax.set_xlabel(f"$Q_{{{qa}}}$ (µL/s)"); ax.set_ylabel(f"$Q_{{{qb}}}$ (µL/s)")


### `ml.pipeline_common`
*Original analysis code.*

In [ ]:
%%module toofast_analysis.ml.pipeline_common
# [vendored] verbatim copy of 2FAST_ML/pipeline/pipeline_common.py (md5 6d761a19afe0dc375db8ced49c1e8617), generated by tools/vendor_ml.py — do not edit by hand.
"""
pipeline_common.py  —  shared engine for the new data-analysis pipeline (30-38)
==============================================================================
Thin layer on top of the EXISTING engines at the repo root (no re-implementation
of the H5 parsing / anchoring / model zoo):

    catalog_2fast     — fresh H5 scan  (parse_h5_file / scan_directory)
    anchor_2fast      — standard-anchored tiers  (nb03)
    spectral_pipeline — normalise / PCA / model zoo / grouped-CV / colour  (nb22-24)

The data-selection phase is SELF-CONTAINED here: it does its OWN fresh scan of the
H5 tree (via catalog_2fast) and caches the result under pipeline/data/ — it does NOT
read the root `spectral_catalogue_full.csv` produced by notebook 20.

Config is centralised so every step 30-38 reads from one place. New step-1/2 pieces:
a configurable flat label (recomputed on the resampled grid) and a negative-spectrum
filter that runs AFTER the flat flag (a flat-but-negative run is a legit 'no particle',
so it is KEPT; only NON-flat negative-dominated runs are dropped).

Runs in the Anaconda 3.11 kernel (C:\\tools\\Anaconda3\\python.exe).
"""
from pathlib import Path
import sys, re
import numpy as np, pandas as pd, h5py
from scipy.signal import savgol_filter, find_peaks

from .. import paths as _paths          # [vendored] repository paths (raw/, data/processed/, figures/)

from . import catalog_2fast                      # noqa: E402
from . import anchor_2fast                       # noqa: E402
from . import spectral_pipeline as sp            # noqa: E402  (re-exported below)

# ── pipeline-wide CONFIG (single source of truth) ─────────────────────────────
DATA_DIR  = _paths.PROCESSED / "ml"     # [vendored] was pipeline/data
FIG_DIR   = _paths.FIGURES / "ml"       # [vendored] was pipeline/figures
DATA_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_ROOT = _paths.RAW   # [vendored] was the flat copy training_full_spec; files: resources/ml_files.csv
CAT_CACHE = DATA_DIR / "catalogue_scan.csv"            # this pipeline's OWN catalogue
#   training_full_spec: 100 LHS (csv_romain provenance, loose at root) + BO folders
#   (<manip>/samples/*_init_*/*_iter_*). All 347 files cover 310-1000 nm, full simplex.
#   No STANDARD files -> no drift anchoring (all tier C).

# --- decision knobs (see README) ---------------------------------------------
WL_RANGE      = (310.0, 1000.0)   # NIR-inclusive full spectrum (every file covers it here)
EXP_TYPES     = ["LHS", "BO_INIT"]   # init only: space-filling, unbiased. BO_ITER points add
                                     # NO new sessions/manips (they densify optima) -> add
                                     # "BO_ITER" only for the step-9 augmentation / ablation.
FULL_SIMPLEX  = True              # n_pumps_free == 5 (no flow pinned at the 2.8 standard)
COVERS_ONLY   = True              # keep only sessions whose spectrometer covers WL_RANGE

# --- particle / no-particle label: PEAK-based (replaces the old std<0.015 rule) ---
#   A spectrum is a PARTICLE if it has a real plasmon peak: a smoothed peak inside the
#   plasmon window whose PROMINENCE >= PROM_MIN and whose ABSORBANCE >= ABS_FLOOR.
#   Rationale: std is an ABSOLUTE measure and misses narrow low-amplitude sphere peaks
#   (dilute spheres near 400 nm) -> they were wrongly labelled 'no particle'. Prominence
#   + a positive height floor + an edge-excluding window fixes that and rejects UV/NIR
#   edge artifacts and negative-baseline wiggles. Calibrated on the selected set
#   (rescues 26 genuine dilute-particle spectra, demotes none).
PEAK_WIN      = (360.0, 990.0)    # plasmon window (exclude 310/1000 nm edge artifacts)
PROM_MIN      = 0.010             # min smoothed peak prominence (absorbance units)
ABS_FLOOR     = 0.020             # min smoothed peak absorbance (rejects noise + neg baselines)
SG_WIN, SG_POLY = 15, 3           # Savitzky-Golay smoothing (matches eda_rg024)
FLAT_STD_THR  = 0.015             # legacy std threshold — REPORTED (spec_std_grid) but no
                                  #   longer the label. Kept for the diagnostic figure only.
NEG_FRAC_MAX  = 0.50              # a spectrum is "negative-dominated" if >50% points < 0
NEG_TOL       = 0.0               # a point counts as "negative" below this
#   ORDER: flag flatness FIRST; a flat run is kept even if negative-dominated (it is the
#   'no particle' class). Only NON-flat negative-dominated runs are dropped.

# tier thresholds (identical to nb03 / nb21)
BIAS_MAX, ANCHOR_DT_MAX = 30.0, 120.0
ABS_CANDS = ["spectra/extinction"]
WL_CANDS  = ["spectra/wavelength"]

FEATURES = sp.FEATURES            # ["Q1".."Q5"]  re-export
SEED     = sp.SEED
N_PC     = 5                      # working PCA decomposition (max-normalised spectra): 5 comps.
                                  #   single source of truth for steps 32/33/34/37 (and outliers).
TRAIN_Q_FLOOR = 0.5               # for the spectral MODELS (32/33/34/37): drop the infeasible edge
                                  #   where AgNO3 (Q1) OR NaBH4 (Q5) < this. The classifier (31) is
                                  #   NOT cut — it needs those edges to learn the feasibility boundary.

def training_cut(df):
    """Boolean keep-mask: recipes above the reagent floor on BOTH gating flows (Q1 & Q5)."""
    return (df.Q1.values >= TRAIN_Q_FLOOR) & (df.Q5.values >= TRAIN_Q_FLOOR)

# ── SINGLE SOURCE OF TRUTH for the spectral-model training set (steps 32/33/34/37) ──
#   "hardcut" (step 31b): flow cut applied FIRST, then negatives and flats removed, then
#       IsolationForest hits FLAGGED (is_shape_outlier) but KEPT -> 151 particle spectra.
#   "clean"   (legacy): step-30 output, filtered here by is_particle & training_cut -> 145.
#   The classifier (step 31) does NOT use this — it needs the flats and the starved edge.
TRAIN_TAG = "hardcut"

def load_training(tag=None):
    """Return (df, S, grid): the particle spectra the spectral models train on.

    Steps 32-48 must call THIS rather than re-deriving the mask, so the dataset is defined
    in one place. Switch `TRAIN_TAG` to "clean" to reproduce the legacy 145-point set."""
    tag = tag or TRAIN_TAG
    if tag == "hardcut":
        df, S, grid = load("hardcut")
        # the legacy step-30 `is_outlier` flag is superseded by `is_shape_outlier` (which
        # flags without deleting). Neutralise it so legacy `~is_outlier` filters downstream
        # do not silently delete 6 more spectra using the old, pre-cut, non-reproducible run.
        df = df.copy(); df["is_outlier"] = False
        return df.reset_index(drop=True), S, grid
    df, S, grid = load(tag)
    m = df.is_particle.values & training_cut(df)
    return df[m].reset_index(drop=True), S[m], grid

# ── H5 spectrum reader ──────────────────────────────────────
def read_spectrum(fp):
    try:
        with h5py.File(fp, "r") as f:
            loc = next((c for c in ABS_CANDS if c in f), None)
            wlk = next((w for w in WL_CANDS if w in f), None)
            if loc is None or wlk is None:
                return None, None
            return f[wlk][()], f[loc][-1]     # wavelengths, LAST (converged) spectrum
    except Exception:
        return None, None

def _coverage(fp):
    """(wl_lo, wl_hi) of an H5 file, or (nan, nan)."""
    wl, _ = read_spectrum(fp)
    if wl is None:
        return np.nan, np.nan
    return float(wl.min()), float(wl.max())

# ── step 0 : SELF-CONTAINED fresh catalogue (scan the H5 tree, cache it) ───────
def build_catalogue(force_rescan=False, wl_range=WL_RANGE):
    """Fresh scan of DATA_ROOT via catalog_2fast, augmented with spectral coverage,
    cached to pipeline/data/catalogue_scan.csv. This pipeline OWNS this file — it does
    not read the notebook-20 catalogue. Pass force_rescan=True to rebuild the cache."""
    if CAT_CACHE.exists() and not force_rescan:
        df = pd.read_csv(CAT_CACHE)
        print(f"[catalogue] cache -> {len(df)} valid files / {df.session.nunique()} sessions "
              f"(force_rescan=True to rebuild)")
        return df

    print(f"[catalogue] fresh scan of {DATA_ROOT} (a few minutes) ...")
    df = catalog_2fast.scan_directory(ml_input_files())   # [vendored] explicit, ordered file list
    df = df[~df.flag_invalid].copy().reset_index(drop=True)
    # robust manip/session from the FILENAME (root-level LHS files have no session/manip
    # folder, so catalog_2fast mis-groups them; derive uniformly from the stem instead).
    df["manip"]   = df.filename.map(lambda s: re.split(r"_(?:sample|init|iter)_", str(s))[0])
    df["session"] = df.manip.map(lambda m: (re.match(r"^(S\d+)", m) or [m, m])[1])
    cov = np.array([_coverage(fp) for fp in df.filepath])
    df["wl_lo"], df["wl_hi"] = cov.T
    lo_ok = wl_range[0] + 10.0; hi_ok = wl_range[1] - 10.0
    df["covers_range"] = (df.wl_lo <= lo_ok) & (df.wl_hi >= hi_ok)
    df.to_csv(CAT_CACHE, index=False)
    print(f"[catalogue] scanned {len(df)} valid files / {df.session.nunique()} sessions "
          f"| cover {wl_range}: {int(df.covers_range.sum())} files -> cached")
    return df

# ── peak-based particle detector ──────────────────────────────────────────────
def peak_metrics(S, grid, win=PEAK_WIN, sg=(SG_WIN, SG_POLY)):
    """Per-spectrum (prominence, height, location) of the most prominent smoothed peak
    inside `win`. Height is the smoothed absorbance at that peak (may be <0 for negative
    baselines, which the ABS_FLOOR then rejects)."""
    inwin = (grid >= win[0]) & (grid <= win[1])
    prom = np.zeros(len(S)); height = np.zeros(len(S)); loc = np.full(len(S), np.nan)
    for i, s in enumerate(S):
        sm = savgol_filter(s, sg[0], sg[1])
        peaks, props = find_peaks(sm, prominence=1e-6)
        if len(peaks) and inwin[peaks].any():
            k = inwin[peaks]
            pr = props["prominences"][k]; hs = sm[peaks][k]; lc = grid[peaks][k]
            j = int(np.argmax(pr))
            prom[i], height[i], loc[i] = pr[j], hs[j], lc[j]
        else:
            height[i] = float(sm[inwin].max())
    return prom, height, loc

def assign_tier(row):                        # confidence label, identical to nb03
    if row["flag_no_anchor"]:            return "C"
    if not row["anchor_is_preceding"]:   return "B"
    if not row["anchor_stable"]:         return "B"
    if abs(row["anchor_bias"]) > BIAS_MAX:      return "B"
    if row["anchor_dt_min"] > ANCHOR_DT_MAX:    return "B"
    return "A"

# ── step 1 : SELECT + anchor-tier + load spectra on a common grid ─────────────
def build_selection(wl_range=WL_RANGE, exp_types=EXP_TYPES,
                    full_simplex=FULL_SIMPLEX, covers_only=COVERS_ONLY,
                    force_rescan=False):
    """Return (df, S, grid): the selected points (tier + spectral flags) and their
    absolute-absorbance spectra resampled onto a common grid. FLATS ARE KEPT (they are
    the 'no-particle' class); the negative filter is applied AFTER flatness (see below).
    Nothing is dropped here — flags only."""
    cat = build_catalogue(force_rescan=force_rescan, wl_range=wl_range)
    cat["timestamp"] = pd.to_datetime(cat["timestamp"], errors="coerce")

    # anchoring context = every file of the covering sessions (needs their standards).
    # If the dataset has no STANDARD runs (e.g. training_full_spec), drift anchoring is
    # impossible -> everything is tier C (the tier is only a confidence label).
    cov = cat.loc[cat.covers_range, "session"].unique()
    ctx = cat[cat.session.isin(cov)].copy()
    if (ctx.exp_type == "STANDARD").any():
        ctx = anchor_2fast.run_anchoring_pipeline(ctx)
        ctx["tier"] = ctx.apply(assign_tier, axis=1)
    else:
        ctx["tier"] = "C"
        print("[anchoring] no STANDARD files -> all tier C (no inter-session drift anchor)")

    sel = ctx.copy()
    if covers_only:
        sel = sel[sel.covers_range]
    sel = sel[sel.exp_type.isin(exp_types)]
    if full_simplex:
        sel = sel[sel.n_pumps_free == 5]
    sel = sel.reset_index(drop=True)

    # common grid from the first readable spectrum
    wl0, _ = read_spectrum(sel.iloc[0].filepath)
    grid = wl0[(wl0 >= wl_range[0]) & (wl0 <= wl_range[1])]

    S, rows = [], []
    for _, r in sel.iterrows():
        wl, a = read_spectrum(r.filepath)
        if wl is None:
            continue
        si = np.interp(grid, wl, a)
        if np.isfinite(si).all():
            S.append(si); rows.append(r)
    S = np.vstack(S)
    out = pd.DataFrame(rows).reset_index(drop=True)

    # step-2 spectral flags on the RESAMPLED spectrum
    prom, height, loc = peak_metrics(S, grid)
    out["spec_std_grid"] = S.std(1)                 # legacy, reported only
    out["peak_prom"]     = prom
    out["peak_height"]   = height
    out["peak_loc"]      = loc
    has_peak = (prom >= PROM_MIN) & (height >= ABS_FLOOR)    # 1) real plasmon peak?
    out["is_flat"]       = ~has_peak                         # 'no particle' class (class 0)
    out["neg_fraction"]  = (S < NEG_TOL).mean(1)
    out["neg_dominated"] = out["neg_fraction"] > NEG_FRAC_MAX                # 2) negativity
    out["neg_drop"]      = out["neg_dominated"] & has_peak   # drop a negative run only if it
                                                             #   claims a peak (untrustworthy);
                                                             #   flat-negatives are kept (class 0)
    out["is_particle"]   = has_peak & (~out["neg_drop"])     # class 1
    return out, S, grid

# ── step 2 : outlier flag among the PARTICLE (non-flat, non-negative) spectra ─
def flag_outliers(df, S, contamination=0.05):
    """IsolationForest on max-normalised PCA scores of the PARTICLE spectra only
    (flats are a legitimate class, negatives already handled). Boolean per row."""
    from sklearn.ensemble import IsolationForest
    part = df.is_particle.values
    out = np.zeros(len(df), bool)
    if part.sum() < 10:
        return out
    Sn, _ = sp.normalise(S[part], "max")
    mu, comps, _ = sp.fit_pca(Sn, N_PC); scores = (Sn - mu) @ comps.T
    pred = IsolationForest(contamination=contamination,
                           random_state=SEED).fit_predict(scores)
    out[np.where(part)[0]] = (pred == -1)
    return out

# ── persistence ───────────────────────────────────────────────────────────────
def save(df, S, grid, tag="selection"):
    df.to_csv(DATA_DIR / f"{tag}.csv", index=False)
    np.savez(DATA_DIR / f"{tag}.npz", S=S, wavelengths=grid)
    print(f"saved -> {DATA_DIR / (tag + '.csv')}  &  {tag}.npz")

def load(tag="selection"):
    df = pd.read_csv(DATA_DIR / f"{tag}.csv")
    z  = np.load(DATA_DIR / f"{tag}.npz")
    return df, z["S"], z["wavelengths"]


# ── [vendored] additions ──────────────────────────────────────────────────────
def ml_input_files():
    """raw/ files of the ML dataset, in the order of the original sorted scan of training_full_spec."""
    import csv as _csv
    with open(_paths.RESOURCE_DIR / "ml_files.csv", encoding="utf-8") as fh:
        return [DATA_ROOT / r["raw_path"] for r in _csv.DictReader(fh)]

def missing_convergence_model():
    raise RuntimeError("data/processed/ml/convergence_model.pkl is missing: deploy the t_conv model with the "
                       "step-36 benchmark first (toofast_analysis.ml.run.step36). The original code silently fitted an "
                       "un-benchmarked RandomForest here (log-R2 0.69 instead of 0.75).")


### `ml.spectral_model`
*Original analysis code.*

In [ ]:
%%module toofast_analysis.ml.spectral_model
# [vendored] verbatim copy of 2FAST_ML/spectral_model.py (md5 db76e7a3b42245b17fa0e5022f3c30bc), generated by tools/vendor_ml.py — do not edit by hand.
"""
spectral_model.py  (notebook 23 engine)
=======================================
Train on the NORMALISED full spectrum: PCA(max-normalised spectra) → model zoo ×
feature engineering, cross-validated GROUPED BY SESSION (honest inter-session
generalisation). Then best-config reconstruction, Q_AgNO3×Q_seed colour maps, and
a learning curve (grouped-CV error vs number of sessions).

Reuses `spectral_pipeline.py` (engine) on the output of nb21 (`spectral_selected.csv`
+ `spectra_matrix.npz`). Runs in the 3.11 kernel. Mirrors 23_spectral_model.ipynb.
"""
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from . import spectral_pipeline as sp
from .eda_rg024 import FIG_DIR

NOTEBOOK   = "23_spectral_model.ipynb"
NORMALIZE  = "max"           # <-- full spectrum normalised to peak 1
N_PC       = 6
CV_K       = 6               # GroupKFold folds (by session)
FEATURE    = "raw"           # primary config for the figures
MODEL_NAME = "GP-Matern52"
TIERS      = ["A", "B", "C"] # which tiers to train on
DROP_FLAT  = True
HERO_PAIR  = ("Q1", "Q5")
MAP_PAIRS  = [("Q1", "Q5"), ("Q1", "Q2"), ("Q4", "Q5"), ("Q2", "Q4")]

def load():
    sel = pd.read_csv("spectral_selected.csv")
    z = np.load("spectra_matrix.npz"); S = z["S"]; wl = z["wavelengths"]
    m = sel.tier.isin(TIERS).values
    if DROP_FLAT:
        m &= (~sel.is_flat).values
    return sel[m].reset_index(drop=True), S[m], wl

# ── figures ───────────────────────────────────────────────────────────────────
def fig_benchmark(res):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
    for a, col, lab in ((ax[0], "spec_rmse", "spectral RMSE (norm.)"),
                        (ax[1], "lmax_rmse", r"derived $\lambda_{\max}$ RMSE (nm)")):
        res.pivot(index="model", columns="feature", values=col).plot.bar(ax=a)
        a.set_ylabel(lab); a.set_xlabel("model"); a.legend(fontsize=7, title="feature")
    fig.tight_layout(); return fig

def fig_recon(wl, S, Sp, colour, idx):
    n = len(idx); fig, ax = plt.subplots(2, 2, figsize=(9, 6)); ax = ax.ravel()
    for a, i, lab in zip(ax, idx, "ABCD"):
        a.plot(wl, S[i], color=colour(S[i]), lw=2, label="experiment")
        a.plot(wl, Sp[i], "k:", lw=1.6, label="prediction (grouped-CV)")
        a.set_xlim(310, 1000); a.set_xlabel("Wavelength (nm)"); a.set_ylabel("Absorbance (norm.)")
        a.text(.92, .9, lab, transform=a.transAxes, fontweight="bold", fontsize=13); a.legend(fontsize=8)
    fig.tight_layout(); return fig

def fig_parity(lam_meas, lam_pred, groups, wl=None, S=None, Sp=None):
    """Paper-ready parity plot of predicted vs measured lambda_max from grouped-CV
    (leave-one-session-out honesty), coloured by session, with R2/RMSE/MAE. If the
    full spectra are passed, also prints the overall spectral R2."""
    ok = np.isfinite(lam_meas) & np.isfinite(lam_pred)
    lm, lp, g = lam_meas[ok], lam_pred[ok], np.asarray(groups)[ok]
    r2 = 1 - np.sum((lm - lp)**2) / np.sum((lm - lm.mean())**2)
    rmse = np.sqrt(np.mean((lm - lp)**2)); mae = np.mean(np.abs(lm - lp))
    fig, ax = plt.subplots(figsize=(5.4, 5.2))
    for i, s in enumerate(sorted(set(g))):
        m = g == s
        ax.scatter(lm[m], lp[m], s=26, color=plt.cm.tab20(i % 20),
                   edgecolor="white", linewidth=0.3)
    lims = [min(lm.min(), lp.min()) - 20, max(lm.max(), lp.max()) + 20]
    ax.plot(lims, lims, "k--", lw=1); ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel(r"measured $\lambda_{\max}$ (nm)")
    ax.set_ylabel(r"predicted $\lambda_{\max}$ (nm, leave-session-out)")
    txt = f"R² = {r2:.2f}\nRMSE = {rmse:.0f} nm\nMAE = {mae:.0f} nm"
    if S is not None and Sp is not None:
        sr2 = 1 - np.sum((S - Sp)**2) / np.sum((S - S.mean(0))**2)
        txt += f"\nspectral R² = {sr2:.2f}"
    ax.text(0.05, 0.95, txt, transform=ax.transAxes, va="top", fontsize=10)
    fig.tight_layout(); return fig

def fig_hero(Q, model, mu, comps, colour, idx, pair):
    fig, ax = plt.subplots(figsize=(6.6, 5.2))
    sp.colour_slice(ax, Q, model, mu, comps, colour, *pair)
    ia, ib = sp.FEATURES.index(pair[0]), sp.FEATURES.index(pair[1])
    for i, lab in zip(idx, "ABCD"):
        ax.add_patch(Rectangle((Q[i,ia]-.15, Q[i,ib]-.15), .3, .3, fill=False, ec="k", lw=1.5))
        ax.annotate(lab, (Q[i,ia], Q[i,ib]), textcoords="offset points", xytext=(6,4),
                    fontweight="bold", fontsize=12)
    ax.set_xlabel(r"$Q_{\mathrm{AgNO_3}}$  (Q1, µL/s)")
    ax.set_ylabel(r"$Q_{\mathrm{seed/NaBH_4}}$  (Q5, µL/s)")
    fig.tight_layout(); return fig

def fig_grid(Q, model, mu, comps, colour, pairs):
    fig, ax = plt.subplots(2, 2, figsize=(10, 8.4)); ax = ax.ravel()
    for a, pr in zip(ax, pairs): sp.colour_slice(a, Q, model, mu, comps, colour, *pr)
    fig.tight_layout(); return fig

def fig_learning(S, Q, wl, groups, lam, feat_fn, factory):
    """Grouped-CV spectral RMSE as sessions are added (drift/coverage diagnostic)."""
    sess = list(pd.Series(groups).drop_duplicates())
    rng = np.random.default_rng(sp.SEED); rng.shuffle(sess)
    ns, rm = [], []
    for k in range(2, len(sess) + 1):
        sub = np.isin(groups, sess[:k])
        Ss, Qs, gs, ls = S[sub], Q[sub], groups[sub], lam[sub]
        mu, comps, _ = sp.fit_pca(Ss, sp.N_PC); sc = (Ss - mu) @ comps.T
        Sp = mu + sp.cv_scores(Qs, sc, feat_fn, factory, gs, 6) @ comps
        ns.append(sub.sum()); rm.append(np.sqrt(np.mean((Ss - Sp)**2)))
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.plot(ns, rm, "o-", color="#0072B2")
    ax.set_xlabel("training points (sessions added)"); ax.set_ylabel("grouped-CV spectral RMSE")
    fig.tight_layout(); return fig

def main():
    sel, S, wl = load()
    Q = sel[sp.FEATURES].values.astype(float); groups = sel.session.values; lam = sel.lambda_max.values
    Sn, amp = sp.normalise(S, NORMALIZE)
    mu, comps, evr = sp.fit_pca(Sn, N_PC); scores = (Sn - mu) @ comps.T
    print(f"{len(sel)} pts / {sel.session.nunique()} sessions | normalise={NORMALIZE} | "
          f"PCA({N_PC})={100*evr.sum():.1f}% | tiers={TIERS} | xgboost={sp.XGBOOST_AVAILABLE}")

    print("\nBenchmark (feature x model, GroupKFold by session):")
    res = sp.benchmark(Sn, Q, scores, mu, comps, wl, groups, lam, k=CV_K)
    res.to_csv(FIG_DIR / "spectral_benchmark.csv", index=False)
    best = res.loc[res.spec_rmse.idxmin()]
    print(f"\nBest: {best.feature}+{best.model} (spec={best.spec_rmse:.4f}, lmax={best.lmax_rmse:.1f})")
    sp.savefig(fig_benchmark(res), "fig23_benchmark", NOTEBOOK)

    feat = sp.FEATURE_SETS[FEATURE]; fac = sp.MODELS[MODEL_NAME]
    Sp = sp.reconstruct(mu, comps, sp.cv_scores(Q, scores, feat, fac, groups, CV_K))
    colour = sp.make_colouriser(wl)
    order = np.argsort(lam); idx = [order[int(f*(len(order)-1))] for f in (0.1,0.4,0.6,0.9)]
    model = sp.SpectralModel(N_PC, feat, fac).fit(Q, scores)
    sp.savefig(fig_recon(wl, Sn, Sp, colour, idx),          "fig23_reconstruction", NOTEBOOK)
    lam_pred = sp.derived_lmax(wl, Sp)
    sp.savefig(fig_parity(lam, lam_pred, groups, wl, Sn, Sp), "fig23_parity", NOTEBOOK)
    sp.savefig(fig_hero(Q, model, mu, comps, colour, idx, HERO_PAIR), "fig23_colormap", NOTEBOOK)
    sp.savefig(fig_grid(Q, model, mu, comps, colour, MAP_PAIRS),      "fig23_colormap_projections", NOTEBOOK)
    sp.savefig(fig_learning(Sn, Q, wl, groups, lam, feat, fac),       "fig23_learning_curve", NOTEBOOK)
    np.savez(FIG_DIR / "spectral_pca_basis.npz", wavelengths=wl, mean=mu, components=comps,
             scores=scores, evr=evr, normalize=NORMALIZE)
    print(f"Done -> {FIG_DIR.resolve()}")

if __name__ == "__main__":
    main()


### `ml.step30_data_selection`
*Original analysis code.*

In [ ]:
%%module toofast_analysis.ml.step30_data_selection
# [vendored] verbatim copy of 2FAST_ML/pipeline/step30_data_selection.py (md5 ba4e40ddd0a501387f9104c1ae3927c6), generated by tools/vendor_ml.py — do not edit by hand.
"""
step30_data_selection.py  —  pipeline steps 1 & 2  (mirrors 30_data_selection.ipynb)
====================================================================================
SELF-CONTAINED selection phase: it does its own fresh H5 scan (via pipeline_common ->
catalog_2fast), not the notebook-20 catalogue.

1. SELECT the LHS + BO_INIT full-simplex points on the 310-1000 nm track (RG track,
   NIR included) and tag each with its anchor-confidence tier (A/B/C).
2. Quality filtering, IN THIS ORDER:
     a. PEAK-based particle label: no real plasmon peak (prominence<0.01 or height<0.02
        in 360-990 nm) = "no particle" -> KEPT (it is the classifier's class 0)
     b. negative filter: drop a negative-dominated run ONLY IF it claims a peak
        (untrustworthy baseline); flat-negatives are kept as no-particle              [NEW]
     c. shape-outlier flag (IsolationForest on particle spectra) -> drop

Outputs (pipeline/data/):
   catalogue_scan.csv                   — this pipeline's own fresh catalogue (cached)
   selection.csv / selection.npz        — everything selected, all flags, nothing dropped
   clean.csv     / clean.npz            — non-flat negatives + outliers removed, flats kept

Run:  C:\\tools\\Anaconda3\\python.exe pipeline\\step30_data_selection.py
      (add --rescan to rebuild the catalogue cache from the raw H5 tree)
"""
import sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from . import pipeline_common as pc


def main(force_rescan=False):
    df, S, grid = pc.build_selection(force_rescan=force_rescan)
    print(f"\nSelected {len(df)} pts / {df.session.nunique()} sessions on "
          f"{grid.min():.0f}-{grid.max():.0f} nm  ({len(grid)} wl)")
    print(f"  exp_type: {df.exp_type.value_counts().to_dict()}")
    print(f"  tiers   : {df.tier.value_counts().to_dict()}")

    df["is_outlier"] = pc.flag_outliers(df, S)

    print(f"\nStep-2 flags (peak-based label, then negatives):")
    print(f"  no-particle (no peak: prom<{pc.PROM_MIN} or h<{pc.ABS_FLOOR}) : {int(df.is_flat.sum())}")
    print(f"  negative-dominated (>{pc.NEG_FRAC_MAX:.0%} pts <0)              : {int(df.neg_dominated.sum())}"
          f"  (no-peak & kept: {int((df.neg_dominated & df.is_flat).sum())})")
    print(f"  negatives DROPPED (has peak but neg baseline)         : {int(df.neg_drop.sum())}")
    print(f"  shape outliers among particles (IsolationForest)      : {int(df.is_outlier.sum())}")

    # clean = drop NON-flat negatives + outliers; KEEP flats (classifier class 0)
    keep = (~df.neg_drop) & (~df.is_outlier)
    clean = df[keep].reset_index(drop=True); Sc = S[keep.values]
    print(f"\nClean set: {len(clean)} pts / {clean.session.nunique()} sessions "
          f"(flats kept: {int(clean.is_flat.sum())}, particles: {int(clean.is_particle.sum())})")

    pc.save(df, S, grid, "selection")
    pc.save(clean, Sc, grid, "clean")

    _figure(df, S, grid)
    print("Done.")


def _figure(df, S, grid):
    fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
    # (a) peak-based split: prominence vs height with the two thresholds
    part = df.is_particle.values; flat = df.is_flat.values
    ax[0].scatter(df.peak_prom[part], df.peak_height[part], s=12, color="#0072B2",
                  alpha=.5, label=f"particle ({int(part.sum())})")
    ax[0].scatter(df.peak_prom[flat], df.peak_height[flat], s=16, color="0.5",
                  marker="x", label=f"no particle ({int(flat.sum())})")
    ax[0].axvline(pc.PROM_MIN, color="#D55E00", ls="--", lw=1)
    ax[0].axhline(pc.ABS_FLOOR, color="#D55E00", ls="--", lw=1,
                  label=f"prom≥{pc.PROM_MIN}, h≥{pc.ABS_FLOOR}")
    ax[0].set_xscale("log"); ax[0].set_yscale("log")
    ax[0].set_xlabel("peak prominence"); ax[0].set_ylabel("peak height (abs)")
    ax[0].set_title("(a) peak-based particle label"); ax[0].legend(fontsize=8)
    # (b) negative-fraction distribution, flats vs non-flats (only non-flats are dropped)
    ax[1].hist(df.loc[df.is_flat, "neg_fraction"], bins=40, color="0.6",
               alpha=0.8, label="flat (kept)")
    ax[1].hist(df.loc[~df.is_flat, "neg_fraction"], bins=40, color="#009E73",
               alpha=0.8, label="non-flat")
    ax[1].axvline(pc.NEG_FRAC_MAX, color="#D55E00", ls="--",
                  label=f"drop non-flat > {pc.NEG_FRAC_MAX:.0%}")
    ax[1].set_xlabel("fraction of points with abs < 0"); ax[1].set_ylabel("count")
    ax[1].set_title("(b) negative filter (after flatness)"); ax[1].legend(fontsize=8)
    # (c) a sample of spectra coloured by final class
    rng = np.random.default_rng(pc.SEED)
    classes = [(df.neg_drop.values, "#D55E00", "negative & non-flat (dropped)"),
               (df.is_flat.values, "0.6", "flat (no particle, kept)"),
               (df.is_particle.values & ~df.is_outlier.values, "#0072B2", "particle")]
    for cls, col, lab in classes:
        idx = np.where(cls)[0]
        if len(idx):
            for i in rng.choice(idx, min(25, len(idx)), replace=False):
                ax[2].plot(grid, S[i], col, lw=0.5, alpha=0.5)
            ax[2].plot([], [], col, label=f"{lab} (n={len(idx)})")
    ax[2].set_xlabel("wavelength (nm)"); ax[2].set_ylabel("absorbance")
    ax[2].set_title("(c) spectra by class"); ax[2].legend(fontsize=8)
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(pc.FIG_DIR / f"fig30_selection.{ext}", dpi=200, bbox_inches="tight")
    print(f"  saved fig30_selection.png/.pdf")


if __name__ == "__main__":
    main(force_rescan="--rescan" in sys.argv)


### `ml.step31b_hardcut_pipeline`
*Original analysis code.*

In [ ]:
%%module toofast_analysis.ml.step31b_hardcut_pipeline
# [vendored] verbatim copy of 2FAST_ML/pipeline/step31b_hardcut_pipeline.py (md5 21c97cf4ee2b7c1d7b849055a3e9c9c9), generated by tools/vendor_ml.py — do not edit by hand.
"""
step31b_hardcut_pipeline.py  —  re-ordered data-cleaning cascade (mirrors 31b_hardcut_pipeline.ipynb)
=====================================================================================================
The pipeline of step 30 flags flats/negatives/outliers on the WHOLE selection, then lets the
downstream steps apply the Q1/Q5 >= 0.5 flow floor. This module re-orders the same operations so
the flow floor comes FIRST, which is what the deployed models actually see:

    0. selection            189 pts  (100 LHS + 89 recycled BO-init, all LHS-sampled)
    1. hard flow cut        Q1 >= 0.5 AND Q5 >= 0.5      -> domain of validity
    2. baseline anomaly     negative-dominated spectra   -> intensity/shape anomaly
    3. flat detection       no plasmon peak              -> genuine 'no particle' INSIDE the domain
    4. outliers             IsolationForest on PCA scores of the surviving particle spectra

Why the order matters: IsolationForest is re-fitted on the POST-CUT particle set, so it no longer
spends its 5 % contamination budget on edge recipes that the flow floor removes anyway.

Every stage returns the spectra it removed so the notebook can plot them for visual verification.

Run:  C:\\tools\\Anaconda3\\python.exe pipeline\\step31b_hardcut_pipeline.py
"""
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from . import pipeline_common as pc

sp   = pc.sp
SEED = pc.SEED
CUT  = pc.TRAIN_Q_FLOOR                 # 0.5 µL/s on Q1 (AgNO3) and Q5 (NaBH4)
COL  = dict(particle="#0072B2", flat="0.45", neg="#D55E00", outlier="#CC79A7")

# ── the cascade ───────────────────────────────────────────────────────────────
def run_cascade(df, S, cut=CUT, contamination=0.05, remove_outliers=False):
    """Apply flow cut -> negatives -> flats -> outliers, in that order.

    `remove_outliers=False` (default) FLAGS the IsolationForest hits in an `is_shape_outlier`
    column instead of deleting them. Rationale: those spectra carry most of the NIR /
    anisotropic evidence in this dataset (see `outlier_coverage`), so deleting them removes
    the only examples of a real region rather than cleaning an artifact. Set True to reproduce
    the delete-them behaviour.

    Returns (stages, summary): `stages` holds the surviving (df, S) and the removed
    (df, S) of every step; `summary` is the cascade table."""
    st, rows = {}, []
    keep = (df.Q1.values >= cut) & (df.Q5.values >= cut)
    rows.append(dict(step="0. selection", removed=0, kept=len(df)))

    st["cut_out"] = (df[~keep].reset_index(drop=True), S[~keep])
    A, SA = df[keep].reset_index(drop=True), S[keep]
    rows.append(dict(step=f"1. flow cut Q1,Q5>={cut}", removed=int((~keep).sum()), kept=len(A)))

    mneg = A.neg_drop.values
    st["neg_out"] = (A[mneg].reset_index(drop=True), SA[mneg])
    B, SB = A[~mneg].reset_index(drop=True), SA[~mneg]
    rows.append(dict(step="2. baseline anomaly (negatives)", removed=int(mneg.sum()), kept=len(B)))

    mflat = B.is_flat.values
    st["flat_out"] = (B[mflat].reset_index(drop=True), SB[mflat])
    C, SC = B[~mflat].reset_index(drop=True), SB[~mflat]
    rows.append(dict(step="3. flat / no particle", removed=int(mflat.sum()), kept=len(C)))

    mout = np.asarray(pc.flag_outliers(C, SC, contamination=contamination))
    st["out_out"] = (C[mout].reset_index(drop=True), SC[mout])
    if remove_outliers:
        D, SD = C[~mout].reset_index(drop=True), SC[~mout]
        D["is_shape_outlier"] = False
        rows.append(dict(step="4. IsolationForest (removed)",
                         removed=int(mout.sum()), kept=len(D)))
    else:
        D, SD = C.copy(), SC
        D["is_shape_outlier"] = mout
        rows.append(dict(step="4. IsolationForest (flagged, kept)", removed=0, kept=len(D)))

    st["final"] = (D, SD)
    return st, pd.DataFrame(rows)

# ── is deleting the IsolationForest hits actually justified? ──────────────────
def outlier_coverage(st, bins=((300, 500), (500, 600), (600, 700), (700, 800), (800, 1000))):
    """What spectral territory do the flagged points hold? If a λmax band exists ONLY among
    them, deleting them removes the only evidence for that band rather than cleaning noise."""
    fin, _ = st["final"]; out, _ = st["out_out"]
    flagged = fin.get("is_shape_outlier", pd.Series(False, index=fin.index)).values
    kept = fin[~flagged]                      # what survives if the hits are deleted
    rows = []
    for lo, hi in bins:
        k = int(((kept.lambda_max >= lo) & (kept.lambda_max < hi)).sum())
        o = int(((out.lambda_max >= lo) & (out.lambda_max < hi)).sum())
        rows.append(dict(band=f"{lo}-{hi} nm", kept_if_deleted=k, in_flagged=o,
                         pct_lost=round(100 * o / max(k + o, 1), 1)))
    return pd.DataFrame(rows)

def fig_outlier_cost(st, grid):
    """(a) λmax coverage with vs without the flagged points, (b) their spectra vs the median."""
    fin, Sfin = st["final"]; out, Sout = st["out_out"]
    flagged = fin.get("is_shape_outlier", pd.Series(False, index=fin.index)).values
    kept = fin[~flagged]
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
    cov = outlier_coverage(st)
    x = np.arange(len(cov))
    ax[0].bar(x - .2, cov.kept_if_deleted, .4, color=COL["particle"], label="kept if deleted")
    ax[0].bar(x + .2, cov.in_flagged, .4, color=COL["outlier"], label="held by the flagged 8")
    for i, r in cov.iterrows():
        if r.in_flagged:
            ax[0].text(i + .2, r.in_flagged + .6, f"{r.pct_lost:.0f}%\nlost",
                       ha="center", fontsize=8, color=COL["outlier"])
    ax[0].set_xticks(x); ax[0].set_xticklabels(cov.band, fontsize=8)
    ax[0].set_ylabel("number of spectra"); ax[0].set_xlabel(r"$\lambda_{max}$ band")
    ax[0].set_title("(a) what deleting the IsolationForest hits costs")
    ax[0].legend(fontsize=8)
    ax[1].plot(grid, np.median(Sfin[~flagged], 0), color="0.5", lw=2, ls="--",
               label=f"median kept (n={int((~flagged).sum())})")
    for i in range(len(out)):
        ax[1].plot(grid, Sout[i], lw=1.1, alpha=.85,
                   label=f"{out.filename.iloc[i]}  λ={out.lambda_max.iloc[i]:.0f}")
    ax[1].set_xlabel("wavelength (nm)"); ax[1].set_ylabel("absorbance")
    ax[1].set_title("(b) the flagged spectra — real anisotropic products, not artifacts")
    ax[1].legend(fontsize=6.5, ncol=1)
    fig.tight_layout()
    return fig

def fig_negatives(st, grid):
    """The negative-dominated baselines removed at step 2."""
    sub, Ssub = st["neg_out"]
    fig = fig_grid(sub, Ssub, grid,
                   f"Step 2 — baseline anomalies: negative-dominated spectra, n={len(sub)}"
                   "   [orange; the shaded band is where absorbance < 0]",
                   colour_by=lambda r: COL["neg"], ncol=5, annot="std")
    if fig is not None:
        for a, i in zip(fig.axes[:len(sub)], range(len(sub))):
            a.fill_between(grid, Ssub[i], 0, where=Ssub[i] < 0,
                           color=COL["neg"], alpha=.18, lw=0)
            a.set_title(a.get_title() + f"\nneg={sub.neg_fraction.iloc[i]:.0%}", fontsize=6.5)
    return fig

# ── generic spectrum-grid figure ──────────────────────────────────────────────
def fig_grid(sub, Ssub, grid, title, colour_by=None, ncol=6, annot="prom"):
    """Plot every spectrum of `sub` with its five flows in the panel title."""
    n = len(sub)
    if n == 0:
        return None
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.9 * ncol, 2.3 * nrow), squeeze=False)
    axes = axes.ravel()
    for a, i in zip(axes, range(n)):
        r = sub.iloc[i]
        c = COL["particle"] if colour_by is None else colour_by(r)
        a.plot(grid, Ssub[i], color=c, lw=1.0)
        a.axhline(0, color="0.75", lw=0.6)
        q = "  ".join(f"{k}={r[k]:.2f}" for k in pc.FEATURES)
        extra = (f"prom={r.peak_prom:.3f}" if annot == "prom"
                 else f"std={r.spec_std_grid:.3f}  λ={r.lambda_max:.0f}nm")
        a.set_title(f"{r.filename}\n{q}\n{extra}", fontsize=6.5)
        a.set_xlim(grid.min(), grid.max()); a.tick_params(labelsize=6.5)
        a.set_xlabel("nm", fontsize=6.5); a.set_ylabel("Abs", fontsize=6.5)
    for a in axes[n:]:
        a.axis("off")
    fig.suptitle(title, fontsize=12)
    # reserve a fixed ~0.45 inch for the suptitle, not a fixed FRACTION (which would
    # swallow a third of a single-row figure)
    fig.tight_layout(rect=[0, 0, 1, 1 - 0.45 / (2.3 * nrow)])
    return fig

def fig_cut_removed(st, grid):
    sub, Ssub = st["cut_out"]
    return fig_grid(sub, Ssub, grid,
                    f"Step 1 — removed by the hard flow cut (Q1 or Q5 < {CUT} µL/s), n={len(sub)}"
                    "   [blue = had a peak, grey = flat]",
                    colour_by=lambda r: COL["flat"] if r.is_flat else COL["particle"])

def fig_remaining_flats(st, grid):
    sub, Ssub = st["flat_out"]
    return fig_grid(sub, Ssub, grid,
                    f"Step 3 — flat spectra SURVIVING the flow cut (no particle inside the "
                    f"domain of validity), n={len(sub)}",
                    colour_by=lambda r: COL["flat"], ncol=5, annot="std")

def fig_outliers(st, grid):
    sub, Ssub = st["out_out"]
    fin, Sfin = st["final"]
    fig = fig_grid(sub, Ssub, grid,
                   f"Step 4 — IsolationForest outliers among the post-cut particles, n={len(sub)}"
                   "   [grey = median surviving particle spectrum]",
                   colour_by=lambda r: COL["outlier"], ncol=4, annot="std")
    if fig is not None and len(fin):
        med = np.median(Sfin, 0)
        for a in fig.axes[:len(sub)]:
            a.plot(grid, med, color="0.6", lw=0.9, ls="--", zorder=0)
    return fig

def fig_cascade(summary, st, grid):
    """(a) waterfall of the cascade, (b) where the removed points sit in Q1xQ5."""
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
    lab = summary.step.tolist()[1:]
    rem = summary.removed.tolist()[1:]
    kept = summary.kept.tolist()
    ax[0].bar(range(len(lab)), rem, color=[COL["flat"], COL["neg"], "0.7", COL["outlier"]])
    for i, (r, k) in enumerate(zip(rem, kept[1:])):
        ax[0].text(i, r + 0.4, f"-{r}\n→{k}", ha="center", fontsize=9)
    ax[0].set_xticks(range(len(lab)))
    ax[0].set_xticklabels([l.split(". ", 1)[1] for l in lab], fontsize=8, rotation=18, ha="right")
    ax[0].set_ylabel("spectra removed")
    ax[0].set_title(f"(a) cleaning cascade  {kept[0]} → {kept[-1]}")
    ax[0].set_ylim(0, max(rem) * 1.25)

    fin, _ = st["final"]
    ax[1].scatter(fin.Q1, fin.Q5, s=16, c=COL["particle"], label=f"kept ({len(fin)})", alpha=.75)
    for key, mk, c, lb in [("cut_out", "X", COL["flat"], "flow cut"),
                           ("neg_out", "s", COL["neg"], "negative baseline"),
                           ("flat_out", "o", "0.35", "flat, inside domain"),
                           ("out_out", "D", COL["outlier"], "IsolationForest")]:
        s, _ = st[key]
        if len(s):
            ax[1].scatter(s.Q1, s.Q5, s=42, marker=mk, facecolor="none",
                          edgecolor=c, lw=1.3, label=f"{lb} ({len(s)})")
    ax[1].axvline(CUT, color="k", ls="--", lw=1, alpha=.6)
    ax[1].axhline(CUT, color="k", ls="--", lw=1, alpha=.6)
    ax[1].set_xlabel(r"$Q_{\mathrm{AgNO_3}}$ (Q1, µL/s)")
    ax[1].set_ylabel(r"$Q_{\mathrm{NaBH_4}}$ (Q5, µL/s)")
    ax[1].set_title("(b) what each step removes, in the gating-flow plane")
    ax[1].legend(fontsize=7.5, loc="upper right")
    fig.tight_layout()
    return fig

def savefig(fig, name):
    if fig is None:
        print(f"  (no figure for {name} — empty set)"); return
    for ext in ("png", "pdf"):
        fig.savefig(pc.FIG_DIR / f"{name}.{ext}", dpi=200, bbox_inches="tight")
    print(f"  saved {name}.png/.pdf")

def main():
    df, S, grid = pc.load("selection")
    print(f"selection: {len(df)} pts / {df.session.nunique()} sessions")
    st, summary = run_cascade(df, S)
    print("\nCascade:\n", summary.to_string(index=False))

    savefig(fig_cascade(summary, st, grid), "fig31b_cascade")
    savefig(fig_cut_removed(st, grid), "fig31b_cut_removed")
    savefig(fig_negatives(st, grid), "fig31b_negatives")
    savefig(fig_remaining_flats(st, grid), "fig31b_remaining_flats")
    savefig(fig_outliers(st, grid), "fig31b_outliers")
    savefig(fig_outlier_cost(st, grid), "fig31b_outlier_cost")
    print("\nCost of deleting the IsolationForest hits:\n",
          outlier_coverage(st).to_string(index=False))

    fin, Sfin = st["final"]
    pc.save(fin, Sfin, grid, tag="hardcut")
    summary.to_csv(pc.DATA_DIR / "hardcut_cascade.csv", index=False)
    for key, name in [("cut_out", "hardcut_removed_flowcut"),
                      ("neg_out", "hardcut_negatives"),
                      ("flat_out", "hardcut_remaining_flats"),
                      ("out_out", "hardcut_outliers")]:
        s, _ = st[key]
        cols = ["filename", "session"] + pc.FEATURES + ["lambda_max", "spec_std_grid",
                                                        "peak_prom", "peak_height", "is_flat",
                                                        "neg_fraction"]
        s[cols].to_csv(pc.DATA_DIR / f"{name}.csv", index=False)
        print(f"  saved {name}.csv ({len(s)} rows)")
    nflag = int(fin.get("is_shape_outlier", pd.Series(False)).sum())
    print(f"\nFINAL: {len(fin)} particle spectra / {fin.session.nunique()} sessions "
          f"({nflag} flagged as shape outliers but KEPT)")

if __name__ == "__main__":
    main()


### `ml.step33_spectral_model`
*Original analysis code.*

In [ ]:
%%module toofast_analysis.ml.step33_spectral_model
# [vendored] verbatim copy of 2FAST_ML/pipeline/step33_spectral_model.py (md5 e9f3eff79b3cf4bb7bdf913d5b7c0f65), generated by tools/vendor_ml.py — do not edit by hand.
"""
step33_spectral_model.py  —  pipeline step 5  (mirrors 33_spectral_model.ipynb)
==============================================================================
Predict the FULL normalised spectrum from the reagent flows, on the PARTICLE spectra
(the classifier of step 31 gates this regressor):

    flows Q --features--> model --> PCA scores --reconstruct--> spectrum

PCA(max-normalised particle spectra) -> one regressor per component on features(Q),
cross-validated GROUPED BY SESSION (honest inter-session generalisation). Benchmarks the
feature-engineering x model zoo, then reconstructs, parity-checks lambda_max, draws
reagent-flow colour maps, and a learning curve. Reuses `spectral_pipeline` (engine) and the
nb23 figure functions (`spectral_model`). Trained model + PCA basis are saved for the
prediction tool (step 35). Kernel: Anaconda 3.11.
"""
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import make_scorer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor)
from . import pipeline_common as pc
from . import spectral_model as sm          # nb23 figure functions (take arrays, reusable)

sp = pc.sp
NORMALIZE = "max"
N_PC      = pc.N_PC            # 5-component max-normalised decomposition (centralised)
CV_K      = 6
# ── reconstruction panels: N RANDOM examples (fresh each call), user-selectable ──
N_RECON    = 4                 # how many reconstruction panels to show
RECON_MODE = "custom"          # "random" | "spread" (by lambda_max) | "custom" (RECON_IDS)
# The four paper examples: a monotone lambda_max ladder driven by the Ag/NaBH4 ratio —
#   A 727 nm (Q1/Q5=3.67) -> B 605 (1.44) -> C 487 (0.79) -> D 411 (0.45).
# They are labelled A,B,C,D on the reconstruction grid, the hero Q1xQ5 colour map AND the
# three mechanistic projections, so the same four runs can be followed across every figure.
RECON_IDS  = ["S25-M02_init_4",     # A
              "S28-M02_init_4",        # B
              "S14-M14_sample_6",      # C
              "S14-M14_sample_1"]      # D
RECON_SEED = None              # None -> a DIFFERENT random draw every call; set an int to freeze
HERO_PAIR = ("Q1", "Q5")
MAP_PAIRS = [("Q1", "Q5"), ("Q1", "Q3"), ("Q5", "Q4"), ("Q2", "Q4")]
NOTEBOOK  = "33_spectral_model.ipynb"

# ── deployed model: PINNED, not argmin-selected ───────────────────────────────
#   The benchmark still runs and is reported, but the argmin of `spec_rmse` is NOT used to
#   pick the deployed model. Reason: the top of the zoo is separated by ~0.005 spec_rmse
#   (noise), and the criterion scores OOF error AT THE TRAINING POINTS only — it says nothing
#   about the interpolated surface. Letting it choose flipped the deployment from
#   clr+GP-Matern52 to chem_v2+ExtraTrees, which is a hair better on the metric (R2 0.663 vs
#   0.640) but visibly WORSE on the colour maps: a tree ensemble is piecewise-constant, so the
#   maps come out blocky and desaturated, and it carries no predictive variance, so the
#   "top-decile GP uncertainty" hatching silently disappears.
#   Set FORCE_MODEL = None to go back to argmin selection.
FORCE_MODEL = ("clr", "GP-Matern52")     # (feature_set, model) or None

def spread_idx(values, n):
    """n indices spread evenly across the sorted range of `values`."""
    order = np.argsort(values)
    return [order[int(f * (len(order) - 1))] for f in np.linspace(0.02, 0.98, n)]

def pick_recon(df, lam, mode=RECON_MODE, n=N_RECON, ids=RECON_IDS, seed=RECON_SEED):
    """Indices of the reconstruction examples. Default: n RANDOM points (reproducible via seed).
    'spread' = evenly across lambda_max; 'custom' = the exact filenames in `ids` (for the paper)."""
    if mode == "custom" and ids:
        idx = [int(df.index[df.filename == f][0]) for f in ids if (df.filename == f).any()]
        if idx:
            return idx
        print("  RECON_IDS matched nothing -> falling back to random")
    if mode == "spread":
        return spread_idx(lam, n)
    return list(np.random.default_rng(seed).choice(len(df), min(n, len(df)), replace=False))

def load():
    return pc.load_training()          # step-31b `hardcut` set (see pipeline_common.TRAIN_TAG)

def savefig(fig, name):
    fig.text(0.995, 0.003, f"source: {NOTEBOOK}", ha="right", va="bottom",
             fontsize=6, color="0.55")
    for ext in ("png", "pdf"):
        fig.savefig(pc.FIG_DIR / f"{name}.{ext}", dpi=200, bbox_inches="tight")
    print(f"  saved {name}.png/.pdf")

# ── N-panel reconstruction grid (user-selectable examples) ────────────────────
def fig_recon_grid(wl, Sn, Sp, colour, lam, idx, names=None, flows=None, ncol=5):
    """Measured vs grouped-CV-predicted spectra for the chosen example indices. Each panel is
    labelled A,B,C,… and titled with the run name, its five reagent flows and measured→predicted
    lambda_max, so the panels can be identified and picked for the figure."""
    n = len(idx); nrow = int(np.ceil(n / ncol))
    fig, ax = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 2.7 * nrow), squeeze=False)
    ax = ax.ravel()
    lam_pred = sp.derived_lmax(wl, Sp)
    for a, i, lab in zip(ax, idx, "ABCDEFGHIJKLMNOPQRSTUVWX"):
        a.plot(wl, Sn[i], color=colour(Sn[i]), lw=1.8, label="experiment")
        a.plot(wl, Sp[i], "k:", lw=1.5, label="prediction (leave-session-out CV)")
        a.set_xlim(wl.min(), wl.max()); a.tick_params(labelsize=7)
        a.set_xlabel("Wavelength (nm)", fontsize=8); a.set_ylabel("Extinction (norm.)", fontsize=8)
        ttl = rf"$\lambda_{{\max}}$ {lam[i]:.0f}$\to${lam_pred[i]:.0f} nm"
        if flows is not None:
            ttl += "\n" + "  ".join(f"{c}={v:.1f}" for c, v in zip(sp.FEATURES, flows[i]))
        if names is not None:
            ttl = f"{names[i]}\n" + ttl
        a.set_title(f"({lab}) " + ttl, fontsize=6.5)
    ax[0].legend(fontsize=6, loc="upper right")
    for a in ax[n:]: a.axis("off")
    fig.tight_layout(); return fig

# ── FULL-DOMAIN recipe->colour slice (colour everywhere except total flow > 14) ─
def colour_slice_full(ax, Q, model, mu, comps, colour, qa, qb, grid_n=60, hatch=True):
    ia, ib = sp.FEATURES.index(qa), sp.FEATURES.index(qb)
    others = [k for k in range(5) if k not in (ia, ib)]
    med = np.median(Q, 0); prop = med[others] / med[others].sum()
    # axes span the TRAINED domain: from the reagent floor on the gating flows (Q1/Q5), else 0
    lo_a = pc.TRAIN_Q_FLOOR if qa in ("Q1", "Q5") else 0.0
    lo_b = pc.TRAIN_Q_FLOOR if qb in ("Q1", "Q5") else 0.0
    ga = np.linspace(lo_a, Q[:, ia].max() * 1.02, grid_n)
    gb = np.linspace(lo_b, Q[:, ib].max() * 1.02, grid_n)
    grid = np.full((grid_n, grid_n, 5), np.nan); mask = np.zeros((grid_n, grid_n), bool)
    for iy, vb in enumerate(gb):
        for ix, va in enumerate(ga):
            rem = sp.Q_TOTAL - va - vb
            if rem <= 0:                     # total flow > 14 -> infeasible -> stays white
                continue
            x = np.empty(5); x[ia] = va; x[ib] = vb; x[others] = rem * prop
            grid[iy, ix] = x; mask[iy, ix] = True
    X = grid[mask]; scr = model.predict(X)
    img = np.ones((grid_n, grid_n, 3)); img[mask] = np.array([colour(mu + s @ comps) for s in scr])
    ax.imshow(img, origin="lower", aspect="auto", extent=[ga[0], ga[-1], gb[0], gb[-1]])
    if hatch:
        sd = model.predict_std(X)
        if sd is not None:
            u = np.full((grid_n, grid_n), np.nan); u[mask] = sd.mean(1)
            ax.contourf(ga, gb, u, levels=[np.nanpercentile(u, 80), np.nanmax(u)],
                        colors="none", hatches=["xx"], alpha=0)
    ax.plot([0, sp.Q_TOTAL], [sp.Q_TOTAL, 0], "k--", lw=1, alpha=.6)   # total-flow limit
    ax.scatter(Q[:, ia], Q[:, ib], s=10, facecolor="none", edgecolor="k", lw=.4, alpha=.4)
    ax.set_xlim(ga[0], ga[-1]); ax.set_ylim(gb[0], gb[-1])
    ax.set_xlabel(f"$Q_{{{qa}}}$ (µL/s)"); ax.set_ylabel(f"$Q_{{{qb}}}$ (µL/s)")

def fig_hero_full(Q, model, mu, comps, colour, idx, pair):
    from matplotlib.patches import Rectangle
    fig, ax = plt.subplots(figsize=(6.8, 5.4))
    colour_slice_full(ax, Q, model, mu, comps, colour, *pair)
    ia, ib = sp.FEATURES.index(pair[0]), sp.FEATURES.index(pair[1])
    for i, lab in zip(idx, "ABCDEFGHIJKLMNO"):
        ax.add_patch(Rectangle((Q[i, ia] - .12, Q[i, ib] - .12), .24, .24, fill=False, ec="k", lw=1.3))
        ax.annotate(lab, (Q[i, ia], Q[i, ib]), textcoords="offset points", xytext=(5, 3),
                    fontweight="bold", fontsize=10)
    ax.set_xlabel(r"$Q_{\mathrm{AgNO_3}}$  (Q1, µL/s)")
    ax.set_ylabel(r"$Q_{\mathrm{NaBH_4}}$  (Q5, µL/s)")
    fig.tight_layout(); return fig

def fig_grid_full(Q, model, mu, comps, colour, pairs):
    fig, ax = plt.subplots(2, 2, figsize=(11, 9)); ax = ax.ravel()
    for a, pr in zip(ax, pairs):
        colour_slice_full(a, Q, model, mu, comps, colour, *pr)
    fig.tight_layout(); return fig

# ═══════════════════════════════════════════════════════════════════════════════
#  MECHANISTIC PROJECTIONS — chemically-constrained slices
# ═══════════════════════════════════════════════════════════════════════════════
#  The raw-flow maps above are hard to read chemically: moving one Q also changes the
#  dilution of everything else (and the median-proportion filler is a statistical
#  convention, not a chemical one). Here the plane is spanned by
#
#       x  =  Q_AgNO3 / Q_NaBH4      (silver load vs nucleation burst — log axis)
#       y  =  ONE swept reagent flow (AcOH, PVP or AscAcid)
#
#  and the remaining flows are pinned by CHEMISTRY:
#       • PVP      held at a constant PVP/Ag FLOW ratio (PVP_TO_AG) — capping per silver
#       • AscAcid  held at the redox stoichiometry of the silver    — see below
#       • AcOH     held at ACOH_FLOW µL/s                           — fixed acidity
#  The instrument runs at a FIXED total flow of 14 µL/s, so these constraints close the
#  recipe exactly. With Q5 = Q1/r and Q_i = k_i·Q1 for the proportional reagents,
#
#       Q1 · (1 + 1/r + Σ k_i)  =  Q_TOTAL − Σ (absolute fixed flows)
#
#  which fixes Q1 — hence all five flows — at every point of the (r, y) plane. A cell is
#  left blank where that solution is infeasible (a flow ≤ 0, or Q1/Q5 below the 0.5 µL/s
#  training floor).
#
#  ── STOICHIOMETRY OF THE ASCORBIC ACID  (the "1x or 0.5x ?" question) ─────────────
#  Ag+ + e- -> Ag0                                    is a ONE-electron reduction.
#  H2A -> dehydroascorbic acid + 2 H+ + 2 e-          delivers TWO electrons.
#  One ascorbate therefore reduces TWO Ag+, so full reduction needs
#
#       n(AscAcid) = 0.5 · n(Ag)        ->  Q4 = 0.5 · Q1
#
#  and because the two stocks sit at the SAME concentration (1.8 each), that molar 0.5
#  is also a flow 0.5. It is 0.5x, NOT 1x. Set ASC_EXCESS > 1 for the practical excess
#  usually used to push the kinetics.
#  NaBH4 reduces as well (BH4- + 8 OH- -> B(OH)4- + 4 H2O + 8 e-), but its stock is 50x
#  more dilute, so it supplies 8·(0.036/1.8)/r = 0.16/r electron-equivalents per Ag:
#  negligible at r > 2, but about a third of the electron budget at r = 0.5. Projection 3
#  draws all three reference lines so the difference is explicit.
STOCK_mM   = np.array([1.8, 1.2, 1256.0, 1.8, 0.036])  # AgNO3, PVP, AcOH, AscAcid, NaBH4
N_E_ASC    = 2         # electrons delivered per ascorbic acid  (H2A -> DHA + 2H+ + 2e-)
N_E_BH4    = 8         # electrons delivered per borohydride    (BH4- -> B(OH)4- + 8e-)
ASC_EXCESS = 1.0       # multiplier on the stoichiometric ascorbic flow (1.0 = exact)
PVP_TO_AG  = 1.0       # constant Q_PVP / Q_AgNO3 flow ratio
ACOH_FLOW  = 1.0       # µL/s, fixed acetic-acid flow

# Q4/Q1 flow ratio at exact stoichiometry, and the NaBH4 electron credit (same units)
ASC_PER_AG = float(STOCK_mM[0] / (N_E_ASC * STOCK_mM[3]))                # 0.50
BH4_PER_AG = float(N_E_BH4 * STOCK_mM[4] / (N_E_ASC * STOCK_mM[3]))      # 0.08  (times 1/r)

RATIO_RANGE  = (0.2, 6.0)   # Q_Ag/Q_NaBH4 sweep (data span 0.17-14.6, p5-p95 = 0.26-3.3)
MECH_N       = 60           # grid points per axis
MECH_LAM     = False        # OPT-IN: overlay predicted-lambda_max iso-lines on the colour
                            #   maps. Off by default — the contours fight the colour, which
                            #   is the message. Flip to True (or pass lam=True) for a
                            #   quantitative read; the level list is MECH_LEVELS.
MECH_LEVELS  = np.arange(400, 951, 50)
OOD_MAX      = 2.0          # grey out where the recipe is farther than this x the training
                            #   set's own nearest-neighbour spacing (extrapolation veil)
HATCH_PCT    = 90           # hatch the top decile of the GP predictive uncertainty
POINT_TOL    = 1.5          # a training run is drawn "on-slice" below this distance
# ── overlays on the mechanistic maps: OFF by default (clean colour, publication figure) ──
#   The colour IS the message; the uncertainty hatching and the extrapolation veil fight it.
#   Both are still implemented — flip these to True (or pass hatch=/veil= per call) to audit
#   where the model is uncertain or extrapolating.
MECH_HATCH   = False        # //-hatch the top decile of the GP predictive uncertainty
MECH_VEIL    = False        # grey veil over recipes far from any training run
STOICH_LINES = ("asc2",)    # only the true stoichiometric line (2e- ascorbic, Q_Asc=0.5 Q_Ag).
                            #   Available: "asc2", "asc1" (1e- variant), "total" (full e-
                            #   balance incl. NaBH4).
STOICH_LABEL_X = 0.76       # where along the x axis (0-1) the "excess reductant" /
                            #   "sub-stoichiometric" callouts sit. 0.62 put them straight on
                            #   top of the boxed example B; 0.76 is the gap between B and A.

# ── STYLE of the mechanistic maps — exposed so figures can be retuned without editing code.
#   Shared with step 46 (its time-resolved and t_conv panels call `mech_decorate`), so a change
#   here propagates to every mechanistic projection in the paper.
#   The two point classes answer "is this run really ON this plane?":
#     OFF = its other flows do not satisfy the pinning condition -> small filled BLACK dot
#     ON  = it does                                              -> open circle
PT_OFF   = dict(s=7.0,  color="k",   alpha=0.55, lw=0)       # off-slice: small black dots
PT_ON    = dict(s=30.0, facecolor="none", edgecolor="k", lw=1.0)   # on-slice: open circles
#   FONT SIZES (pt). Every one is read at call time, so `s33.FS_TICK = 9` in the notebook (or
#   `s33.mech_style(FS_TICK=9, ...)`) takes effect on the next figure without a re-import.
FS_TICK  = 8.0              # tick labels (both axes)
FS_LABEL = None             # axis labels; None -> matplotlib's axes.labelsize
FS_TITLE = 8.5              # panel title = the pinning condition (fig_mech_single uses FS_TITLE_1)
FS_TITLE_1 = 9.0            # title of the one-panel version
FS_LEGEND = 6.4             # the stoichiometry legend
FS_ANNOT = 6.3              # the "excess reductant" / "sub-stoichiometric" callouts
FS_MOLAR = 7.0              # the molar top axis (ticks); its label uses FS_TICK
FS_MARK  = 10.0             # the boxed A-D letters
MARK_SHAPE = "square"       # A-D marker: "square" (matches fig_hero_full) or "circle"
MARK_SIZE  = 150.0          # A-D marker area (pt^2); solid = run ON the plane, dashed = projected
FS_FOOT  = 6.0              # the footnote under the figure (blank = ..., open circles = ...)
FS_CLABEL = 6.2             # inline labels of the lambda_max iso-lines (only if MECH_LAM)
TITLE_PAD = 26              # gap (pt) between the molar top axis and the panel title
# LEGEND — position, box and text
LEGEND_LOC = "upper left"   # any matplotlib loc string, or an (x, y) tuple in axes fraction
LEGEND_KW = dict(frameon=True, framealpha=.93, facecolor="white", edgecolor="0.4",
                 borderpad=.5, handlelength=2.4)
#   frameon=True is REQUIRED: the house style (eda_rg024.set_pub_style, applied when
#   spectral_pipeline is imported) sets legend.frameon=False globally, and black text on a dark
#   colour map is unreadable without the box.
STOICH_LEGEND = {           # legend entry per stoichiometry line (see STOICH_LINES)
    "asc2":  r"ascorbic stoich. ($2e^-$): $Q_{Asc}=0.5\,Q_{Ag}$",
    "asc1":  r"if $1e^-$ per ascorbic: $Q_{Asc}=Q_{Ag}$",
    "total": r"full $e^-$ balance incl. NaBH$_4$",
}
ANNOT_ABOVE = "excess reductant\n(full Ag reduction)"      # callout above the stoich. line
ANNOT_BELOW = "sub-stoichiometric\n(Ag$^+$ left over)"     # callout below it ("" = none)
SHOW_FOOT   = True          # the footnote under the figure (False for the paper caption)

STOICH_LEGEND_CONC = {      # same, used when MECH_UNITS == "conc"
    "asc2":  r"ascorbic stoich. ($2e^-$): $n_{Asc}=0.5\,n_{Ag}$",
    "asc1":  r"if $1e^-$ per ascorbic: $n_{Asc}=n_{Ag}$",
    "total": r"full $e^-$ balance incl. NaBH$_4$",
}

_STYLE_KEYS = ("FS_TICK", "FS_LABEL", "FS_TITLE", "FS_TITLE_1", "FS_LEGEND", "FS_ANNOT",
               "FS_MOLAR", "FS_MARK", "FS_FOOT", "FS_CLABEL", "TITLE_PAD", "LEGEND_LOC",
               "LEGEND_KW", "STOICH_LEGEND", "STOICH_LEGEND_CONC", "ANNOT_ABOVE", "ANNOT_BELOW",
               "SHOW_FOOT", "PT_OFF", "PT_ON", "STOICH_LINES", "STOICH_LABEL_X", "MECH_UNITS",
               "MARK_SHAPE", "MARK_SIZE")

def mech_style(**kw):
    """Set several style knobs at once: `s33.mech_style(FS_TICK=9, LEGEND_LOC="lower right")`.
    Unknown names raise, so a typo never silently does nothing. Returns the current values of
    every knob (call with no argument to print them)."""
    for k, v in kw.items():
        if k not in _STYLE_KEYS:
            raise KeyError(f"{k!r} is not a style knob; choose from {_STYLE_KEYS}")
        globals()[k] = v
    return {k: globals()[k] for k in _STYLE_KEYS}

# ── AXIS UNITS: flows (µL/s) or reactor concentrations (mM) ─────────────────────────
#   The instrument runs at a FIXED total flow (every training run sums to 14.000 µL/s) and the
#   stocks are fixed, so the concentration of reagent i in the reactor after mixing is
#       C_i = STOCK_mM[i] · Q_i / Q_TOTAL            (mM)
#   — a pure LINEAR rescaling of each flow, and n(Ag)/n(NaBH4) = (STOCK_Ag/STOCK_BH4)·Q1/Q5 = 50·r.
#   So a "concentration" map is the SAME plane as the flow map: the model is evaluated on the same
#   recipes, the cached cubes are reused, and only the tick positions / labels / titles change.
#   MECH_UNITS = "conc" switches every mechanistic figure (nb33 statics, nb46 colour + t_conv, nb47)
#   to molar-ratio x / mM y; the flow ratio moves to the secondary top axis.
MECH_UNITS = "flow"         # "flow" (µL/s, flow ratio) | "conc" (mM in the reactor, molar ratio)
# x graduations of the molar-ratio axis (MECH_UNITS == "conc"): labelled ticks + unlabelled minor ones
CONC_XTICKS = (10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 150, 200, 300, 500)
CONC_YBINS       = 12     # max number of LABELLED y ticks (concentration axis), full-size panels
CONC_YBINS_SMALL = 6      #   small panels (fs < 0.9)
CONC_YMINOR      = 2      # minor ticks: this many sub-intervals between two labelled ticks
CONC_XTICKS_SMALL =(10, 20, 30, 50, 100, 200, 300)          # same, on small panels (fs < 0.9)
CONC_XTICKS_MINOR =tuple(range(10, 100, 5)) + tuple(range(100, 501, 25))
REAGENT_TEX = {"Q1": r"AgNO_3", "Q2": r"PVP", "Q3": r"AcOH", "Q4": r"Asc", "Q5": r"NaBH_4"}

def conc_mM(flow_key, q):
    """Reactor concentration (mM) of the reagent fed by `flow_key` at flow `q` (µL/s)."""
    return STOCK_mM[sp.FEATURES.index(flow_key)] * np.asarray(q, float) / sp.Q_TOTAL

def flow_of_conc(flow_key, c):
    """Inverse of `conc_mM`: the flow (µL/s) that gives reactor concentration `c` (mM)."""
    return np.asarray(c, float) * sp.Q_TOTAL / STOCK_mM[sp.FEATURES.index(flow_key)]

def molar_per_ag(flow_key, k):
    """Molar ratio n(reagent)/n(Ag) for a FLOW ratio Q_reagent/Q_Ag = k."""
    return k * STOCK_mM[sp.FEATURES.index(flow_key)] / STOCK_mM[0]


class MechProj(dict):
    """A projection definition. `proj["cond"]` / `proj["ylab"]` resolve at READ time to the flow or
    the concentration wording according to `MECH_UNITS`, so every caller (here, step 46, step 47)
    follows the switch without being edited. The raw strings stay under cond_flow / cond_conc."""
    def __getitem__(self, k):
        if k in ("cond", "ylab"):
            return dict.__getitem__(self, f"{k}_{'conc' if MECH_UNITS == 'conc' else 'flow'}")
        return dict.__getitem__(self, k)
    def get(self, k, default=None):
        try:
            return self[k]
        except KeyError:
            return default


def make_proj(key, pvp_to_ag=None, acoh_flow=None, asc_excess=None):
    """Build one of the three projections ("acoh", "pvp", "asc") with the pinned values given (the
    module knobs by default). Both wordings of the pinning condition are generated here."""
    pvp_to_ag  = PVP_TO_AG  if pvp_to_ag  is None else float(pvp_to_ag)
    acoh_flow  = ACOH_FLOW  if acoh_flow  is None else float(acoh_flow)
    asc_excess = ASC_EXCESS if asc_excess is None else float(asc_excess)
    a_asc = ASC_PER_AG * asc_excess
    # concentration wording of each pinned quantity
    pvp_f = r"$Q_{PVP}/Q_{Ag}=%g$" % pvp_to_ag
    # molar value + the flow ratio in brackets: the user sets PVP/Ag as a FLOW ratio, and "0.67" alone
    # did not read as "the PVP = 1 map"
    pvp_c = r"$n_{PVP}/n_{Ag}=%.2g$ ($Q_{PVP}/Q_{Ag}=%g$)" % (molar_per_ag("Q2", pvp_to_ag), pvp_to_ag)
    asc_f = r"$Q_{Asc}=%g\,Q_{Ag}$ (stoich.)" % a_asc
    asc_c = r"$n_{Asc}=%g\,n_{Ag}$ (stoich.)" % molar_per_ag("Q4", a_asc)
    aco_f = r"$Q_{AcOH}=%g$ µL/s" % acoh_flow
    aco_c = r"[AcOH] $=%.0f$ mM" % conc_mM("Q3", acoh_flow)
    if key == "acoh":
        p = dict(axis="Q3", prop={"Q2": pvp_to_ag, "Q4": a_asc}, fixed={},
                 cond_flow=f"{pvp_f},   {asc_f}", cond_conc=f"{pvp_c},   {asc_c}")
    elif key == "pvp":
        p = dict(axis="Q2", prop={"Q4": a_asc}, fixed={"Q3": acoh_flow},
                 cond_flow=f"{aco_f},   {asc_f}", cond_conc=f"{aco_c},   {asc_c}")
    elif key == "asc":
        p = dict(axis="Q4", prop={"Q2": pvp_to_ag}, fixed={"Q3": acoh_flow}, stoich=True,
                 cond_flow=f"{pvp_f},   {aco_f}", cond_conc=f"{pvp_c},   {aco_c}")
    else:
        raise KeyError(key)
    name = {"Q3": r"\mathrm{AcOH}", "Q2": r"\mathrm{PVP}", "Q4": r"\mathrm{AscAcid}"}[p["axis"]]
    p["ylab_flow"] = r"$Q_{%s}$  (µL/s)" % name
    p["ylab_conc"] = r"$[%s]$  (mM)" % name
    p["base"] = key
    p["pvp_to_ag"] = pvp_to_ag
    return MechProj(p)

def mech_file_tag(key):
    """Explicit name for figure files: the PVP/Ag value is spelled out on every plane that pins it,
    the default included ("acoh" -> "acoh_pvp1", "acoh_pvp0.5" unchanged, "pvp" unchanged)."""
    p = MECH_PROJ[key]
    if "Q2" in p["prop"] and "_pvp" not in key:
        return f"{key}_pvp{p['pvp_to_ag']:g}"
    return key


MECH_PROJ = {k: make_proj(k) for k in ("acoh", "pvp", "asc")}
MECH_ORDER = ["acoh", "pvp", "asc"]

def mech_variant(key, pvp_to_ag=None, acoh_flow=None, asc_excess=None):
    """Register a re-pinned copy of projection `key` in MECH_PROJ and return its name, e.g.
    `mech_variant("acoh", pvp_to_ag=0.5)` -> "acoh_pvp0.5". Every step-33/46 function that takes a
    projection key (cubes, colour maps, t_conv maps) then works on it unchanged."""
    tag = "".join(f"_{n}{v:g}" for n, v in (("pvp", pvp_to_ag), ("acoh", acoh_flow),
                                            ("asc", asc_excess)) if v is not None)
    name = key + tag
    MECH_PROJ[name] = make_proj(key, pvp_to_ag, acoh_flow, asc_excess)
    return name

def mech_recipe_at(proj, r, y, q_total=None):
    """Close the 5-flow recipe from (r = Q1/Q5, y = the swept flow) under `proj`'s chemical
    constraints and the fixed total flow. Broadcasts over arrays; returns (..., 5) in the
    Q1..Q5 order (no feasibility check — see mech_recipes)."""
    q_total = float(sp.Q_TOTAL if q_total is None else q_total)
    r = np.asarray(r, float); y = np.asarray(y, float)
    ax_i  = sp.FEATURES.index(proj["axis"])
    prop  = {sp.FEATURES.index(k): v for k, v in proj["prop"].items()}
    fixed = {sp.FEATURES.index(k): v for k, v in proj["fixed"].items()}
    with np.errstate(divide="ignore", invalid="ignore"):
        Q1 = (q_total - sum(fixed.values()) - y) / (1.0 + 1.0 / r + sum(prop.values()))
    G = np.zeros(np.broadcast(r, y).shape + (5,))
    G[..., 0] = Q1
    G[..., 4] = Q1 / r
    for i, k in prop.items():  G[..., i] = k * Q1
    for i, v in fixed.items(): G[..., i] = v
    G[..., ax_i] = y
    return G

def mech_recipes(proj, rr, yy, q_total=None):
    """(ny, nx, 5) recipe cube over the (ratio x swept-flow) grid + feasibility mask."""
    R, Y = np.meshgrid(rr, yy)
    G = mech_recipe_at(proj, R, Y, q_total)
    ok = np.isfinite(G).all(-1) & (G > 0).all(-1)
    ok &= (G[..., 0] >= pc.TRAIN_Q_FLOOR) & (G[..., 4] >= pc.TRAIN_Q_FLOOR)
    return G, ok

def stoich_y(proj, rr, a, q_total=None):
    """y(r) along which the SWEPT flow equals a·Q1 (a scalar or an array over rr).
    From y = a·Q1 and Q1 = (T - sum(fixed) - y)/A0:   y = a(T - sum(fixed))/(A0 + a)."""
    q_total = float(sp.Q_TOTAL if q_total is None else q_total)
    a  = np.asarray(a, float)
    A0 = 1.0 + 1.0 / np.asarray(rr, float) + sum(proj["prop"].values())
    return a * (q_total - sum(proj["fixed"].values())) / (A0 + a)

def domain_scale(Q):
    """(per-flow std, median nearest-neighbour spacing, KD-tree) of the training recipes.
    Distances are then expressed in units of that spacing: 1.0 = as far from the data as a
    typical training point is from its own nearest neighbour."""
    from scipy.spatial import cKDTree
    s = Q.std(0)
    tree = cKDTree(Q / s)
    d, _ = tree.query(Q / s, k=2)
    return s, float(np.median(d[:, 1])), tree

def slice_distance(proj, Q, cache=None):
    """How far each recipe in `Q` sits FROM the slice `proj`, in the same units as
    `domain_distance` (multiples of the training set's median nearest-neighbour spacing):
    the distance between the run and the slice recipe closed at that run's own (ratio, swept
    flow). 0 = the run lies exactly on the plane. This is the honest test of "is this point
    really on this slice" — `domain_distance` answers the different question "does the slice
    pass anywhere near the data"."""
    s, scale, _ = cache or domain_scale(Q)
    G = mech_recipe_at(proj, Q[:, 0] / Q[:, 4], Q[:, sp.FEATURES.index(proj["axis"])])
    return np.linalg.norm((Q - G) / s, axis=1) / scale


# ── MODEL UNCERTAINTY, referenced to the experimental noise ──────────────────────────────
#   Every deployed GP has kernel  A·Matern + WhiteKernel(noise). sklearn's predictive std INCLUDES
#   the white noise, so  sigma_tot^2 = sigma_noise^2 + sigma_epi^2  and the epistemic part — what more
#   data would remove — is  sigma_epi = sqrt(sigma_tot^2 - sigma_noise^2). The fitted noise is the
#   run-to-run irreproducibility the GP attributes to the data, so
#       U = sigma_epi / sigma_noise
#   reads: U < 1  the model knows the recipe better than a repeat of the experiment would scatter;
#          U > 1  the model's own ignorance dominates -> "uncertain zone".
#   No percentile, no arbitrary scale: the threshold 1 is the repeatability of the experiment itself.
def _white_noise(k):
    """noise_level of the WhiteKernel inside a (sum/product) kernel, 0 if none."""
    from sklearn.gaussian_process.kernels import WhiteKernel
    if isinstance(k, WhiteKernel):
        return float(k.noise_level)
    return sum(_white_noise(getattr(k, a)) for a in ("k1", "k2") if hasattr(k, a))

def _gp_split(pipe, X):
    """(total predictive variance, noise variance) of one fitted GP pipeline, in y units."""
    gp = pipe.steps[-1][1] if hasattr(pipe, "steps") else pipe
    _, sd = pipe.predict(X, return_std=True)
    ystd = float(np.ravel(getattr(gp, "_y_train_std", 1.0))[0]) if gp.normalize_y else 1.0
    return sd ** 2, _white_noise(gp.kernel_) * ystd ** 2

def gp_uncertainty_ratio(models, X):
    """U = sigma_epi / sigma_noise for one GP pipeline or a list of them (one per PCA score: the
    variances are summed, the PCA basis being orthonormal -> spectral-norm uncertainty)."""
    models = models if isinstance(models, (list, tuple)) else [models]
    epi = np.zeros(len(X)); noi = 0.0
    for m in models:
        tot, nz = _gp_split(m, X)
        epi += np.clip(tot - nz, 0, None); noi += nz
    return np.sqrt(epi / max(noi, 1e-30))

def spectral_uncertainty_ratio(smodel, Q):
    """U for a `sp.SpectralModel` (one GP per PCA component) at recipes Q."""
    return gp_uncertainty_ratio(smodel.models, smodel.feat(np.atleast_2d(Q)))


def domain_distance(X, Q, cache=None):
    s, scale, tree = cache or domain_scale(Q)
    d, _ = tree.query(np.asarray(X, float).reshape(-1, 5) / s)
    return (d / scale).reshape(np.shape(X)[:-1])

# ── one mechanistic panel ─────────────────────────────────────────────────────
def mech_slice(ax, Q, model, mu, comps, colour, wl, proj, n=MECH_N, hatch=None,
               lam=None, points=True, cache=None, mark=None, veil=None):
    """Perceived-colour map over (Q_Ag/Q_NaBH4) x (one swept flow), every other flow pinned
    by chemistry. Overlays: predicted-lambda_max iso-lines, GP-uncertainty hatching, a grey
    extrapolation veil, the training runs (filled = on this slice), and — for the ascorbic
    projection — the redox stoichiometry lines."""
    lam   = MECH_LAM   if lam   is None else lam    # resolved HERE, so setting s33.MECH_LAM
    hatch = MECH_HATCH if hatch is None else hatch  #   after import still takes effect
    veil  = MECH_VEIL  if veil  is None else veil
    rr = np.geomspace(*RATIO_RANGE, n)
    ax_i = sp.FEATURES.index(proj["axis"])
    yy = np.linspace(0.0, proj.get("ymax") or Q[:, ax_i].max() * 1.02, n)
    G, ok = mech_recipes(proj, rr, yy)
    Sp = mu + model.predict(G[ok]) @ comps
    lx = np.log10(rr); ext = [lx[0], lx[-1], yy[0], yy[-1]]

    img = np.ones((n, n, 3)); img[ok] = np.array([colour(s) for s in Sp])
    ax.imshow(img, origin="lower", aspect="auto", extent=ext)

    # OUT-OF-DOMAIN: recipes the declared validity floor (Q_AgNO3, Q_NaBH4 >= TRAIN_Q_FLOOR)
    # excludes. mech_recipes() already drops them from `ok`, so they were rendered as plain
    # white — indistinguishable from the figure background. Mark them explicitly instead:
    # on these ratio-space slices the floor only bites in tiny corners (0-0.4 % of the grid),
    # and a silently blank corner reads as "no data" rather than "outside the stated domain".
    fin = np.isfinite(G).all(-1) & (G > 0).all(-1)
    below = fin & ~((G[..., 0] >= pc.TRAIN_Q_FLOOR) & (G[..., 4] >= pc.TRAIN_Q_FLOOR))
    # left blank (white) by the `ok` mask; drawn as a plain black boundary only — no fill and
    # no hatching, so the map stays clean while the domain edge is still explicit.
    if below.any():
        ax.contour(lx, yy, below.astype(float), levels=[0.5], colors="k",
                   linewidths=1.0, zorder=2.7)

    if lam:                                       # predicted-lambda_max iso-lines
        import matplotlib.patheffects as pe
        L = np.full((n, n), np.nan); L[ok] = sp.derived_lmax(wl, Sp)
        cs = ax.contour(lx, yy, L, levels=MECH_LEVELS, colors="k", linewidths=.8, alpha=.6)
        for c in ax.clabel(cs, fmt="%.0f nm", fontsize=FS_CLABEL, inline=True):
            c.set_path_effects([pe.withStroke(linewidth=1.8, foreground="w")])

    if hatch:                                     # GP predictive uncertainty (top decile)
        sd = model.predict_std(G[ok])
        if sd is not None:
            u = np.full((n, n), np.nan); u[ok] = sd.mean(1)
            with plt.rc_context({"hatch.linewidth": 0.5, "hatch.color": "0.30"}):
                ax.contourf(lx, yy, u, levels=[np.nanpercentile(u, HATCH_PCT), np.nanmax(u)],
                            colors="none", hatches=["//"], alpha=0)

    if veil:                                      # extrapolation veil (MECH_VEIL)
        d = np.full((n, n), np.nan)
        d[ok] = domain_distance(G[ok], Q, cache)
        vl = np.zeros((n, n, 4)); vl[..., :3] = 0.62
        vl[..., 3] = np.where(np.nan_to_num(d, nan=0.0) > OOD_MAX, 0.40, 0.0)
        ax.imshow(vl, origin="lower", aspect="auto", extent=ext)
        if np.nanmax(d) > OOD_MAX > np.nanmin(d):
            ax.contour(lx, yy, d, levels=[OOD_MAX], colors="w", linewidths=1.0,
                       linestyles=[(0, (4, 2))])

    mech_decorate(ax, proj, rr, yy, Q, cache, stoich=True, points=points, mark=mark)
    return dict(rr=rr, yy=yy, G=G, ok=ok)


# ── shared axis decoration (static map AND the time-resolved panels of step 46) ──
def mech_decorate(ax, proj, rr, yy, Q=None, cache=None, stoich=True, points=True,
                  fs=1.0, legend=True, molar=True, mark=None):
    """Log-ratio ticks, the molar top axis, the axis labels, the redox stoichiometry lines and the
    training-run overlay. `fs` scales every font (use ~0.75 in a small matrix panel); `points`
    needs `Q` (the training flows). Both `mech_slice` here and `step46.mech_panel` call this, so the
    static and the time-resolved maps carry exactly the same annotations."""
    n = len(rr); lx = np.log10(rr)
    if stoich and proj.get("stoich"):             # redox reference lines (projection 3)
        a_tot  = np.clip(ASC_PER_AG - BH4_PER_AG / rr, 0, None)
        curves = {                                # (y(r), linestyle); text from STOICH_LEGEND
            "asc2":  (stoich_y(proj, rr, ASC_PER_AG), "-"),
            "asc1":  (stoich_y(proj, rr, 1.0), "--"),
            "total": (stoich_y(proj, rr, a_tot), ":"),
        }
        leg_txt = STOICH_LEGEND_CONC if MECH_UNITS == "conc" else STOICH_LEGEND
        for k in STOICH_LINES:
            y_c, ls = curves[k]
            ax.plot(lx, y_c, color="k",  ls=ls, lw=3.4 * fs, alpha=.30, zorder=3.9)
            ax.plot(lx, y_c, color="w",  ls=ls, lw=1.9 * fs, label=leg_txt[k], zorder=4.0)
        if legend:
            ax.legend(fontsize=FS_LEGEND * fs, loc=LEGEND_LOC, **LEGEND_KW).set_zorder(6)
            # which side of the switch you are on
            y_s = curves["asc2"][0]; j = int(STOICH_LABEL_X * n)
            for dy, txt, va in ((+0.9, ANNOT_ABOVE, "bottom"), (-0.55, ANNOT_BELOW, "top")):
                if not txt:
                    continue
                ax.annotate(txt, (lx[j], y_s[j] + dy), fontsize=FS_ANNOT * fs, ha="center", va=va,
                            zorder=6, bbox=dict(boxstyle="round,pad=0.28", fc="white",
                                                ec="0.55", alpha=.86))

    if points and Q is not None:                  # training runs, split by slice distance
        ax_i = sp.FEATURES.index(proj["axis"])
        r_d = Q[:, 0] / Q[:, 4]; y_d = Q[:, ax_i]
        near = slice_distance(proj, Q, cache) < POINT_TOL   # is the RUN on this plane?
        dd = None
        inx  = (r_d >= rr[0]) & (r_d <= rr[-1]) & (y_d <= yy[-1])
        off = dict(PT_OFF); off["s"] = off["s"] * fs ** 2
        on  = dict(PT_ON);  on["s"]  = on["s"]  * fs ** 2; on["lw"] = on["lw"] * fs
        ax.scatter(np.log10(r_d[inx & ~near]), y_d[inx & ~near], **off)   # black dots: off-slice
        ax.scatter(np.log10(r_d[inx & near]),  y_d[inx & near],  **on)    # circles: on-slice

    # the labelled A,B,C,D examples — the SAME runs as the reconstruction grid and the hero Q1xQ5
    # map. Drawn whenever `mark` and `Q` are given, even with points=False (the t_conv panels).
    # A run whose other flows do not satisfy this projection's pinning condition does not literally
    # lie on the plane: its ring is DASHED and its label parenthesised, so a projected point is
    # never mistaken for an on-plane one.
    if mark and Q is not None:
        import matplotlib.patheffects as pe
        ax_i = sp.FEATURES.index(proj["axis"])
        r_d = Q[:, 0] / Q[:, 4]; y_d = Q[:, ax_i]
        on_pl = slice_distance(proj, Q, cache) < POINT_TOL
        mk = {"square": "s", "circle": "o"}[MARK_SHAPE]
        for i, lab in mark:
            if not (rr[0] <= r_d[i] <= rr[-1] and y_d[i] <= yy[-1]):
                continue                           # outside this panel's window
            x_i = np.log10(r_d[i]); ls = "-" if on_pl[i] else (0, (2.5, 1.5))
            # white halo under a black ring: readable on pale colloid colours AND dark viridis.
            # zorder 7 > the stoichiometry annotations (6), which otherwise bury a marker
            ax.scatter([x_i], [y_d[i]], s=MARK_SIZE * fs ** 2, marker=mk, facecolor="none",
                       edgecolor="w", lw=3.2 * fs, zorder=7.0)
            ax.scatter([x_i], [y_d[i]], s=MARK_SIZE * fs ** 2, marker=mk, facecolor="none",
                       edgecolor="k", lw=1.5 * fs, linestyle=ls, zorder=7.05)
            ax.annotate(lab if on_pl[i] else f"({lab})", (x_i, y_d[i]), textcoords="offset points",
                        xytext=(0.62 * np.sqrt(MARK_SIZE) * fs, 0.62 * np.sqrt(MARK_SIZE) * fs),
                        fontweight="bold", fontsize=FS_MARK * fs, zorder=7.1,
                        path_effects=[pe.withStroke(linewidth=2.4, foreground="w")])

    ax.set_xlim(lx[0], lx[-1]); ax.set_ylim(yy[0], yy[-1])
    lab_fs = (plt.rcParams["axes.labelsize"] if (FS_LABEL is None and fs == 1.0)
              else (FS_LABEL or FS_TICK) * fs)
    if MECH_UNITS == "conc":
        return _decorate_conc(ax, proj, rr, yy, fs, lab_fs, molar)
    tk = [t for t in (0.2, 0.3, 0.5, 0.7, 1, 1.5, 2, 3, 5, 8, 12) if rr[0] <= t <= rr[-1]]
    ax.set_xticks(np.log10(tk))
    ax.set_xticklabels([f"{t:g}" for t in tk], fontsize=FS_TICK * fs)
    ax.tick_params(labelsize=FS_TICK * fs)
    ax.set_xlabel(r"$Q_{\mathrm{AgNO_3}}\,/\,Q_{\mathrm{NaBH_4}}$   (flow ratio)", fontsize=lab_fs)
    ax.set_ylabel(proj["ylab"], fontsize=lab_fs)
    if not molar:                                 # crowded matrix panels: top axis on row 1 only
        return None
    sec = ax.twiny(); sec.set_xlim(ax.get_xlim())
    st = [t for t in (0.2, 0.5, 1, 2, 5, 10) if rr[0] <= t <= rr[-1]]
    sec.set_xticks(np.log10(st))
    sec.set_xticklabels([f"{t * STOCK_mM[0] / STOCK_mM[4]:.0f}" for t in st],
                        fontsize=FS_MOLAR * fs)
    sec.set_xlabel(r"molar  $n(\mathrm{Ag})\,/\,n(\mathrm{NaBH_4})$",
                   fontsize=FS_TICK * fs, labelpad=3)
    sec.tick_params(length=2, labelsize=FS_MOLAR * fs)
    return sec


def _decorate_conc(ax, proj, rr, yy, fs, lab_fs, molar):
    """Axis ticks/labels for MECH_UNITS == "conc". The DATA coordinates stay (log10 flow ratio,
    flow): with a fixed total flow both are linear images of the molar ratio and the reactor
    concentration, so only where the ticks sit and what they read changes."""
    from matplotlib.ticker import MaxNLocator
    m = STOCK_mM[0] / STOCK_mM[4]                      # molar ratio per unit flow ratio (= 50)
    lo, hi = rr[0] * m * (1 - 1e-6), rr[-1] * m * (1 + 1e-6)   # tolerance: 0.2*50 = 10.000000001
    # small panels (matrices, the 3x4 variant grid: fs < 0.9) get the sparse set, or labels collide
    mk = [t for t in (CONC_XTICKS if fs >= 0.9 else CONC_XTICKS_SMALL) if lo <= t <= hi]
    ax.set_xticks(np.log10(np.array(mk) / m))
    ax.set_xticklabels([f"{t:g}" for t in mk], fontsize=FS_TICK * fs)
    mn = [t for t in CONC_XTICKS_MINOR if lo <= t <= hi and t not in mk]
    ax.set_xticks(np.log10(np.array(mn) / m), minor=True)
    ax.tick_params(axis="x", which="minor", length=3)
    ax.set_xlabel(r"$[\mathrm{AgNO_3}]\,/\,[\mathrm{NaBH_4}]$   (molar ratio)", fontsize=lab_fs)
    key = proj["axis"]
    c0, c1 = conc_mM(key, yy[0]), conc_mM(key, yy[-1])
    nb = CONC_YBINS if fs >= 0.9 else CONC_YBINS_SMALL
    ct = [c for c in MaxNLocator(nbins=nb, steps=[1, 2, 2.5, 5, 10]).tick_values(c0, c1)
          if c0 - 1e-9 <= c <= c1 + 1e-9]
    ax.set_yticks(flow_of_conc(key, ct))
    ax.set_yticklabels([f"{c:g}" for c in np.round(ct, 6)], fontsize=FS_TICK * fs)
    if len(ct) > 1:                                    # unlabelled minor ticks between the labels
        st = (ct[1] - ct[0]) / CONC_YMINOR
        mn = [c for c in np.arange(ct[0] - st * CONC_YMINOR, c1 + st / 2, st)
              if c0 - 1e-9 <= c <= c1 + 1e-9 and np.min(np.abs(np.array(ct) - c)) > st / 4]
        ax.set_yticks(flow_of_conc(key, mn), minor=True)
        ax.tick_params(axis="y", which="minor", length=3)
    ax.set_ylim(yy[0], yy[-1])
    ax.tick_params(labelsize=FS_TICK * fs)
    ax.set_ylabel(proj["ylab"], fontsize=lab_fs)
    if not molar:
        return None
    sec = ax.twiny(); sec.set_xlim(ax.get_xlim())     # the flow ratio, for traceability
    st = [t for t in (0.2, 0.5, 1, 2, 5, 10) if rr[0] <= t <= rr[-1]]
    sec.set_xticks(np.log10(st))
    sec.set_xticklabels([f"{t:g}" for t in st], fontsize=FS_MOLAR * fs)
    sec.set_xlabel(r"flow ratio  $Q_{\mathrm{AgNO_3}}/Q_{\mathrm{NaBH_4}}$",
                   fontsize=FS_TICK * fs, labelpad=3)
    sec.tick_params(length=2, labelsize=FS_MOLAR * fs)
    return sec


_MECH_FOOT = ("blank = outside the declared domain "
              r"($Q_{AgNO_3}$ or $Q_{NaBH_4} < 0.5$ µL/s)"
              "   ·   open circles = training runs ON this slice"
              "   ·   small black dots = runs at the same $(r, y)$ but a different balance on the "
              "pinned reagents")
_MARK_FOOT = ("   ·   boxed A–D = the reconstruction examples of the same figure, "
              "projected onto this slice")

def fig_mech_grid(Q, model, mu, comps, colour, wl, keys=None, mark=None):
    """The three requested projections side by side. `mark` = [(row_index, 'A'), ...]."""
    keys = keys or MECH_ORDER
    cache = domain_scale(Q)
    fig, ax = plt.subplots(1, len(keys), figsize=(5.8 * len(keys), 5.6))
    ax = np.atleast_1d(ax)
    for a, k, tag in zip(ax, keys, "123456"):
        mech_slice(a, Q, model, mu, comps, colour, wl, MECH_PROJ[k], cache=cache, mark=mark)
        a.set_title(f"({tag})  " + MECH_PROJ[k]["cond"], fontsize=FS_TITLE, pad=TITLE_PAD)
    if SHOW_FOOT:
        fig.text(0.005, 0.004, _MECH_FOOT + (_MARK_FOOT if mark else ""),
                 fontsize=FS_FOOT, color="0.4", ha="left", va="bottom")
    fig.tight_layout(); return fig

def fig_mech_single(Q, model, mu, comps, colour, wl, key, cache=None, mark=None):
    """One mechanistic projection, full page width (for the paper)."""
    fig, ax = plt.subplots(figsize=(6.8, 5.8))
    mech_slice(ax, Q, model, mu, comps, colour, wl, MECH_PROJ[key],
               cache=cache or domain_scale(Q), mark=mark)
    ax.set_title(MECH_PROJ[key]["cond"], fontsize=FS_TITLE_1, pad=TITLE_PAD)
    # two lines on the single-panel version: the one-liner runs into the "source:" stamp that
    # savefig() puts at the bottom-right of this narrower figure
    if SHOW_FOOT:
        foot = _MECH_FOOT + (_MARK_FOOT if mark else "")
        fig.text(0.005, 0.010, foot.replace("   ·   ", "\n", 1) if len(foot) > 90 else foot,
                 fontsize=FS_FOOT, color="0.4", ha="left", va="bottom")
    fig.tight_layout(); return fig

# ══════════════════════════════════════════════════════════════════════════════
# SHAPE METRICS — report the whole predicted curve, not just its peak position
# ══════════════════════════════════════════════════════════════════════════════
#   lambda_max is a lossy and FRAGILE summary of a curve the model predicts in full: on this
#   set 20 runs have the right peak (<20 nm) but a wrong shape (SAM>15 deg), against only 1 the
#   other way round. It is also quantised — a 5-PC reconstruction can only place its argmax on
#   the fixed bumps of the basis, which is what banded the old parity plot.
#
#   SAM (spectral angle) = the angle between measured and predicted spectra seen as vectors.
#   Amplitude-invariant, so it measures shape alone; reported in degrees because the cosine
#   compresses every broad positive spectrum into 0.8-1.0 and hides the differences.
#
#   Two references make the number readable, and BOTH must be reported with it:
#     * mean-spectrum baseline — what "no model at all" already scores (~26 deg here)
#     * repeatability floor    — how much two runs of the SAME recipe differ (~9-14 deg here).
#       This is the honest target: a model at the floor cannot be improved with this data.
SAM_REPEAT_DQ = 0.40        # recipes closer than this (in per-flow std units) count as repeats

def sam(A, B):
    """Spectral angle (degrees) between matching rows of A and B."""
    num = (A * B).sum(1)
    den = np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1)
    return np.degrees(np.arccos(np.clip(num / np.maximum(den, 1e-12), -1, 1)))

def _sam_matrix(A, B):
    An = A / np.maximum(np.linalg.norm(A, axis=1, keepdims=True), 1e-12)
    Bn = B / np.maximum(np.linalg.norm(B, axis=1, keepdims=True), 1e-12)
    return np.degrees(np.arccos(np.clip(An @ Bn.T, -1, 1)))

def repeatability_floor(Q, Sn, dq=SAM_REPEAT_DQ):
    """Median spectral angle between runs of near-identical recipes = the experimental noise
    the model is competing against. Returns (median_deg, n_pairs, best_pair_deg_or_nan)."""
    s = Q.std(0)
    dQ = np.sqrt((((Q[:, None, :] - Q[None, :, :]) / s) ** 2).sum(-1))
    np.fill_diagonal(dQ, np.inf)
    i, j = np.where(np.triu(dQ < dq, 1))
    if len(i) == 0:
        return np.nan, 0, np.nan
    v = _sam_matrix(Sn, Sn)[i, j]
    return float(np.median(v)), len(i), float(v[int(np.argmin(dQ[i, j]))])

def shape_report(Sn, Sp, Q, wl, seed=None):
    """Every number the shape panel needs, as a dict."""
    rng = np.random.default_rng(pc.SEED if seed is None else seed)
    m_sam = sam(Sn, Sp)
    base_sam = sam(Sn, np.repeat(Sn.mean(0)[None, :], len(Sn), 0))
    shuf_sam = sam(Sn, Sn[rng.permutation(len(Sn))])
    D = _sam_matrix(Sp, Sn)
    rank = (D < D[np.arange(len(Sn)), np.arange(len(Sn))][:, None]).sum(1)
    ssr = ((Sn - Sp) ** 2).sum(0); sst = ((Sn - Sn.mean(0)) ** 2).sum(0)
    floor, n_pairs, best = repeatability_floor(Q, Sn)
    return dict(sam=m_sam, base=base_sam, shuf=shuf_sam, rank=rank,
                r2_wl=1 - ssr / np.maximum(sst, 1e-12), wl=wl,
                floor=floor, n_pairs=n_pairs, floor_best=best,
                r2_global=1 - np.sum((Sn - Sp) ** 2) / np.sum((Sn - Sn.mean(0)) ** 2))

def fig_shape_metrics(rep, Sn, Sp, wl, df=None):
    """(a) SAM distribution against both references, (b) R2 per wavelength,
    (c) retrieval curve, (d) the model's own error ladder."""
    fig, ax = plt.subplots(2, 2, figsize=(12.4, 8.2)); ax = ax.ravel()
    s, b, sh = rep["sam"], rep["base"], rep["shuf"]

    bins = np.linspace(0, 70, 36)
    ax[0].hist(sh, bins=bins, color="0.85", label=f"random spectrum (med {np.median(sh):.0f}°)")
    ax[0].hist(b,  bins=bins, color="#E69F00", alpha=.75,
               label=f"mean spectrum, no model (med {np.median(b):.0f}°)")
    ax[0].hist(s,  bins=bins, color="#0072B2", alpha=.85,
               label=f"MODEL (med {np.median(s):.1f}°)")
    if np.isfinite(rep["floor"]):
        ax[0].axvline(rep["floor"], color="#009E73", lw=2.4,
                      label=f"repeatability floor ({rep['floor']:.1f}°, n={rep['n_pairs']} pairs)")
        if np.isfinite(rep["floor_best"]):
            ax[0].axvline(rep["floor_best"], color="#009E73", lw=1.6, ls=":",
                          label=f"identical recipe, 2 sessions ({rep['floor_best']:.1f}°)")
    ax[0].set_xlabel("spectral angle to the measurement (deg)")
    ax[0].set_ylabel("runs"); ax[0].legend(fontsize=7.2)
    ax[0].set_title("(a) shape agreement — the model sits at the experimental floor", fontsize=9)

    ax[1].plot(wl, rep["r2_wl"], color="#0072B2", lw=1.5)
    ax[1].axhline(0, color="0.7", lw=.8)
    ax[1].set_xlim(wl.min(), wl.max()); ax[1].set_ylim(-0.2, 1)
    ax[1].set_xlabel("wavelength (nm)"); ax[1].set_ylabel("$R^2$ at this wavelength")
    ax[1].set_title(f"(b) where the model is good  (global $R^2$ = {rep['r2_global']:.2f})",
                    fontsize=9)

    n = len(s); ks = np.arange(1, min(31, n + 1))
    acc = [100 * (rep["rank"] < k).mean() for k in ks]
    ax[2].plot(ks, acc, color="#0072B2", lw=1.8, label="prediction identifies its own run")
    ax[2].plot(ks, 100 * ks / n, color="0.6", ls="--", lw=1.2, label="chance")
    ax[2].set_xlabel("top-k"); ax[2].set_ylabel("% of runs retrieved")
    ax[2].legend(fontsize=7.5)
    ax[2].set_title(f"(c) retrieval: top-1 = {acc[0]:.0f} % vs {100/n:.1f} % by chance\n"
                    "low values = the spectrum→recipe map is degenerate at this noise level",
                    fontsize=9)

    o = np.argsort(s); pick = o[[int(f * (len(o) - 1)) for f in (0.05, 0.5, 0.95)]]
    for k, i in enumerate(pick):
        off = k * 1.15
        ax[3].plot(wl, Sn[i] + off, color="#0072B2", lw=1.4)
        ax[3].plot(wl, Sp[i] + off, color="#D55E00", lw=1.2, ls="--")
        lab = f"{s[i]:.0f}°" + (f"  {df.filename.iloc[i]}" if df is not None else "")
        ax[3].text(wl.min() + 8, off + 1.02, lab, fontsize=7.5)
    ax[3].set_xlim(wl.min(), wl.max()); ax[3].set_yticks([])
    ax[3].set_xlabel("wavelength (nm)")
    ax[3].set_title("(d) p5 / median / p95 of the model's own error\n"
                    "blue = measured, dashed orange = predicted", fontsize=9)
    fig.tight_layout(); return fig

# ── hyper-parameter tuning of the chosen model (grouped-CV, spectral-RMSE objective) ──
def _neg_spec_rmse(y_true, y_pred):
    return -np.sqrt(np.mean((y_true - y_pred) ** 2))     # PCA is orthonormal -> score-RMSE ∝ spectral RMSE
SPEC_SCORER = make_scorer(_neg_spec_rmse, greater_is_better=True)

def _tune_grids():
    g = {
        "RandomForest": (RandomForestRegressor(random_state=SEED, n_jobs=sp.N_JOBS),
            {"n_estimators": [200, 400, 600], "max_depth": [8, 12, 20, None],
             "min_samples_leaf": [1, 2, 4], "max_features": ["sqrt", 0.5, 1.0]}),
        "ExtraTrees": (ExtraTreesRegressor(random_state=SEED, n_jobs=sp.N_JOBS),
            {"n_estimators": [200, 400, 600], "max_depth": [8, 12, 20, None],
             "min_samples_leaf": [1, 2, 4]}),
        "GradientBoosting": (GradientBoostingRegressor(random_state=SEED),
            {"n_estimators": [200, 400], "max_depth": [2, 3, 5],
             "learning_rate": [0.02, 0.05, 0.1], "subsample": [0.7, 0.9, 1.0]}),
        "Ridge": (Ridge(), {"alpha": [1e-2, 0.1, 1, 10, 100, 1000]}),
    }
    if sp.XGBOOST_AVAILABLE:
        import xgboost as xgb
        g["XGBoost"] = (xgb.XGBRegressor(random_state=SEED, n_jobs=sp.N_JOBS),
            {"n_estimators": [200, 400], "max_depth": [3, 5, 7], "learning_rate": [0.02, 0.05, 0.1],
             "subsample": [0.7, 0.9], "colsample_bytree": [0.7, 1.0]})
    return g

SEED = pc.SEED

def tune_spectral(Q, scores, groups, feature, model, n_iter=25):
    """RandomizedSearchCV on the FULL spectrum (MultiOutput over the PCA scores), grouped-CV by
    session, spectral-RMSE objective. GP: its kernel is already fit by marginal likelihood, so only
    the noise floor `alpha` is searched; tree/linear models get a real grid.
    Returns (best_params, tuned_cv_rmse) — compare to the benchmark's default RMSE."""
    X = StandardScaler().fit_transform(sp.FEATURE_SETS[feature](Q))
    cv = GroupKFold(n_splits=min(CV_K, len(np.unique(groups))))
    if model == "GP-Matern52":
        from sklearn.gaussian_process import GaussianProcessRegressor
        base = GaussianProcessRegressor(kernel=sp.make_kernel(), normalize_y=True,
                                        n_restarts_optimizer=4, random_state=SEED)
        grid = {"estimator__alpha": [1e-10, 1e-8, 1e-6, 1e-4, 1e-2]}
    else:
        grids = _tune_grids()
        if model not in grids:
            print(f"no tuning grid for {model}"); return None, np.nan
        base, gg = grids[model]
        grid = {f"estimator__{k}": v for k, v in gg.items()}
    n_comb = int(np.prod([len(v) for v in grid.values()]))
    search = RandomizedSearchCV(MultiOutputRegressor(base), grid,
                                n_iter=min(n_iter, n_comb), cv=cv, scoring=SPEC_SCORER,
                                random_state=SEED, n_jobs=sp.N_JOBS)
    search.fit(X, scores, groups=groups)
    best = {k.replace("estimator__", ""): v for k, v in search.best_params_.items()}
    return best, -search.best_score_

def main():
    df, S, wl = load()
    Q = df[sp.FEATURES].values.astype(float)
    groups = df.session.values; lam = df.lambda_max.values
    Sn, amp = sp.normalise(S, NORMALIZE)
    mu, comps, evr = sp.fit_pca(Sn, N_PC); scores = (Sn - mu) @ comps.T
    print(f"{len(df)} particle spectra / {df.session.nunique()} sessions | "
          f"normalise={NORMALIZE} | PCA({N_PC})={100*evr.sum():.0f}% | xgboost={sp.XGBOOST_AVAILABLE}")

    print("\nBenchmark (feature x model, GroupKFold by session):")
    res = sp.benchmark(Sn, Q, scores, mu, comps, wl, groups, lam, k=CV_K)
    res.to_csv(pc.DATA_DIR / "spectral_benchmark.csv", index=False)
    argmin = res.loc[res.spec_rmse.idxmin()]
    print(f"\nBenchmark argmin: {argmin.feature}+{argmin.model} "
          f"(spec_rmse={argmin.spec_rmse:.4f}, lmax_rmse={argmin.lmax_rmse:.1f} nm)")
    if FORCE_MODEL is None:
        best = argmin
    else:
        f_, m_ = FORCE_MODEL
        sel = res[(res.feature == f_) & (res.model == m_)]
        if sel.empty:
            print(f"  WARNING: FORCE_MODEL {f_}+{m_} not in the benchmark -> using the argmin")
            best = argmin
        else:
            best = sel.iloc[0]
            print(f"DEPLOYED (pinned, see FORCE_MODEL): {best.feature}+{best.model} "
                  f"(spec_rmse={best.spec_rmse:.4f}, lmax_rmse={best.lmax_rmse:.1f} nm) "
                  f"— {best.spec_rmse - argmin.spec_rmse:+.4f} spec_rmse vs the argmin")
    savefig(sm.fig_benchmark(res), "fig33_benchmark")

    feat = sp.FEATURE_SETS[best.feature]; fac = sp.MODELS[best.model]
    Sp = sp.reconstruct(mu, comps, sp.cv_scores(Q, scores, feat, fac, groups, CV_K))
    sr2 = 1 - np.sum((Sn - Sp) ** 2) / np.sum((Sn - Sn.mean(0)) ** 2)
    lam_pred = sp.derived_lmax(wl, Sp)
    lr2 = 1 - np.sum((lam - lam_pred) ** 2) / np.sum((lam - lam.mean()) ** 2)
    print(f"[{best.feature}+{best.model}, grouped-CV] spectral R2={sr2:.2f} | "
          f"lambda_max R2={lr2:.2f} RMSE={np.sqrt(np.mean((lam-lam_pred)**2)):.1f} nm")

    colour = sp.make_colouriser(wl)
    idx = pick_recon(df, lam)                       # A,B,C,... examples (random / spread / custom)
    print(f"reconstruction examples ({RECON_MODE}): "
          f"{[f'{l}={n}' for l, n in zip('ABCDEFGH', df.filename.values[idx])]}")
    model = sp.SpectralModel(N_PC, feat, fac).fit(Q, scores)
    # SAME points labelled A,B,C,... on the reconstruction grid AND the hero colour map
    savefig(fig_recon_grid(wl, Sn, Sp, colour, lam, idx, names=df.filename.values, flows=Q),
            "fig33_reconstruction")
    savefig(sm.fig_parity(lam, lam_pred, groups, wl, Sn, Sp),     "fig33_parity")
    savefig(fig_hero_full(Q, model, mu, comps, colour, idx, HERO_PAIR), "fig33_colormap")
    savefig(fig_grid_full(Q, model, mu, comps, colour, MAP_PAIRS),           "fig33_colormap_projections")
    # chemically-constrained projections (ratio Ag/NaBH4 vs one swept reagent)
    cache = domain_scale(Q)
    mark = list(zip(idx, "ABCDEFGHIJKLMNO"))      # SAME A,B,C,D as the recon grid + hero map
    savefig(fig_mech_grid(Q, model, mu, comps, colour, wl, mark=mark),
            "fig33_mechanistic_projections")
    for key in MECH_ORDER:
        savefig(fig_mech_single(Q, model, mu, comps, colour, wl, key, cache, mark=mark),
                f"fig33_mech_{key}")
    savefig(sm.fig_learning(Sn, Q, wl, groups, lam, feat, fac),   "fig33_learning")

    # shape metrics — the headline the paper should lead with (see the block above)
    rep = shape_report(Sn, Sp, Q, wl)
    savefig(fig_shape_metrics(rep, Sn, Sp, wl, df), "fig33_shape_metrics")
    print(f"\nSHAPE: median SAM = {np.median(rep['sam']):.1f} deg   "
          f"(no-model baseline {np.median(rep['base']):.1f} deg, "
          f"random {np.median(rep['shuf']):.1f} deg)")
    print(f"  repeatability floor      : {rep['floor']:.1f} deg over {rep['n_pairs']} "
          f"near-identical recipe pairs; identical recipe across sessions = "
          f"{rep['floor_best']:.1f} deg")
    print(f"  runs below 15 deg        : {100*(rep['sam'] < 15).mean():.0f} %")
    print(f"  retrieval top-1 / top-10 : {100*(rep['rank'] < 1).mean():.0f} % / "
          f"{100*(rep['rank'] < 10).mean():.0f} %  (chance {100/len(Sn):.1f} % / "
          f"{1000/len(Sn):.1f} %)")

    # persist for the prediction tool (step 35)
    np.savez(pc.DATA_DIR / "spectral_pca_basis.npz", wavelengths=wl, mean=mu, components=comps,
             evr=evr, normalize=NORMALIZE)
    joblib.dump(dict(model=model, feature=best.feature, mean=mu, components=comps,
                     wavelengths=wl, normalize=NORMALIZE, n_pc=N_PC),
                pc.DATA_DIR / "spectral_model.pkl")
    print(f"saved data/spectral_model.pkl + spectral_pca_basis.npz | Done.")

if __name__ == "__main__":
    main()


### `ml.step36_convergence_model`
*Original analysis code.*

In [ ]:
%%module toofast_analysis.ml.step36_convergence_model
# [vendored] verbatim copy of 2FAST_ML/pipeline/step36_convergence_model.py (md5 980c7fbef9c8b71392226cf30cf8998c), generated by tools/vendor_ml.py — do not edit by hand.
"""
step36_convergence_model.py  —  pipeline step 8  (mirrors 36_convergence_model.ipynb)
====================================================================================
Predict the CONVERGENCE TIME of a synthesis from its flows — handling CENSORING.

Convergence time is **recomputed** per run from the spectral time series with a PLATEAU criterion
(see `convergence_time_v2`) that removes the instrument's 10-spectrum acquisition floor: the old
`eda_rg024.convergence_time` only starts checking after 10 spectra, so anything that stabilised
earlier was reported as converging at ~10 s (an artifact). We instead take the earliest time the
spectrum enters the final steady-state plateau. Many runs **never converge** within the observed
window (status 'never') = right-censored; naively regressing on the converged subset alone biases
the estimate. So we use a transparent **two-part model**:

  1. CONVERGENCE classifier  : P(the run converges within its window)   — target = status 'ok'
  2. TIME regressor          : log(conv_time_s) on the CONVERGED subset  — conditional on (1)

Both cross-validated GROUPED BY SESSION. Caveat: convergence time depends heavily on instrument
termination settings and session, so the composition signal is expected to be weaker than for the
spectrum — treat this as exploratory. (A proper survival/AFT model, e.g. lifelines, is the
principled upgrade if the two-part fit looks promising.)

The per-run convergence is cached to data/convergence.csv (H5 time-series I/O — run it yourself).
Kernel: Anaconda 3.11.
"""
from pathlib import Path
from itertools import combinations
import numpy as np, pandas as pd, h5py, joblib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              ExtraTreesClassifier, GradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_predict, GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from . import pipeline_common as pc
from . import eda_rg024 as eda

sp = pc.sp
SEED = pc.SEED
CV_K = 6
NOTEBOOK = "36_convergence_model.ipynb"
TS_ABS  = ["spectra/extinction"]
TS_AMX  = ["spectra/extinction_max"]
TS_TIME = ["spectra/timestamp"]

# ── convergence criterion (recomputed, NO 10-spectrum floor) ──────────────────
#   The instrument's own criterion (eda_rg024.convergence_time) only starts checking after
#   >= 10 spectra, so any run that stabilised earlier is reported as "converged at the 10th
#   spectrum" (~10 s) — an acquisition-floor artifact, not real kinetics. We instead detect the
#   PLATEAU directly: a run truly converges only if its last TAIL spectra form a steady state
#   (mutually within tolerance); the convergence time is then the EARLIEST time from which the
#   spectrum stays inside that plateau (matches the final spectrum), with no minimum-count floor.
AMP_THR  = 0.02     # |Δabs_max| / abs_max tolerance
CORR_THR = 0.997    # Pearson(spectrum_i, final) tolerance
TAIL     = 3        # last TAIL spectra must be a real steady state for status 'ok'

def convergence_time_v2(A, amx, t, amp=AMP_THR, corr=CORR_THR, tail=TAIL):
    """Earliest onset of the steady-state plateau (seconds), free of the 10-spectrum floor.
    Returns (time, index, status) with status in {'ok','never','too_short'}."""
    A = np.asarray(A, float); amx = np.asarray(amx, float); t = np.asarray(t, float)
    n = len(t)
    if n < 2:
        return np.nan, n - 1, "too_short"
    final = A[-1]; af = amx[-1]
    def match(i):
        d = abs(amx[i] - af) / max(abs(af), 1e-9)
        r = np.corrcoef(A[i], final)[0, 1]
        return (d < amp) and (r > corr)
    k = min(tail, n)
    if not all(match(i) for i in range(n - k, n)):     # tail is still drifting -> censored
        return np.nan, n - 1, "never"
    for i in range(n):                                 # earliest i with a stable tail from i..n
        if all(match(j) for j in range(i, n)):
            return float(t[i]), i, "ok"
    return float(t[-1]), n - 1, "ok"

def savefig(fig, name):
    fig.text(0.995, 0.003, f"source: {NOTEBOOK}", ha="right", va="bottom", fontsize=6, color="0.55")
    for ext in ("png", "pdf"):
        fig.savefig(pc.FIG_DIR / f"{name}.{ext}", dpi=200, bbox_inches="tight")
    print(f"  saved {name}.png/.pdf")

# ── per-run convergence from the H5 time series ───────────────────────────────
def conv_for_file(fp):
    try:
        with h5py.File(fp, "r") as f:
            ak = next((k for k in TS_ABS if k in f), None)
            mk = next((k for k in TS_AMX if k in f), None)
            tk = next((k for k in TS_TIME if k in f), None)
            if ak is None or mk is None:
                return np.nan, "no_ts", np.nan
            A = f[ak][()]; amx = f[mk][()]
            if tk is not None:
                ts = [t.decode() if isinstance(t, bytes) else t for t in f[tk][()]]
                try:
                    tsec = (pd.to_datetime(ts) - pd.to_datetime(ts[0])).total_seconds().values
                except Exception:
                    tsec = np.arange(len(amx), dtype=float)
            else:
                tsec = np.arange(len(amx), dtype=float)
    except Exception as e:
        return np.nan, f"err:{e}", np.nan
    t, idx, status = convergence_time_v2(A, amx, tsec)   # plateau-based, no 10-spectrum floor
    return t, status, float(tsec[-1])          # conv_time, status, total window length

def load_conv(force=False):
    """Clean PARTICLE points (pc.load("clean") + is_particle: 156 runs, NO flow cut) + their cached
    convergence time / status / window length.

    DELIBERATELY NOT the step-31b `hardcut` set used by the spectral/growth models (151 runs). Tried
    on 2026-10-08: LOSO log-R2 0.75 -> 0.72, R2(s) 0.61 -> 0.52 (the flow cut removes the low-Ag /
    low-NaBH4 recipes, i.e. most of the slow tail), median errors unchanged. The user preferred the
    clean set; this is a documented choice, not an oversight. Overlap with hardcut = 143 runs.

    The cache is keyed by filename and topped up for any run it lacks: it used to be all-or-nothing,
    went stale when the run set changed, and silently dropped a new run (S14-M14_sample_17)."""
    cache = pc.DATA_DIR / "convergence.csv"
    df, S, grid = pc.load("clean"); df = df[df.is_particle].reset_index(drop=True)
    c = pd.read_csv(cache) if cache.exists() and not force else pd.DataFrame(
        columns=["conv_time_s", "conv_status", "window_s", "filename"])
    miss = df[~df.filename.isin(c.filename)]
    if len(miss):
        print(f"computing convergence for {len(miss)} runs (H5 time-series I/O) ...")
        rec = pd.DataFrame([conv_for_file(fp) for fp in miss.filepath],
                           columns=["conv_time_s", "conv_status", "window_s"])
        rec["filename"] = miss.filename.values
        c = pd.concat([c, rec], ignore_index=True)
        c.to_csv(cache, index=False)
    df = df.merge(c, on="filename", how="left")
    df["converged"] = (df.conv_status == "ok")
    return df

# ── model zoos (benchmarked like the spectral model, not hard-coded) ──────────
CLASSIFIERS = {
    "LogReg":           lambda: make_pipeline(StandardScaler(),
                            LogisticRegression(max_iter=2000, class_weight="balanced")),
    "RandomForest":     lambda: RandomForestClassifier(400, max_depth=10,
                            class_weight="balanced", random_state=SEED, n_jobs=sp.N_JOBS),
    "ExtraTrees":       lambda: ExtraTreesClassifier(400, max_depth=10,
                            class_weight="balanced", random_state=SEED, n_jobs=sp.N_JOBS),
    "GradientBoosting": lambda: GradientBoostingClassifier(random_state=SEED),
}
REGRESSORS = sp.MODELS          # the shared regressor zoo (Ridge/RF/ET/GB/GP/MLP/XGB)

def _cv(groups):
    return GroupKFold(n_splits=min(CV_K, len(np.unique(groups))))

def cv_class(X, y, groups, fac):
    return cross_val_predict(fac(), X, y, cv=_cv(groups), groups=groups,
                             method="predict_proba")[:, 1]

def cv_reg(X, y, groups, fac):
    return cross_val_predict(fac(), X, y, cv=_cv(groups), groups=groups)

def benchmark_conv(X, y, groups):
    """Converge-vs-not classifier zoo, grouped-CV by session."""
    rows = []
    for name, fac in CLASSIFIERS.items():
        try:
            p = cv_class(X, y, groups, fac)
        except Exception as e:
            print(f"  conv {name:16s} FAILED: {e}"); continue
        rows.append(dict(model=name, roc_auc=roc_auc_score(y, p),
                         ap=average_precision_score(y, p)))
        print(f"  conv {name:16s} ROC-AUC={rows[-1]['roc_auc']:.3f} AP={rows[-1]['ap']:.3f}")
    return pd.DataFrame(rows)

def benchmark_time(X, y, groups):
    """log-time regressor zoo (converged subset), grouped-CV by session."""
    rows = []
    for name, fac in REGRESSORS.items():
        try:
            pred = cv_reg(X, y, groups, fac)
        except Exception as e:
            print(f"  time {name:16s} FAILED: {e}"); continue
        r2 = 1 - np.sum((y - pred) ** 2) / np.sum((y - y.mean()) ** 2)
        rmse_s = np.sqrt(np.mean((np.expm1(y) - np.expm1(pred)) ** 2))
        rows.append(dict(model=name, r2=r2, rmse_s=rmse_s))
        print(f"  time {name:16s} log-R2={r2:.3f}  RMSE={rmse_s:5.1f} s")
    return pd.DataFrame(rows)

TIME_GAMMA = 0.4     # PowerNorm exponent (<1): expands colour contrast at LOW times (0-20 s)

def time_norm(vmax, gamma=TIME_GAMMA):
    """Non-linear colour norm: fast colour change for short times (0-20 s), slow beyond."""
    return mcolors.PowerNorm(gamma=gamma, vmin=0.0, vmax=vmax)

def time_slice(ax, Q, rf, qa, qb, grid_n=60, vmax=None, gamma=TIME_GAMMA):
    ia, ib = sp.FEATURES.index(qa), sp.FEATURES.index(qb)
    others = [k for k in range(5) if k not in (ia, ib)]
    med = np.median(Q, 0); prop = med[others] / med[others].sum()
    ga = np.linspace(0, Q[:, ia].max() * 1.02, grid_n); gb = np.linspace(0, Q[:, ib].max() * 1.02, grid_n)
    Z = np.full((grid_n, grid_n), np.nan); pts, ij = [], []
    for iy, vb in enumerate(gb):
        for ix, va in enumerate(ga):
            rem = sp.Q_TOTAL - va - vb
            if rem <= 0: continue
            x = np.empty(5); x[ia] = va; x[ib] = vb; x[others] = rem * prop
            pts.append(x); ij.append((iy, ix))
    pred = np.expm1(rf.predict(np.array(pts)))            # back to seconds
    for (iy, ix), v in zip(ij, pred): Z[iy, ix] = v
    if vmax is None:
        vmax = np.nanpercentile(Z, 98)
    im = ax.imshow(Z, origin="lower", aspect="auto", cmap="viridis",
                   norm=time_norm(vmax, gamma), extent=[ga[0], ga[-1], gb[0], gb[-1]])
    ax.plot([0, sp.Q_TOTAL], [sp.Q_TOTAL, 0], "w--", lw=1, alpha=.7)
    ax.scatter(Q[:, ia], Q[:, ib], s=8, facecolor="none", edgecolor="w", lw=.3, alpha=.4)
    ax.set_xlabel(f"$Q_{{{qa}}}$ (µL/s)"); ax.set_ylabel(f"$Q_{{{qb}}}$ (µL/s)")
    return im

ALL_PAIRS = list(combinations(["Q1", "Q2", "Q3", "Q4", "Q5"], 2))   # all 10 flow-pair axes

def fig_time_maps(Q, rf_time, pairs=ALL_PAIRS, ncol=5, vmax=None):
    """Predicted convergence-time projections on EVERY pair of reagent-flow axes (others filled at
    their median proportion; blank where total flow > 14). Shared non-linear colour scale so the
    0-20 s zone gets most of the contrast."""
    if vmax is None:
        vmax = float(np.nanpercentile(np.expm1(rf_time.predict(Q)), 95))
    n = len(pairs); nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.9 * ncol, 2.7 * nrow), squeeze=False)
    axes = axes.ravel(); im = None
    for a, pr in zip(axes, pairs):
        im = time_slice(a, Q, rf_time, *pr, vmax=vmax)
        a.tick_params(labelsize=7); a.xaxis.label.set_size(8); a.yaxis.label.set_size(8)
    for a in axes[n:]: a.axis("off")
    if im is not None:
        ticks = [t for t in (2, 5, 10, 20, 40, 80) if t <= vmax]
        cb = fig.colorbar(im, ax=axes.tolist(), label="predicted convergence time (s)",
                          fraction=0.02, ticks=ticks)
        cb.ax.set_yticklabels([str(t) for t in ticks])
    return fig

# ── figures ───────────────────────────────────────────────────────────────────
def fig_overview(df):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
    ok = df[df.converged]; counts = df.conv_status.value_counts().to_dict()
    ax[0].hist(ok.conv_time_s, bins=25, color="#0072B2")
    ax[0].set_xlabel("convergence time (s)"); ax[0].set_ylabel("count")
    ax[0].set_title(f"(a) converged runs (n={len(ok)})")
    ax[0].text(0.95, 0.95, f"status: {counts}", transform=ax[0].transAxes, ha="right", va="top",
               fontsize=8, color="0.3")
    sc = ax[1].scatter(df.lambda_max, df.conv_time_s, c=df.Q5, cmap="viridis", s=26,
                       edgecolor="white", lw=.3)
    ax[1].set_xlabel(r"$\lambda_{max}$ (nm)"); ax[1].set_ylabel("convergence time (s)")
    ax[1].set_title("(b) time vs λmax (colour = NaBH4)"); fig.colorbar(sc, ax=ax[1], label="Q5 NaBH4")
    fig.tight_layout(); return fig

def fig_model(df, p_conv, y_conv, logt_pred, ok, rf_time):
    Q = df[sp.FEATURES].values.astype(float)
    fig, ax = plt.subplots(2, 2, figsize=(12, 9)); ax = ax.ravel()
    # (a) convergence classifier
    roc = roc_auc_score(y_conv, p_conv); apv = average_precision_score(y_conv, p_conv)
    ax[0].hist([p_conv[y_conv], p_conv[~y_conv]], bins=15, stacked=True,
               color=["#009E73", "#D55E00"], label=["converged", "never"])
    ax[0].set_xlabel("P(converge)"); ax[0].set_ylabel("count")
    ax[0].set_title(f"(a) convergence classifier  ROC-AUC={roc:.2f}  AP={apv:.2f}"); ax[0].legend(fontsize=8)
    # (b) time regressor parity (log-time -> seconds), converged only
    t_true = np.expm1(np.log1p(ok.conv_time_s.values)); t_pred = np.expm1(logt_pred)
    r2 = 1 - np.sum((np.log1p(t_true) - logt_pred) ** 2) / np.sum((np.log1p(t_true) - np.log1p(t_true).mean()) ** 2)
    ax[1].scatter(t_true, t_pred, s=26, color="#0072B2", edgecolor="white", lw=.3)
    lims = [0, max(t_true.max(), t_pred.max()) * 1.05]; ax[1].plot(lims, lims, "k--", lw=1)
    ax[1].set_xlim(lims); ax[1].set_ylim(lims)
    ax[1].set_xlabel("measured time (s)"); ax[1].set_ylabel("predicted time (s, leave-session-out)")
    ax[1].set_title(f"(b) time regressor (converged)  log-time R²={r2:.2f}")
    # (c) flow importance on log-time — own RF, robust to whatever model is deployed
    yok = np.log1p(ok.conv_time_s.values)
    rf_imp = RandomForestRegressor(400, max_depth=12, min_samples_leaf=2, random_state=SEED,
                                   n_jobs=sp.N_JOBS).fit(ok[sp.FEATURES].values.astype(float), yok)
    lab = ["AgNO3", "PVP", "AcOH", "AscAcid", "NaBH4"]; imp = rf_imp.feature_importances_
    o = np.argsort(imp); ax[2].barh([lab[i] for i in o], imp[o], color="#009E73")
    ax[2].set_xlabel("RF importance"); ax[2].set_title("(c) flows driving convergence time")
    # (d) predicted-time map over Q1 x Q5 (non-linear scale: contrast concentrated at short times)
    vmax = float(np.nanpercentile(ok.conv_time_s.values, 95))
    im = time_slice(ax[3], Q[df.converged.values], rf_time, "Q1", "Q5", vmax=vmax)
    ax[3].set_title("(d) predicted time (s) — Q1×Q5")
    ticks = [t for t in (2, 5, 10, 20, 40, 80) if t <= vmax]
    cb = fig.colorbar(im, ax=ax[3], label="time (s)", ticks=ticks)
    cb.ax.set_yticklabels([str(t) for t in ticks])
    fig.tight_layout(); return fig

def main():
    df = load_conv()
    Q = df[sp.FEATURES].values.astype(float); groups = df.session.values
    print(f"{len(df)} particle runs | status: {df.conv_status.value_counts().to_dict()}")
    print(f"converged fraction = {df.converged.mean():.2f}")

    # 1) convergence classifier zoo (all particle runs)
    print("\nConvergence classifier benchmark (GroupKFold by session):")
    resc = benchmark_conv(Q, df.converged.values.astype(int), groups)
    resc.to_csv(pc.DATA_DIR / "convergence_classifier_benchmark.csv", index=False)
    best_clf = resc.loc[resc.roc_auc.idxmax(), "model"]

    # 2) time regressor zoo (converged subset, log seconds)
    ok = df[df.converged].reset_index(drop=True)
    Xok = ok[sp.FEATURES].values.astype(float); gok = ok.session.values
    yok = np.log1p(ok.conv_time_s.values)
    print("\nTime regressor benchmark (converged, GroupKFold by session):")
    rest = benchmark_time(Xok, yok, gok)
    rest.to_csv(pc.DATA_DIR / "convergence_time_benchmark.csv", index=False)
    best_reg = rest.loc[rest.r2.idxmax(), "model"]
    print(f"\nDeployed: classifier={best_clf} | time-regressor={best_reg}")

    p_conv    = cv_class(Q, df.converged.values.astype(int), groups, CLASSIFIERS[best_clf])
    logt_pred = cv_reg(Xok, yok, gok, REGRESSORS[best_reg])
    reg      = REGRESSORS[best_reg]().fit(Xok, yok)
    clf_conv = CLASSIFIERS[best_clf]().fit(Q, df.converged.values.astype(int))
    joblib.dump(dict(reg=reg, reg_name=best_reg, clf=clf_conv, clf_name=best_clf,
                     features=sp.FEATURES, source="step36-benchmark"),
                pc.DATA_DIR / "convergence_model.pkl")

    savefig(fig_overview(df), "fig36_overview")
    savefig(fig_model(df, p_conv, df.converged.values, logt_pred, ok, reg), "fig36_model")
    savefig(fig_time_maps(Xok, reg), "fig36_time_maps")   # projections on all 10 flow-pair axes
    print("saved data/convergence_model.pkl | Done.")

if __name__ == "__main__":
    main()


### `ml.step39_growth_trajectory`
*Original analysis code.*

In [ ]:
%%module toofast_analysis.ml.step39_growth_trajectory
# [vendored] verbatim copy of 2FAST_ML/pipeline/step39_growth_trajectory.py (md5 c2e31d0b470641bbd4f5f6a6492d2bcd), generated by tools/vendor_ml.py — do not edit by hand.
"""
step39_growth_trajectory.py  —  pipeline step 11 (bonus)  (mirrors 39_growth_trajectory.ipynb)
==============================================================================================
Predict the WHOLE growth movie, not just the converged spectrum. Every H5 stores the reaction as
a time series of spectra, so instead of keeping only the last one we model the full transient.

Idea (validated in the probe): intermediate spectra live on the SAME low-dim PCA manifold as the
finals, and each run is a SMOOTH trajectory of scores that mostly slides along PC1 (size/redshift).
So rather than one model per second (fragile, ragged time grids), we:

  1. extract every time-spectrum, resample to the wl grid, drop pre-nucleation noise (peak < ABS_FLOOR);
  2. parametrise each by a NORMALISED progress  τ = t / t_convergence  (decouples "how far" from
     "how fast", which is session/kinetics-dependent);  t_conv from step-36 convergence_time_v2;
  3. project onto a PCA(5) basis fit on ALL valid growth spectra -> score trajectories s(τ);
  4. train ONE model  (flows, τ) -> scores  (leave-one-session-out CV), reusing the PCA+regressor
     machinery, and reconstruct the movie by sweeping τ (map τ->seconds via the convergence model).

Extraction is cached to data/growth.npz + growth.csv (H5 time-series I/O — run it yourself).
Kernel: Anaconda 3.11.
"""
from pathlib import Path
import numpy as np, pandas as pd, h5py
import matplotlib.pyplot as plt
from . import pipeline_common as pc
from . import step36_convergence_model as s36

sp = pc.sp
NORMALIZE = "max"
N_PC = pc.N_PC
AMP_FLOOR = pc.ABS_FLOOR        # peak-height floor: start the trajectory when a real peak exists
CV_K = 6
SEC_PER_FRAME = 1.05            # acquisition cadence: corr(n_frames, duration)=1.00, ~1.05 s/frame
                               #   across all runs -> index n and elapsed seconds are interchangeable.
                               #   Used to map a NOVEL recipe's predicted t_conv (s) to a convergence
                               #   INDEX, so the interactive movie draws to the right length.
# DEPLOYED MODEL = ExtraTrees (default hyper-parameters, `sp.MODELS["ExtraTrees"]`): it reconstructs
# the growth movie with the sharp spectral detail the user preferred, is a NATIVE multi-output model
# (one cheap joint fit per index, exploits amplitude/shape correlation) AND is ~20x faster to fit than
# GP-Matern52 (full per-index fit ~11 s vs ~4 min). GP wins the AGGREGATE spectral RMSE (dominated by
# PC1 amplitude ~72% var) but that metric over-weights amplitude and under-weights the visual shape
# detail — which is why the GP-as-"winner" deployment looked degraded for ExtraTrees. We deploy
# ExtraTrees and run ONE honest full-trajectory benchmark at the end (reusing the already-fit models,
# no retraining) to quantify where every model actually stands.
DEFAULT_MODEL = "ExtraTrees"
CANDIDATE_MODELS = ("Ridge", "RandomForest", "ExtraTrees", "GradientBoosting", "GP-Matern52")
NOTEBOOK = "39_growth_trajectory.ipynb"

def savefig(fig, name):
    fig.text(0.995, 0.003, f"source: {NOTEBOOK}", ha="right", va="bottom", fontsize=6, color="0.55")
    for ext in ("png", "pdf"):
        fig.savefig(pc.FIG_DIR / f"{name}.{ext}", dpi=200, bbox_inches="tight")
    print(f"  saved {name}.png/.pdf")

# ── read a run's full time series ───────────────────────────
def read_ts(fp):
    try:
        with h5py.File(fp, "r") as f:
            ak = next((k for k in s36.TS_ABS if k in f), None)
            wk = next((w for w in pc.WL_CANDS if w in f), None)
            mk = next((k for k in s36.TS_AMX if k in f), None)
            tk = next((k for k in s36.TS_TIME if k in f), None)
            if ak is None or wk is None:
                return None
            A = f[ak][()]; w = f[wk][()]
            amx = f[mk][()] if mk else A.max(1)
            if tk is not None:
                ts = [t.decode() if isinstance(t, bytes) else t for t in f[tk][()]]
                try:
                    tsec = (pd.to_datetime(ts) - pd.to_datetime(ts[0])).total_seconds().values
                except Exception:
                    tsec = np.arange(len(amx), dtype=float)
            else:
                tsec = np.arange(len(amx), dtype=float)
    except Exception:
        return None
    return np.asarray(A, float), np.asarray(amx, float), np.asarray(w, float), np.asarray(tsec, float)

# ── build the trajectory table (cached) ───────────────────────────────────────
def extract(force=False):
    """Return (meta, S_traj, grid): one row per (run, time) point above the peak floor, its flows,
    elapsed time t and normalised progress tau, plus the resampled max-un-normalised spectrum."""
    npz = pc.DATA_DIR / "growth.npz"; csv = pc.DATA_DIR / "growth.csv"
    if npz.exists() and csv.exists() and not force:
        meta = pd.read_csv(csv); z = np.load(npz)
        return meta, z["S"], z["wavelengths"]
    dfp, S, grid = pc.load_training()   # step-31b `hardcut` set (pipeline_common.TRAIN_TAG)
    print(f"extracting growth trajectories for {len(dfp)} runs (H5 time-series I/O) ...")
    rows, specs = [], []
    for r in dfp.itertuples():
        out = read_ts(r.filepath)
        if out is None:
            continue
        A, amx, w, tsec = out
        tconv, _, status = s36.convergence_time_v2(A, amx, tsec)
        denom = tconv if (status == "ok" and tconv and tconv > 1e-6) else max(tsec[-1], 1e-6)
        for j in range(len(A)):
            si = np.interp(grid, w, A[j])
            if not np.isfinite(si).all() or si.max() < AMP_FLOOR:
                continue                                    # skip pre-nucleation / noise frames
            specs.append(si)
            rows.append(dict(run=r.filename, session=r.session,
                             Q1=r.Q1, Q2=r.Q2, Q3=r.Q3, Q4=r.Q4, Q5=r.Q5,
                             t=float(tsec[j]), tau=float(tsec[j] / denom),
                             conv_status=status, is_final=(j == len(A) - 1)))
    meta = pd.DataFrame(rows); S_traj = np.vstack(specs)
    meta.to_csv(csv, index=False)
    np.savez(npz, S=S_traj, wavelengths=grid)
    print(f"  {len(meta)} frames / {meta.run.nunique()} runs -> cached growth.npz/.csv")
    return meta, S_traj, grid

def fit_basis(S_traj):
    Sn, _ = sp.normalise(S_traj, NORMALIZE)
    mu, comps, evr = sp.fit_pca(Sn, N_PC)
    return Sn, mu, comps, evr, (Sn - mu) @ comps.T

# ── validation figure (run this BEFORE training) ──────────────────────────────
def fig_trajectories(meta, scores, n_runs=15, seed=0):
    runs = pd.unique(meta.run)
    sel = np.random.default_rng(seed).choice(runs, min(n_runs, len(runs)), replace=False)
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    for run in sel:
        m = (meta.run == run).values; o = np.argsort(meta.t.values[m])
        p = scores[m][o]
        ax[0].plot(p[:, 0], p[:, 1], "-", color="0.75", lw=.8, alpha=.7, zorder=1)
    sc = ax[0].scatter(scores[:, 0], scores[:, 1], c=meta.tau.clip(0, 1.2).values,
                       cmap="viridis", s=8, zorder=2)
    fin = meta.is_final.values
    ax[0].scatter(scores[fin, 0], scores[fin, 1], marker="*", s=70,
                  facecolor="none", edgecolor="k", lw=.6, zorder=3, label="final")
    fig.colorbar(sc, ax=ax[0], label=r"progress $\tau = t / t_{conv}$")
    ax[0].set_xlabel("PC1 (size/redshift)"); ax[0].set_ylabel("PC2")
    ax[0].set_title("(a) growth trajectories in PCA space (★ = converged)"); ax[0].legend(fontsize=8)
    for run in sel:
        m = (meta.run == run).values; o = np.argsort(meta.tau.values[m])
        ax[1].plot(meta.tau.values[m][o], scores[m, 0][o], "-", lw=1, alpha=.7)
    ax[1].set_xlabel(r"progress $\tau$"); ax[1].set_ylabel("PC1 (size/redshift)")
    ax[1].set_xlim(0, 1.2); ax[1].set_title("(b) PC1 vs progress — the growth curves to learn")
    fig.tight_layout(); return fig

# ── model: (flows, tau) -> scores, leave-one-session-out ──────────────────────
def features_tau(Q, tau, feat=sp.f_clr):
    """Composition features augmented with the progress variable tau."""
    return np.column_stack([feat(np.asarray(Q, float)), np.asarray(tau, float)])

def cv_trajectory(meta, Sn, scores, mu, comps, grid, model=DEFAULT_MODEL, k=CV_K):
    """Leave-one-session-out spectral RMSE over the WHOLE trajectory (every frame)."""
    from sklearn.model_selection import cross_val_predict, GroupKFold
    Q = meta[list(sp.FEATURES)].values.astype(float); tau = meta.tau.values
    X = features_tau(Q, tau); groups = meta.session.values
    cv = GroupKFold(n_splits=min(k, meta.session.nunique()))
    Sp = np.column_stack([cross_val_predict(sp.MODELS[model](), X, scores[:, kk],
                          cv=cv, groups=groups) for kk in range(scores.shape[1])])
    Spec = mu + Sp @ comps
    rmse = np.sqrt(np.mean((Sn - Spec) ** 2))
    return rmse, Spec

# ── fit the full model + reconstruct a predicted growth movie ─────────────────
def fit_full(meta, scores, model=DEFAULT_MODEL):
    """One regressor per PCA score on (flows, tau), fit on all frames — for movie generation."""
    X = features_tau(meta[list(sp.FEATURES)].values.astype(float), meta.tau.values)
    return [sp.MODELS[model]().fit(X, scores[:, k]) for k in range(scores.shape[1])]

def predict_movie(models, mu, comps, Q, taus):
    X = features_tau(np.tile(np.asarray(Q, float), (len(taus), 1)), taus)
    sc = np.column_stack([m.predict(X) for m in models])
    return mu + sc @ comps                                  # (len(taus), n_wl)

def fig_movie(models, mu, comps, grid, Q, taus=None, real=None):
    """Predicted growth movie for a recipe: spectra swept over progress tau, coloured by tau.
    Pass real=(taus_r, Sn_r) to overlay a measured run's frames (dotted)."""
    taus = np.linspace(0.1, 1.0, 10) if taus is None else np.asarray(taus)
    M = predict_movie(models, mu, comps, Q, taus)
    fig, ax = plt.subplots(figsize=(7.4, 4.7))
    for t, spec in zip(taus, M):
        ax.plot(grid, spec, color=plt.cm.viridis(min(t, 1.0)), lw=1.6)
    if real is not None:
        tr, Sr = real
        for t, spec in zip(tr, Sr):
            ax.plot(grid, spec, ":", color=plt.cm.viridis(min(t, 1.0)), lw=1.0, alpha=.8)
    ax.set_xlim(grid.min(), grid.max()); ax.axhline(0, color="0.8", lw=.6)
    ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("Extinction (norm.)")
    q = "  ".join(f"{c}={v:.1f}" for c, v in zip(sp.FEATURES, np.ravel(Q)))
    ax.set_title(f"predicted growth movie   ({q})", fontsize=9)
    sm = plt.cm.ScalarMappable(cmap="viridis"); sm.set_array([0, 1])
    fig.colorbar(sm, ax=ax, label=r"progress $\tau$")
    fig.tight_layout(); return fig

# ══════════════════════════════════════════════════════════════════════════════
# PER-INDEX growth model (the user's design): one model per acquisition index n.
#   • index by SPECTRUM NUMBER (not seconds) -> defined for every run, no ragged grid;
#   • runs shorter than n are FORWARD-FILLED with their last (stable) spectrum;
#   • NON-normalised spectra -> the model shows the REAL amplitude growth;
#   • at prediction time, iterate index 1,2,… until spectrum_n ≈ spectrum_{n-1} (converged).
# ══════════════════════════════════════════════════════════════════════════════
N_PC_RAW = 8          # non-normalised PCA keeps an extra amplitude dimension

def build_sequences(force=False):
    """Per run: its ordered NON-normalised resampled frames (all of them). Returns
    (runs_df with offset/n_frames, F = concatenated frames, grid). Cached. The per-frame elapsed
    time (seconds from the run start) is ALSO cached (npz['times'], parallel to F) so the growth
    figures can put the acquisition-index axis onto a REAL seconds axis — read it with
    `frame_times()`."""
    npz = pc.DATA_DIR / "growth_seq.npz"; csv = pc.DATA_DIR / "growth_seq.csv"
    if npz.exists() and csv.exists() and not force:
        runs = pd.read_csv(csv); z = np.load(npz)
        # backward-compat: older caches have no 'times' -> force a rebuild so seconds are available
        if "times" in z.files:
            return runs, z["frames"], z["wavelengths"]
        print("  growth_seq cache has no per-frame times -> rebuilding once to add them ...")
    dfp, S, grid = pc.load_training()   # step-31b `hardcut` set (pipeline_common.TRAIN_TAG)
    print(f"building growth sequences for {len(dfp)} runs (non-normalised, all frames) ...")
    frames, times, rows, off = [], [], [], 0
    for r in dfp.itertuples():
        out = read_ts(r.filepath)
        if out is None:
            continue
        A, amx, w, tsec = out
        G = np.vstack([np.interp(grid, w, a) for a in A])
        ok = np.isfinite(G).all(1)
        G = G[ok]; ts = np.asarray(tsec, float)[:len(ok)][ok]
        if len(G) == 0:
            continue
        frames.append(G); times.append(ts)
        rows.append(dict(run=r.filename, session=r.session, Q1=r.Q1, Q2=r.Q2, Q3=r.Q3,
                         Q4=r.Q4, Q5=r.Q5, n_frames=len(G), offset=off)); off += len(G)
    F = np.vstack(frames); T = np.concatenate(times); runs = pd.DataFrame(rows)
    runs.to_csv(csv, index=False); np.savez(npz, frames=F, wavelengths=grid, times=T)
    print(f"  {len(F)} frames / {len(runs)} runs, max index = {int(runs.n_frames.max())} -> cached")
    return runs, F, grid

def frame_times():
    """Per-frame elapsed time (s from run start), parallel to `F` from build_sequences (npz['times']).
    Slice with a run's offset/n_frames to get that run's real acquisition timestamps."""
    z = np.load(pc.DATA_DIR / "growth_seq.npz")
    if "times" not in z.files:
        build_sequences(force=True); z = np.load(pc.DATA_DIR / "growth_seq.npz")
    return z["times"]

def run_frames_times(runs, F, T, i):
    """(frames, times_seconds) for run row `i` of `runs`."""
    r = runs.loc[i]; a = int(r.offset); b = a + int(r.n_frames)
    return F[a:b], (T[a:b] if T is not None else np.arange(int(r.n_frames), dtype=float))

def raw_basis(F, n_pc=N_PC_RAW):
    """PCA on the NON-normalised frames (PC1 ~ overall amplitude / growth)."""
    mu = F.mean(0); U, sv, Vt = np.linalg.svd(F - mu, full_matrices=False)
    return mu, Vt[:n_pc], (sv**2 / (sv**2).sum())[:n_pc]

def frames_at_index(n, runs, F):
    """(n_runs, n_wl): each run's frame at index n, FORWARD-FILLED with its last frame."""
    idx = runs.offset.values + np.minimum(n, runs.n_frames.values - 1)
    return F[idx]

def suggest_max_index(runs, min_growing=5):
    """Smallest index n beyond which fewer than `min_growing` runs are still contributing genuine
    (non-forward-filled) information — a practical cap so GP fits aren't wasted on indices where
    almost every row is just a repeated copy of an already-known endpoint."""
    mx = int(runs.n_frames.max())
    counts = np.array([(runs.n_frames.values > n).sum() for n in range(mx)])
    over = np.where(counts < min_growing)[0]
    return int(over[0]) if len(over) else mx

# Models where sklearn's native multi-output fit(X, Y_2d) is SAFE and CHEAP (one joint model per
# index instead of one per component): Ridge/RandomForest/ExtraTrees/MLP genuinely support it and
# a joint fit can exploit cross-component correlations (amplitude and shape evolve together during
# growth) — this is the behaviour the user found visually better for ExtraTrees, and it's ~8x
# cheaper to train (1 model/index vs 8). GaussianProcessRegressor's native multi-output support
# instead SHARES one kernel/length-scale across every column — wrong given PC1 (~72% var, amplitude)
# vs PC2-8 (minor shape, different scales) — so GP (and GradientBoosting, which sklearn's API does
# not let fit 2D y at all) are FORCED through the per-component `sp.SpectralModel` path regardless.
NATIVE_MULTIOUTPUT = {"Ridge", "RandomForest", "ExtraTrees", "MLP"}

class _NativeMultiOutputModel:
    """Adapter: ONE native multi-output estimator wrapped to expose the same `.predict(Qraw)`
    interface as `sp.SpectralModel` (applies `feat` internally), so downstream code (predict_growth,
    trajectory_rmse, ...) doesn't care which fitting strategy was used."""
    def __init__(self, feat_fn, estimator):
        self.feat = feat_fn; self.est = estimator
    def fit(self, Qraw, Y):
        self.est.fit(self.feat(Qraw), Y); return self
    def predict(self, Qraw):
        return self.est.predict(self.feat(Qraw))

def fit_per_index(runs, F, mu, comps, feat=sp.f_clr, model=DEFAULT_MODEL, max_index=None,
                  factory=None, per_component=None):
    """One model per acquisition index. For `model` in `NATIVE_MULTIOUTPUT` (Ridge/RandomForest/
    ExtraTrees/MLP): ONE joint native multi-output fit per index (cheap, can exploit cross-component
    correlation). Otherwise (notably GP-Matern52, GradientBoosting): `sp.SpectralModel` — an
    INDEPENDENT single-output regressor per PCA component (steps 32-38's convention), required
    because GP's native multi-output sharing a single kernel across components is wrong here.
    Override the auto-choice with `per_component=True/False`. Pass `factory` (e.g. from
    `make_tuned_factory`) to override the plain `sp.MODELS[model]` default — always pass the matching
    `model=` name alongside it so the native-vs-per-component choice stays correct."""
    Qraw = runs[list(sp.FEATURES)].values.astype(float)
    mx = int(runs.n_frames.max()); mx = mx if max_index is None else min(mx, max_index)
    fac = factory if factory is not None else sp.MODELS[model]
    if per_component is None:
        per_component = model not in NATIVE_MULTIOUTPUT
    n_comp = comps.shape[0]
    models = []
    for n in range(mx):
        Y = (frames_at_index(n, runs, F) - mu) @ comps.T
        if per_component:
            models.append(sp.SpectralModel(n_comp, feat, fac).fit(Qraw, Y))
        else:
            models.append(_NativeMultiOutputModel(feat, fac()).fit(Qraw, Y))
    return models

def predict_growth(models, mu, comps, Q, feat=sp.f_clr, tol_corr=0.9995, min_iter=3):
    """Iterate the per-index models (SpectralModel or _NativeMultiOutputModel -> take RAW flows) for
    a recipe until spectrum_n ≈ spectrum_{n-1} (converged). Returns the NON-normalised predicted
    growth sequence.

    NOTE: `corrcoef` is SCALE-INVARIANT, so this stops when the SHAPE stabilises even if amplitude is
    still growing -> the last frame can UNDERSHOOT the true amplitude. For the on-screen movie prefer
    `growth_to_convergence` (amplitude-aware stop) and `anchor_final` (nb33-anchored endpoint)."""
    Qraw = np.atleast_2d(np.asarray(Q, float)); specs = []
    for m in models:
        specs.append(mu + m.predict(Qraw)[0] @ comps)
        if len(specs) > min_iter and np.corrcoef(specs[-1], specs[-2])[0, 1] > tol_corr:
            break
    return np.array(specs)

def growth_to_convergence(models, mu, comps, Q, tol_rel=2e-3, min_iter=3):
    """Iterate the per-index models until the AMPLITUDE (not just shape) stabilises: stop when the
    relative L2 change ‖s_n - s_{n-1}‖ / ‖s_n‖ < `tol_rel`. Returns (specs, k) where specs are the
    non-normalised predicted frames 0..k (k = convergence index). Amplitude-aware, so it does not
    undershoot the final the way the correlation stop does — used to drive the seconds axis for a
    NOVEL recipe (the movie's last frame is the converged one, mapped to the predicted t_conv)."""
    Qraw = np.atleast_2d(np.asarray(Q, float)); specs = []
    for m in models:
        s = mu + m.predict(Qraw)[0] @ comps
        specs.append(s)
        if len(specs) > min_iter:
            d = np.linalg.norm(specs[-1] - specs[-2]) / max(np.linalg.norm(specs[-1]), 1e-9)
            if d < tol_rel:
                break
    return np.array(specs), len(specs) - 1

# ── nb33 endpoint anchoring ───────────────────────────────────────────────────
_nb33_cache = None
def _nb33():
    """Cached nb33 endpoint model bundle (data/spectral_model.pkl): the dedicated, tuned predictor of
    the max-NORMALISED converged spectrum (SpectralModel, clr features, PCA-5 on normalised finals)."""
    global _nb33_cache
    if _nb33_cache is None:
        import joblib
        _nb33_cache = joblib.load(pc.DATA_DIR / "spectral_model.pkl")
    return _nb33_cache

def predict_nb33_shape(Q):
    """nb33's predicted converged spectrum for a recipe, renormalised to unit max = a pure SHAPE
    (on the same wl grid as the growth frames). This is the shape the growth endpoint is anchored to."""
    M = _nb33()
    Xn = np.atleast_2d(np.asarray(Q, float))
    spec = M["mean"] + M["model"].predict(Xn)[0] @ M["components"]
    return spec / max(float(spec.max()), 1e-9)

def anchor_final(growth_final, Q):
    """Anchor a growth-model endpoint to nb33: keep the growth model's AMPLITUDE (its own max, which
    it predicts well) but take the SHAPE from nb33 (which beats the raw growth basis on the normalised
    endpoint). `anchored = growth_final.max() * nb33_shape(Q)`. This is the 'nb39 owns the path, nb33
    owns the destination' choice."""
    return float(np.asarray(growth_final).max()) * predict_nb33_shape(Q)

def _cv_predict_scores(Qraw, Y, feat, model_name, factory, groups, k):
    """Cross-validated score predictions, choosing the CHEAPER native multi-output path (one fit per
    fold) for `model_name in NATIVE_MULTIOUTPUT`, and the per-component `sp.cv_scores` path
    (independent single-output fit per component per fold) for everything else — this MUST match
    `fit_per_index`'s choice so the benchmark ranking reflects what actually gets deployed."""
    if model_name in NATIVE_MULTIOUTPUT:
        from sklearn.model_selection import cross_val_predict, GroupKFold
        X = feat(Qraw); cv = GroupKFold(n_splits=min(k, len(np.unique(groups))))
        return cross_val_predict(factory(), X, Y, cv=cv, groups=groups)
    return sp.cv_scores(Qraw, Y, feat, factory, groups, k)

def benchmark_at_indices(runs, F, mu, comps, feat=sp.f_clr, indices=None, k=CV_K,
                         models=("Ridge", "RandomForest", "ExtraTrees", "GradientBoosting",
                                 "GP-Matern52")):
    """Compare model families at a few representative indices (default: 10/35/60/90% of
    `suggest_max_index` — genuinely-growing regime, not the near-duplicate tail). Uses the SAME
    native-vs-per-component split as `fit_per_index` (`_cv_predict_scores`), so this ranking is
    trustworthy for what actually gets deployed AND cheap (native multi-output for Ridge/
    RandomForest/ExtraTrees/MLP, one fit per fold; per-component only where required, e.g. GP).
    Same leave-session-out protocol as `cv_per_index`, restricted to a few indices for speed."""
    Qraw = runs[list(sp.FEATURES)].values.astype(float); groups = runs.session.values
    if indices is None:
        cap = suggest_max_index(runs, min_growing=10)
        indices = sorted({max(1, int(cap * f)) for f in (0.10, 0.35, 0.60, 0.90)})
    rows = []
    for n in indices:
        true = frames_at_index(n, runs, F); Y = (true - mu) @ comps.T
        n_growing = int((runs.n_frames.values > n).sum())
        for mname in models:
            try:
                Sp = _cv_predict_scores(Qraw, Y, feat, mname, sp.MODELS[mname], groups, k)
            except Exception as e:
                print(f"  index {n:3d} | {mname:16s} FAILED: {e}"); continue
            rmse = float(np.sqrt(np.mean((true - (mu + Sp @ comps)) ** 2)))
            rows.append(dict(index=n, model=mname, rmse=rmse, n_growing=n_growing))
            print(f"  index {n:3d} (n_growing={n_growing:3d}) | {mname:16s} rmse={rmse:.4f}")
    return pd.DataFrame(rows)

def fig_benchmark_indices(bench):
    """RMSE vs acquisition index, one line per model — visual ExtraTrees-vs-GP comparison."""
    fig, ax = plt.subplots(figsize=(7, 4.4))
    for mname, g in bench.groupby("model"):
        g = g.sort_values("index")
        ax.plot(g["index"], g["rmse"], "o-", label=mname)
    ax.set_xlabel("acquisition index n"); ax.set_ylabel("leave-session-out spectral RMSE (a.u.)")
    ax.set_title("Model benchmark at representative growth indices"); ax.legend(fontsize=9)
    fig.tight_layout(); return fig

# ── fine-tune the winning model at the representative indices ─────────────────
#   Grids for the whole zoo, so whichever model wins the §3 benchmark can be tuned. GP: only `alpha`
#   (the fixed jitter) is tuned — the kernel length-scale/variance are already optimised internally at
#   every .fit() via n_restarts_optimizer (a fitting BUDGET, not a hyperparameter; same convention as
#   model_spectral_csv_romain.py's nb13 tuning).
PARAM_GRIDS_TUNE = {
    "Ridge": {"alpha": [1e-2, 0.1, 1.0, 10.0, 100.0]},
    "RandomForest": {
        "n_estimators": [100, 200, 300, 400], "max_depth": [8, 12, 16, 20, None],
        "min_samples_split": [2, 3, 5, 8], "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", None]},
    "ExtraTrees": {
        "n_estimators": [100, 200, 300, 400], "max_depth": [8, 12, 16, 20, None],
        "min_samples_split": [2, 3, 5, 8], "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", None]},
    "GradientBoosting": {
        "n_estimators": [100, 200, 300], "max_depth": [2, 3, 4],
        "learning_rate": [0.03, 0.05, 0.1], "subsample": [0.7, 0.8, 1.0]},
    "GP-Matern52": {"alpha": [1e-10, 1e-8, 1e-6, 1e-4, 1e-2]},
}

def _bare_estimator(model):
    from sklearn.linear_model import Ridge
    from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                                  GradientBoostingRegressor)
    from sklearn.gaussian_process import GaussianProcessRegressor
    return {
        "Ridge": lambda: Ridge(),
        "RandomForest": lambda: RandomForestRegressor(random_state=pc.SEED, n_jobs=sp.N_JOBS),
        "ExtraTrees": lambda: ExtraTreesRegressor(random_state=pc.SEED, n_jobs=sp.N_JOBS),
        "GradientBoosting": lambda: GradientBoostingRegressor(random_state=pc.SEED),
        "GP-Matern52": lambda: GaussianProcessRegressor(kernel=sp.make_kernel(),
                              normalize_y=True, random_state=pc.SEED),
    }[model]

def _grid_size(g):
    n = 1
    for v in g.values(): n *= len(v)
    return n

def tune_at_indices(runs, F, mu, comps, model, feat=sp.f_clr, indices=None, n_iter=15, k=CV_K):
    """Hyperparameter-tune any zoo `model` SEPARATELY at each of a handful of representative indices
    (same defaults as `benchmark_at_indices`), grouped CV by session, scored by spectral RMSE. Tuning
    PER INDEX (never pooling indices) matches deployment and avoids feeding the search contradictory
    same-flows/different-score examples across indices (the step-40 pitfall). For `model in
    NATIVE_MULTIOUTPUT` (Ridge/RandomForest/ExtraTrees/MLP) the search runs DIRECTLY on the bare
    estimator with the 2D target (one fit per candidate, matching how it's deployed and ~8x cheaper);
    otherwise (GP-Matern52, GradientBoosting) it's wrapped in `MultiOutputRegressor` for the required
    per-component independence. Features are StandardScaler-transformed. Returns {index: {...}}."""
    from sklearn.model_selection import RandomizedSearchCV, GroupKFold
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import make_scorer

    def _neg_rmse(y_true, y_pred):
        return -np.sqrt(np.mean((y_true - y_pred) ** 2))
    scorer = make_scorer(_neg_rmse, greater_is_better=True)

    X = StandardScaler().fit_transform(feat(runs[list(sp.FEATURES)].values.astype(float)))
    groups = runs.session.values
    if indices is None:
        cap = suggest_max_index(runs, min_growing=10)
        indices = sorted({max(1, int(cap * f)) for f in (0.10, 0.35, 0.60, 0.90)})
    gkf = GroupKFold(n_splits=min(k, runs.session.nunique()))
    grid = PARAM_GRIDS_TUNE[model]
    native = model in NATIVE_MULTIOUTPUT
    pg = grid if native else {f"estimator__{kk}": v for kk, v in grid.items()}

    results = {}
    for n in indices:
        Y = (frames_at_index(n, runs, F) - mu) @ comps.T
        est = _bare_estimator(model)() if native else MultiOutputRegressor(_bare_estimator(model)())
        search = RandomizedSearchCV(est, pg, n_iter=min(n_iter, _grid_size(grid)), cv=gkf,
                                    scoring=scorer, random_state=pc.SEED, n_jobs=sp.N_JOBS)
        search.fit(X, Y, groups=groups)
        best = {k.replace("estimator__", ""): v for k, v in search.best_params_.items()}
        results[n] = dict(best_params=best, rmse=-search.best_score_)
        print(f"  index {n:3d} | tuned {model} spec_rmse={-search.best_score_:.4f}  {best}")
    return results

def aggregate_tuned_params(results):
    """Collapse per-index tuned hyperparameters into ONE recommended set (median for numeric
    values, most-common for categorical ones), so a single factory can be used to fit every index
    rather than juggling per-index hyperparameters."""
    from collections import Counter
    keys = next(iter(results.values()))["best_params"].keys()
    agg = {}
    for k in keys:
        vals = [r["best_params"][k] for r in results.values()]
        numeric = [v for v in vals if isinstance(v, (int, float)) and not isinstance(v, bool)]
        if len(numeric) == len(vals):
            allint = all(isinstance(v, int) for v in numeric)
            agg[k] = int(np.median(numeric)) if allint else float(np.median(numeric))
        else:
            agg[k] = Counter(vals).most_common(1)[0][0]
    return agg

def make_tuned_factory(model, params):
    """Zero-arg factory producing `model` with tuned hyperparameters, for `fit_per_index(factory=...)`.
    Scale-sensitive models (Ridge, GP) are wrapped in a StandardScaler pipeline, matching the scaling
    used during tuning; tree models are bare."""
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import Ridge
    from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                                  GradientBoostingRegressor)
    from sklearn.gaussian_process import GaussianProcessRegressor
    if model == "Ridge":
        return lambda: make_pipeline(StandardScaler(), Ridge(**params))
    if model == "RandomForest":
        return lambda: RandomForestRegressor(random_state=pc.SEED, n_jobs=sp.N_JOBS, **params)
    if model == "ExtraTrees":
        return lambda: ExtraTreesRegressor(random_state=pc.SEED, n_jobs=sp.N_JOBS, **params)
    if model == "GradientBoosting":
        return lambda: GradientBoostingRegressor(random_state=pc.SEED, **params)
    if model == "GP-Matern52":
        return lambda: make_pipeline(StandardScaler(), GaussianProcessRegressor(
            kernel=sp.make_kernel(), normalize_y=True, random_state=pc.SEED,
            n_restarts_optimizer=4, **params))
    raise ValueError(f"no tuned factory for {model!r}")

def cv_per_index(runs, F, mu, comps, feat=sp.f_clr, model=DEFAULT_MODEL, indices=None, k=CV_K,
                 factory=None):
    """Leave-one-session-out spectral RMSE (non-normalised) at sampled indices -> growth-quality
    curve. Uses the same native-vs-per-component split as `fit_per_index` (`_cv_predict_scores`).
    Pass `factory` to validate tuned hyperparameters — always pass the matching `model=` name too."""
    Qraw = runs[list(sp.FEATURES)].values.astype(float); groups = runs.session.values
    fac = factory if factory is not None else sp.MODELS[model]
    mx = int(runs.n_frames.max())
    indices = indices if indices is not None else list(range(0, mx, max(1, mx // 15)))
    rows = []
    for n in indices:
        true = frames_at_index(n, runs, F); Y = (true - mu) @ comps.T
        Sp = _cv_predict_scores(Qraw, Y, feat, model, fac, groups, k)
        rows.append(dict(index=n, rmse=float(np.sqrt(np.mean((true - (mu + Sp @ comps))**2))),
                         n_growing=int((runs.n_frames.values > n).sum())))
    return pd.DataFrame(rows)

def fig_growth_real(models, mu, comps, grid, Q, real=None, feat=sp.f_clr):
    """Predicted NON-normalised growth movie for a recipe (real amplitude), coloured by index."""
    specs = predict_growth(models, mu, comps, Q, feat); nn = len(specs)
    fig, ax = plt.subplots(figsize=(7.6, 4.8))
    for n, spec in enumerate(specs):
        ax.plot(grid, spec, color=plt.cm.plasma(n / max(nn - 1, 1)), lw=1.4)
    if real is not None:
        sel = np.linspace(0, len(real) - 1, min(10, len(real))).astype(int)   # subsample for clarity
        for n in sel:
            ax.plot(grid, real[n], ":", color=plt.cm.plasma(n / max(len(real) - 1, 1)), lw=1, alpha=.8)
    ax.set_xlim(grid.min(), grid.max()); ax.axhline(0, color="0.8", lw=.6)
    ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("Extinction (a.u., NOT normalised)")
    q = "  ".join(f"{c}={v:.1f}" for c, v in zip(sp.FEATURES, np.ravel(Q)))
    ax.set_title(f"predicted growth ({nn} steps to convergence)   ({q})", fontsize=9)
    smp = plt.cm.ScalarMappable(cmap="plasma"); smp.set_array([1, nn])
    fig.colorbar(smp, ax=ax, label="acquisition index n"); fig.tight_layout(); return fig

def match_run(runs, Q, F=None, T=None, tol=0.05):
    """If the proposed recipe `Q` coincides (within `tol` µL/s on every flow) with a MEASURED run,
    return (run_name, measured_frames_or_None, times_or_None); else (None, None, None). Lets the
    interactive tool overlay the real growth + real convergence time when the user types a known
    recipe."""
    Qv = np.asarray(Q, float)
    QQ = runs[list(sp.FEATURES)].values.astype(float)
    d = np.abs(QQ - Qv).max(1)
    i = int(np.argmin(d))
    if d[i] > tol:
        return None, None, None
    r = runs.iloc[i]
    real = times = None
    if F is not None and "offset" in runs.columns:
        a, b = int(r.offset), int(r.offset) + int(r.n_frames)
        real = F[a:b]
        times = T[a:b] if T is not None else None
    return r.run, real, times

def _draw_growth(ax, models, mu, comps, grid, Q, feat=sp.f_clr, real=None, times=None,
                 t_pred=None, anchor=True, n_show=12, normalize_final=False):
    """Core growth plotter shared by every growth figure. Two modes, both coloured by REAL SECONDS
    (so the colour axis matches the t_conv annotations) with the nb33-ANCHORED endpoint:

    • MEASURED run (`real` given): predicted and measured spectra are drawn at the SAME acquisition
      indices (index-aligned, one colour scale), coloured by that frame's real elapsed time `times`
      (seconds). Solid = predicted, dotted = measured -> a like-for-like overlay that matches the
      §7 full-trajectory RMSE.
    • NOVEL recipe (`real` None): the growth is iterated to AMPLITUDE convergence and its index axis
      is mapped linearly onto [0, t_pred] seconds, so the last (anchored) frame sits at the predicted
      convergence time.

    `normalize_final=True`: rescale every REAL frame by the REAL final's own max, and every PREDICTED
    frame by the PREDICTED final's own max — independently. Both curves then end at exactly 1.0 by
    construction, so any leftover mismatch is pure SHAPE / relative-growth-rate error, decoupled from
    an absolute-amplitude bias between the predicted and measured endpoints.
    Returns the seconds horizon used for the colour normalisation."""
    from matplotlib.colors import Normalize
    Qraw = np.atleast_2d(np.asarray(Q, float)); ncomp_last = len(models) - 1
    if real is not None and len(real):
        nf = len(real)
        t = np.asarray(times, float) if times is not None else np.arange(nf, dtype=float)
        horizon = max(float(t[-1]), 1e-9)
        idx = np.unique(np.linspace(0, nf - 1, min(n_show, nf)).astype(int))
        real_final_amp = pred_final_amp = None
        if normalize_final:
            real_final_amp = max(float(real[-1].max()), 1e-9)
            pred_final = mu + models[ncomp_last].predict(Qraw)[0] @ comps
            if anchor:
                pred_final = anchor_final(pred_final, Q)
            pred_final_amp = max(float(pred_final.max()), 1e-9)
        for n in idx:
            pred = mu + models[min(n, ncomp_last)].predict(Qraw)[0] @ comps
            if anchor and n == nf - 1:
                pred = anchor_final(pred, Q)
            real_n = real[n]
            if normalize_final:
                pred = pred / pred_final_amp; real_n = real_n / real_final_amp
            col = plt.cm.plasma(t[n] / horizon)
            ax.plot(grid, pred, color=col, lw=1.5)
            ax.plot(grid, real_n, ":", color=col, lw=1.0, alpha=.85)
    else:
        # NOVEL recipe: draw to the PREDICTED convergence INDEX (t_pred / SEC_PER_FRAME) so the number
        # of frames tracks the predicted t_conv instead of `growth_to_convergence`'s amplitude-plateau
        # (which drifts far past a short recipe's real length — e.g. 83 frames for a 2 s recipe). Fall
        # back to the amplitude-plateau stop only when no t_pred is given.
        if t_pred is not None:
            kconv = int(np.clip(round(float(t_pred) / SEC_PER_FRAME), 3, ncomp_last))
        else:
            _, kconv = growth_to_convergence(models, mu, comps, Q)
        sel_idx = np.unique(np.linspace(0, kconv, min(n_show, kconv + 1)).astype(int))
        specs = [mu + models[min(int(n), ncomp_last)].predict(Qraw)[0] @ comps for n in sel_idx]
        if anchor:
            specs[-1] = anchor_final(specs[-1], Q)
        if normalize_final:
            final_amp = max(float(specs[-1].max()), 1e-9)
            specs = [s / final_amp for s in specs]
        horizon = max(float(t_pred), 1e-9) if t_pred is not None else float(kconv) * SEC_PER_FRAME
        for n, s in zip(sel_idx, specs):
            ax.plot(grid, s, color=plt.cm.plasma(min(n * SEC_PER_FRAME / horizon, 1.0)), lw=1.6)
    ax.set_xlim(grid.min(), grid.max()); ax.axhline(0, color="0.8", lw=.6)
    if normalize_final:
        ax.axhline(1.0, color="0.8", lw=.5, ls="--")
    return horizon

def fig_predicted_growth(models, mu, comps, grid, Q, feat=sp.f_clr, real=None, times=None,
                         t_measured=None, t_predicted=None, run_name=None, anchor=True,
                         normalize_final=False):
    """INTERACTIVE-TOOL figure: predicted NON-normalised growth (real amplitude) for a recipe the USER
    proposes (arbitrary Q1..Q5), coloured by REAL SECONDS, with the nb33-anchored endpoint. The
    flow-model predicted `t_conv` is always shown; if the recipe matches a known run its measured
    frames (index-aligned, dotted) and measured `t_conv` are shown too. Pass `normalize_final=True` to
    rescale real/predicted independently by their OWN final amplitude (final = 1 for both; see
    `_draw_growth`)."""
    fig, ax = plt.subplots(figsize=(8.0, 5.0))
    horizon = _draw_growth(ax, models, mu, comps, grid, Q, feat=feat, real=real, times=times,
                           t_pred=t_predicted, anchor=anchor, normalize_final=normalize_final)
    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Extinction, final-normalised (a.u.)" if normalize_final
                 else "Extinction (a.u., NOT normalised)")
    q = "  ".join(f"{c}={v:.2f}" for c, v in zip(sp.FEATURES, np.ravel(Q)))
    parts = []
    if t_predicted is not None: parts.append(f"predicted $t_{{conv}}$ = {t_predicted:.0f}s")
    if t_measured is not None:
        parts.append(f"measured $t_{{conv}}$ = {t_measured:.0f}s"
                     + (f" (matches {run_name})" if run_name else ""))
    ttl_head = "predicted growth, final-normalised (real & pred. own-final = 1)" if normalize_final \
        else "predicted growth (endpoint anchored to nb33)"
    ttl = f"{ttl_head}\n{q}"
    if parts:
        ttl += "\n" + "   |   ".join(parts)
    ax.set_title(ttl, fontsize=9)
    from matplotlib.colors import Normalize
    smp = plt.cm.ScalarMappable(cmap="plasma", norm=Normalize(0, horizon)); smp.set_array([])
    fig.colorbar(smp, ax=ax, label="elapsed time (s)"); fig.tight_layout(); return fig

def pick_growth_runs(runs, n=4, mode="random", ids=None, seed=None):
    """Indices into `runs` for the growth-movie grid. mode='random' draws a FRESH sample each call
    (pass seed for a reproducible draw); 'custom' uses the given run names; 'longest' takes the
    n runs with the most frames (most informative movies)."""
    if mode == "custom" and ids:
        sel = runs.index[runs.run.isin(ids)].tolist()
        if sel:
            return sel
        print("  no run matched the given ids -> falling back to random")
    if mode == "longest":
        return runs.sort_values("n_frames", ascending=False).index[:n].tolist()
    rng = np.random.default_rng(seed)
    return list(rng.choice(runs.index, min(n, len(runs)), replace=False))

def fig_growth_grid(models, mu, comps, grid, runs, F, sel, T=None, feat=sp.f_clr, ncol=2,
                    show_conv_time=True, held_sessions=None, anchor=True, normalize_final=False):
    """Grid of predicted-vs-measured growth movies for the chosen `runs` rows (see pick_growth_runs).
    Predicted (solid) and measured (dotted) spectra are drawn INDEX-ALIGNED and coloured by each
    frame's REAL elapsed time (seconds, from `T` = frame_times()); the endpoint is nb33-anchored.
    Each panel is titled with the run name, flows, and its MEASURED vs flow-model PREDICTED
    convergence time — now on the same seconds axis as the colour.

    Pass `held_sessions` (the set held out of training) to TAG each panel `in-sample` vs `HELD-OUT`:
    a model fit on the train split reconstructs its own training recipes near-perfectly (ExtraTrees
    ~interpolates them), which flatters the fit — the honest panels are the HELD-OUT ones, and they
    will look like §7's out-of-sample benchmark.

    Pass `normalize_final=True` for the FINAL-NORMALISED view: real frames are rescaled by the REAL
    final's own max, predicted frames by the PREDICTED final's own max (independently), so both curves
    reach exactly 1.0 at the last index — isolating shape/relative-growth error from any absolute-
    amplitude mismatch between prediction and measurement."""
    from matplotlib.colors import Normalize
    n = len(sel); nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(6.2 * ncol, 4.4 * nrow), squeeze=False)
    axes = axes.ravel()
    for a, i in zip(axes, sel):
        r = runs.loc[i]
        Q = r[list(sp.FEATURES)].values.astype(float)
        real, times = run_frames_times(runs, F, T, i)
        horizon = _draw_growth(a, models, mu, comps, grid, Q, feat=feat, real=real, times=times,
                               anchor=anchor, normalize_final=normalize_final)
        smp = plt.cm.ScalarMappable(cmap="plasma", norm=Normalize(0, horizon)); smp.set_array([])
        fig.colorbar(smp, ax=a, label="elapsed time (s)")
        a.set_xlabel("Wavelength (nm)", fontsize=8)
        a.set_ylabel("Extinction, final-norm." if normalize_final else "Extinction (a.u.)", fontsize=8)
        q = "  ".join(f"{c}={v:.1f}" for c, v in zip(sp.FEATURES, Q))
        held = held_sessions is not None and r.session in set(held_sessions)
        tag = ("  [HELD-OUT]" if held else "  [in-sample]") if held_sessions is not None else ""
        ttl = f"{r.run}{tag}\n{q}"
        if show_conv_time:
            tm = lookup_real_conv_time(r.run); tp = predict_conv_time(Q)
            ttl += f"\n$t_{{conv}}$: measured={'%.0f' % tm + 's' if tm is not None else 'n/a'}" \
                   f"   predicted={tp:.0f}s"
        a.set_title(ttl, fontsize=7.5, color=("#b00" if held else "black"))
        a.tick_params(labelsize=7)
    for a in axes[n:]: a.axis("off")
    fig.tight_layout(); return fig

# ── side-by-side model comparison: same recipe, GP vs ExtraTrees ──────────────
def fit_per_index_multi(runs, F, mu, comps, models=("GP-Matern52", "ExtraTrees"),
                        factories=None, max_index=None):
    """Fit per-index models for EACH of `models`. Pass `factories={name: factory}` to use tuned
    hyperparameters for one (or both); any model not in `factories` gets plain sp.MODELS[name]
    defaults. Returns {name: [fitted model per index]}."""
    factories = factories or {}
    return {name: fit_per_index(runs, F, mu, comps, model=name,
                                factory=factories.get(name), max_index=max_index)
            for name in models}

def fit_zoo(runs, F, mu, comps, models=CANDIDATE_MODELS, max_index=None, feat=sp.f_clr):
    """Train the WHOLE candidate zoo ONCE (one per-index model set per family), with DEFAULT
    hyper-parameters (fair, apples-to-apples). This is the single heavy training step of the
    notebook: every later section (deploy, interactive tool, end-of-notebook benchmark) REUSES these
    fitted models instead of retraining — that is what makes 'one real benchmark at the end' cheap.
    Trees/Ridge/MLP get a cheap native joint multi-output fit; GP/GradientBoosting the required
    per-component fit. Prints per-model wall-clock so the cost is visible. Returns {name: [models]}."""
    import time
    out = {}
    for name in models:
        t0 = time.time()
        out[name] = fit_per_index(runs, F, mu, comps, model=name, max_index=max_index, feat=feat)
        print(f"  trained {name:16s} ({len(out[name])} per-index models)  {time.time() - t0:5.1f} s")
    return out

def rank_from_table(table):
    """Rank models by the compare-table's MEAN full-trajectory RMSE row (the end-of-notebook
    benchmark's verdict) — lower is better. Returns a sorted Series {model: mean_rmse}."""
    return table.loc["MEAN"].sort_values()

# ── on-disk model persistence with a TRAINING FINGERPRINT ─────────────────────
#   Save fitted per-index models so re-running the notebook doesn't retrain them — but ONLY reuse the
#   cache when nothing that affects training has changed. The fingerprint hashes: the training data
#   actually consumed (Qraw of runs_tr, the frames F, the PCA basis mu/comps), the index cap, the
#   feature function, the estimator's FULL config (str(factory()) captures every hyper-parameter), AND
#   the SOURCE CODE of the fitting path (fit_per_index / _NativeMultiOutputModel / SpectralModel /
#   frames_at_index / raw_basis). Change any of these -> fingerprint differs -> automatic refit.
def _model_fingerprint(runs_tr, F, mu, comps, model, max_index, feat):
    import hashlib, inspect
    h = hashlib.sha256()
    h.update(np.ascontiguousarray(mu, dtype=np.float64).tobytes())
    h.update(np.ascontiguousarray(comps, dtype=np.float64).tobytes())
    h.update(runs_tr[["run", "session", "offset", "n_frames", *sp.FEATURES]]
             .to_csv(index=False).encode())
    h.update(np.array(F.shape, dtype=np.int64).tobytes())
    h.update(hashlib.sha256(np.ascontiguousarray(F, dtype=np.float64).tobytes()).digest())
    h.update(f"{model}|{max_index}|{getattr(feat, '__name__', repr(feat))}".encode())
    try:
        h.update(str(sp.MODELS[model]()).encode())            # full estimator config = all hyperparams
    except Exception:
        h.update(model.encode())
    h.update(repr(sorted(NATIVE_MULTIOUTPUT)).encode())
    for obj in (fit_per_index, _NativeMultiOutputModel, sp.SpectralModel, frames_at_index, raw_basis):
        try:
            h.update(inspect.getsource(obj).encode())
        except Exception:
            pass
    return h.hexdigest()

def fit_or_load_model(runs_tr, F, mu, comps, model, max_index=None, feat=sp.f_clr, force=False,
                      path=None, verbose=True):
    """Fit ONE model's per-index set, or reload it from disk if a cached fit exists whose training
    fingerprint matches (see `_model_fingerprint`). `force=True` always refits. Cache lives at
    data/growth_zoo_<model>.pkl (fixed name + internal fingerprint, so stale files don't accumulate)."""
    import joblib, time
    path = path or (pc.DATA_DIR / f"growth_zoo_{model}.pkl")
    fp = _model_fingerprint(runs_tr, F, mu, comps, model, max_index, feat)
    if path.exists() and not force:
        try:
            blob = joblib.load(path)
            if blob.get("fp") == fp:
                if verbose:
                    print(f"  loaded {model:16s} from cache ({len(blob['models'])} idx) — training unchanged")
                return blob["models"]
            reason = "training changed"
        except Exception:
            reason = "unreadable cache"
        if verbose:
            print(f"  {model:16s}: {reason} -> refitting")
    t0 = time.time()
    models = fit_per_index(runs_tr, F, mu, comps, model=model, max_index=max_index, feat=feat)
    joblib.dump({"fp": fp, "models": models, "model": model, "max_index": max_index}, path)
    if verbose:
        print(f"  fitted {model:16s} ({len(models)} idx)  {time.time() - t0:5.1f} s -> cached {path.name}")
    return models

def fit_or_load_zoo(runs_tr, F, mu, comps, spec, feat=sp.f_clr, force=False, verbose=True):
    """Fit-or-load a whole zoo. `spec` = {model_name: max_index_or_None} (each model may have its own
    cap — e.g. the deployed ExtraTrees uncapped, the GP capped). Returns {name: [per-index models]}.
    Set `force=True` (or delete data/growth_zoo_*.pkl) to force a full retrain."""
    return {name: fit_or_load_model(runs_tr, F, mu, comps, name, max_index=mx, feat=feat,
                                    force=force, verbose=verbose)
            for name, mx in spec.items()}

# ══════════════════════════════════════════════════════════════════════════════
# SCALE-BY-FINAL growth model (self-contained alternative to nb33 anchoring).
#   Normalise every frame by its run's FINAL amplitude (one large, well-defined scalar per run — NOT
#   the per-frame max, which is ~0 at nucleation and would amplify noise), learn the well-behaved
#   normalised trajectory + a separate amplitude scaler A(Q). Reconstruct  frame_n = A(Q)·norm_n.
#   Validated: fixes the per-frame-decomposition blow-up AND reaches nb33-class endpoint shape (with
#   GP; ExtraTrees is a notch behind but fast and self-contained — no dependency on nb33).
# ══════════════════════════════════════════════════════════════════════════════
class _ScaleByFinalIndexModel:
    """Per-index adapter exposing the SAME `.predict(Qraw)->scores` interface as `sp.SpectralModel`,
    so every existing figure/benchmark works unchanged. With the AUGMENTED basis (mu=0, comps=[mu_n;
    comps_n]) the usual `mu + scores @ comps` reconstructs A(Q)·(mu_n + s_n @ comps_n): the scores it
    returns are [A(Q), A(Q)·s_n], i.e. amplitude times the normalised-shape scores."""
    def __init__(self, feat, shape_model, amp_reg):
        self.feat = feat; self.shape = shape_model; self.amp = amp_reg
    def predict(self, Qraw):
        A = np.expm1(self.amp.predict(self.feat(np.atleast_2d(Qraw))))          # (n,)  amplitude
        s = self.shape.predict(Qraw)                                            # (n, k) shape scores
        return np.column_stack([A, A[:, None] * s])                            # (n, 1+k)

def fit_scale_by_final(runs_tr, F, feat=sp.f_clr, model="ExtraTrees", amp_model="ExtraTrees",
                       max_index=None, n_pc=N_PC_RAW):
    """Fit the scale-by-final growth model on the train split. Returns (models, mu, comps) — plug
    compatible with fit_per_index's output (so fig_growth_grid / predict_growth / compare_table /
    trajectory_rmse all accept it as (models, mu, comps)). `mu` is zeros and `comps` is the augmented
    basis [mu_n; comps_n]; each per-index `models[n]` is a `_ScaleByFinalIndexModel`. The endpoint is
    self-contained (no nb33) — call the figures with `anchor=False`."""
    Qraw = runs_tr[list(sp.FEATURES)].values.astype(float)
    fmax = np.array([float(F[int(r.offset) + int(r.n_frames) - 1].max()) for r in runs_tr.itertuples()])
    fmax = np.maximum(fmax, 1e-9)
    # normalised-basis fit on the train frames (each run / its own final amplitude)
    blocks = [F[int(r.offset):int(r.offset) + int(r.n_frames)] / fm
              for r, fm in zip(runs_tr.itertuples(), fmax)]
    Fn = np.vstack(blocks)
    mu_n = Fn.mean(0); _, sv, Vt = np.linalg.svd(Fn - mu_n, full_matrices=False)
    comps_n = Vt[:n_pc]
    # amplitude scaler A(Q) in log space (final amplitudes span ~0.09..1.1)
    amp_reg = sp.MODELS[amp_model](); amp_reg.fit(feat(Qraw), np.log1p(fmax))
    # per-index normalised-shape models
    per_component = model not in NATIVE_MULTIOUTPUT
    mx = int(runs_tr.n_frames.max()); mx = mx if max_index is None else min(mx, max_index)
    models = []
    for n in range(mx):
        idx = runs_tr.offset.values + np.minimum(n, runs_tr.n_frames.values - 1)
        Y = (F[idx] / fmax[:, None] - mu_n) @ comps_n.T
        if per_component:
            base = sp.SpectralModel(n_pc, feat, sp.MODELS[model]).fit(Qraw, Y)
        else:
            base = _NativeMultiOutputModel(feat, sp.MODELS[model]()).fit(Qraw, Y)
        models.append(_ScaleByFinalIndexModel(feat, base, amp_reg))
    mu_aug = np.zeros(comps_n.shape[1]); comps_aug = np.vstack([mu_n, comps_n])
    return models, mu_aug, comps_aug

def _scale_fingerprint(runs_tr, F, model, amp_model, max_index, n_pc, feat):
    import hashlib, inspect
    h = hashlib.sha256()
    h.update(runs_tr[["run", "session", "offset", "n_frames", *sp.FEATURES]]
             .to_csv(index=False).encode())
    h.update(np.array(F.shape, dtype=np.int64).tobytes())
    h.update(hashlib.sha256(np.ascontiguousarray(F, dtype=np.float64).tobytes()).digest())
    h.update(f"{model}|{amp_model}|{max_index}|{n_pc}|{getattr(feat,'__name__',repr(feat))}".encode())
    for m in (model, amp_model):
        try: h.update(str(sp.MODELS[m]()).encode())
        except Exception: h.update(m.encode())
    for obj in (fit_scale_by_final, _ScaleByFinalIndexModel, _NativeMultiOutputModel, sp.SpectralModel):
        try: h.update(inspect.getsource(obj).encode())
        except Exception: pass
    return h.hexdigest()

def fit_or_load_scale_by_final(runs_tr, F, feat=sp.f_clr, model="ExtraTrees", amp_model="ExtraTrees",
                               max_index=None, n_pc=N_PC_RAW, force=False, path=None, verbose=True):
    """Fit-or-load the scale-by-final model with the same fingerprint guard as fit_or_load_model.
    Cached at data/growth_scalebyfinal_<model>.pkl. Returns (models, mu, comps)."""
    import joblib, time
    path = path or (pc.DATA_DIR / f"growth_scalebyfinal_{model}.pkl")
    fp = _scale_fingerprint(runs_tr, F, model, amp_model, max_index, n_pc, feat)
    if path.exists() and not force:
        try:
            blob = joblib.load(path)
            if blob.get("fp") == fp:
                if verbose:
                    print(f"  loaded scale-by-final {model:12s} from cache — training unchanged")
                return blob["models"], blob["mu"], blob["comps"]
            reason = "training changed"
        except Exception:
            reason = "unreadable cache"
        if verbose: print(f"  scale-by-final {model}: {reason} -> refitting")
    t0 = time.time()
    models, mu, comps = fit_scale_by_final(runs_tr, F, feat=feat, model=model, amp_model=amp_model,
                                           max_index=max_index, n_pc=n_pc)
    joblib.dump({"fp": fp, "models": models, "mu": mu, "comps": comps}, path)
    if verbose:
        print(f"  fitted scale-by-final {model:12s} ({len(models)} idx)  {time.time()-t0:5.1f}s -> cached")
    return models, mu, comps

# ══════════════════════════════════════════════════════════════════════════════
# SHAPE-SCALE growth model — "nb33 per iteration" (the user's construction).
#   Normalise each frame by its OWN max (a pure unit-max SHAPE at every index, exactly nb33's target),
#   fit a shared clean shape basis on PEAKED frames, and per index learn (a) a shape model Q->shape and
#   (b) an amplitude model Q->log(frame max). Reconstruct  frame_n = A(n,Q)·shape(n,Q).
#   The normalisation denominator is FLOORED at ABS_FLOOR: a pre-nucleation frame has ~0 (or slightly
#   negative) amplitude and thus NO shape — flooring stops it blowing up (index 0 otherwise reaches
#   |recon|~1e5). Validated: shape reproduces nb33 at t=40/final (0.155/0.142), full-trajectory
#   absolute RMSE 0.097 (matches scale-by-final), abs RMSE at t=1/5/10/40 = 0.037/0.052/0.075/0.074.
# ══════════════════════════════════════════════════════════════════════════════
def fit_shape_scale(runs_tr, F, feat=sp.f_clr, model="ExtraTrees", amp_model="ExtraTrees",
                    max_index=None, n_pc=N_PC, floor=None):
    """The user's per-iteration nb33 model. Returns (models, mu, comps) plug-compatible with every
    figure/benchmark (mu=0, comps=[mu_s; comps_s]); each `models[n]` is a `_ScaleByFinalIndexModel`
    holding this index's own shape model and amplitude regressor. `floor` (default `pc.ABS_FLOOR`)
    guards the per-frame normalisation. `n_pc` defaults to the nb33 decomposition (`pc.N_PC`)."""
    floor = pc.ABS_FLOOR if floor is None else floor
    Qraw = runs_tr[list(sp.FEATURES)].values.astype(float)
    # shared shape basis, fit on PEAKED train frames only (no nucleation noise in the basis)
    tr_idx = np.concatenate([np.arange(int(r.offset), int(r.offset) + int(r.n_frames))
                             for r in runs_tr.itertuples()])
    peaked = F[tr_idx][F[tr_idx].max(1) > floor]
    Sn, _ = sp.normalise(peaked, "max")
    mu_s = Sn.mean(0); _, sv, Vt = np.linalg.svd(Sn - mu_s, full_matrices=False)
    comps_s = Vt[:n_pc]
    per_component = model not in NATIVE_MULTIOUTPUT
    mx = int(runs_tr.n_frames.max()); mx = mx if max_index is None else min(mx, max_index)
    models = []
    for n in range(mx):
        fr = frames_at_index(n, runs_tr, F); a = fr.max(1)
        Y = (fr / np.maximum(a, floor)[:, None] - mu_s) @ comps_s.T          # per-frame unit-max shape
        if per_component:
            shape_m = sp.SpectralModel(n_pc, feat, sp.MODELS[model]).fit(Qraw, Y)
        else:
            shape_m = _NativeMultiOutputModel(feat, sp.MODELS[model]()).fit(Qraw, Y)
        amp = sp.MODELS[amp_model](); amp.fit(feat(Qraw), np.log1p(np.maximum(a, 0)))  # per-index A(n,Q)
        models.append(_ScaleByFinalIndexModel(feat, shape_m, amp))
    mu_aug = np.zeros(comps_s.shape[1]); comps_aug = np.vstack([mu_s, comps_s])
    return models, mu_aug, comps_aug

def _shapescale_fingerprint(runs_tr, F, model, amp_model, max_index, n_pc, floor, feat):
    import hashlib, inspect
    h = hashlib.sha256()
    h.update(runs_tr[["run", "session", "offset", "n_frames", *sp.FEATURES]]
             .to_csv(index=False).encode())
    h.update(np.array(F.shape, dtype=np.int64).tobytes())
    h.update(hashlib.sha256(np.ascontiguousarray(F, dtype=np.float64).tobytes()).digest())
    h.update(f"{model}|{amp_model}|{max_index}|{n_pc}|{floor}|{getattr(feat,'__name__',repr(feat))}".encode())
    for m in (model, amp_model):
        try: h.update(str(sp.MODELS[m]()).encode())
        except Exception: h.update(m.encode())
    for obj in (fit_shape_scale, _ScaleByFinalIndexModel, _NativeMultiOutputModel, sp.SpectralModel):
        try: h.update(inspect.getsource(obj).encode())
        except Exception: pass
    return h.hexdigest()

def fit_or_load_shape_scale(runs_tr, F, feat=sp.f_clr, model="ExtraTrees", amp_model="ExtraTrees",
                            max_index=None, n_pc=N_PC, floor=None, force=False, path=None, verbose=True):
    """Fit-or-load the shape-scale ('nb33 per iteration') model, fingerprint-guarded like the others.
    Cached at data/growth_shapescale_<model>.pkl. Returns (models, mu, comps)."""
    import joblib, time
    floor_v = pc.ABS_FLOOR if floor is None else floor
    path = path or (pc.DATA_DIR / f"growth_shapescale_{model}.pkl")
    fp = _shapescale_fingerprint(runs_tr, F, model, amp_model, max_index, n_pc, floor_v, feat)
    if path.exists() and not force:
        try:
            blob = joblib.load(path)
            if blob.get("fp") == fp:
                if verbose:
                    print(f"  loaded shape-scale {model:12s} from cache — training unchanged")
                return blob["models"], blob["mu"], blob["comps"]
            reason = "training changed"
        except Exception:
            reason = "unreadable cache"
        if verbose: print(f"  shape-scale {model}: {reason} -> refitting")
    t0 = time.time()
    models, mu, comps = fit_shape_scale(runs_tr, F, feat=feat, model=model, amp_model=amp_model,
                                        max_index=max_index, n_pc=n_pc, floor=floor)
    joblib.dump({"fp": fp, "models": models, "mu": mu, "comps": comps}, path)
    if verbose:
        print(f"  fitted shape-scale {model:12s} ({len(models)} idx)  {time.time()-t0:5.1f}s -> cached")
    return models, mu, comps

# ══════════════════════════════════════════════════════════════════════════════
# TAU-REPARAMETRISED growth model (progress tau = t / t_conv) — alternative, NOT the deployed model.
#   Aligns every run on normalised progress tau∈[0,1] (fixes the short/long-run confusion of the raw
#   per-index design: at index 50 short runs are forward-filled finals while long runs are mid-growth).
#   Reconstruct  frame(tau) = A_final(Q) · g(tau,Q) · shape(tau,Q):
#     • shape(tau,Q): one regressor on [feat(Q), tau] -> shape-PCA scores (basis on peaked norm frames)
#     • A_final(Q):   one regressor Q -> log final amplitude
#     • g(tau,Q):     one regressor on [feat(Q), tau] -> a(tau)/A_final (the growth curve, per recipe)
#   Bonus interpretable output: a power-law GROWTH RATE p per recipe (a/A_final ≈ tau^p; p<1 concave,
#   >1 convex), predictable from Q (held-out R²≈0.6). Validated held-out abs RMSE ≈0.065 (< 0.10 of the
#   per-index model) using the MEASURED t_conv for tau; a novel recipe uses predict_conv_time(Q).
# ══════════════════════════════════════════════════════════════════════════════
class TauGrowthModel:
    """Bundle for the tau-reparametrised growth model. `predict_at(Q, tau)` -> raw spectrum at progress
    tau; `predict_trajectory(Q, taus)` -> (len(taus), n_wl); `growth_rate(Q)` -> power-law exponent p
    (None if not fit). tau->seconds for a novel recipe: tau = t / predict_conv_time(Q)."""
    def __init__(self, mu_s, comps_s, shape_m, g_m, Af_m, p_m=None, feat=sp.f_clr):
        self.mu_s = mu_s; self.comps_s = comps_s
        self.shape_m = shape_m; self.g_m = g_m; self.Af_m = Af_m; self.p_m = p_m; self.feat = feat
    def _X(self, Q, taus):
        Q2 = np.atleast_2d(np.asarray(Q, float)); taus = np.atleast_1d(np.asarray(taus, float))
        return np.column_stack([np.repeat(self.feat(Q2), len(taus), axis=0), taus])
    def predict_trajectory(self, Q, taus):
        X = self._X(Q, taus)
        sh = self.mu_s + self.shape_m.predict(X) @ self.comps_s
        A = np.expm1(self.Af_m.predict(self.feat(np.atleast_2d(Q)))[0]) * self.g_m.predict(X)
        return A[:, None] * sh
    def predict_at(self, Q, tau):
        return self.predict_trajectory(Q, [tau])[0]
    def growth_rate(self, Q):
        return None if self.p_m is None else float(self.p_m.predict(self.feat(np.atleast_2d(Q)))[0])

def _run_final_amp(meta, amp):
    """Per-run final amplitude (amp of the latest-t frame). Returns dict run->amp."""
    out = {}
    for r in pd.unique(meta.run):
        m = (meta.run == r).values
        out[r] = float(amp[m][np.argmax(meta.t.values[m])])
    return out

def fit_tau_growth(meta, S, feat=sp.f_clr, model="ExtraTrees", n_pc=N_PC, held_sessions=None,
                   floor=None, fit_rate=True):
    """Fit the tau-reparametrised growth model on the frames whose session is NOT in `held_sessions`
    (all frames if None). Uses the tau column of `extract()`'s meta. Returns a `TauGrowthModel`."""
    floor = pc.ABS_FLOOR if floor is None else floor
    amp = S.max(1); tau = np.clip(meta.tau.values, 0, 1.2)
    Q = meta[list(sp.FEATURES)].values.astype(float); runs = meta.run.values
    tr = np.ones(len(meta), bool) if not held_sessions else ~meta.session.isin(set(held_sessions)).values
    # shape basis on peaked TRAIN frames
    pk = tr & (amp > floor)
    Sn = S[pk] / amp[pk][:, None]
    mu_s = Sn.mean(0); _, sv, Vt = np.linalg.svd(Sn - mu_s, full_matrices=False); comps_s = Vt[:n_pc]
    X = np.column_stack([feat(Q), tau])
    shape_m = sp.MODELS[model](); shape_m.fit(X[tr], (S[tr] / np.maximum(amp[tr], floor)[:, None] - mu_s) @ comps_s.T)
    rf = _run_final_amp(meta, amp); Af_frame = np.array([rf[r] for r in runs])
    g_m = sp.MODELS[model](); g_m.fit(X[tr], amp[tr] / np.maximum(Af_frame[tr], 1e-9))
    tr_runs = pd.unique(runs[tr]); Qrun = np.array([Q[runs == r][0] for r in tr_runs])
    Af_m = sp.MODELS[model](); Af_m.fit(feat(Qrun), np.log1p([rf[r] for r in tr_runs]))
    p_m = None
    if fit_rate:
        def _p(r):
            m = runs == r; t = tau[m]; g = amp[m] / max(rf[r], 1e-9)
            ok = (t > 0.05) & (t < 1.0) & (g > 0.02)
            return float(np.polyfit(np.log(t[ok]), np.log(g[ok]), 1)[0]) if ok.sum() >= 4 else np.nan
        ps = np.array([_p(r) for r in tr_runs]); okp = np.isfinite(ps)
        if okp.sum() >= 5:
            p_m = sp.MODELS[model](); p_m.fit(feat(Qrun[okp]), ps[okp])
    return TauGrowthModel(mu_s, comps_s, shape_m, g_m, Af_m, p_m, feat)

def tau_holdout_rmse(model, meta, S, held_sessions, floor=None):
    """Absolute spectral RMSE of the tau model over the HELD-OUT frames (tau from measured t_conv), plus
    the same broken into 5 tau bins. Returns (mean_rmse, DataFrame per bin)."""
    amp = S.max(1); tau = np.clip(meta.tau.values, 0, 1.2)
    Q = meta[list(sp.FEATURES)].values.astype(float)
    ho = meta.session.isin(set(held_sessions)).values
    errs = np.array([np.sqrt(np.mean((model.predict_at(Q[i], tau[i]) - S[i]) ** 2)) for i in np.where(ho)[0]])
    tho = tau[ho]; rows = []
    for lo, hi in [(0, .2), (.2, .4), (.4, .6), (.6, .8), (.8, 1.01)]:
        b = (tho >= lo) & (tho < hi)
        if b.any(): rows.append(dict(tau_bin=f"{lo:.1f}-{hi:.1f}", rmse=float(errs[b].mean()), n=int(b.sum())))
    return float(errs.mean()), pd.DataFrame(rows)

def _tau_fingerprint(meta, S, model, n_pc, held_sessions, floor, feat, fit_rate):
    import hashlib, inspect
    h = hashlib.sha256()
    h.update(meta[["run", "session", "t", "tau", *sp.FEATURES]].round(6).to_csv(index=False).encode())
    h.update(hashlib.sha256(np.ascontiguousarray(S, dtype=np.float64).tobytes()).digest())
    h.update(f"{model}|{n_pc}|{sorted(held_sessions or [])}|{floor}|{fit_rate}|{getattr(feat,'__name__',repr(feat))}".encode())
    try: h.update(str(sp.MODELS[model]()).encode())
    except Exception: h.update(model.encode())
    for obj in (fit_tau_growth, TauGrowthModel, _run_final_amp):
        try: h.update(inspect.getsource(obj).encode())
        except Exception: pass
    return h.hexdigest()

def fit_or_load_tau_growth(meta, S, feat=sp.f_clr, model="ExtraTrees", n_pc=N_PC, held_sessions=None,
                           floor=None, fit_rate=True, force=False, path=None, verbose=True):
    """Fit-or-load the tau growth model, fingerprint-guarded. Cached at data/growth_tau_<model>.pkl."""
    import joblib, time
    floor_v = pc.ABS_FLOOR if floor is None else floor
    path = path or (pc.DATA_DIR / f"growth_tau_{model}.pkl")
    fp = _tau_fingerprint(meta, S, model, n_pc, held_sessions, floor_v, feat, fit_rate)
    if path.exists() and not force:
        try:
            blob = joblib.load(path)
            if blob.get("fp") == fp:
                if verbose: print(f"  loaded tau-growth {model:12s} from cache — training unchanged")
                return blob["m"]
            reason = "training changed"
        except Exception:
            reason = "unreadable cache"
        if verbose: print(f"  tau-growth {model}: {reason} -> refitting")
    t0 = time.time()
    m = fit_tau_growth(meta, S, feat=feat, model=model, n_pc=n_pc, held_sessions=held_sessions,
                       floor=floor, fit_rate=fit_rate)
    joblib.dump({"fp": fp, "m": m}, path)
    if verbose: print(f"  fitted tau-growth {model:12s}  {time.time()-t0:5.1f}s -> cached")
    return m

def fig_tau_recon(model, meta, S, grid, sel_runs, ncol=2):
    """Measured vs tau-model reconstruction for the given run names, on a real-seconds axis (tau from
    the run's own measured t_conv). Panel title carries the recipe and its predicted growth-rate p."""
    amp = S.max(1)
    n = len(sel_runs); nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(6.2 * ncol, 4.4 * nrow), squeeze=False); axes = axes.ravel()
    from matplotlib.colors import Normalize
    for a, run in zip(axes, sel_runs):
        m = (meta.run == run).values
        if not m.any(): a.axis("off"); continue
        Q = meta.loc[m, list(sp.FEATURES)].values[0].astype(float)
        t = meta.t.values[m]; tau = np.clip(meta.tau.values[m], 0, 1.2); o = np.argsort(t)
        t, tau = t[o], tau[o]; real = S[m][o]
        horizon = max(float(t[-1]), 1e-9)
        sub = np.unique(np.linspace(0, len(t) - 1, min(12, len(t))).astype(int))
        for k in sub:
            col = plt.cm.plasma(t[k] / horizon)
            a.plot(grid, model.predict_at(Q, tau[k]), color=col, lw=1.5)
            a.plot(grid, real[k], ":", color=col, lw=1.0, alpha=.85)
        a.set_xlim(grid.min(), grid.max()); a.axhline(0, color="0.8", lw=.6)
        smp = plt.cm.ScalarMappable(cmap="plasma", norm=Normalize(0, horizon)); smp.set_array([])
        fig.colorbar(smp, ax=a, label="elapsed time (s)")
        p = model.growth_rate(Q); q = "  ".join(f"{c}={v:.1f}" for c, v in zip(sp.FEATURES, Q))
        a.set_title(f"{run}\n{q}" + (f"\ngrowth-rate p={p:.2f}" if p is not None else ""), fontsize=8)
        a.set_xlabel("Wavelength (nm)", fontsize=8); a.set_ylabel("Extinction (a.u.)", fontsize=8)
    for a in axes[n:]: a.axis("off")
    fig.tight_layout(); return fig

# ── LEAVE-ONE-SESSION-OUT evaluation of the tau growth model ───────────────────
def loso_tau_growth(meta, S, feat=sp.f_clr, model="ExtraTrees", n_pc=N_PC, floor=None, basis="max"):
    """Leave-ONE-session-out absolute spectral RMSE of the tau growth pipeline: for each session, refit
    on the other 12 and score its frames (τ from the run's MEASURED t_conv — the spectral-model ceiling,
    same setting as the fixed-split numbers). This is the HONEST generalisation estimate; a single fixed
    held-out split can be lucky/unlucky.

    `basis` selects the decomposition being compared:
      • 'max'  — max-normalised SHAPE basis + amplitude split  A_final(Q)·g(τ,Q)·shape(τ,Q) (deployed style)
      • 'raw'  — NON-normalised PCA on the raw frames, predict the full spectrum's scores directly from
                 (feat(Q), τ) and reconstruct (no shape/amplitude split) — the 'base pca non normalisée'.
    Returns a DataFrame (session, n_frames, n_runs, rmse); `.attrs` carries frame-weighted & session means."""
    floor = pc.ABS_FLOOR if floor is None else floor
    amp = S.max(1); tau = np.clip(meta.tau.values, 0, 1.2)
    Q = meta[list(sp.FEATURES)].values.astype(float); sess = meta.session.values; runs = meta.run.values
    X = np.column_stack([feat(Q), tau])
    rf = {r: float(amp[runs == r][np.argmax(meta.t.values[runs == r])]) for r in np.unique(runs)}
    Af_frame = np.array([rf[r] for r in runs])
    rows = []
    for s in sorted(np.unique(sess)):
        tr = sess != s; te = sess == s
        if basis == "raw":                                   # non-normalised PCA, direct full-spectrum
            mu_r = S[tr].mean(0); _, _, Vt = np.linalg.svd(S[tr] - mu_r, full_matrices=False)
            comps_r = Vt[:n_pc]
            mdl = sp.MODELS[model](); mdl.fit(X[tr], (S[tr] - mu_r) @ comps_r.T)
            recon = mu_r + mdl.predict(X[te]) @ comps_r
        else:                                                # max-normalised shape + amplitude split
            pk = tr & (amp > floor); Sn = S[pk] / amp[pk][:, None]
            mu_s = Sn.mean(0); _, _, Vt = np.linalg.svd(Sn - mu_s, full_matrices=False); comps_s = Vt[:n_pc]
            shp = sp.MODELS[model](); shp.fit(X[tr], (S[tr] / np.maximum(amp[tr], floor)[:, None] - mu_s) @ comps_s.T)
            g = sp.MODELS[model](); g.fit(X[tr], amp[tr] / np.maximum(Af_frame[tr], 1e-9))
            tr_runs = np.unique(runs[tr]); Qr = np.array([Q[runs == r][0] for r in tr_runs])
            Af = sp.MODELS[model](); Af.fit(feat(Qr), np.log1p([rf[r] for r in tr_runs]))
            sh = mu_s + shp.predict(X[te]) @ comps_s
            A = np.expm1(Af.predict(feat(Q[te]))) * g.predict(X[te])
            recon = A[:, None] * sh
        rows.append(dict(session=s, n_frames=int(te.sum()), n_runs=int(np.unique(runs[te]).size),
                         rmse=float(np.sqrt(np.mean((recon - S[te]) ** 2)))))
    df = pd.DataFrame(rows).sort_values("rmse", ascending=False).reset_index(drop=True)
    df.attrs["weighted_mean"] = float(np.average(df.rmse, weights=df.n_frames))
    df.attrs["session_mean"] = float(df.rmse.mean())
    df.attrs["basis"] = basis
    return df

def fig_loso(df, ref=None):
    """Bar chart of per-session leave-one-out RMSE (sorted worst->best) with the frame-weighted mean
    line. Pass `ref` (e.g. another basis's DataFrame) to overlay a second bar series for comparison."""
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    x = np.arange(len(df)); w = 0.38 if ref is not None else 0.7
    ax.bar(x - (w/2 if ref is not None else 0), df.rmse, w, label=f"basis={df.attrs.get('basis','?')}", color="#0072B2")
    ax.axhline(df.attrs["weighted_mean"], color="#0072B2", ls="--", lw=1,
               label=f"weighted mean {df.attrs['weighted_mean']:.3f}")
    if ref is not None:
        rr = ref.set_index("session").reindex(df.session).reset_index()
        ax.bar(x + w/2, rr.rmse, w, label=f"basis={ref.attrs.get('basis','?')}", color="#D55E00", alpha=.85)
        ax.axhline(ref.attrs["weighted_mean"], color="#D55E00", ls="--", lw=1,
                   label=f"weighted mean {ref.attrs['weighted_mean']:.3f}")
    ax.set_xticks(x); ax.set_xticklabels(df.session, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("leave-one-session-out spectral RMSE"); ax.legend(fontsize=8)
    ax.set_title("Honest generalisation: leave-one-session-out (τ from measured t_conv)")
    fig.tight_layout(); return fig

def _pick_loso_recipes(meta, n, seed=0, min_frames=12):
    """`n` run names, one from each of n DISTINCT sessions (the longest run in each), so every recipe
    is genuinely LOSO (its whole session is held out) and the sessions are diverse."""
    g = meta.groupby("run").agg(session=("session", "first"), nf=("t", "size")).reset_index()
    g = g[g.nf >= min_frames]
    rng = np.random.default_rng(seed)
    sess = rng.permutation(g.session.unique())[:n]
    return [g[g.session == s].sort_values("nf").run.values[-1] for s in sess]

def fig_loso_raw_growth(meta, S, grid, recipes=None, n_recipes=6, n_pc=5, n_show=10, feat=sp.f_clr,
                        seed=0, ncol=3):
    """TEST 1 — growth model on a NON-normalised PCA basis (`n_pc`=5), shown on `n_recipes` recipes in
    LEAVE-ONE-SESSION-OUT. For each recipe the raw-basis model (predict full-spectrum scores from
    [feat(Q), τ], reconstruct directly — no shape/amplitude split) is refit with the recipe's whole
    session HELD OUT, then its growth is drawn (solid) vs the measured frames (dotted), coloured by
    real elapsed seconds."""
    from matplotlib.colors import Normalize
    amp = S.max(1); tau = np.clip(meta.tau.values, 0, 1.2)
    Q = meta[list(sp.FEATURES)].values.astype(float); sess = meta.session.values; runs = meta.run.values
    X = np.column_stack([feat(Q), tau])
    recipes = recipes or _pick_loso_recipes(meta, n_recipes, seed)
    cache = {}
    def model_without(s):
        if s not in cache:
            tr = sess != s; mu_r = S[tr].mean(0)
            _, _, Vt = np.linalg.svd(S[tr] - mu_r, full_matrices=False); comps = Vt[:n_pc]
            m = sp.MODELS["ExtraTrees"](); m.fit(X[tr], (S[tr] - mu_r) @ comps.T)
            cache[s] = (mu_r, comps, m)
        return cache[s]
    n = len(recipes); nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 3.9 * nrow), squeeze=False); axes = axes.ravel()
    for a, run in zip(axes, recipes):
        m = (meta.run == run).values
        s = meta.session.values[m][0]; Qv = Q[m][0]
        mu_r, comps, mdl = model_without(s)
        t = meta.t.values[m]; tr_ = tau[m]; real = S[m]; o = np.argsort(t)
        t, tr_, real = t[o], tr_[o], real[o]; horizon = max(float(t[-1]), 1e-9)
        sub = np.unique(np.linspace(0, len(t) - 1, min(n_show, len(t))).astype(int))
        for k in sub:
            pred = mu_r + mdl.predict(np.column_stack([feat(Qv[None]), [tr_[k]]]))[0] @ comps
            col = plt.cm.plasma(t[k] / horizon)
            a.plot(grid, pred, color=col, lw=1.5); a.plot(grid, real[k], ":", color=col, lw=1.0, alpha=.85)
        a.set_xlim(grid.min(), grid.max()); a.axhline(0, color="0.8", lw=.6); a.tick_params(labelsize=7)
        smp = plt.cm.ScalarMappable(cmap="plasma", norm=Normalize(0, horizon)); smp.set_array([])
        fig.colorbar(smp, ax=a, label="s")
        q = "  ".join(f"{c}={v:.1f}" for c, v in zip(sp.FEATURES, Qv))
        a.set_title(f"{run}  [{s} HELD-OUT]\n{q}", fontsize=7.5, color="#b00")
        a.set_xlabel("nm", fontsize=8); a.set_ylabel("Ext (a.u.)", fontsize=8)
    for a in axes[n:]: a.axis("off")
    fig.suptitle(f"TEST 1 — NON-normalised {n_pc}-PC growth basis, leave-one-session-out "
                 "(solid=predicted, dotted=measured)", fontsize=10)
    fig.tight_layout(rect=[0, 0, 1, 0.97]); return fig

def fig_loso_buildup(meta, S, grid, recipes=None, n_recipes=4, n_show=8, n_pc=5, feat=sp.f_clr, seed=1):
    """TEST 2 — build up the reconstruction on the NORMALISED basis, in LEAVE-ONE-SESSION-OUT, for
    `n_recipes` recipes. Each ROW is a recipe; the three COLUMNS show the SAME predicted growth series
    (`n_show` representative τ points) in three scalings, so the decomposition is visible:
      1. normalised SHAPE (peak≈1) — the shape model shape(τ,Q);
      2. × growth KINETICS g(τ,Q) — the learned amplitude fraction a(τ)/A_final (relative growth 0→1);
      3. × Ext_max A_final(Q) — the learned final-amplitude model → real absolute intensity.
    Measured frames overlaid (dotted) in the matching scaling. All three models are refit with the
    recipe's whole session HELD OUT (no data leak). Floor guards the per-frame normalisation."""
    from matplotlib.colors import Normalize
    floor = pc.ABS_FLOOR
    amp = S.max(1); tau = np.clip(meta.tau.values, 0, 1.2)
    Q = meta[list(sp.FEATURES)].values.astype(float); sess = meta.session.values; runs = meta.run.values
    X = np.column_stack([feat(Q), tau])
    rf = {r: float(amp[runs == r][np.argmax(meta.t.values[runs == r])]) for r in np.unique(runs)}
    Af_frame = np.array([rf[r] for r in runs])
    recipes = recipes or _pick_loso_recipes(meta, n_recipes, seed)
    cache = {}
    def models_without(s):
        if s not in cache:
            tr = sess != s; pk = tr & (amp > floor)
            Sn = S[pk] / amp[pk][:, None]; mu_s = Sn.mean(0)
            _, _, Vt = np.linalg.svd(Sn - mu_s, full_matrices=False); comps_s = Vt[:n_pc]
            shp = sp.MODELS["ExtraTrees"](); shp.fit(X[tr], (S[tr] / np.maximum(amp[tr], floor)[:, None] - mu_s) @ comps_s.T)
            g = sp.MODELS["ExtraTrees"](); g.fit(X[tr], amp[tr] / np.maximum(Af_frame[tr], 1e-9))
            tr_runs = np.unique(runs[tr]); Qr = np.array([Q[runs == r][0] for r in tr_runs])
            Af = sp.MODELS["ExtraTrees"](); Af.fit(feat(Qr), np.log1p([rf[r] for r in tr_runs]))
            cache[s] = (mu_s, comps_s, shp, g, Af)
        return cache[s]
    titles = ["(1) normalised shape", "(2) × growth kinetics g(τ)", "(3) × Ext_max (absolute)"]
    nr = len(recipes)
    fig, axes = plt.subplots(nr, 3, figsize=(5.0 * 3, 3.4 * nr), squeeze=False)
    for ri, run in enumerate(recipes):
        m = (meta.run == run).values; s = meta.session.values[m][0]; Qv = Q[m][0]
        mu_s, comps_s, shp, g, Af = models_without(s)
        t = meta.t.values[m]; tr_ = tau[m]; real = S[m]; o = np.argsort(t)
        t, tr_, real = t[o], tr_[o], real[o]; horizon = max(float(t[-1]), 1e-9)
        A_final_meas = max(float(real[-1].max()), 1e-9)
        A_final_pred = float(np.expm1(Af.predict(feat(Qv[None]))[0]))
        sub = np.unique(np.linspace(0, len(t) - 1, min(n_show, len(t))).astype(int))
        for k in sub:
            x = np.column_stack([feat(Qv[None]), [tr_[k]]])
            shape = mu_s + shp.predict(x)[0] @ comps_s
            gk = float(g.predict(x)[0]); col = plt.cm.plasma(t[k] / horizon)
            rk = real[k]
            axes[ri, 0].plot(grid, shape, color=col, lw=1.4)
            axes[ri, 0].plot(grid, rk / max(rk.max(), 1e-9), ":", color=col, lw=.9, alpha=.8)
            axes[ri, 1].plot(grid, gk * shape, color=col, lw=1.4)
            axes[ri, 1].plot(grid, rk / A_final_meas, ":", color=col, lw=.9, alpha=.8)
            axes[ri, 2].plot(grid, A_final_pred * gk * shape, color=col, lw=1.4)
            axes[ri, 2].plot(grid, rk, ":", color=col, lw=.9, alpha=.8)
        for ci in range(3):
            axes[ri, ci].set_xlim(grid.min(), grid.max()); axes[ri, ci].axhline(0, color="0.85", lw=.5)
            axes[ri, ci].tick_params(labelsize=7)
            if ri == 0: axes[ri, ci].set_title(titles[ci], fontsize=9)
            if ri == nr - 1: axes[ri, ci].set_xlabel("nm", fontsize=8)
        axes[ri, 0].set_ylabel(f"{run}\n[{s} HELD-OUT]", fontsize=7.5, color="#b00")
    fig.suptitle("TEST 2 — normalised basis, LOSO: predicted shape → × kinetics → × Ext_max "
                 "(solid=pred, dotted=measured)", fontsize=10)
    fig.tight_layout(rect=[0, 0, 1, 0.97]); return fig

def _pick_loso_recipes_many(meta, n, seed=0, min_frames=5):
    """`n` run names spread across sessions (round-robin, longest-first within each session), for when
    `n` exceeds the number of distinct sessions — repeats sessions rather than lowering diversity."""
    g = meta.groupby("run").agg(session=("session", "first"), nf=("t", "size")).reset_index()
    g = g[g.nf >= min_frames]
    rng = np.random.default_rng(seed)
    sessions = list(rng.permutation(g.session.unique()))
    remaining = g.copy(); picked = []
    while len(picked) < n and len(remaining):
        progressed = False
        for s in sessions:
            if len(picked) >= n:
                break
            cand = remaining[remaining.session == s].sort_values("nf", ascending=False)
            if len(cand):
                picked.append(cand.iloc[0].run)
                remaining = remaining[remaining.run != cand.iloc[0].run]
                progressed = True
        if not progressed:
            break
    return picked[:n]

def fig_live_growth(meta, S, grid, recipes=None, n_recipes=20, n_pc=N_PC, floor=None, feat=sp.f_clr,
                    seed=2, ncol=4, scale="absolute"):
    """'Recorded LIVE' LOSO comparison of the normalised-shape × growth-kinetics (× Ext_max) pipeline,
    for many recipes at once. Each panel: EVERY measured frame (dotted — already ~1 acquisition/second,
    see SEC_PER_FRAME) vs a DENSE predicted growth series sampled at the SAME ~1 Hz cadence out to the
    recipe's flow-model PREDICTED convergence time (solid) — what the prediction would look like if it
    had itself been recorded live, from composition alone. Title shows MEASURED (H5 plateau) vs PREDICTED
    (flow model) convergence time. Each recipe's whole SESSION is held out of training (LOSO; session
    models cached, so repeated sessions don't refit).

    `scale`:
      • 'absolute' — reconstruct A_final(Q)·g(τ)·shape(τ); measured shown at real absorbance.
      • 'relative' — reconstruct g(τ)·shape(τ) with the FINAL kept NORMALISED (peak≈1, no Ext_max
        scaling); measured shown as frame / its own measured final amplitude. Isolates shape + growth
        kinetics from the absolute-amplitude model."""
    from matplotlib.colors import Normalize
    floor = pc.ABS_FLOOR if floor is None else floor
    amp = S.max(1); tau = np.clip(meta.tau.values, 0, 1.2)
    Q = meta[list(sp.FEATURES)].values.astype(float); sess = meta.session.values; runs = meta.run.values
    X = np.column_stack([feat(Q), tau])
    rf = {r: float(amp[runs == r][np.argmax(meta.t.values[runs == r])]) for r in np.unique(runs)}
    Af_frame = np.array([rf[r] for r in runs])
    recipes = recipes or _pick_loso_recipes_many(meta, n_recipes, seed)
    cache = {}
    def models_without(s):
        if s not in cache:
            tr = sess != s; pk = tr & (amp > floor)
            Sn = S[pk] / amp[pk][:, None]; mu_s = Sn.mean(0)
            _, _, Vt = np.linalg.svd(Sn - mu_s, full_matrices=False); comps_s = Vt[:n_pc]
            shp = sp.MODELS["ExtraTrees"](); shp.fit(X[tr], (S[tr] / np.maximum(amp[tr], floor)[:, None] - mu_s) @ comps_s.T)
            g = sp.MODELS["ExtraTrees"](); g.fit(X[tr], amp[tr] / np.maximum(Af_frame[tr], 1e-9))
            tr_runs = np.unique(runs[tr]); Qr = np.array([Q[runs == r][0] for r in tr_runs])
            Af = sp.MODELS["ExtraTrees"](); Af.fit(feat(Qr), np.log1p([rf[r] for r in tr_runs]))
            cache[s] = (mu_s, comps_s, shp, g, Af)
        return cache[s]

    n = len(recipes); nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 3.7 * nrow), squeeze=False); axes = axes.ravel()
    for a, run in zip(axes, recipes):
        m = (meta.run == run).values; s = meta.session.values[m][0]; Qv = Q[m][0]
        mu_s, comps_s, shp, g, Af = models_without(s)
        t = meta.t.values[m]; o = np.argsort(t); t = t[o]; real = S[m][o]
        A_meas_final = max(float(real[-1].max()), 1e-9)
        t_meas = lookup_real_conv_time(run); t_pred = predict_conv_time(Qv)
        horizon = max(float(t[-1]), t_pred, 1e-9)
        # DENSE predicted series at the acquisition cadence, out to the PREDICTED convergence time
        t_grid = np.arange(0.0, t_pred + SEC_PER_FRAME, SEC_PER_FRAME)
        tau_grid = np.clip(t_grid / max(t_pred, 1e-9), 0, 1.0)
        Xp = np.column_stack([np.repeat(feat(Qv[None]), len(tau_grid), axis=0), tau_grid])
        shapes = mu_s + shp.predict(Xp) @ comps_s
        gk = g.predict(Xp); Afv = float(np.expm1(Af.predict(feat(Qv[None]))[0]))
        if scale == "relative":                      # final stays normalised (peak~1), no Ext_max
            pred_specs = gk[:, None] * shapes
            meas_specs = real / A_meas_final
        else:
            pred_specs = Afv * gk[:, None] * shapes
            meas_specs = real
        for k in range(len(t)):                      # ALL measured frames, ~1/s cadence
            a.plot(grid, meas_specs[k], ":", color=plt.cm.plasma(min(t[k] / horizon, 1.0)),
                  lw=.7, alpha=.8, rasterized=True)
        for k in range(len(t_grid)):                  # ALL predicted frames, dense ~1 Hz to t_pred
            a.plot(grid, pred_specs[k], "-", color=plt.cm.plasma(min(t_grid[k] / horizon, 1.0)),
                  lw=.9, alpha=.85, rasterized=True)
        a.set_xlim(grid.min(), grid.max()); a.axhline(0, color="0.85", lw=.5); a.tick_params(labelsize=6)
        smp = plt.cm.ScalarMappable(cmap="plasma", norm=Normalize(0, horizon)); smp.set_array([])
        fig.colorbar(smp, ax=a, label="s")
        tm_s = f"{t_meas:.0f}s" if t_meas is not None else "n/a"
        a.set_title(f"{run}  [{s} HELD-OUT]\n$t_{{conv}}$ measured={tm_s}  predicted={t_pred:.0f}s",
                   fontsize=7, color="#b00")
        a.set_xlabel("nm", fontsize=7); a.set_ylabel("Ext (a.u.)", fontsize=7)
    for a in axes[n:]: a.axis("off")
    recon_lbl = ("shape×kinetics, final NORMALISED" if scale == "relative"
                 else "shape×kinetics×Ext_max")
    fig.suptitle(f"LIVE-RECORDING comparison, {n} recipes, leave-one-session-out — "
                 f"dotted=ALL measured frames, solid=DENSE predicted ({recon_lbl}) "
                 "to predicted $t_{conv}$", fontsize=10)
    fig.tight_layout(rect=[0, 0, 1, 0.96]); return fig

# ══════════════════════════════════════════════════════════════════════════════
# DISCRETE TIME-SLICE growth model — INDEPENDENT model per fixed checkpoint (0/5/10/20/40 s / final),
# shape via GP-Matern52 (nb33's own choice), NOT a single model pooled over continuous τ.
#   Rationale: the τ-bonus model (§9-13) pools ALL frames into ONE ExtraTrees regressor with τ as an
#   extra feature — cheap, but forces one estimator to fit both the fast early transient and the slow
#   plateau. Instead, fit a SEPARATE shape-GP + amplitude regressor at each checkpoint (mirroring how
#   nb33 dedicates one tuned GP to the endpoint). Validated in LOSO: weighted-mean absolute RMSE
#   ≈ 0.063 vs ≈ 0.079 for the pooled τ-continuous ExtraTrees model — a real (~19%), not marginal, gain.
#   Caveat: this evaluates ONLY at the 6 checkpoints per recipe, not the full continuous trajectory
#   (the τ-model's LOSO number averages over every acquired frame) — not perfectly apples-to-apples,
#   but the size of the gap is far beyond the ~2-5% moved by any other lever tried (basis, features,
#   growth-law family, PC count).
# ══════════════════════════════════════════════════════════════════════════════
DEFAULT_SLICES_SEC = (0, 5, 10, 20, 40, "final")

def _slice_to_index(lab, max_idx):
    return max_idx if lab == "final" else min(int(round(lab / SEC_PER_FRAME)), max_idx)

class SliceGrowthModel:
    """Bundle of INDEPENDENT per-checkpoint models: `bundles[label] = (mu_s, comps_s, shape_model,
    amp_model)`. `predict_at(Q, label)` reconstructs `A(Q)·shape(Q)` at that one checkpoint — there is
    NO interpolation between checkpoints (that is the point: each is its own dedicated fit)."""
    def __init__(self, slices_sec, bundles, feat=sp.f_clr):
        self.slices_sec = list(slices_sec); self.bundles = bundles; self.feat = feat
    def predict_at(self, Q, label):
        mu_s, comps_s, shape_m, amp_m = self.bundles[label]
        Xq = np.atleast_2d(np.asarray(Q, float))
        sh = mu_s + shape_m.predict(Xq)[0] @ comps_s
        A = float(np.expm1(amp_m.predict(self.feat(Xq))[0]))
        return A * sh

def _fit_slice_bundle(Qraw, fr, floor, n_pc, shape_model, amp_model, feat, train_mask=None):
    """One checkpoint's (mu_s, comps_s, shape_model, amp_model), fit on `Qraw`/`fr` restricted to
    `train_mask` (None = use all rows). The shape basis is fit on that same train subset's PEAKED
    frames only (falls back to the whole train subset if too few are above `floor`)."""
    tr = np.ones(len(fr), bool) if train_mask is None else train_mask
    a = fr.max(1); Sn = fr / np.maximum(a, floor)[:, None]
    pk = tr & (a > floor)
    if pk.sum() < 10:
        pk = tr
    mu_s = Sn[pk].mean(0); _, _, Vt = np.linalg.svd(Sn[pk] - mu_s, full_matrices=False); comps_s = Vt[:n_pc]
    Y = (Sn[tr] - mu_s) @ comps_s.T
    per_component = shape_model not in NATIVE_MULTIOUTPUT
    shp = (sp.SpectralModel(n_pc, feat, sp.MODELS[shape_model]).fit(Qraw[tr], Y) if per_component
          else _NativeMultiOutputModel(feat, sp.MODELS[shape_model]()).fit(Qraw[tr], Y))
    amp = sp.MODELS[amp_model](); amp.fit(feat(Qraw[tr]), np.log1p(np.maximum(a[tr], 0)))
    return mu_s, comps_s, shp, amp

def fit_slice_growth(runs_tr, F, slices_sec=DEFAULT_SLICES_SEC, n_pc=N_PC, floor=None,
                     shape_model="GP-Matern52", amp_model="ExtraTrees", feat=sp.f_clr):
    """Fit one INDEPENDENT (shape-GP, amplitude) bundle per checkpoint in `slices_sec` (seconds, or the
    string 'final' for each run's own converged frame). Returns a `SliceGrowthModel`."""
    floor = pc.ABS_FLOOR if floor is None else floor
    Qraw = runs_tr[list(sp.FEATURES)].values.astype(float)
    max_idx = int(runs_tr.n_frames.max()) - 1
    bundles = {}
    for lab in slices_sec:
        fr = frames_at_index(_slice_to_index(lab, max_idx), runs_tr, F)
        bundles[lab] = _fit_slice_bundle(Qraw, fr, floor, n_pc, shape_model, amp_model, feat)
    return SliceGrowthModel(slices_sec, bundles, feat)

def _slice_fingerprint(runs_tr, F, slices_sec, n_pc, floor, shape_model, amp_model, feat):
    import hashlib, inspect
    h = hashlib.sha256()
    h.update(runs_tr[["run", "session", "offset", "n_frames", *sp.FEATURES]].to_csv(index=False).encode())
    h.update(np.array(F.shape, dtype=np.int64).tobytes())
    h.update(hashlib.sha256(np.ascontiguousarray(F, dtype=np.float64).tobytes()).digest())
    h.update(f"{slices_sec}|{n_pc}|{floor}|{shape_model}|{amp_model}|{getattr(feat,'__name__',repr(feat))}".encode())
    for m in (shape_model, amp_model):
        try: h.update(str(sp.MODELS[m]()).encode())
        except Exception: h.update(m.encode())
    for obj in (fit_slice_growth, _fit_slice_bundle, SliceGrowthModel, sp.SpectralModel, _NativeMultiOutputModel):
        try: h.update(inspect.getsource(obj).encode())
        except Exception: pass
    return h.hexdigest()

def fit_or_load_slice_growth(runs_tr, F, slices_sec=DEFAULT_SLICES_SEC, n_pc=N_PC, floor=None,
                             shape_model="GP-Matern52", amp_model="ExtraTrees", feat=sp.f_clr,
                             force=False, path=None, verbose=True):
    """Fit-or-load the discrete time-slice model, fingerprint-guarded like the others. Cached at
    data/growth_slices_<shape_model>.pkl."""
    import joblib, time
    floor_v = pc.ABS_FLOOR if floor is None else floor
    path = path or (pc.DATA_DIR / f"growth_slices_{shape_model}.pkl")
    fp = _slice_fingerprint(runs_tr, F, slices_sec, n_pc, floor_v, shape_model, amp_model, feat)
    if path.exists() and not force:
        try:
            blob = joblib.load(path)
            if blob.get("fp") == fp:
                if verbose: print(f"  loaded slice-growth {shape_model:12s} from cache — training unchanged")
                return blob["m"]
            reason = "training changed"
        except Exception:
            reason = "unreadable cache"
        if verbose: print(f"  slice-growth {shape_model}: {reason} -> refitting")
    t0 = time.time()
    m = fit_slice_growth(runs_tr, F, slices_sec=slices_sec, n_pc=n_pc, floor=floor,
                         shape_model=shape_model, amp_model=amp_model, feat=feat)
    joblib.dump({"fp": fp, "m": m}, path)
    if verbose: print(f"  fitted slice-growth {shape_model:12s} ({len(slices_sec)} checkpoints)  "
                      f"{time.time()-t0:5.1f}s -> cached")
    return m

def loso_slice_growth(runs, F, slices_sec=DEFAULT_SLICES_SEC, n_pc=N_PC, floor=None,
                      shape_model="GP-Matern52", amp_model="ExtraTrees", feat=sp.f_clr):
    """Leave-one-session-out absolute RMSE of the discrete time-slice model, evaluated ONLY at the
    checkpoints in `slices_sec` (not the full continuous trajectory — see `loso_tau_growth`/`fig_loso`
    for that full-frame number; the two use different sampling density per recipe, so compare with that
    caveat). Returns a DataFrame (session, n_frames, n_runs, rmse) compatible with `fig_loso`."""
    floor = pc.ABS_FLOOR if floor is None else floor
    sess = runs.session.values; Qraw = runs[list(sp.FEATURES)].values.astype(float)
    max_idx = int(runs.n_frames.max()) - 1
    rows = []
    for s in sorted(np.unique(sess)):
        tr = sess != s; te = sess == s
        errs = []
        for lab in slices_sec:
            fr = frames_at_index(_slice_to_index(lab, max_idx), runs, F)
            mu_s, comps_s, shp, amp = _fit_slice_bundle(Qraw, fr, floor, n_pc, shape_model, amp_model, feat, tr)
            sh_pred = mu_s + shp.predict(Qraw[te]) @ comps_s
            A_pred = np.expm1(amp.predict(feat(Qraw[te])))
            recon = A_pred[:, None] * sh_pred
            errs.extend(np.sqrt(np.mean((recon - fr[te]) ** 2, axis=1)).tolist())
        rows.append(dict(session=s, n_frames=len(errs), n_runs=int(te.sum()), rmse=float(np.mean(errs))))
    df = pd.DataFrame(rows).sort_values("rmse", ascending=False).reset_index(drop=True)
    df.attrs["weighted_mean"] = float(np.average(df.rmse, weights=df.n_frames))
    df.attrs["session_mean"] = float(df.rmse.mean())
    df.attrs["basis"] = f"slices+{shape_model}"
    return df

def slice_checkpoints(bundles, Qv, slices_sec, t_pred, feat=sp.f_clr):
    """(checkpoint_times_s, checkpoint_spectra) for one recipe from its per-session `bundles`. Each
    checkpoint's absolute reconstruction A(Q)·shape(Q); 'final' is placed at the predicted t_conv. The
    'final' checkpoint's shape (GP on the final frames) reproduces nb33 exactly."""
    tc, Sc = [], []
    for lab in slices_sec:
        mu_s, comps_s, shp, amp = bundles[lab]
        sh = mu_s + shp.predict(np.atleast_2d(Qv))[0] @ comps_s
        A = float(np.expm1(amp.predict(feat(np.atleast_2d(Qv)))[0]))
        tc.append(t_pred if lab == "final" else float(lab)); Sc.append(A * sh)
    o = np.argsort(tc)
    return np.array(tc)[o], np.array(Sc)[o]

def fig_slice_growth_recon(runs, F, grid, recipes=None, n_recipes=6, slices_sec=DEFAULT_SLICES_SEC,
                          n_pc=N_PC, floor=None, shape_model="GP-Matern52", amp_model="ExtraTrees",
                          feat=sp.f_clr, seed=3, ncol=3, interpolate=False, scale="absolute"):
    """LOSO reconstruction from the discrete GP checkpoints. Two axes of control:

    `interpolate` — False: draw only the independent per-checkpoint predictions (thick lines, the raw
    output of each slice model, no continuity). True: **linearly interpolate the checkpoint spectra in
    time** onto the acquisition cadence, giving a continuous growth movie that passes through each GP
    checkpoint — the 'most precise' reconstruction (GP-accurate anchors + a smooth path between them).
    `scale` — 'absolute': real Ext; 'relative': divide every frame by the FINAL checkpoint's own max so
    the predicted final is NORMALISED (peak≈1) — and, because the final checkpoint reproduces nb33, that
    normalised final IS nb33's predicted spectrum. Measured frames (dotted) are shown in the matching
    scale. Each recipe's whole session is held out (LOSO; per-session bundles cached)."""
    from matplotlib.colors import Normalize
    from scipy.interpolate import interp1d
    floor = pc.ABS_FLOOR if floor is None else floor
    sess = runs.session.values; Qraw = runs[list(sp.FEATURES)].values.astype(float)
    max_idx = int(runs.n_frames.max()) - 1
    if recipes is None:
        g = runs[runs.n_frames >= 12]
        rng = np.random.default_rng(seed)
        chosen_sessions = rng.permutation(g.session.unique())[:n_recipes]
        recipes = [g[g.session == s].sort_values("n_frames").run.values[-1] for s in chosen_sessions]
    cache = {}
    def bundles_without(s):
        if s not in cache:
            tr = sess != s
            cache[s] = {lab: _fit_slice_bundle(Qraw, frames_at_index(_slice_to_index(lab, max_idx), runs, F),
                                               floor, n_pc, shape_model, amp_model, feat, tr)
                       for lab in slices_sec}
        return cache[s]
    n = len(recipes); nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.8 * ncol, 3.7 * nrow), squeeze=False); axes = axes.ravel()
    for a, run in zip(axes, recipes):
        i = int(runs.index[runs.run == run][0]); r = runs.loc[i]
        s = r.session; Qv = Qraw[runs.index.get_loc(i)]
        bundles = bundles_without(s)
        real = F[int(r.offset):int(r.offset) + int(r.n_frames)]
        t_real = np.arange(len(real)) * SEC_PER_FRAME
        t_meas = lookup_real_conv_time(run); t_pred = predict_conv_time(Qv)
        horizon = max(float(t_real[-1]), t_pred, 1e-9)
        tc, Sc = slice_checkpoints(bundles, Qv, slices_sec, t_pred, feat)
        fin_amp = max(float(Sc[-1].max()), 1e-9)                     # final checkpoint's own max
        meas_final = max(float(real[-1].max()), 1e-9)
        def rescale(M, meas):     # -> (predicted, measured) in the requested scale
            return (M / fin_amp, meas / meas_final) if scale == "relative" else (M, meas)
        # measured (dotted)
        _, meas_plot = rescale(np.zeros((1, len(grid))), real)
        for k in range(len(real)):
            a.plot(grid, meas_plot[k], ":", color=plt.cm.plasma(min(t_real[k] / horizon, 1.0)),
                  lw=.6, alpha=.6, rasterized=True)
        # predicted
        if interpolate:
            t_grid = np.arange(0.0, tc[-1] + SEC_PER_FRAME, SEC_PER_FRAME)
            dense = interp1d(tc, Sc, axis=0, bounds_error=False, fill_value=(Sc[0], Sc[-1]))(t_grid)
            dense_plot, _ = rescale(dense, real)
            for k in range(len(t_grid)):
                a.plot(grid, dense_plot[k], "-", color=plt.cm.plasma(min(t_grid[k] / horizon, 1.0)),
                      lw=.9, alpha=.85, rasterized=True)
        else:
            Sc_plot, _ = rescale(Sc, real)
            for tk, spec in zip(tc, Sc_plot):
                a.plot(grid, spec, "-", color=plt.cm.plasma(min(tk / horizon, 1.0)), lw=2.2)
        a.set_xlim(grid.min(), grid.max()); a.axhline(0, color="0.85", lw=.5); a.tick_params(labelsize=7)
        if scale == "relative": a.axhline(1.0, color="0.8", lw=.5, ls="--")
        smp = plt.cm.ScalarMappable(cmap="plasma", norm=Normalize(0, horizon)); smp.set_array([])
        fig.colorbar(smp, ax=a, label="s")
        tm_s = f"{t_meas:.0f}s" if t_meas is not None else "n/a"
        a.set_title(f"{run}  [{s} HELD-OUT]\n$t_{{conv}}$ measured={tm_s}  predicted={t_pred:.0f}s",
                   fontsize=7.5, color="#b00")
        a.set_xlabel("nm", fontsize=7)
        a.set_ylabel("Ext, final-norm." if scale == "relative" else "Ext (a.u.)", fontsize=7)
    for a in axes[n:]: a.axis("off")
    mode = ("interpolated between checkpoints" if interpolate else "checkpoints only, no interpolation")
    sc_lbl = "final-NORMALISED (final = nb33)" if scale == "relative" else "absolute"
    fig.suptitle(f"Discrete time-slice {shape_model} model, LOSO — {mode}, {sc_lbl}; "
                 f"checkpoints t={list(slices_sec)}s (thin dotted=measured)", fontsize=9)
    fig.tight_layout(rect=[0, 0, 1, 0.95]); return fig

# ── nb33-per-iteration SHAPE diagnostic (normalised, leave-session-out) ────────
def shape_rmse_per_index(runs, F, grid, indices, model="GP-Matern52", n_pc=N_PC, k=CV_K, floor=None):
    """Leave-session-out NORMALISED-shape spectral RMSE at each acquisition `index` (nb33 protocol:
    normalise='max', shared PCA(`n_pc`) shape basis on peaked frames, grouped CV). Returns a DataFrame
    (index, spec_rmse, lmax_rmse, n_growing, n_noisy). The t=1/5 rows read huge NOT because the model
    fails but because the *measured* normalised target is amplified noise (no particle yet)."""
    floor = pc.ABS_FLOOR if floor is None else floor
    amp = F.max(1); peaked = F[amp > floor]
    Sn0, _ = sp.normalise(peaked, "max")
    mu_s, comps_s, _ = sp.fit_pca(Sn0, n_pc)
    Q = runs[list(sp.FEATURES)].values.astype(float); groups = runs.session.values
    rows = []
    for n in indices:
        fr = frames_at_index(n, runs, F); a = fr.max(1)
        Sn = fr / np.maximum(a, floor)[:, None]
        scores = (Sn - mu_s) @ comps_s.T
        Sp = _cv_predict_scores(Q, scores, sp.f_clr, model, sp.MODELS[model], groups, k)
        recon = mu_s + Sp @ comps_s
        lam = sp.derived_lmax(grid, Sn); lamp = sp.derived_lmax(grid, recon)
        rows.append(dict(index=n, spec_rmse=float(np.sqrt(np.mean((Sn - recon) ** 2))),
                         lmax_rmse=float(np.sqrt(np.nanmean((lam - lamp) ** 2))),
                         n_growing=int((runs.n_frames.values > n).sum()),
                         n_noisy=int((a < floor).sum())))
    return pd.DataFrame(rows)

def fig_shape_recon(runs, F, grid, indices, model="ExtraTrees", n_runs=3, n_pc=N_PC, floor=None, seed=0):
    """Per-iteration NORMALISED-shape reconstruction (nb33 protocol): for a few long runs, MEASURED
    unit-max shape (black) vs MODEL-predicted shape (red) at each of `indices`. Makes the key point
    visible: the model predicts a SMOOTH, sensible shape at every t, while the *measured* normalised
    target is amplified noise at early t (there is barely a particle yet) — so the huge early-t
    spec_rmse is a metric artifact, not a model failure. Shape models are fit on ALL runs here (this is
    an illustration, not the held-out benchmark)."""
    floor = pc.ABS_FLOOR if floor is None else floor
    amp = F.max(1); Sn0, _ = sp.normalise(F[amp > floor], "max")
    mu_s, comps_s, _ = sp.fit_pca(Sn0, n_pc)
    Q = runs[list(sp.FEATURES)].values.astype(float)
    per_component = model not in NATIVE_MULTIOUTPUT
    shp = {}
    for n in indices:
        fr = frames_at_index(n, runs, F); a = fr.max(1)
        Y = (fr / np.maximum(a, floor)[:, None] - mu_s) @ comps_s.T
        shp[n] = (sp.SpectralModel(n_pc, sp.f_clr, sp.MODELS[model]).fit(Q, Y) if per_component
                  else _NativeMultiOutputModel(sp.f_clr, sp.MODELS[model]()).fit(Q, Y))
    sel = runs.sort_values("n_frames", ascending=False).index[:n_runs]
    fig, axes = plt.subplots(len(sel), len(indices), squeeze=False,
                             figsize=(2.9 * len(indices), 2.5 * len(sel)))
    for ri, i in enumerate(sel):
        r = runs.loc[i]; q = r[list(sp.FEATURES)].values.astype(float)
        for ci, n in enumerate(indices):
            a = axes[ri, ci]; k = min(n, int(r.n_frames) - 1)
            meas = F[int(r.offset) + k]; meas_n = meas / max(meas.max(), 1e-9)
            pred = mu_s + shp[n].predict(np.atleast_2d(q))[0] @ comps_s
            a.plot(grid, meas_n, "k-", lw=1.2, label="measured shape")
            a.plot(grid, pred, "r:", lw=1.4, label="predicted shape")
            a.set_xlim(grid.min(), grid.max()); a.tick_params(labelsize=6)
            ff = " (fwd-fill)" if n > int(r.n_frames) - 1 else ""
            if ri == 0:
                a.set_title(f"t={n}" + (" (final)" if n == indices[-1] else ""), fontsize=9)
            a.text(0.03, 0.88, f"idx {k}{ff}", transform=a.transAxes, fontsize=6, color="0.4")
            if ci == 0: a.set_ylabel(f"{r.run}\nnorm. ext.", fontsize=7)
            if ri == len(sel) - 1: a.set_xlabel("nm", fontsize=7)
    axes[0, 0].legend(fontsize=6, loc="lower center")
    fig.suptitle("Normalised SHAPE per iteration (nb33 protocol) — black = measured, red = predicted",
                 fontsize=10)
    fig.tight_layout(rect=[0, 0, 1, 0.97]); return fig

_clean_df_cache = None
def _clean_df():
    """Cached metadata table for `lookup_real_conv_time` (avoids re-reading from disk on every
    call — cheap alone, but adds up across many recipes).

    Reads `selection` (189 rows), NOT `clean`: this is a LOOKUP table, so it must be a superset
    of whatever training set is active. 6 of the 151 `hardcut` runs are absent from clean.csv
    (they were dropped as step-30 outliers), and those lookups would silently miss."""
    global _clean_df_cache
    if _clean_df_cache is None:
        _clean_df_cache = pc.load("selection")[0]
    return _clean_df_cache

def lookup_real_conv_time(run_name):
    """Real (measured) convergence time (s) for a run, recomputed from its own H5 time series via
    step36's plateau criterion. Re-reads the ONE H5 file (metadata lookup is cached)."""
    df = _clean_df()
    row = df[df.filename == run_name]
    if row.empty:
        return None
    out = read_ts(row.iloc[0].filepath)
    if out is None:
        return None
    A, amx, w, tsec = out
    t, _, status = s36.convergence_time_v2(A, amx, tsec)
    return float(t) if status == "ok" else None

def predict_conv_time(Q):
    """Flow-based predicted convergence time (s): loads `data/convergence_model.pkl` (step 36/38)
    if present, else builds a fast RF fallback once (via step38's `_build_convergence`)."""
    import joblib
    cpkl = pc.DATA_DIR / "convergence_model.pkl"
    Cconv = joblib.load(cpkl) if cpkl.exists() else pc.missing_convergence_model()   # [vendored] no silent RF fallback
    return float(np.expm1(Cconv["reg"].predict(np.atleast_2d(np.asarray(Q, float)))[0]))

def fig_growth_compare(models_by_name, mu, comps, grid, Q, real=None, feat=sp.f_clr,
                       t_measured=None, t_predicted=None):
    """Side-by-side predicted growth movies for the SAME recipe — one panel per model in
    `models_by_name` (e.g. {'GP-Matern52': [...], 'ExtraTrees': [...]}) — for a direct visual
    comparison of how each model family reconstructs the growth. Measured frames (if given) are
    overlaid as dotted lines, subsampled, on every panel. `t_measured`/`t_predicted` (seconds, from
    `lookup_real_conv_time`/`predict_conv_time`) are shown in the figure title alongside each
    panel's own "steps to convergence" (its internal stopping heuristic, not a real time)."""
    names = list(models_by_name); n = len(names)
    fig, axes = plt.subplots(1, n, figsize=(6.4 * n, 4.6), squeeze=False); axes = axes[0]
    for a, name in zip(axes, names):
        specs = predict_growth(models_by_name[name], mu, comps, Q, feat)
        for k, s in enumerate(specs):
            a.plot(grid, s, color=plt.cm.plasma(k / max(len(specs) - 1, 1)), lw=1.5)
        if real is not None:
            rsel = np.linspace(0, len(real) - 1, min(10, len(real))).astype(int)
            for k in rsel:
                a.plot(grid, real[k], ":", color=plt.cm.plasma(k / max(len(real) - 1, 1)),
                      lw=1, alpha=.8)
        a.set_xlim(grid.min(), grid.max()); a.axhline(0, color="0.8", lw=.6)
        a.set_xlabel("Wavelength (nm)"); a.set_ylabel("Extinction (a.u.)")
        a.set_title(f"{name}  ({len(specs)} steps to convergence)", fontsize=10)
    q = "  ".join(f"{c}={v:.1f}" for c, v in zip(sp.FEATURES, np.ravel(Q)))
    title = q
    parts = []
    if t_measured is not None: parts.append(f"measured t_conv = {t_measured:.1f} s")
    if t_predicted is not None: parts.append(f"flow-model predicted t_conv = {t_predicted:.1f} s")
    if parts:
        title += "\n" + "   |   ".join(parts)
    fig.suptitle(title, fontsize=9)
    fig.tight_layout(); return fig

# ── quantitative top-k model comparison on defined recipes ────────────────────
def top_models(bench, k=4):
    """The k model names with the lowest MEAN spectral RMSE across the benchmark indices, plus the
    full ranking Series. Feeds the head-to-head comparison so it uses the actual data-driven best."""
    ranking = bench.groupby("model")["rmse"].mean().sort_values()
    return list(ranking.index[:k]), ranking

def trajectory_rmse(models, mu, comps, Q, real, feat=sp.f_clr):
    """Full-trajectory spectral RMSE for ONE recipe: predict the spectrum at each real acquisition
    index with that index's model (forward-filled past the fitted cap) and compare to the measured
    frame. Honest per-recipe number (uses the fitted per-index models directly, not predict_growth's
    early-stop heuristic).

    Computes each DISTINCT per-index prediction ONCE: frames beyond the fitted index cap all reuse
    the SAME last model on the SAME Q (forward-fill), so re-predicting that identical point once per
    frame — up to ~244 times for the longest runs — was pure waste and the main reason this felt like
    it hung (up to ~172 redundant predict calls per recipe per model beyond a ~72-index cap)."""
    Qraw = np.atleast_2d(np.asarray(Q, float))
    n_frames = len(real); n_models = len(models)
    idx_per_frame = np.minimum(np.arange(n_frames), n_models - 1)
    cache = {}
    errs = np.empty(n_frames)
    for n in range(n_frames):
        mi = int(idx_per_frame[n])
        if mi not in cache:
            cache[mi] = mu + models[mi].predict(Qraw)[0] @ comps
        errs[n] = np.sqrt(np.mean((real[n] - cache[mi]) ** 2))
    return float(errs.mean())

def resolve_recipes(runs_sub, defined=None, n=4, mode="longest"):
    """Row indices (into `runs_sub`) of the recipes to compare. `defined` = explicit run names
    (the 'recettes définies'); otherwise pick `n` via `pick_growth_runs` (default: longest = most
    informative movies)."""
    if defined:
        idx = [int(runs_sub.index[runs_sub.run == name][0]) for name in defined
               if (runs_sub.run == name).any()]
        missing = [name for name in defined if not (runs_sub.run == name).any()]
        if missing:
            print(f"  not in this (held-out) subset, skipped: {missing}")
        if idx:
            return idx
        print("  no defined recipe matched -> falling back to auto-pick")
    return pick_growth_runs(runs_sub, n=n, mode=mode)

def compare_table(models_by_name, mu, comps, runs_sub, F, recipe_idx, feat=sp.f_clr):
    """Per-recipe full-trajectory RMSE for every model -> a DataFrame (recipe rows, model columns)
    with a final 'mean' row. The quantitative backbone of the visual comparison."""
    rows = []
    for i in recipe_idx:
        r = runs_sub.loc[i]; Q = r[list(sp.FEATURES)].values.astype(float)
        real = F[int(r.offset):int(r.offset) + int(r.n_frames)]
        d = {"run": r.run}
        d.update({name: trajectory_rmse(mdl, mu, comps, Q, real, feat)
                  for name, mdl in models_by_name.items()})
        rows.append(d)
    df = pd.DataFrame(rows).set_index("run")
    df.loc["MEAN"] = df.mean()
    return df

def fig_compare_grid(models_by_name, mu, comps, grid, runs_sub, F, recipe_idx, feat=sp.f_clr,
                     show_conv_time=True, table=None):
    """Grid: one ROW per recipe (from `runs_sub`), one COLUMN per model. Each cell = predicted vs
    measured growth, INDEX-ALIGNED (predicted at the same acquisition indices as the measured frames,
    one colour scale) + that model's per-recipe full-trajectory RMSE. This is the BENCHMARK, so the
    endpoint is NOT anchored — it compares the RAW model families head-to-head. Row label carries the
    run name and (optionally) measured vs flow-predicted convergence time. Pass `table` (from
    `compare_table`) to REUSE its already-computed RMSE values instead of recomputing them."""
    names = list(models_by_name); nc = len(names); nr = len(recipe_idx)
    fig, axes = plt.subplots(nr, nc, figsize=(3.9 * nc, 3.0 * nr), squeeze=False)
    for ri, i in enumerate(recipe_idx):
        r = runs_sub.loc[i]; Q = r[list(sp.FEATURES)].values.astype(float)
        Qraw = np.atleast_2d(Q)
        real = F[int(r.offset):int(r.offset) + int(r.n_frames)]
        rsel = np.unique(np.linspace(0, len(real) - 1, min(8, len(real))).astype(int))
        row_lbl = r.run
        if show_conv_time:
            tm = lookup_real_conv_time(r.run); tp = predict_conv_time(Q)
            row_lbl += (f"\nt_conv meas={tm:.0f}s" if tm is not None else "\nt_conv meas=n/a") + \
                       f" pred={tp:.0f}s"
        for ci, name in enumerate(names):
            a = axes[ri, ci]; mdl = models_by_name[name]
            for k in rsel:                                    # index-aligned: same k for both
                col = plt.cm.plasma(k / max(len(real) - 1, 1))
                pred = mu + mdl[min(k, len(mdl) - 1)].predict(Qraw)[0] @ comps
                a.plot(grid, pred, color=col, lw=1.1)
                a.plot(grid, real[k], ":", color=col, lw=0.7, alpha=.7)
            rmse = table.loc[r.run, name] if table is not None else \
                trajectory_rmse(mdl, mu, comps, Q, real, feat)
            a.set_xlim(grid.min(), grid.max()); a.tick_params(labelsize=6); a.axhline(0, color="0.85", lw=.5)
            a.set_title((f"{name}\n" if ri == 0 else "") + f"RMSE={rmse:.4f}", fontsize=8)
            if ci == 0:
                a.set_ylabel(row_lbl + "\nExtinction (a.u.)", fontsize=7)
            if ri == nr - 1:
                a.set_xlabel("Wavelength (nm)", fontsize=7)
    fig.suptitle("Top-model comparison on held-out recipes  (index-aligned: solid = predicted, "
                 "dotted = measured at the SAME acquisition index; raw models, endpoint NOT anchored)",
                 fontsize=9)
    fig.tight_layout(rect=[0, 0, 1, 0.98]); return fig

def main():
    meta, S_traj, grid = extract()
    Sn, mu, comps, evr, scores = fit_basis(S_traj)
    print(f"{len(meta)} frames / {meta.run.nunique()} runs | PCA({N_PC})={100*evr.sum():.0f}% "
          f"| median frames/run={int(meta.groupby('run').size().median())}")
    savefig(fig_trajectories(meta, scores), "fig39_trajectories")
    # NOTE: model training below is heavy; run it explicitly in the notebook.
    # rmse, _ = cv_trajectory(meta, Sn, scores, mu, comps, grid)
    # print(f"leave-session-out trajectory spectral RMSE = {rmse:.4f}")
    print("Done (extraction + trajectory figure).")

if __name__ == "__main__":
    main()


### `ml.step40_growth_prediction`
*Original analysis code.*

In [ ]:
%%module toofast_analysis.ml.step40_growth_prediction
# [vendored] verbatim copy of 2FAST_ML/pipeline/step40_growth_prediction.py (md5 963c1b796e254264ccd275ed0d07207a), generated by tools/vendor_ml.py — do not edit by hand.
"""
step40_growth_prediction.py  —  clean growth-PREDICTION engine (mirrors 40_growth_prediction.ipynb)
===================================================================================================
The ONE prediction strategy we converged on:

  • at a few PROGRESS checkpoints τ = t/t_conv (0/0.2/0.4/0.6/0.8/1.0), one INDEPENDENT GP-Matern52
    predicts the spectrum RELATIVE TO THE FINAL (frame / the run's own final max), on a PCA basis of
    relative frames. ONE joint target (shape and growth together) — every band grows MONOTONICALLY.
  • reconstruction = interpolate those relative checkpoints in τ, then map τ→seconds via t_conv(Q). Final
    peak 1 (≈ nb33); the growth rate g(τ) is the relative spectrum's max — a smooth SIGMOID.

Two artifacts drove this design: (1) a peak-normalised shape × scalar growth rate made blue bands
rise-then-fall (fixed by the relative-to-final joint target); (2) checkpoints in fixed SECONDS made the
growth RATE stall then jump, because a given second is a different growth fraction for fast vs slow
recipes (fixed by PROGRESS-τ checkpoints).

Settled by measurement, not assumption (details at `N_PC_REL` and `GrowthPredictor._lmax`):
  • BASIS SIZE 6, NOT 4. Held-out accuracy improves monotonically with the number of components, and 4
    is the worst setting for the late-growth λmax error. Four is right for the max-NORMALISED endpoint
    spectra (nb32/33, `pc.N_PC`, the BO cosine cost); these targets are relative-to-final frames and
    carry the growth AMPLITUDE too, which needs more directions.
  • NO SMOOTHING OF THE TRAINING SPECTRA. Every Savitzky-Golay window makes the held-out spectral RMSE
    worse. The PCA truncation already IS the smoother — the basis is fit on every frame of every
    training run, so per-frame noise is orthogonal to the leading components and the projection
    discards it. The knob exists (`SG_SMOOTH`, `smooth_frames`) and defaults to off.
  • THE λmax READER IS GATED. An interior maximum inside 380-800 nm on a frame that has reached 30 % of
    its own final height, else NaN and back-filled. A bare argmax returned the window edge on the
    low-amplitude early checkpoints and `maximum.accumulate` then locked the whole λ(τ) curve there,
    warping one showcase recipe by a bogus 313 nm.
  • THE HONEST λmax NUMBER IS ~68 nm RMSE / 17 nm MEDIAN on unseen sessions (`lmax_table_loso`,
    `fig_lmax_parity`) — the same ceiling as nb33's dedicated endpoint model (62.7 nm grouped CV). The
    interactive tool draws with the all-data deployment model, where it looks like 9 nm; that number
    does not transfer to a novel composition.

Thin layer over the validated `step39_growth_trajectory`. Kernel: Anaconda 3.11.
"""
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from scipy.interpolate import interp1d
from . import pipeline_common as pc
from . import step39_growth_trajectory as s39
sp = pc.sp
from .spectral_pipeline import f_clr, MODELS

SLICES = (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)   # checkpoints in PROGRESS τ = t/t_conv (τ=1 = converged frame).
                                          #   Progress, not seconds: a fixed second means different growth
                                          #   fractions for fast vs slow recipes, which made the growth
                                          #   RATE stall then jump; τ-checkpoints give a smooth sigmoid.
Q_TOTAL = float(sp.Q_TOTAL)            # recipes constrained to sum to 14 µL/s
CADENCE = s39.SEC_PER_FRAME            # ~1.05 s / acquisition
NOTEBOOK = "40_growth_prediction.ipynb"

# ── COLOUR MAPS of the growth reconstructions — exposed so figures can be retuned in one place.
#   Every reconstruction panel encodes ELAPSED TIME as colour: the predicted spectra ramp along
#   CMAP_PRED, the measured ones along CMAP_MEAS (a light->dark grey, so the two families stay
#   distinguishable in print and for colour-blind readers), and the per-τ A-D spectra of §7 use
#   CMAP_TAU. Set e.g. `s40.CMAP_PRED = "viridis"` before drawing to change every panel at once.
CMAP_PRED = "plasma"        # predicted spectra, coloured by elapsed time
CMAP_MEAS = "gray_r"        # measured spectra, coloured by elapsed time
CMAP_TAU  = "viridis"       # the A-D spectra extracted at each progress τ (§7)
MEAS_LO, MEAS_HI = 0.35, 0.95   # grey range used on CMAP_MEAS: avoids invisible near-white lines
PRED_ALPHA_LO, PRED_ALPHA_HI = 0.35, 0.95   # predicted opacity ramps with time

def _cm(name):
    """Resolve a colormap NAME (or pass a Colormap through) — so the knobs accept either."""
    return plt.get_cmap(name) if isinstance(name, str) else name

def savefig(fig, name):
    fig.text(0.995, 0.003, f"source: {NOTEBOOK}", ha="right", va="bottom", fontsize=6, color="0.55")
    for ext in ("png", "pdf"):
        fig.savefig(pc.FIG_DIR / f"{name}.{ext}", dpi=200, bbox_inches="tight")
    print(f"  saved {name}.png/.pdf")

def load():
    """(runs, F, grid) — the per-run non-normalised growth frames (from step39's cache)."""
    return s39.build_sequences()

# ══════════════════════════════════════════════════════════════════════════════
# MODEL: per-checkpoint GP predicting the spectrum RELATIVE TO THE FINAL  (frame / run's own final max)
#   One JOINT target per checkpoint (shape AND growth fraction together) instead of a peak-normalised
#   shape times a separate scalar growth rate. Why: the separated form re-normalises each intermediate
#   frame to peak 1 and multiplies by a scalar g(τ); because the intermediate shape's blue shoulder is
#   relatively high (peak-normalised) while g saturates, a band like 420 nm would RISE THEN FALL —
#   unphysical (the data grow monotonically per band). Predicting the relative-to-final spectrum
#   directly keeps every band monotone and needs no re-normalisation. The final checkpoint target is
#   final/final_max = the max-normalised final ≈ nb33. Movie is capped at the predicted t_conv.
# ══════════════════════════════════════════════════════════════════════════════
# ── basis size + input smoothing: both settled by the ablation in `ablate_basis` ───────────────
# 5 grouped folds by session, error measured against the RAW measured frames (so smoothing cannot
# cheat). Spectral RMSE on the relative-to-final target:
#
#     savgol window \ n_pc     3       4       5       6       8
#              none        0.1984  0.1959  0.1951  0.1950  0.1934
#                 9        0.1986  0.1963  0.1954  0.1953  0.1939
#                15        0.1989  0.1966  0.1957  0.1955  0.1942
#                31        0.1993  0.1971  0.1962  0.1959  0.1946
#
# TWO CONCLUSIONS, both mildly counter-intuitive:
#  (1) DROPPING TO 4 COMPONENTS WOULD HURT. Accuracy improves monotonically with the basis size, and
#      n_pc=4 is also the WORST setting for the late-growth λmax error (61.4 nm vs 59.3 at 6). Four
#      components is the right number for the max-NORMALISED endpoint spectra (nb32/33, `pc.N_PC`, and
#      the BO cosine cost) — but these targets are relative-to-final frames, which carry the AMPLITUDE
#      of the growth as well as the shape, and that needs more directions. n_pc=8 is very slightly
#      better again (-0.0016 RMSE, -0.4 nm λmax) for 12 more GPs per fit; 6 is the plateau, so 6 stays.
#  (2) SMOOTHING THE TRAINING SPECTRA DOES NOT HELP — every window makes the spectral RMSE worse,
#      monotonically in the window length, and buys ~1 nm of λmax (inside the noise of a descriptor
#      whose own measured trajectory jumps >40 nm in 30 % of runs). The reason is that THE PCA
#      TRUNCATION IS ALREADY THE SMOOTHER: the basis is fit on every frame of every training run, so
#      per-frame noise is orthogonal to the first few components and is discarded by the projection.
#      Savitzky-Golay in front of it only blurs the genuinely narrow features too.
N_PC_REL = 6                                     # relative frames carry amplitude too -> a few more PCs

def _frame_at_tau(F, offset, n_frames, tau):
    """The run's REAL frame at progress fraction τ∈[0,1] (τ=1 = final). No forward-fill needed."""
    return F[int(offset) + int(round(np.clip(tau, 0, 1) * (int(n_frames) - 1)))]

LMAX_WIN = (380.0, 800.0)      # plasmon window for the peak-position (λmax) tracking. Set from the
                               #   DATA: the pipeline's own λmax label spans 379-798 nm over these 145
                               #   runs (3 above 760). The old upper bound of 760 clipped those 3 and,
                               #   with no interior-maximum test, pinned λ exactly ON the 760 edge for
                               #   any spectrum still rising there; opening it to 900+ swings the other
                               #   way and hands λ to the broad ~830 nm NIR shoulder that the PCA
                               #   reconstruction shows on the low-amplitude early checkpoints.
LMAX_MIN_SHIFT = 20.0          # only peak-align (warp) recipes whose λmax red-shifts by ≥ this (nm)
LMAX_H_MIN = 0.30              # λ gate on the relative-to-final scale (that run's final peak = 1), so
                               #   this reads "trust a checkpoint's peak POSITION only once the frame
                               #   has reached 30 % of the final peak HEIGHT". Below it the predicted
                               #   spectrum is a low-amplitude PCA reconstruction whose tallest point
                               #   wanders into the NIR (~776-832 nm); those checkpoints are marked NaN
                               #   and back-filled, so early frames are simply not warped rather than
                               #   translated by a meaningless amount.

class GrowthPredictor:
    """`mu`,`comps` = PCA basis of relative-to-final frames; `ckpt` = {τ: SpectralModel} GP per PROGRESS
    checkpoint predicting the relative-to-final spectrum; `slices` are τ fractions; `grid` the wl axis.
    Reconstruction = interpolate the relative checkpoints in τ, then map τ→seconds via t_conv. With
    `warp=True` the interpolation is done in a PEAK-ALIGNED frame using the predicted λmax(τ) growth, so
    the plasmon peak TRANSLATES smoothly (seed→NP red-shift) instead of two fixed peaks cross-fading."""
    def __init__(self, mu, comps, ckpt, grid, slices=SLICES, feat=f_clr):
        self.mu = mu; self.comps = comps; self.ckpt = ckpt; self.grid = grid
        self.slices = tuple(sorted(slices)); self.feat = feat

    def _rel_checkpoints(self, Q):
        S = np.array([self.mu + self.ckpt[t].predict(np.atleast_2d(Q))[0] @ self.comps for t in self.slices])
        return np.array(self.slices), S

    def _lmax(self, spec):
        """λmax (nm) = the position of the tallest point of the smoothed spectrum inside the plasmon
        window — but NaN unless that point is an INTERIOR maximum and clears `LMAX_H_MIN`.

        Both gates matter, and the missing edge gate was a real bug. A bare `argmax` inside a window
        returns the window EDGE whenever the spectrum has no interior maximum there, which is exactly
        the case for the early (τ≈0) checkpoints: the relative-to-final spectrum is then a
        low-amplitude curve with a monotone tail. With the old (380, 760) window that put λmax(τ=0) at
        759.9 nm for 13/145 recipes, and `maximum.accumulate` in `lambda_max_curve` LOCKED the whole
        λ(τ) curve at that bogus value — printing nonsense in §2 and, worse, driving the peak-aligned
        interpolation with a garbage shift (up to 348 nm) so the reconstructed movie's peak really was
        displaced. Returning NaN lets `lambda_max_curve` discard the reading instead of trusting it.

        Tallest rather than most-prominent on purpose: the warp aligns the peak you actually SEE, and a
        prominence ranking hands that role to a broad NIR shoulder near 830 nm on the PCA-reconstructed
        early checkpoints."""
        from scipy.signal import savgol_filter
        w = (self.grid >= LMAX_WIN[0]) & (self.grid <= LMAX_WIN[1])
        sm = savgol_filter(spec[w], 15, 3)
        j = int(np.argmax(sm))
        if j == 0 or j == len(sm) - 1 or sm[j] < LMAX_H_MIN:
            return np.nan                             # window edge / not enough material yet
        return float(self.grid[w][j])

    def lambda_max_curve(self, Q):
        """(τ_checkpoints, λmax at each) — the learned peak-position growth for a recipe. Checkpoints
        with no real peak (NaN, typically the earliest ones before there is enough material) are filled
        from the first trustworthy reading, and the curve is then monotone-accumulated (seed→NP
        red-shift). Returns λ = all-NaN when the recipe never shows a peak, so callers can skip the
        peak-aligned interpolation rather than warp by a meaningless amount."""
        tau_c, S = self._rel_checkpoints(Q)
        return tau_c, self._lam_curve(S)

    def _lam_curve(self, S):
        """The gated, filled, monotone λ(τ) curve for a stack of checkpoint spectra (see
        `lambda_max_curve`); all-NaN when not a single checkpoint has a real peak."""
        lam = np.array([self._lmax(s) for s in S], float)
        ok = np.isfinite(lam)
        if not ok.any():
            return np.full(len(S), np.nan)
        first = int(np.argmax(ok))
        lam[:first] = lam[first]                      # back-fill the peak-less early checkpoints
        for i in range(first + 1, len(lam)):          # forward-fill any interior gap
            if not np.isfinite(lam[i]):
                lam[i] = lam[i - 1]
        return np.maximum.accumulate(lam)

    def _shift(self, spec, delta):
        return np.interp(self.grid - delta, self.grid, spec, left=spec[0], right=spec[-1])

    def predict(self, Q, t_conv, cadence=CADENCE, warp=True):
        """(times_s, specs) — relative-to-final movie on the acquisition cadence, τ=0..1 ↔ 0..t_conv,
        final peak 1 (≈ nb33). `warp=True` (default): interpolate adjacent checkpoints in a PEAK-ALIGNED
        frame driven by the smooth λmax(τ) growth, so the plasmon peak translates monotonically to the
        red instead of a spurious secondary peak cross-fading in. Auto-disabled for recipes whose
        checkpoints barely red-shift (< LMAX_MIN_SHIFT nm)."""
        tau_c, S = self._rel_checkpoints(Q)
        t_grid = np.arange(0.0, max(t_conv, cadence) + cadence, cadence)
        tq = np.clip(t_grid / max(t_conv, 1e-9), 0, 1)
        lam_c = self._lam_curve(S)
        if not warp or not np.isfinite(lam_c).all() or (lam_c[-1] - lam_c[0]) < LMAX_MIN_SHIFT:
            movie = interp1d(tau_c, S, axis=0, bounds_error=False, fill_value=(S[0], S[-1]))(tq)
        else:
            from scipy.interpolate import PchipInterpolator
            lam_smooth = PchipInterpolator(tau_c, lam_c)
            movie = np.empty((len(tq), S.shape[1]))
            for i, tt in enumerate(tq):
                j = int(np.clip(np.searchsorted(tau_c, tt), 1, len(tau_c) - 1))
                a = (tt - tau_c[j - 1]) / (tau_c[j] - tau_c[j - 1] + 1e-9); lt = float(lam_smooth(tt))
                movie[i] = ((1 - a) * self._shift(S[j - 1], lt - lam_c[j - 1])
                            + a * self._shift(S[j], lt - lam_c[j]))
        return t_grid, movie / max(float(movie[-1].max()), 1e-9)

    def growth_rate(self, Q, tau, t_conv=None):
        """g(τ) = predicted overall extinction fraction (relative spectrum's max) at progress τ∈[0,1]."""
        tau_c, S = self._rel_checkpoints(Q)
        M = interp1d(tau_c, S, axis=0, bounds_error=False, fill_value=(S[0], S[-1]))(
            np.atleast_1d(np.asarray(tau, float)))
        return np.clip(M.max(1), 0, None)

SG_SMOOTH = None               # Savitzky-Golay window applied to the TRAINING frames, or None. Kept as
                               #   a knob only because it is the obvious thing to try; the ablation
                               #   above says leave it off (the PCA truncation already denoises).

def smooth_frames(F, window=SG_SMOOTH, poly=3):
    """Savitzky-Golay along the wavelength axis, or `F` untouched when `window` is None."""
    if not window:
        return F
    from scipy.signal import savgol_filter
    return savgol_filter(F, int(window), poly, axis=1)


def _fit_relative_basis(runs_tr, F, n_pc=N_PC_REL):
    """PCA basis of the relative-to-final frames (each run's frames / its own final max)."""
    fm = {r.run: max(float(F[int(r.offset) + int(r.n_frames) - 1].max()), 1e-9) for r in runs_tr.itertuples()}
    blocks = [F[int(r.offset):int(r.offset) + int(r.n_frames)] / fm[r.run] for r in runs_tr.itertuples()]
    Rel = np.vstack(blocks); mu = Rel.mean(0)
    _, _, Vt = np.linalg.svd(Rel - mu, full_matrices=False)
    return mu, Vt[:n_pc], fm

def _grid():
    return s39.build_sequences()[2]                     # cached wavelength axis

def fit_predictor(runs_tr, F, slices=SLICES, shape_model="GP-Matern52", feat=f_clr, n_pc=N_PC_REL,
                  grid=None):
    """Fit a `GrowthPredictor`: PCA on relative-to-final frames + one GP per PROGRESS checkpoint τ on the
    relative target `frame_at(τ) / run_final_max` (frame_at(τ) is the run's real frame at that progress).
    `grid` (the wl axis) is stored for the λmax peak-alignment; loaded from cache if not given."""
    mu, comps, fm = _fit_relative_basis(runs_tr, F, n_pc)
    Q = runs_tr[list(sp.FEATURES)].values.astype(float)
    ckpt = {}
    for tau in slices:
        rel = np.array([_frame_at_tau(F, r.offset, r.n_frames, tau) / fm[r.run] for r in runs_tr.itertuples()])
        ckpt[tau] = sp.SpectralModel(n_pc, feat, MODELS[shape_model]).fit(Q, (rel - mu) @ comps.T)
    return GrowthPredictor(mu, comps, ckpt, grid if grid is not None else _grid(), slices, feat)

def fit_deploy(runs, F, slices=SLICES, shape_model="GP-Matern52"):
    """The DEPLOYMENT predictor, fit on EVERY run (novel recipes / the tool). For honest per-recipe
    numbers on KNOWN runs, use `fig_known_recipes` (leave-one-session-out). Cached on disk."""
    import joblib, hashlib, time
    key = hashlib.sha256(("v2warp" + str(slices) + shape_model + str(F.shape) +
                          str(runs.run.tolist())).encode()).hexdigest()[:12]
    path = pc.DATA_DIR / f"growth40_deploy_{key}.pkl"
    if path.exists():
        print(f"  loaded deploy predictor from cache ({path.name})"); return joblib.load(path)
    t0 = time.time(); m = fit_predictor(runs, F, slices, shape_model)
    joblib.dump(m, path); print(f"  fitted deploy predictor  {time.time()-t0:5.1f}s -> cached"); return m

def growth_rate_table(model, Q, t_conv=None):
    """Growth rate g(τ) = relative-spectrum max at progress τ, on a clean τ grid + power-law exponent p
    (g ≈ τ^p; p<1 fast-then-saturating). Monotone sigmoid by construction (τ-indexed checkpoints)."""
    tg = np.array([0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 1.0]); g = model.growth_rate(Q, tg)
    ok = (tg > 0) & (tg < 1) & (g > 1e-3)
    p = float(np.polyfit(np.log(tg[ok]), np.log(g[ok]), 1)[0]) if ok.sum() >= 2 else np.nan
    return pd.DataFrame({"tau": tg, "growth_fraction_g": np.round(g, 3)}), p

# ── convergence-time prediction error statistics ──────────────────────────────
def conv_time_table(runs):
    """Per-run measured vs flow-model predicted convergence time (s), for every run with a measured
    t_conv. Columns: run, session, t_meas, t_pred, err(=pred-meas), abs_err."""
    rows = []
    for r in runs.itertuples():
        tm = s39.lookup_real_conv_time(r.run)
        if tm is None:
            continue
        Q = np.array([r.Q1, r.Q2, r.Q3, r.Q4, r.Q5], float)
        tp = s39.predict_conv_time(Q)
        rows.append(dict(run=r.run, session=r.session, t_meas=tm, t_pred=tp,
                         err=tp - tm, abs_err=abs(tp - tm)))
    return pd.DataFrame(rows)

def conv_time_stats(df):
    e = df.err.values; ae = df.abs_err.values; tm = df.t_meas.values; tp = df.t_pred.values
    big = tm >= 2.0
    return dict(n=len(df), MAE=float(ae.mean()), medAE=float(np.median(ae)),
                RMSE=float(np.sqrt(np.mean(e ** 2))), bias=float(e.mean()),
                R2=float(1 - np.sum(e ** 2) / np.sum((tm - tm.mean()) ** 2)),
                corr=float(np.corrcoef(tm, tp)[0, 1]),
                med_rel_err=float(np.median(ae[big] / tm[big])) if big.any() else np.nan)

REAGENTS = {"Q1": "AgNO3", "Q2": "PVP", "Q3": "acetic acid", "Q4": "ascorbic acid", "Q5": "NaBH4"}

def shap_conv_time(nbg=60, seed=None):
    """SHAP analysis of the deployed convergence-time model (`convergence_model.pkl`, predicting
    log1p(t_conv) from the 5 raw flows). Returns (shap_values, X, labels, importance_df, explainer).
    SHAP values are in the model's log1p(t_conv) output space (additive).

    Explainer is chosen from the model, NOT hard-coded: `TreeExplainer` (exact, fast) for a tree
    ensemble, otherwise `KernelExplainer` on a k-means background. This used to assume a
    RandomForest and broke the moment step 36 deployed the benchmark winner (a GP inside a
    `Pipeline`, which TreeExplainer rejects). With only 5 features the kernel path enumerates all
    2^5 coalitions, so it is exact too — just slower (~1 min)."""
    import joblib, shap
    from . import step36_convergence_model as s36
    blob = joblib.load(pc.DATA_DIR / "convergence_model.pkl")
    reg = blob["reg"]
    ok = s36.load_conv()
    ok = ok[ok.converged].reset_index(drop=True)
    X = ok[list(sp.FEATURES)].values.astype(float)
    labels = [f"{f}  {REAGENTS[f]}" for f in sp.FEATURES]

    est = reg[-1] if hasattr(reg, "steps") else reg          # unwrap a Pipeline for the type test
    is_tree = est.__class__.__name__ in (
        "RandomForestRegressor", "ExtraTreesRegressor", "GradientBoostingRegressor",
        "DecisionTreeRegressor", "XGBRegressor", "LGBMRegressor")
    if is_tree and not hasattr(reg, "steps"):
        ex = shap.TreeExplainer(reg); sv = ex.shap_values(X)
    else:
        print(f"  {blob.get('reg_name', est.__class__.__name__)} is not a bare tree ensemble "
              f"-> KernelExplainer on {nbg} background points (exact for 5 features, ~1 min)")
        bg = shap.kmeans(X, min(nbg, len(X)))
        ex = shap.KernelExplainer(lambda A: reg.predict(np.asarray(A, float)), bg)
        sv = np.asarray(ex.shap_values(X, nsamples=2 ** X.shape[1], silent=True))
    mabs = np.abs(sv).mean(0)
    dirn = ["longer t_conv" if np.corrcoef(X[:, i], sv[:, i])[0, 1] > 0 else "shorter t_conv"
            for i in range(X.shape[1])]
    imp = pd.DataFrame({"flow": sp.FEATURES, "reagent": [REAGENTS[f] for f in sp.FEATURES],
                        "mean_abs_shap": np.round(mabs, 4),
                        "higher_flow_gives": dirn}).sort_values("mean_abs_shap", ascending=False)
    return sv, X, labels, imp, ex

def fig_shap_conv_time(sv, X, labels, imp):
    """Two panels: (left) mean|SHAP| importance bar; (right) beeswarm — each dot a recipe, coloured by
    the flow's value, x = its SHAP contribution to log(t_conv). Shows both HOW MUCH and IN WHICH
    DIRECTION each reagent drives the predicted convergence time."""
    import shap
    order = imp.flow.tolist()[::-1]                        # least->most important (bottom->top)
    idx = [list(sp.FEATURES).index(f) for f in order]
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
    # left: importance bar
    y = np.arange(len(order))
    ax[0].barh(y, imp.set_index("flow").loc[order, "mean_abs_shap"].values, color="#0072B2")
    ax[0].set_yticks(y); ax[0].set_yticklabels([f"{f}  {REAGENTS[f]}" for f in order], fontsize=11)
    ax[0].set_xlabel("mean |SHAP|  (impact on log $t_{conv}$)", fontsize=12)
    ax[0].set_title("Feature importance", fontsize=13); ax[0].tick_params(labelsize=10)
    # right: beeswarm on its own axes
    plt.sca(ax[1])
    shap.summary_plot(sv[:, idx], X[:, idx], feature_names=[labels[i] for i in idx],
                      show=False, plot_size=None, color_bar=True, sort=False)
    ax[1].set_xlabel("SHAP value  (→ longer predicted $t_{conv}$)", fontsize=12)
    ax[1].set_title("Direction & spread (beeswarm)", fontsize=13)
    name = _conv_model().get("reg_name", "deployed model")    # read from the pkl, never hard-coded
    fig.suptitle(f"SHAP analysis — convergence-time model ({name}, log1p $t_{{conv}}$ ~ 5 flows)",
                 fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.95]); return fig

def fig_shap_dependence(sv, X, cols=("Q5", "Q3")):
    """SHAP dependence: for each named flow, how its SHAP contribution to log(t_conv) varies with the
    flow value (the learned dose-response of that reagent on convergence time)."""
    fig, ax = plt.subplots(1, len(cols), figsize=(5.2 * len(cols), 4.2), squeeze=False); ax = ax[0]
    for a, c in zip(ax, cols):
        j = list(sp.FEATURES).index(c)
        a.scatter(X[:, j], sv[:, j], s=16, alpha=.6, edgecolor="k", lw=.3, color="#D55E00")
        a.axhline(0, color="0.6", lw=.8, ls="--")
        a.set_xlabel(f"{c}  {REAGENTS[c]}  (µL/s)", fontsize=12)
        a.set_ylabel("SHAP → log $t_{conv}$", fontsize=12)
        a.set_title(f"{REAGENTS[c]} dose–response", fontsize=12); a.tick_params(labelsize=10)
    fig.suptitle("SHAP dependence — how each reagent's effect on convergence time varies with dose",
                 fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.94]); return fig

def conv_time_table_loso(runs=None, k=None):
    """Per-run measured vs predicted t_conv with the run's whole SESSION held out — the honest
    protocol. `conv_time_table` uses the single globally-fit `convergence_model.pkl`, which
    step 36 trains on EVERY run, so its numbers (and nb40 §3/§8) are IN-SAMPLE.

    `k=None` means TRUE leave-one-session-out (k = number of sessions). Do not lower it to chase
    a nicer number: R2 in *seconds* is dominated by the slow tail and swings with the split
    (0.599 at k=3, 0.752 at k=5, 0.618 at k>=8), while log-R2 — the space the model is actually
    fit in — is stable at 0.73-0.76. Quote log-R2, or R2(s) at full leave-one-session-out."""
    from sklearn.model_selection import GroupKFold
    from . import step36_convergence_model as s36
    import joblib
    name = joblib.load(pc.DATA_DIR / "convergence_model.pkl").get("reg_name", "GP-Matern52")
    ok = s36.load_conv(); ok = ok[ok.converged].reset_index(drop=True)
    k = ok.session.nunique() if k is None else k
    if runs is not None:                       # restrict to a given run set, keep the full fit pool
        ok = ok[ok.filename.isin(set(runs.run))].reset_index(drop=True) if len(
            set(runs.run) & set(ok.filename)) else ok
    X = ok[list(sp.FEATURES)].values.astype(float)
    y = np.log1p(ok.conv_time_s.values); g = ok.session.values
    pred = np.full(len(ok), np.nan)
    for tr, te in GroupKFold(n_splits=min(k, len(np.unique(g)))).split(X, groups=g):
        pred[te] = s36.REGRESSORS[name]().fit(X[tr], y[tr]).predict(X[te])
    t_pred = np.expm1(pred)
    tm = ok.conv_time_s.values
    log_r2 = float(1 - np.sum((pred - y) ** 2) / np.sum((y - y.mean()) ** 2))
    out = pd.DataFrame(dict(run=ok.filename, session=ok.session, t_meas=tm, t_pred=t_pred,
                            err=t_pred - tm, abs_err=np.abs(t_pred - tm)))
    out.attrs["log_r2"] = log_r2; out.attrs["model"] = name; out.attrs["folds"] = k
    return out, name

def fig_conv_time(df, stats, protocol="in-sample (globally-fit model)"):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
    fig.suptitle(f"convergence time — {protocol}", fontsize=9, y=1.02,
                 color=("#B22222" if protocol.startswith("in-sample") else "0.25"))
    hi = max(df.t_meas.max(), df.t_pred.max()) * 1.05
    ax[0].scatter(df.t_meas, df.t_pred, s=18, alpha=.6, edgecolor="k", lw=.3)
    ax[0].plot([0, hi], [0, hi], "k--", lw=1, label="y = x")
    ax[0].set_xlim(0, hi); ax[0].set_ylim(0, hi); ax[0].set_aspect("equal")
    ax[0].set_xlabel("measured $t_{conv}$ (s)"); ax[0].set_ylabel("predicted $t_{conv}$ (s)")
    ax[0].set_title(f"parity — R²={stats['R2']:.2f}, MAE={stats['MAE']:.1f}s, "
                    f"median|err|={stats['medAE']:.1f}s"); ax[0].legend(fontsize=8)
    ax[1].hist(df.err, bins=25, color="#0072B2", alpha=.85)
    ax[1].axvline(0, color="k", lw=1)
    ax[1].axvline(stats["bias"], color="#D55E00", ls="--", lw=1.2, label=f"bias={stats['bias']:.1f}s")
    ax[1].set_xlabel("predicted − measured $t_{conv}$ (s)"); ax[1].set_ylabel("count")
    ax[1].set_title(f"error distribution — RMSE={stats['RMSE']:.1f}s"); ax[1].legend(fontsize=8)
    fig.tight_layout(); return fig

def fig_conv_time_log(df):
    """Publication (SI) parity of t_conv in log1p(t) — EXACTLY the space the GP is fit in, so the
    plotted distance to y = x is the model's own residual and log-R2 is read straight off the axes.
    (Earlier version used log(t) axes: same picture above ~5 s, distorted below.)

    On linear axes the five runs > 100 s carry ~2/3 of the squared error and squash everything else
    into a corner; in log1p a constant relative error (on 1 + t) is a constant distance from y = x.
    (a) parity in log1p, secondary axes in seconds. (b) histogram of the log1p residual
    log1p(pred) - log1p(meas); |residual| <= ln 2 means (1 + t) predicted within a factor 2.
    Runs with t_meas = 0 (converged on the FIRST frame) are still DROPPED (user's choice, kept for
    consistency with the earlier figure) although log1p would allow them.
    No title (publication style); stats are written inside the panels."""
    df = df[df.t_meas >= s39.SEC_PER_FRAME * 0.5]
    st = conv_time_stats(df); lr2 = _log_r2(df)
    ym, yp = np.log1p(df.t_meas.values), np.log1p(df.t_pred.values)
    C_F = "#0072B2"
    fig, ax = plt.subplots(1, 2, figsize=(10.5, 4.6))

    lo = min(ym.min(), yp.min()) - 0.2
    hi = max(ym.max(), yp.max()) + 0.2
    xx = np.array([lo, hi])
    ax[0].plot(xx, xx, "k--", lw=1, zorder=1, label="y = x")
    ax[0].scatter(ym, yp, s=20, alpha=.7, color=C_F, edgecolor="k", lw=.3, zorder=2)
    ax[0].set_xlim(lo, hi); ax[0].set_ylim(lo, hi); ax[0].set_aspect("equal")
    ax[0].set_xlabel("measured ln(1 + $t_{conv}$ / s)"); ax[0].set_ylabel("predicted ln(1 + $t_{conv}$ / s)")
    sec_ticks = [t for t in (1, 2, 5, 10, 20, 50, 100, 200) if lo <= np.log1p(t) <= hi]
    sx = ax[0].secondary_xaxis("top", functions=(np.expm1, np.log1p))
    sy = ax[0].secondary_yaxis("right", functions=(np.expm1, np.log1p))
    for sec, axis in ((sx, sx.xaxis), (sy, sy.yaxis)):
        sec.set_ticks(sec_ticks)
        axis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
        axis.set_minor_locator(plt.NullLocator())
        sec.tick_params(labelsize=7.5, colors="0.4")
    ax[0].text(1.0, 1.08, "s", transform=ax[0].transAxes, ha="left", fontsize=7.5, color="0.4")
    ax[0].legend(fontsize=8, loc="upper left")
    ax[0].text(0.97, 0.04, f"n = {st['n']}\nlog-R² = {lr2:.2f}\n"
                           f"RMSE = {np.sqrt(np.mean((yp - ym) ** 2)):.2f} (ln units)\n"
                           f"median |err| = {st['medAE']:.1f} s",
               transform=ax[0].transAxes, ha="right", va="bottom", fontsize=8.5)
    ax[0].text(-0.14, 1.02, "(a)", transform=ax[0].transAxes, fontsize=12, fontweight="bold")

    res = yp - ym
    in2 = np.mean(np.abs(res) <= np.log(2))
    ax[1].hist(res, bins=28, color=C_F, alpha=.85)
    ax[1].axvline(0, color="k", lw=1)
    med = np.median(res)
    ax[1].axvline(med, color="#D55E00", ls="--", lw=1.2, label=f"median residual = {med:+.2f}")
    ax[1].set_xlabel("ln(1 + $t_{pred}$ / s) − ln(1 + $t_{meas}$ / s)"); ax[1].set_ylabel("count")
    ax[1].legend(fontsize=8, loc="upper left")
    ax[1].text(0.97, 0.96, f"{100 * in2:.0f} % within |residual| ≤ ln 2\n(1 + t within a factor 2)",
               transform=ax[1].transAxes, ha="right", va="top", fontsize=8.5)
    ax[1].text(-0.12, 1.02, "(b)", transform=ax[1].transAxes, fontsize=12, fontweight="bold")
    fig.tight_layout(); return fig

# ── convergence time: does the model only fail on the SLOW tail? ───────────────
CONV_FAST_CUT = 50.0    # s — "reasonable" growth times; ~85 % of the runs (126/149) sit below it.

def _log_r2(d):
    """R2 in the log1p(t_conv) space the model is actually fit in, for any subset of a conv table."""
    y, p = np.log1p(d.t_meas.values), np.log1p(d.t_pred.values)
    return float(1 - np.sum((p - y) ** 2) / np.sum((y - y.mean()) ** 2))

def conv_time_branch_table(df, cut=CONV_FAST_CUT):
    """Error statistics of a conv table (from `conv_time_table_loso`) split at a measured-t_conv cut:
    whole set / fast branch (t_meas < cut) / slow tail (t_meas >= cut).

    READ R2 WITH CARE. It is normalised by the variance of the branch it is computed on, and the slow
    tail carries almost all the spread of t_conv — so restricting to the fast branch MECHANICALLY
    lowers R2 (0.62 -> 0.50 at cut=50 s) even though every error measured in seconds gets ~2x smaller.
    The scale-free number to compare across branches is `med_rel_err` (median |err| / t_meas)."""
    rows = []
    for label, m in (("all", np.ones(len(df), bool)),
                     (f"fast   t_meas < {cut:g} s", df.t_meas.values < cut),
                     (f"slow   t_meas >= {cut:g} s", df.t_meas.values >= cut)):
        d = df[m]
        if not len(d):
            continue
        st = conv_time_stats(d)
        rows.append(dict(branch=label, n=st["n"], R2=st["R2"], logR2=_log_r2(d), MAE=st["MAE"],
                         medAE=st["medAE"], RMSE=st["RMSE"], bias=st["bias"],
                         med_rel_err=st["med_rel_err"]))
    out = pd.DataFrame(rows)
    for c in ("R2", "logR2", "MAE", "medAE", "RMSE", "bias", "med_rel_err"):
        out[c] = out[c].round(3)
    return out

def fig_conv_time_cut(df, cut=CONV_FAST_CUT, protocol="leave-session-out (one fold per session)"):
    """Full-range parity vs the same model restricted to the 'reasonable' growth times (t_meas < cut).

    (a) every run, fast branch and slow tail coloured apart, cut drawn on both axes.
    (b) the SAME predictions zoomed to t_meas < cut — the parity plot to compare against (a).
    (c) signed relative error vs measured t_conv (log x): the diagnostic that settles whether the
        model is 'good for reasonable times and bad for long ones', or just proportionally wrong
        everywhere with the seconds growing along with t_conv."""
    fast, slow = df[df.t_meas < cut], df[df.t_meas >= cut]
    s_all, s_fast = conv_time_stats(df), conv_time_stats(fast)
    C_F, C_S = "#0072B2", "#D55E00"
    fig, ax = plt.subplots(1, 3, figsize=(15.5, 4.6))
    fig.suptitle(f"convergence time — {protocol}", fontsize=9, y=1.02, color="0.25")

    # (a) full range
    hi = max(df.t_meas.max(), df.t_pred.max()) * 1.05
    ax[0].plot([0, hi], [0, hi], "k--", lw=1, zorder=0, label="y = x")
    ax[0].scatter(fast.t_meas, fast.t_pred, s=20, alpha=.65, color=C_F, edgecolor="k", lw=.3,
                  label=f"t < {cut:g} s  (n={len(fast)})")
    ax[0].scatter(slow.t_meas, slow.t_pred, s=26, alpha=.75, color=C_S, edgecolor="k", lw=.3,
                  label=f"t ≥ {cut:g} s  (n={len(slow)})")
    ax[0].axvline(cut, color="0.6", lw=.8, ls=":"); ax[0].axhline(cut, color="0.6", lw=.8, ls=":")
    ax[0].set_xlim(0, hi); ax[0].set_ylim(0, hi); ax[0].set_aspect("equal")
    ax[0].set_xlabel("measured $t_{conv}$ (s)"); ax[0].set_ylabel("predicted $t_{conv}$ (s)")
    ax[0].set_title(f"(a) all runs — R²={s_all['R2']:.2f}, MAE={s_all['MAE']:.1f}s, "
                    f"med|err|={s_all['medAE']:.1f}s", fontsize=11)
    ax[0].legend(fontsize=8, loc="upper left")

    # (b) zoom on the fast branch — same predictions, restricted set
    ax[1].plot([0, cut], [0, cut], "k--", lw=1, zorder=0, label="y = x")
    ax[1].scatter(fast.t_meas, fast.t_pred, s=22, alpha=.7, color=C_F, edgecolor="k", lw=.3)
    off = fast[fast.t_pred > cut]                      # predicted slow although measured fast
    if len(off):
        ax[1].scatter(off.t_meas, np.full(len(off), cut * .985), s=40, marker="^", color=C_F,
                      edgecolor="k", lw=.4, label=f"predicted > {cut:g} s (n={len(off)}, clipped)")
    ax[1].set_xlim(0, cut); ax[1].set_ylim(0, cut); ax[1].set_aspect("equal")
    ax[1].set_xlabel("measured $t_{conv}$ (s)"); ax[1].set_ylabel("predicted $t_{conv}$ (s)")
    ax[1].set_title(f"(b) t_meas < {cut:g} s — R²={s_fast['R2']:.2f}, MAE={s_fast['MAE']:.1f}s, "
                    f"med|err|={s_fast['medAE']:.1f}s", fontsize=11)
    ax[1].legend(fontsize=8, loc="upper left")

    # (c) relative error — the scale-free comparison
    for d, c, lab in ((fast, C_F, "fast"), (slow, C_S, "slow")):
        ax[2].scatter(d.t_meas, 100 * d.err / d.t_meas, s=20, alpha=.7, color=c, edgecolor="k", lw=.3)
        if len(d):
            ax[2].hlines(100 * np.median(d.err / d.t_meas), d.t_meas.min(), d.t_meas.max(),
                         color=c, lw=2, label=f"{lab}: median rel. err "
                                              f"{100 * np.median(d.err / d.t_meas):+.0f} %, "
                                              f"median |rel| {100 * np.median(np.abs(d.err / d.t_meas)):.0f} %")
    ax[2].axhline(0, color="k", lw=.8); ax[2].axvline(cut, color="0.6", lw=.8, ls=":")
    ax[2].set_xscale("log"); ax[2].set_ylim(-110, 260)
    ax[2].set_xlabel("measured $t_{conv}$ (s)")
    ax[2].set_ylabel("relative error  (pred − meas) / meas  (%)")
    ax[2].set_title("(c) relative error vs speed", fontsize=11); ax[2].legend(fontsize=8, loc="upper right")
    fig.tight_layout(); return fig

# ── prediction figures ────────────────────────────────────────────────────────
def _draw_predict(ax, model, grid, Q, t_conv, real=None):
    t_grid, specs = model.predict(Q, t_conv)
    horizon = max(t_grid[-1], 1e-9)
    if real is not None and len(real):
        mf = max(float(real[-1].max()), 1e-9); tr = np.arange(len(real)) * CADENCE
        for k in range(len(real)):
            ax.plot(grid, real[k] / mf, ":", color=_cm(CMAP_PRED)(min(tr[k] / horizon, 1.0)),
                    lw=.6, alpha=.6, rasterized=True)
    for k in range(len(t_grid)):
        ax.plot(grid, specs[k], "-", color=_cm(CMAP_PRED)(min(t_grid[k] / horizon, 1.0)),
                lw=.9, alpha=.9, rasterized=True)
    ax.set_xlim(grid.min(), grid.max()); ax.axhline(0, color="0.85", lw=.5)
    ax.axhline(1.0, color="0.8", lw=.5, ls="--"); ax.tick_params(labelsize=7)
    smp = plt.cm.ScalarMappable(cmap=_cm(CMAP_PRED), norm=Normalize(0, horizon)); smp.set_array([])
    ax.figure.colorbar(smp, ax=ax, label="elapsed time (s)"); return horizon

def fig_predict_one(model, grid, Q, t_conv, title="", real=None):
    fig, ax = plt.subplots(figsize=(7.8, 4.9))
    _draw_predict(ax, model, grid, Q, t_conv, real)
    ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("Extinction, final-normalised")
    ax.set_title(title, fontsize=9); fig.tight_layout(); return fig

def _run_Q(runs, run):
    r = runs[runs.run == run].iloc[0]
    return np.array([r.Q1, r.Q2, r.Q3, r.Q4, r.Q5], float)

# ── per-panel annotation of the reconstruction figures: flows + λmax + spectral angle ──────────────
FLOW_LABELS = {"Q1": r"AgNO$_3$", "Q2": "PVP", "Q3": "AcOH", "Q4": "AA", "Q5": r"NaBH$_4$"}

def recipe_metrics(model, real, Q, taus=None):
    """Scores of ONE reconstruction, on the same scales as §9/§9b. λmax = the gated reader
    (`GrowthPredictor._lmax`) on the measured and predicted FINAL frames (both final-normalised, so
    the value printed is the one a reader sees on the plot). Spectral angle (deg) between predicted
    and measured at MATCHED progress τ: at τ = 1 (`ang_final`) and averaged over `KIN_TAUS`
    (`ang_mean`, the per-run version of fig40_kinetic_metrics). τ-matching makes the angles
    independent of the t_conv prediction, which is reported separately."""
    taus = KIN_TAUS if taus is None else np.asarray(taus, float)
    nf = len(real)
    rel = real / max(float(real[-1].max()), 1e-9)
    _, specs = model.predict(Q, (nf - 1) * CADENCE)
    k = np.clip((taus * (nf - 1)).round().astype(int), 0, nf - 1)
    ang = _sam(rel[k], specs[np.clip(k, 0, len(specs) - 1)])
    return dict(lam_meas=model._lmax(rel[-1]), lam_pred=model._lmax(specs[-1]),
                ang_final=float(ang[-1]), ang_mean=float(ang.mean()))

def _fmt_lam(x):
    return f"{x:.0f}" if np.isfinite(x) else "n/a"

def _annotate_panel(a, run, session, tm, tp, Q, met, fs_title, fs_info, title_color="k", bold=False,
                    show_session=True, show_run=True):
    """Title = run + held-out session + t_conv; just under it, a smaller block with the five flows,
    λmax measured | predicted and the spectral angles. Everything sits ABOVE the axes: a call-out
    inside the panel always lands on someone's red-shifted peak."""
    tm_s = f"{tm:.0f} s" if tm is not None else "n/a"
    flows = "  ".join(f"{FLOW_LABELS[c]} {q:.2f}" for c, q in zip(("Q1", "Q2", "Q3", "Q4", "Q5"), Q))
    info = (f"{flows}  µL/s\n"
            rf"$\lambda_{{max}}$ meas {_fmt_lam(met['lam_meas'])} | pred {_fmt_lam(met['lam_pred'])} nm"
            rf"    $\theta_{{\tau=1}}$ {met['ang_final']:.1f}° | $\langle\theta\rangle_\tau$ {met['ang_mean']:.1f}°")
    a.text(0.5, 1.015, info, transform=a.transAxes, ha="center", va="bottom", fontsize=fs_info,
           linespacing=1.35)
    head = (f"{run}   [{session} held out]\n" if show_session else f"{run}\n") if show_run else ""
    a.set_title(head +
                f"$t_{{conv}}$:  measured {tm_s}   |   predicted {tp:.0f} s",
                fontsize=fs_title, color=title_color, fontweight="bold" if bold else "normal",
                pad=2 * fs_info * 1.35 + 6)

def _metrics_row(run, session, Q, tm, tp, met):
    return dict(run=run, session=session, **dict(zip(("Q1", "Q2", "Q3", "Q4", "Q5"), np.round(Q, 3))),
                t_conv_meas=tm, t_conv_pred=tp, **met)

def _draw_growth_panel(a, grid, real, t_grid, specs, lw_meas=1.0, lw_pred=1.8):
    """One measured-vs-predicted growth panel, final-normalised. MEASURED frames: dark-grey gradient
    (light grey early -> near-black late, thin solid); PREDICTED: bright plasma gradient with opacity
    ramping up over time. Both encode elapsed time as a fraction of the predicted horizon (= the
    predicted t_conv), so the two families are easy to tell apart. Returns the horizon (s)."""
    mf = max(float(real[-1].max()), 1e-9); horizon = max(t_grid[-1], 1e-9)
    tr = np.arange(len(real)) * CADENCE
    for k in range(len(real)):
        f = min(tr[k] / horizon, 1.0)
        a.plot(grid, real[k] / mf, "-", color=_cm(CMAP_MEAS)(MEAS_LO + (MEAS_HI - MEAS_LO) * f),
               lw=lw_meas, alpha=.75, rasterized=True, zorder=1)
    for k in range(len(t_grid)):
        f = min(t_grid[k] / horizon, 1.0)
        a.plot(grid, specs[k], "-", color=_cm(CMAP_PRED)(f), lw=lw_pred,
               alpha=PRED_ALPHA_LO + (PRED_ALPHA_HI - PRED_ALPHA_LO) * f, rasterized=True, zorder=2)
    a.set_xlim(grid.min(), grid.max()); a.set_ylim(bottom=0.0)      # Ext starts at 0
    a.axhline(1.0, color="0.7", lw=.6, ls="--")
    return horizon

def fig_known_recipes(runs, F, grid, recipes, slices=SLICES, ncol=2, shape_model="GP-Matern52", feat=f_clr):
    """Final-normalised LOSO prediction for KNOWN runs (each run's whole session held out — no leak).
    PREDICTED spectra use a bright PLASMA gradient with growing opacity over time; MEASURED spectra a
    separate DARK-GREY gradient — so the two are easy to tell apart despite both encoding elapsed time.
    Extinction axis starts at 0; the measured vs predicted convergence time is called out in a box."""
    from matplotlib.lines import Line2D
    sess = runs.session.values; n = len(recipes); nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(6.4 * ncol, 4.7 * nrow), squeeze=False); axes = axes.ravel()
    cache = {}; rows = []
    def predictor_without(s):
        if s not in cache:
            cache[s] = fit_predictor(runs[sess != s].reset_index(drop=True), F, slices, shape_model, feat)
        return cache[s]
    for a, run in zip(axes, recipes):
        r = runs[runs.run == run].iloc[0]; s = r.session; Q = _run_Q(runs, run)
        model = predictor_without(s)
        real = F[int(r.offset):int(r.offset) + int(r.n_frames)]
        tm = s39.lookup_real_conv_time(run); tp = predict_conv_time_without(s, Q)
        t_grid, specs = model.predict(Q, tp)
        horizon = _draw_growth_panel(a, grid, real, t_grid, specs)
        a.tick_params(labelsize=12); a.set_xlabel("Wavelength (nm)", fontsize=13)
        a.set_ylabel("Extinction, final-normalised", fontsize=13)
        # convergence times, flows and scores go ABOVE the axes, not in a floating box: an
        # upper-right call-out sat exactly where a red-shifted recipe's predicted peak is and hid it
        # (S25-M02_init_4 peaks near 755 nm at 1.0, and the box made it look as if the prediction
        # topped out at 0.87).
        met = recipe_metrics(model, real, Q)
        _annotate_panel(a, run, s, tm, tp, Q, met, fs_title=13, fs_info=10.5, title_color="#b00",
                        bold=True)
        rows.append(_metrics_row(run, s, Q, tm, tp, met))
        sm = plt.cm.ScalarMappable(cmap=_cm(CMAP_PRED), norm=Normalize(0, horizon)); sm.set_array([])
        cb = fig.colorbar(sm, ax=a, label="elapsed time (s)"); cb.ax.tick_params(labelsize=11)
        cb.set_label("elapsed time (s)", fontsize=12)
        a.legend(handles=[Line2D([0], [0], color=_cm(CMAP_PRED)(.75), lw=2.4, label="predicted"),
                          Line2D([0], [0], color=_cm(CMAP_MEAS)(.8), lw=2.0, label="measured")],
                 fontsize=11, loc="upper left", framealpha=.85)
    for a in axes[n:]: a.axis("off")
    fig.suptitle("Final-normalised growth prediction (leave-one-session-out) — "
                 "bright = predicted, dark grey = measured; colour/opacity = elapsed time", fontsize=15)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.recipe_metrics = pd.DataFrame(rows)      # the numbers printed on the panels, for the caption
    return fig

# ── SI: a 3×3 grid of RANDOMLY drawn recipes, same leave-one-session-out reconstruction ──────────
# The four A-D recipes of `fig_known_recipes` are author-chosen; the SI shows the model on a random
# draw so a reader can judge how representative they are. Fixed seed -> the same nine every run.
SI_GRID_SEED = 40           # seed of the random draw (change to see a different nine)
SI_GRID_MIN_FRAMES = 12     # skip runs too short to show a growth (same floor as nb40 §5)

def sample_recipes(runs, n=9, seed=SI_GRID_SEED, exclude=(), min_frames=SI_GRID_MIN_FRAMES):
    """`n` run ids drawn at random (fixed `seed`) among runs with >= `min_frames` frames, never one
    of `exclude` (the main-figure recipes). Plain uniform draw — no stratification by session."""
    pool = runs[(runs.n_frames >= min_frames) & ~runs.run.isin(list(exclude))].run.values
    rng = np.random.default_rng(seed)
    return sorted(rng.choice(pool, size=min(n, len(pool)), replace=False).tolist())

def fig_recipe_grid(runs, F, grid, recipes, ncol=3, slices=SLICES, shape_model="GP-Matern52", feat=f_clr,
                    panel=(4.2, 3.6)):
    """Compact grid (default 3×3) of leave-one-session-out growth reconstructions — the SI companion
    of `fig_known_recipes`, same drawing conventions (grey = measured, plasma = predicted, colour /
    opacity = elapsed time) but ONE shared colourbar in progress units τ = t / t_conv(predicted), since
    every panel has its own horizon; each panel's title carries its measured and predicted t_conv.
    No figure title (caption goes in the paper); axis labels only on the outer panels."""
    from matplotlib.lines import Line2D
    sess = runs.session.values; n = len(recipes); nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(panel[0] * ncol + 0.9, panel[1] * nrow),
                             squeeze=False, sharex=True, sharey=True)
    cache = {}; rows = []
    def predictor_without(s):
        if s not in cache:
            cache[s] = fit_predictor(runs[sess != s].reset_index(drop=True), F, slices, shape_model, feat)
        return cache[s]
    for i, (a, run) in enumerate(zip(axes.ravel(), recipes)):
        r = runs[runs.run == run].iloc[0]; s = r.session; Q = _run_Q(runs, run)
        real = F[int(r.offset):int(r.offset) + int(r.n_frames)]
        tm = s39.lookup_real_conv_time(run); tp = predict_conv_time_without(s, Q)
        model = predictor_without(s)
        t_grid, specs = model.predict(Q, tp)
        _draw_growth_panel(a, grid, real, t_grid, specs, lw_meas=0.8, lw_pred=1.4)
        met = recipe_metrics(model, real, Q)
        _annotate_panel(a, run, s, tm, tp, Q, met, fs_title=8.5, fs_info=6.8, show_session=False,
                        show_run=False)
        rows.append(_metrics_row(run, s, Q, tm, tp, met))
        a.tick_params(labelsize=8)
        a.text(0.02, 0.96, f"({chr(97 + i)})", transform=a.transAxes, ha="left", va="top",
               fontsize=10, fontweight="bold")
    for a in axes.ravel()[n:]:
        a.axis("off")
    for a in axes[-1]:
        a.set_xlabel("Wavelength (nm)", fontsize=10)
    for a in axes[:, 0]:
        a.set_ylabel("Extinction, final-normalised", fontsize=10)
    # legend on the FIGURE, below the grid: inside a panel it always lands on someone's spectrum
    fig.legend(handles=[Line2D([0], [0], color=_cm(CMAP_PRED)(.75), lw=2.0, label="predicted"),
                        Line2D([0], [0], color=_cm(CMAP_MEAS)(.8), lw=1.6, label="measured")],
               fontsize=9, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.47, 0.0))
    fig.tight_layout(rect=[0, 0.03, 0.93, 1])
    sm = plt.cm.ScalarMappable(cmap=_cm(CMAP_PRED), norm=Normalize(0, 1)); sm.set_array([])
    cax = fig.add_axes([0.945, 0.15, 0.014, 0.7])
    cb = fig.colorbar(sm, cax=cax); cb.set_label(r"progress  $\tau = t\,/\,t_{conv}$ (predicted)", fontsize=10)
    cb.ax.tick_params(labelsize=8)
    fig.recipe_metrics = pd.DataFrame(rows)      # the numbers printed on the panels, for the caption
    return fig

# ══════════════════════════════════════════════════════════════════════════════
# HOW ACCURATE IS THE PEAK POSITION?  (the honest, out-of-sample λmax number)
#   The interactive tool (§6) draws with the DEPLOYMENT model, which has seen every run — its λmax
#   lands within ~9 nm of the measurement, and that number is meaningless for a NOVEL recipe. This
#   measures λmax the way it will actually behave: grouped folds by session, so the queried recipe's
#   whole session is unseen. For reference, nb33's own grouped-CV λmax RMSE on the CONVERGED spectrum
#   is 62.7 nm (data/spectral_benchmark.csv, the best of its whole feature × model zoo) — i.e. a
#   visible peak-position error on an unseen composition is the accuracy ceiling of this dataset, not
#   a defect of the growth model.
# ══════════════════════════════════════════════════════════════════════════════
def lmax_table_loso(runs, F, grid, k=5, slices=SLICES, shape_model="GP-Matern52", feat=f_clr,
                    warp=True):
    """Per-frame measured vs predicted λmax for every run, with the run's whole SESSION held out
    (`k` grouped folds). λmax is read on BOTH sides with the same gated reader (`GrowthPredictor._lmax`
    on the final-normalised frame), so frames with no trustworthy peak drop out instead of contributing
    a window-edge artifact. Columns: run, session, tau, lam_meas, lam_pred, err, abs_err."""
    from sklearn.model_selection import GroupKFold
    rows = []
    for tr_i, te_i in GroupKFold(n_splits=min(k, runs.session.nunique())).split(
            runs, groups=runs.session.values):
        m = fit_predictor(runs.iloc[tr_i].reset_index(drop=True), F, slices, shape_model, feat,
                          grid=grid)
        te = runs.iloc[te_i].reset_index(drop=True)
        for r in te.itertuples():
            a, n = int(r.offset), int(r.n_frames)
            real = F[a:a + n]
            rel = real / max(float(real[-1].max()), 1e-9)
            Q = np.array([r.Q1, r.Q2, r.Q3, r.Q4, r.Q5], float)
            _, specs = m.predict(Q, (n - 1) * CADENCE, warp=warp)
            for j in range(n):
                lm = m._lmax(rel[j]); lp = m._lmax(specs[min(j, len(specs) - 1)])
                if not (np.isfinite(lm) and np.isfinite(lp)):
                    continue
                rows.append(dict(run=r.run, session=r.session, tau=j / max(n - 1, 1),
                                 lam_meas=lm, lam_pred=lp, err=lp - lm, abs_err=abs(lp - lm)))
    d = pd.DataFrame(rows)
    d["is_final"] = d.tau >= 1.0 - 1e-9
    return d


# ══════════════════════════════════════════════════════════════════════════════
# KINETIC SHAPE METRIC — the platform's own added value, scored the same way as nb33
# ══════════════════════════════════════════════════════════════════════════════
#   nb33 scores ONE spectrum per recipe (the converged one). This system measures the whole
#   growth in line, so the honest question is not "is the endpoint right" but "is the spectrum
#   right AT EVERY STAGE of the growth". Same metric as nb33 (spectral angle, amplitude-
#   invariant, in degrees) evaluated on a τ grid, against the same two references:
#     * a no-model baseline (the mean trajectory over the training runs)
#     * the experimental REPEATABILITY FLOOR at that same τ, from near-identical recipe pairs
#   A model inside the floor at every τ predicts the whole kinetics as well as a rerun does.
KIN_TAUS   = np.array([0.2, 0.4, 0.6, 0.8, 1.0])
KIN_REP_DQ = 0.40        # recipes closer than this (per-flow std units) count as repeats

def _sam(A, B):
    num = (A * B).sum(-1)
    den = np.linalg.norm(A, axis=-1) * np.linalg.norm(B, axis=-1)
    return np.degrees(np.arccos(np.clip(num / np.maximum(den, 1e-12), -1, 1)))

def _rel_frames_at(F, r, taus):
    """The run's frames at progress `taus`, each divided by the run's FINAL peak (so the stack
    keeps the growth in amplitude, not just the shape)."""
    G = F[int(r.offset):int(r.offset) + int(r.n_frames)]
    k = np.clip((taus * (int(r.n_frames) - 1)).round().astype(int), 0, int(r.n_frames) - 1)
    return G[k] / max(float(G[-1].max()), 1e-9)

def kinetic_repeat_floor(runs, F, taus=KIN_TAUS, dq=KIN_REP_DQ):
    """Spectral angle at each τ between runs of near-identical recipes = the experimental
    kinetic noise. Returns (floor_per_tau, n_pairs)."""
    Q = runs[["Q1", "Q2", "Q3", "Q4", "Q5"]].values.astype(float)
    s = Q.std(0)
    dQ = np.sqrt((((Q[:, None, :] - Q[None, :, :]) / s) ** 2).sum(-1))
    np.fill_diagonal(dQ, np.inf)
    i, j = np.where(np.triu(dQ < dq, 1))
    if len(i) == 0:
        return np.full(len(taus), np.nan), 0
    R = np.stack([_rel_frames_at(F, r, taus) for r in runs.itertuples()])   # (n, tau, wl)
    return np.median(_sam(R[i], R[j]), axis=0), len(i)

def kinetic_report(runs, F, grid, k=5, taus=KIN_TAUS, slices=SLICES,
                   shape_model="GP-Matern52", feat=f_clr, warp=True):
    """Leave-one-SESSION-out spectral angle at each τ, plus the two references.
    Returns a dict ready for `fig_kinetic_metrics`."""
    from sklearn.model_selection import GroupKFold
    n = len(runs)
    meas = np.stack([_rel_frames_at(F, r, taus) for r in runs.itertuples()])
    pred = np.full_like(meas, np.nan)
    for tr_i, te_i in GroupKFold(n_splits=min(k, runs.session.nunique())).split(
            runs, groups=runs.session.values):
        m = fit_predictor(runs.iloc[tr_i].reset_index(drop=True), F, slices, shape_model,
                          feat, grid=grid)
        for pos in te_i:
            r = runs.iloc[pos]
            Q = np.array([r.Q1, r.Q2, r.Q3, r.Q4, r.Q5], float)
            nf = int(r.n_frames)
            _, specs = m.predict(Q, (nf - 1) * CADENCE, warp=warp)
            kk = np.clip((taus * (nf - 1)).round().astype(int), 0, len(specs) - 1)
            pred[pos] = specs[kk]
    sam_model = _sam(meas, pred)                                  # (n, tau)
    base = np.stack([meas.mean(0)] * n)                           # mean trajectory, no model
    sam_base = _sam(meas, base)
    floor, n_pairs = kinetic_repeat_floor(runs, F, taus)
    return dict(taus=taus, sam=sam_model, base=sam_base, floor=floor, n_pairs=n_pairs,
                meas=meas, pred=pred, runs=runs, grid=grid)

def fig_kinetic_metrics(rep, ncol_examples=3):
    """(a) spectral angle vs progress with both references, (b) the growth curve g(τ),
    (c) example trajectories measured vs predicted."""
    taus, sam_m, sam_b, floor = rep["taus"], rep["sam"], rep["base"], rep["floor"]
    grid, meas, pred = rep["grid"], rep["meas"], rep["pred"]
    fig = plt.figure(figsize=(13.6, 7.6))
    gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 0.9])
    ax0 = fig.add_subplot(gs[0, :2]); ax1 = fig.add_subplot(gs[0, 2])

    med = np.median(sam_m, 0); p25 = np.percentile(sam_m, 25, 0); p75 = np.percentile(sam_m, 75, 0)
    ax0.fill_between(taus, p25, p75, color="#0072B2", alpha=.20, label="model, IQR")
    ax0.plot(taus, med, "o-", color="#0072B2", lw=2.2, label="model (leave-session-out)")
    ax0.plot(taus, np.median(sam_b, 0), "s--", color="#E69F00", lw=1.6,
             label="mean trajectory (no model)")
    if np.isfinite(floor).any():
        ax0.plot(taus, floor, "^-", color="#009E73", lw=2.0,
                 label=f"repeatability floor (n={rep['n_pairs']} pairs)")
        ax0.fill_between(taus, 0, floor, color="#009E73", alpha=.10)
    ax0.set_xlabel(r"growth progress  $\tau = t / t_{conv}$")
    ax0.set_ylabel("spectral angle to the measurement (deg)")
    ax0.set_ylim(0, max(np.median(sam_b, 0).max() * 1.15, 5))
    ax0.legend(fontsize=8)
    ax0.set_title("(a) the spectrum is predicted at every stage of the growth, not just at the end\n"
                  "green band = within experimental repeatability", fontsize=9.5)

    gm = meas.max(-1); gp = pred.max(-1)          # growth curve g(tau) = frame peak / final peak
    ax1.plot(taus, np.median(gm, 0), "o-", color="#0072B2", lw=2, label="measured")
    ax1.fill_between(taus, np.percentile(gm, 25, 0), np.percentile(gm, 75, 0),
                     color="#0072B2", alpha=.18)
    ax1.plot(taus, np.median(gp, 0), "s--", color="#D55E00", lw=1.8, label="predicted")
    ax1.set_xlabel(r"$\tau$"); ax1.set_ylabel(r"$g(\tau)$ = peak / final peak")
    ax1.legend(fontsize=8); ax1.set_title("(b) growth curve", fontsize=9.5)

    order = np.argsort(np.median(sam_m, 1))
    pick = order[[int(f * (len(order) - 1)) for f in (0.05, 0.5, 0.95)]][:ncol_examples]
    for c, i in enumerate(pick):
        a = fig.add_subplot(gs[1, c])
        for t in range(len(taus)):
            sh = 0.6 * (1 - t / max(len(taus) - 1, 1))
            a.plot(grid, meas[i, t], color=(sh, sh, sh), lw=1.2)
            a.plot(grid, pred[i, t], color="#D55E00", lw=1.0, ls="--")
        a.set_xlim(grid.min(), grid.max()); a.tick_params(labelsize=7)
        a.set_xlabel("wavelength (nm)", fontsize=8)
        a.set_ylabel("extinction / final peak", fontsize=8)
        a.set_title(f"{rep['runs'].run.iloc[i]}\nmedian SAM over τ = "
                    f"{np.median(sam_m[i]):.1f}°", fontsize=8)
    fig.text(0.005, 0.004, "grey = measured (dark → light with increasing τ)   ·   "
             "dashed orange = predicted, session held out",
             fontsize=6.5, color="0.4", ha="left", va="bottom")
    fig.tight_layout(); return fig


def lmax_stats(d):
    e = d.err.values; lm = d.lam_meas.values
    return dict(n=len(d), MAE=float(np.abs(e).mean()), medAE=float(np.median(np.abs(e))),
                RMSE=float(np.sqrt((e ** 2).mean())), bias=float(e.mean()),
                R2=float(1 - np.sum(e ** 2) / np.sum((lm - lm.mean()) ** 2)),
                pct_within_25nm=float(100 * np.mean(np.abs(e) <= 25)))


def fig_lmax_parity(d, nb33_rmse=62.7):
    """(a) λmax parity on the CONVERGED frame, held-out sessions, with nb33's grouped-CV RMSE as the
    reference line; (b) |λmax error| vs progress τ — the peak position is least certain early, when
    there is barely any material and the plasmon band is still forming."""
    fin = d[d.is_final]; sf = lmax_stats(fin); sa = lmax_stats(d)
    fig, ax = plt.subplots(1, 2, figsize=(11.4, 4.5))
    lo, hi = 370, 810
    ax[0].scatter(fin.lam_meas, fin.lam_pred, s=22, alpha=.65, edgecolor="k", lw=.3, color="#0072B2")
    ax[0].plot([lo, hi], [lo, hi], "k--", lw=1, label="y = x")
    ax[0].fill_between([lo, hi], [lo - 25, hi - 25], [lo + 25, hi + 25], color="0.6", alpha=.18,
                       label="±25 nm")
    ax[0].set_xlim(lo, hi); ax[0].set_ylim(lo, hi); ax[0].set_aspect("equal")
    ax[0].set_xlabel(r"measured $\lambda_{max}$ (nm)")
    ax[0].set_ylabel(r"predicted $\lambda_{max}$ (nm)")
    ax[0].set_title(f"(a) converged frame, session held out\nRMSE={sf['RMSE']:.0f} nm  "
                    f"MAE={sf['MAE']:.0f}  bias={sf['bias']:+.0f}  "
                    f"within 25 nm: {sf['pct_within_25nm']:.0f}%\n"
                    f"(nb33 endpoint model, grouped CV: {nb33_rmse:.0f} nm)", fontsize=9)
    ax[0].legend(fontsize=8, loc="upper left")
    bins = np.linspace(0, 1, 11)
    ib = np.clip(np.digitize(d.tau.values, bins) - 1, 0, len(bins) - 2)
    med = [np.median(d.abs_err.values[ib == i]) if (ib == i).any() else np.nan
           for i in range(len(bins) - 1)]
    p90 = [np.percentile(d.abs_err.values[ib == i], 90) if (ib == i).any() else np.nan
           for i in range(len(bins) - 1)]
    ctr = 0.5 * (bins[1:] + bins[:-1])
    ax[1].plot(ctr, med, "o-", color="#0072B2", label="median")
    ax[1].plot(ctr, p90, "s--", color="#D55E00", label="90th percentile")
    ax[1].set_xlabel(r"progress $\tau = t / t_{conv}$")
    ax[1].set_ylabel(r"$|\lambda_{max}$ error$|$ (nm)")
    ax[1].set_title(f"(b) all {sa['n']} held-out frames — MAE={sa['MAE']:.0f} nm", fontsize=9)
    ax[1].legend(fontsize=8)
    fig.suptitle("Peak-position accuracy of the growth model on UNSEEN sessions — the honest number "
                 "behind the interactive tool", fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.95]); return fig


# ══════════════════════════════════════════════════════════════════════════════
# IS THE MODEL MORE ACCURATE NEAR THE BARYCENTRE OF THE PARAMETER SPACE?
#   Claim to test: held-out error grows with the distance of the recipe from the centre of the
#   design. No new fit — it re-uses the leave-session-out errors already computed by
#   `kinetic_report` (§9b, spectral angle per run and τ) and `lmax_table_loso` (§9, λmax).
#   Distance = Mahalanobis in the 5 flows (pseudo-inverse covariance: the flows sum to 14, so the
#   design is 4-D). Two distances, because "near the centre" has a confound:
#     d_centre  distance to the design barycentre (the claim)
#     d_nn      distance to the NEAREST TRAINING recipe of that run's own fold (how much the GP
#               extrapolates; edges of an LHS design are also sparser)
#   Statistics are session-aware: runs of one session share a day / stock solutions, so CIs come
#   from a CLUSTER bootstrap over sessions, not over runs.
# ══════════════════════════════════════════════════════════════════════════════
FLOWS = ("Q1", "Q2", "Q3", "Q4", "Q5")

def _mahal_VI(Q):
    return np.linalg.pinv(np.cov(Q, rowvar=False))

def _mahal(A, B, VI):
    """Pairwise Mahalanobis distances between the rows of A (n,5) and B (m,5) -> (n,m)."""
    D = A[:, None, :] - B[None, :, :]
    return np.sqrt(np.einsum("nmi,ij,nmj->nm", D, VI, D).clip(0))

def centroid_table(kin, lam=None, k=5, centre="design"):
    """One row per run: d_centre, d_nn and the held-out errors. `kin` = `kinetic_report(...)` (must
    be run with the same `k`, so the folds — hence d_nn — are the ones its predictions came from);
    `lam` = `lmax_table_loso(...)` (optional, adds the converged-frame |Δλmax|). `centre`: "design"
    = mean of the sampled recipes, "simplex" = equal flows (14/5 each) — the two differ by < 0.3 µL/s
    per flow here, the choice does not change the conclusion."""
    from sklearn.model_selection import GroupKFold
    runs = kin["runs"]
    Q = runs[list(FLOWS)].values.astype(float)
    VI = _mahal_VI(Q)
    c = Q.mean(0) if centre == "design" else np.full(5, Q_TOTAL / 5)
    d_c = _mahal(Q, c[None, :], VI)[:, 0]
    d_nn = np.full(len(runs), np.nan)
    for tr_i, te_i in GroupKFold(n_splits=min(k, runs.session.nunique())).split(
            runs, groups=runs.session.values):
        d_nn[te_i] = _mahal(Q[te_i], Q[tr_i], VI).min(1)
    sam = np.asarray(kin["sam"], float)
    df = pd.DataFrame(dict(run=runs.run.values, session=runs.session.values, d_centre=d_c, d_nn=d_nn,
                           ang_mean=np.nanmean(sam, 1), ang_final=sam[:, -1]))
    if lam is not None:
        fin = lam[lam.is_final].groupby("run").abs_err.first().rename("lam_err")
        df = df.merge(fin, left_on="run", right_index=True, how="left")
    return df

def _spearman(x, y):
    from scipy.stats import spearmanr
    ok = np.isfinite(x) & np.isfinite(y)
    return spearmanr(x[ok], y[ok]).correlation if ok.sum() > 3 else np.nan

def _partial_spearman(x, y, z):
    """Spearman ρ(x, y) after removing the (rank-linear) effect of z from both."""
    from scipy.stats import rankdata
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
    rx, ry, rz = (rankdata(v[ok]) for v in (x, y, z))
    res = lambda a: a - np.polyval(np.polyfit(rz, a, 1), rz)
    return np.corrcoef(res(rx), res(ry))[0, 1]

def _tercile_ratio(d, e):
    """median error of the OUTER third of recipes (largest d) / median error of the INNER third."""
    ok = np.isfinite(d) & np.isfinite(e); d, e = d[ok], e[ok]
    lo, hi = np.quantile(d, [1 / 3, 2 / 3])
    return np.median(e[d >= hi]) / max(np.median(e[d <= lo]), 1e-12)

def centroid_stats(df, metrics=("ang_mean", "ang_final", "lam_err"), n_boot=2000, seed=0):
    """Per error metric: Spearman ρ with d_centre (point + 95 % session-cluster-bootstrap CI), the same
    for d_nn, the partial ρ(d_centre, error | d_nn), the median error in the inner / middle / outer
    third of recipes by d_centre, and the outer/inner ratio with its CI. A CI on ρ(d_centre) that
    excludes 0 is the evidence for the claim; the partial ρ says whether it survives once the
    'there is simply less data out there' effect is removed."""
    rng = np.random.default_rng(seed)
    sess = df.session.values; us = np.unique(sess)
    idx_by_s = {s: np.where(sess == s)[0] for s in us}
    out = []
    for m in metrics:
        if m not in df:
            continue
        e = df[m].values.astype(float); dc = df.d_centre.values; dn = df.d_nn.values
        f = {"rho_centre": lambda i: _spearman(dc[i], e[i]),
             "rho_nn": lambda i: _spearman(dn[i], e[i]),
             "rho_centre|nn": lambda i: _partial_spearman(dc[i], e[i], dn[i]),
             "outer/inner": lambda i: _tercile_ratio(dc[i], e[i])}
        allidx = np.arange(len(df))
        boots = {key: [] for key in f}
        for _ in range(n_boot):
            i = np.concatenate([idx_by_s[s] for s in rng.choice(us, len(us), replace=True)])
            for key, fn in f.items():
                boots[key].append(fn(i))
        ok = np.isfinite(dc) & np.isfinite(e)
        q = np.quantile(dc[ok], [1 / 3, 2 / 3])
        row = dict(metric=m, n=int(ok.sum()),
                   med_inner=np.median(e[ok & (dc <= q[0])]),
                   med_middle=np.median(e[ok & (dc > q[0]) & (dc < q[1])]),
                   med_outer=np.median(e[ok & (dc >= q[1])]))
        for key, fn in f.items():
            b = np.asarray(boots[key], float)
            row[key] = fn(allidx)
            row[key + "_lo"], row[key + "_hi"] = np.nanquantile(b, [0.025, 0.975])
        out.append(row)
    return pd.DataFrame(out)

_METRIC_LABEL = {"ang_mean": r"spectral angle, mean over $\tau$ (deg)",
                 "ang_final": r"spectral angle at $\tau = 1$ (deg)",
                 "lam_err": r"$|\Delta\lambda_{max}|$ at $\tau = 1$ (nm)"}

def fig_centroid_accuracy(df, stats, kin=None, metrics=("ang_mean", "ang_final", "lam_err"), nbins=5):
    """One panel per error metric: held-out error of every run vs its Mahalanobis distance to the
    design barycentre, with the binned median (quantile bins) and IQR band, Spearman ρ [95 % session
    bootstrap CI] in the corner, and — for the angles — the experimental repeatability floor
    (`kin['floor']`) as a dashed line. Last panel: d_nn vs d_centre, to show how much of 'far from the
    centre' is 'far from any training recipe'. No figure title (caption in the paper)."""
    metrics = [m for m in metrics if m in df]
    fig, axes = plt.subplots(1, len(metrics) + 1, figsize=(4.2 * (len(metrics) + 1), 3.8))
    for a, m in zip(axes, metrics):
        d, e = df.d_centre.values, df[m].values.astype(float)
        ok = np.isfinite(e); d, e = d[ok], e[ok]
        a.scatter(d, e, s=14, color="0.45", alpha=.6, lw=0)
        edges = np.quantile(d, np.linspace(0, 1, nbins + 1))
        b = np.clip(np.searchsorted(edges, d, side="right") - 1, 0, nbins - 1)
        ctr = np.array([np.median(d[b == j]) for j in range(nbins)])
        q = np.array([np.percentile(e[b == j], [25, 50, 75]) for j in range(nbins)])
        a.fill_between(ctr, q[:, 0], q[:, 2], color="#0072B2", alpha=.18, lw=0)
        a.plot(ctr, q[:, 1], "o-", color="#0072B2", lw=1.8, label="binned median (IQR)")
        if kin is not None and m.startswith("ang"):
            fl = np.nanmean(kin["floor"]) if m == "ang_mean" else kin["floor"][-1]
            a.axhline(fl, color="#D55E00", ls="--", lw=1.2, label="repeatability floor")
        s = stats.set_index("metric").loc[m]
        a.text(0.03, 0.97, rf"$\rho_s$ = {s.rho_centre:.2f}  [{s.rho_centre_lo:.2f}, {s.rho_centre_hi:.2f}]"
               f"\nouter/inner third: ×{s['outer/inner']:.2f}\n"
               rf"partial $\rho_s$ | $d_{{nn}}$ = {s['rho_centre|nn']:.2f}",
               transform=a.transAxes, ha="left", va="top", fontsize=8,
               bbox=dict(boxstyle="round", fc="white", ec="0.8", alpha=.9))
        a.set_xlabel("Mahalanobis distance to the design barycentre"); a.set_ylabel(_METRIC_LABEL[m])
        a.set_ylim(bottom=0); a.legend(fontsize=7.5, loc="upper right")
    a = axes[-1]
    a.scatter(df.d_centre, df.d_nn, s=14, color="0.45", alpha=.6, lw=0)
    a.set_xlabel("Mahalanobis distance to the design barycentre")
    a.set_ylabel("distance to nearest training recipe ($d_{nn}$)")
    a.text(0.03, 0.97, rf"$\rho_s$ = {_spearman(df.d_centre.values, df.d_nn.values):.2f}",
           transform=a.transAxes, ha="left", va="top", fontsize=8)
    for i, a in enumerate(axes):
        a.text(-0.02, 1.03, f"({chr(97 + i)})", transform=a.transAxes, fontsize=10, fontweight="bold",
               ha="right", va="bottom")
    fig.tight_layout(); return fig


# ══════════════════════════════════════════════════════════════════════════════
# ABLATION: basis size × smoothing of the training spectra (the numbers quoted at N_PC_REL)
# ══════════════════════════════════════════════════════════════════════════════
def ablate_basis(runs, F, grid, n_pcs=(3, 4, 5, 6, 8), windows=(None, 9, 15, 31), k=5,
                 shape_model="GP-Matern52", feat=f_clr, verbose=True):
    """Grouped-fold (by session) spectral RMSE for each (n_pc, savgol window) pair. The model is FIT on
    the optionally-smoothed frames but always SCORED against the RAW measured frames, so smoothing
    cannot flatter itself. Predictions are the plain τ-interpolation of the checkpoints (warp off) so
    the comparison isolates the basis. ~20 min for the full grid — the result is recorded above
    `N_PC_REL`, so run this only to re-derive it."""
    from sklearn.model_selection import GroupKFold
    folds = list(GroupKFold(n_splits=min(k, runs.session.nunique())).split(
        runs, groups=runs.session.values))
    rows = []
    for w in windows:
        Fs = smooth_frames(F, w)
        for n_pc in n_pcs:
            se = []
            for tr_i, te_i in folds:
                tr = runs.iloc[tr_i].reset_index(drop=True)
                mu, comps, fm = _fit_relative_basis(tr, Fs, n_pc)
                Qtr = tr[list(sp.FEATURES)].values.astype(float)
                CK = {}
                for tau in SLICES:
                    rel = np.array([Fs[int(r.offset) + int(round(tau * (int(r.n_frames) - 1)))]
                                    / fm[r.run] for r in tr.itertuples()])
                    CK[tau] = sp.SpectralModel(n_pc, feat, MODELS[shape_model]).fit(
                        Qtr, (rel - mu) @ comps.T)
                te = runs.iloc[te_i].reset_index(drop=True)
                Qte = te[list(sp.FEATURES)].values.astype(float)
                P = np.array([mu + CK[t].predict(Qte) @ comps for t in SLICES])
                for i, r in enumerate(te.itertuples()):
                    a, n = int(r.offset), int(r.n_frames)
                    real = F[a:a + n]                          # RAW, never smoothed
                    true = real / max(float(real[-1].max()), 1e-9)
                    pred = interp1d(np.array(SLICES), P[:, i, :], axis=0, bounds_error=False,
                                    fill_value=(P[0, i], P[-1, i]))(np.arange(n) / max(n - 1, 1))
                    pred = pred / max(float(pred[-1].max()), 1e-9)
                    se.append(((pred - true) ** 2).ravel())
            rmse = float(np.sqrt(np.concatenate(se).mean()))
            rows.append(dict(n_pc=n_pc, savgol=(0 if not w else w), spec_rmse=rmse))
            if verbose:
                print(f"  n_pc={n_pc} savgol={rows[-1]['savgol']:2d}  spec_rmse={rmse:.4f}")
    d = pd.DataFrame(rows)
    return d, d.pivot(index="savgol", columns="n_pc", values="spec_rmse")


def fig_ablate_basis(d):
    """Spectral RMSE vs basis size, one line per smoothing window — the picture behind "keep 6
    components and do not smooth"."""
    fig, ax = plt.subplots(figsize=(6.6, 4.3))
    for w, g in d.groupby("savgol"):
        g = g.sort_values("n_pc")
        ax.plot(g.n_pc, g.spec_rmse, "o-",
                label="no smoothing" if w == 0 else f"Savitzky-Golay {int(w)}")
    ax.axvline(N_PC_REL, color="0.6", ls=":", lw=1)
    ax.annotate(f"deployed\nn_pc={N_PC_REL}", (N_PC_REL, ax.get_ylim()[1]), fontsize=8,
                ha="center", va="top", color="0.4")
    ax.set_xlabel("PCA components of the relative-to-final basis")
    ax.set_ylabel("held-out spectral RMSE (relative units)")
    ax.set_title("Basis size and input smoothing (5 grouped folds, scored on RAW frames)", fontsize=10)
    ax.legend(fontsize=8)
    fig.tight_layout(); return fig


def normalise_recipe(Q, total=Q_TOTAL):
    """Rescale a 5-flow recipe so it sums to `total` (14 µL/s)."""
    Q = np.clip(np.asarray(Q, float), 0, None); s = Q.sum()
    return Q * total / s if s > 0 else np.full(5, total / 5)

# ══════════════════════════════════════════════════════════════════════════════
# A single, real TRAIN / TEST split (by SESSION) — parity of t_conv + reconstruct every test recipe
# ══════════════════════════════════════════════════════════════════════════════
def train_test_split_sessions(runs, test_frac=0.25, seed=0):
    """Split SESSIONS (not runs) into train/test so no test recipe shares a session with training.
    Returns (train_runs, test_runs, test_sessions)."""
    rng = np.random.default_rng(seed); sessions = runs.session.unique()
    n_test = max(1, int(round(test_frac * len(sessions))))
    test_sess = set(rng.choice(sessions, n_test, replace=False))
    tr = runs[~runs.session.isin(test_sess)].reset_index(drop=True)
    te = runs[runs.session.isin(test_sess)].reset_index(drop=True)
    return tr, te, sorted(test_sess)

def fig_conv_time_test(test_runs):
    """Parity plot of predicted vs measured t_conv on the held-out TEST recipes (the flow-model t_conv is
    trained elsewhere; this shows its accuracy on this split), returns (fig, stats).
    Uses the LEAVE-ONE-SESSION-OUT table restricted to `test_runs` — it used to call the in-sample
    `conv_time_table` (global pkl, trained on these very runs)."""
    df, _ = conv_time_table_loso(); df = df[df.run.isin(set(test_runs.run))]; st = conv_time_stats(df)
    return fig_conv_time(df, st, protocol="leave-one-session-out"), st

def fig_test_reconstructions(model, test_runs, F, grid, ncol=4, max_panels=None):
    """Final-normalised reconstruction of EVERY test recipe (all of them by default), predicted with the
    TRAIN-ONLY `model` (so the whole test set is out-of-sample). Solid = predicted, dotted = measured
    (each frame / its own measured final). No colorbars (would swamp a big grid); colour = elapsed time."""
    sel = list(test_runs.run.values)
    if max_panels is not None:
        sel = sel[:max_panels]
    from matplotlib.colors import Normalize
    n = len(sel); nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.3 * ncol, 2.6 * nrow), squeeze=False); axes = axes.ravel()
    for a, run in zip(axes, sel):
        r = test_runs[test_runs.run == run].iloc[0]; Q = _run_Q(test_runs, run)
        real = F[int(r.offset):int(r.offset) + int(r.n_frames)]
        tp = predict_conv_time_without(r.session, Q); tm = s39.lookup_real_conv_time(run)
        tg, specs = model.predict(Q, tp); horizon = max(tg[-1], 1e-9)
        mf = max(float(real[-1].max()), 1e-9); tr = np.arange(len(real)) * CADENCE
        for k in range(len(real)):
            a.plot(grid, real[k] / mf, ":", color=_cm(CMAP_PRED)(min(tr[k] / horizon, 1.0)), lw=.5, alpha=.6, rasterized=True)
        for k in range(len(tg)):
            a.plot(grid, specs[k], "-", color=_cm(CMAP_PRED)(min(tg[k] / horizon, 1.0)), lw=.7, alpha=.9, rasterized=True)
        a.set_xlim(grid.min(), grid.max()); a.axhline(0, color="0.9", lw=.4); a.tick_params(labelsize=5)
        tm_s = f"{tm:.0f}" if tm is not None else "n/a"
        a.set_title(f"{run}\n{r.session}  $t_c$ m={tm_s} p={tp:.0f}s", fontsize=6, color="#b00")
        a.set_xlabel(""); a.set_ylabel("")
    for a in axes[n:]: a.axis("off")
    fig.suptitle(f"TRAIN/TEST split — reconstruction of all {n} test recipes (out-of-sample), "
                 "final-normalised (solid=pred, dotted=measured, colour=elapsed s)", fontsize=10)
    fig.tight_layout(rect=[0, 0, 1, 0.98]); return fig

# ── batched convergence-time (fast for grids) ─────────────────────────────────
_conv_cache = None
def _conv_model():
    """The deployed t_conv model. Warns if it is the un-benchmarked step-38 fallback: nothing
    ever rebuilds an existing pkl, so a fallback fit silently shadows the benchmark winner
    (RandomForest, held-out R2 0.543, stayed deployed for six weeks over GP-Matern52, 0.632)."""
    global _conv_cache
    if _conv_cache is None:
        import joblib
        cpkl = pc.DATA_DIR / "convergence_model.pkl"
        if cpkl.exists():
            _conv_cache = joblib.load(cpkl)
            if _conv_cache.get("source") != "step36-benchmark":
                print(f"  WARNING: convergence_model.pkl was not produced by the step-36 benchmark "
                      f"(source={_conv_cache.get('source', 'unknown')}, "
                      f"reg={_conv_cache.get('reg_name')}). Run step 36 to deploy the benchmarked "
                      f"model.")
        else:
            _conv_cache = pc.missing_convergence_model()   # [vendored] no silent RF fallback
    return _conv_cache

_conv_loso_cache = {}

def predict_conv_time_without(session, Q):
    """Predicted t_conv (s) for recipe `Q` from a t_conv model refit WITHOUT `session` — the same
    regressor family as the deployed pkl, trained on every converged run of the other sessions.
    Use it whenever a KNOWN run is displayed: `s39.predict_conv_time` uses the global pkl, which step
    36 fits on ALL runs, so the t_conv printed on (and the time axis of) a reconstructed growth would
    be in-sample even though its spectra are leave-one-session-out. Cached per session."""
    if session not in _conv_loso_cache:
        from . import step36_convergence_model as s36
        name = _conv_model().get("reg_name", "GP-Matern52")
        ok = s36.load_conv(); ok = ok[ok.converged & (ok.session != session)]
        _conv_loso_cache[session] = s36.REGRESSORS[name]().fit(
            ok[list(sp.FEATURES)].values.astype(float), np.log1p(ok.conv_time_s.values))
    return float(np.expm1(_conv_loso_cache[session].predict(np.atleast_2d(np.asarray(Q, float)))[0]))

def predict_conv_time_batch(Qmat):
    """Flow-model predicted t_conv (s) for many recipes at once (loads the model once)."""
    return np.expm1(_conv_model()["reg"].predict(np.atleast_2d(np.asarray(Qmat, float))))

# ── Q1×Q5 perceived-colour maps at INTERMEDIATE times (nb33 style, but time-resolved) ──
def _spectra_at_tau_batch(model, Qpix, tau):
    """(npix, n_wl) relative-to-final predicted spectra at progress `tau` for a batch of recipes — the
    τ-checkpoints interpolated at τ (progress is the checkpoint axis, so no t_conv needed)."""
    Qpix = np.atleast_2d(np.asarray(Qpix, float))
    rel_ck = np.array([model.mu + model.ckpt[t].predict(Qpix) @ model.comps for t in model.slices])
    tau_c = np.array(model.slices)
    return interp1d(tau_c, rel_ck, axis=0, bounds_error=False, fill_value=(rel_ck[0], rel_ck[-1]))(tau)

def fig_colormap_time(model, runs, F, grid, pair=("Q1", "Q5"), taus=(0.3, 0.6, 1.0), points=None,
                      grid_n=44, holdout_points=True, shape_model="GP-Matern52"):
    """nb33-style perceived-colour map over the (Q1, Q5) plane, but at several INTERMEDIATE progress
    fractions `taus` (τ=1 = final = nb33's map). Other flows fill proportionally to the data median so
    the recipe sums to 14. `points` = {label: Q} recipes A,B,C,D marked on every panel; their predicted
    spectra are extracted into the second returned value.

    `holdout_points=True` (default, NO LEAK): the model used for BOTH the map and the extracted A–D
    spectra is refit with the WHOLE SESSION of every marked point held out of training, so A–D are
    genuinely out-of-sample. Pass `holdout_points=False` to use the deployment `model` you passed in
    (faster, but A–D are then in-sample). Returns (fig, spectra_dict[label][tau])."""
    Qdata = runs[list(sp.FEATURES)].values.astype(float)
    colour = sp.make_colouriser(grid)
    ia, ib = sp.FEATURES.index(pair[0]), sp.FEATURES.index(pair[1])
    others = [k for k in range(5) if k not in (ia, ib)]
    med = np.median(Qdata, 0); prop = med[others] / med[others].sum()
    lo_a = pc.TRAIN_Q_FLOOR if pair[0] in ("Q1", "Q5") else 0.0
    lo_b = pc.TRAIN_Q_FLOOR if pair[1] in ("Q1", "Q5") else 0.0
    ga = np.linspace(lo_a, Qdata[:, ia].max() * 1.02, grid_n)
    gb = np.linspace(lo_b, Qdata[:, ib].max() * 1.02, grid_n)
    gridQ = np.full((grid_n, grid_n, 5), np.nan); mask = np.zeros((grid_n, grid_n), bool)
    for iy, vb in enumerate(gb):
        for ix, va in enumerate(ga):
            rem = Q_TOTAL - va - vb
            if rem <= 0:
                continue
            x = np.empty(5); x[ia] = va; x[ib] = vb; x[others] = rem * prop
            gridQ[iy, ix] = x; mask[iy, ix] = True
    Qpix = gridQ[mask]
    if points is None:                              # 4 spread recipes (one per distinct session)
        g = runs[runs.n_frames >= 12]
        rng = np.random.default_rng(7)
        sess4 = rng.permutation(g.session.unique())[:4]
        chosen = [g[g.session == s].sort_values("n_frames").run.values[-1] for s in sess4]
        points = {lab: _run_Q(runs, r) for lab, r in zip("ABCD", chosen)}
    # NO-LEAK: refit the model with the marked points' whole sessions held out, use it for everything.
    leak = "in-sample"
    if holdout_points:
        pt_sess = set()
        for Qp in points.values():
            d = np.abs(runs[list(sp.FEATURES)].values.astype(float) - Qp).max(1)
            pt_sess.add(runs.iloc[int(np.argmin(d))].session)
        model = fit_predictor(runs[~runs.session.isin(pt_sess)].reset_index(drop=True), F,
                              slices=model.slices, shape_model=shape_model, feat=model.feat)
        leak = f"A–D sessions held out ({sorted(pt_sess)})"
    n = len(taus)
    fig, axes = plt.subplots(1, n, figsize=(4.8 * n, 4.4), squeeze=False); axes = axes[0]
    spectra = {lab: {} for lab in points}
    for a, tau in zip(axes, taus):
        S = _spectra_at_tau_batch(model, Qpix, tau)
        img = np.ones((grid_n, grid_n, 3)); img[mask] = np.array([colour(s) for s in S])
        a.imshow(img, origin="lower", aspect="auto", extent=[ga[0], ga[-1], gb[0], gb[-1]])
        a.plot([0, Q_TOTAL], [Q_TOTAL, 0], "k--", lw=1, alpha=.5)
        for lab, Qp in points.items():
            a.scatter(Qp[ia], Qp[ib], s=44, facecolor="none", edgecolor="k", lw=1.4)
            a.annotate(lab, (Qp[ia], Qp[ib]), textcoords="offset points", xytext=(5, 3),
                       fontweight="bold", fontsize=11)
            spectra[lab][tau] = _spectra_at_tau_batch(model, Qp[None], tau)[0]
        a.set_xlim(ga[0], ga[-1]); a.set_ylim(gb[0], gb[-1])
        a.set_xlabel(f"$Q_{{{pair[0]}}}$ (µL/s)"); a.set_ylabel(f"$Q_{{{pair[1]}}}$ (µL/s)")
        a.set_title(f"τ = {tau:.2f}" + ("  (final)" if abs(tau - 1) < 1e-6 else ""), fontsize=10)
    fig.suptitle(f"Perceived colour over ({pair[0]}, {pair[1]}) at intermediate growth times "
                 f"(other flows median-filled to sum 14) — {leak}", fontsize=9.5)
    fig.tight_layout(rect=[0, 0, 1, 0.95]); return fig, spectra

def fig_point_spectra(spectra, grid, taus):
    """Companion to fig_colormap_time: the extracted predicted spectra of points A,B,C,D, one panel per
    point, curves coloured by τ (final-normalised)."""
    labs = list(spectra); n = len(labs)
    fig, axes = plt.subplots(1, n, figsize=(3.6 * n, 3.4), squeeze=False); axes = axes[0]
    for a, lab in zip(axes, labs):
        for tau in taus:
            a.plot(grid, spectra[lab][tau], color=_cm(CMAP_TAU)(tau), lw=1.6, label=f"τ={tau:.2f}")
        a.set_xlim(grid.min(), grid.max()); a.axhline(0, color="0.85", lw=.5)
        a.set_title(f"point {lab}", fontsize=10); a.set_xlabel("nm", fontsize=8)
        a.set_ylabel("Ext, final-norm.", fontsize=8); a.tick_params(labelsize=7)
    axes[0].legend(fontsize=7)
    fig.suptitle("Extracted predicted spectra at points A–D across growth times", fontsize=10)
    fig.tight_layout(rect=[0, 0, 1, 0.95]); return fig


### `ml.step42_kinetic_descriptors`
*Original analysis code.*

In [ ]:
%%module toofast_analysis.ml.step42_kinetic_descriptors
# [vendored] verbatim copy of 2FAST_ML/pipeline/step42_kinetic_descriptors.py (md5 501c7ea4d839041904ab67c71508d610), generated by tools/vendor_ml.py — do not edit by hand.
"""
step42_kinetic_descriptors.py  —  pipeline step 12  (mirrors 42_kinetic_descriptors.ipynb)
==========================================================================================
Task B of the manuscript brief: promote the KINETICS of each synthesis from a single scalar
(`growth_time`) to a small set of physically-meaningful trajectory descriptors, and test whether
they carry information NOT recoverable from the endpoint spectrum.

For every particle run we read its spectral time series and build a growth signal g(t) = spectral
AREA(t) (the quantity the manuscript already quotes maximum growth rates for), then extract:

    induction_s        lag before growth takes off      (t at which g reaches 5% of its span)
    t50_s / t90_s      time to 50% / 90% of final area   (progress-based, baseline-subtracted)
    t_maxrate_s        time of steepest growth           (inflection of the sigmoid)
    initial_rate       area slope over [induction, t50]  (au·nm/s)
    max_growth_rate    peak of d(area)/dt                (au·nm/s — manuscript's 30.4 vs 4.1)
    growth_time_s      convergence time                  (reused from step36 convergence.csv)
    dlambda_redshift   lambda_final - lambda_initial     (nm — trajectory of the plasmon position)
    lambda_overshoot   max(lambda(t)) - lambda_final     (nm — transient blue/red excursion)

These are then correlated with the experimental inputs — raw flows Q1..Q5 and the mechanistic
chemistry ratios (reductant/Ag, NaBH4/Ag, NaBH4/AscAcid, PVP/Ag, acid/Ag) — with:

  1. Pearson AND Spearman heatmaps  (linear + monotone association);
  2. PARTIAL correlation controlling for lambda_max (isolates input information the ENDPOINT does
     not already carry — the core non-redundancy test);
  3. mutual information kinetic <-> endpoint descriptors (redundancy check);
  4. a leave-one-session-out redundancy model: predict each flow from endpoint-only vs
     endpoint+kinetic feature sets, compare grouped-CV R^2.

Everything runs in the Anaconda 3.11 kernel (numpy/scipy/pandas/scikit-learn). The per-run
extraction reads the H5 time series once and caches to pipeline/data/kinetics.csv — run it yourself.
"""
from pathlib import Path
import numpy as np, pandas as pd, h5py
import matplotlib as mpl, matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from . import pipeline_common as pc
from . import step36_convergence_model as s36
from . import eda_rg024 as eda            # STOCK / REAGENTS for the mechanistic chemistry ratios

sp = pc.sp
SEED = pc.SEED
CV_K = 6
NOTEBOOK = "42_kinetic_descriptors.ipynb"
_trapz = getattr(np, "trapezoid", np.trapz)   # numpy>=2.0 renamed trapz

# ── descriptor construction knobs ─────────────────────────────────────────────
FRAC_IND   = 0.05      # "growth has started": g reaches 5% of its baseline->final span
FRAC_50    = 0.50
FRAC_90    = 0.90
TAIL       = 3         # final plateau = median of the last TAIL frames (matches step36)
MIN_FRAMES = 5         # runs with fewer frames have no resolvable kinetics -> descriptors NaN
GROWTH_MIN = 0.05      # min relative area span (g_final-g_base)/g_final to call it "grew" at all
LAM_WIN    = pc.PEAK_WIN   # (360,990) plasmon window: argmax OUTSIDE it is a UV/NIR edge artifact,
                           #   not a peak -> lambda(t) must be read inside it (esp. pre-nucleation)

KIN_DESCRIPTORS = ["induction_s", "t50_s", "t90_s", "t_maxrate_s", "initial_rate",
                   "max_growth_rate", "growth_time_s", "dlambda_redshift", "lambda_overshoot"]
KIN_LABELS = {
    "induction_s":      "induction (s)",
    "t50_s":            r"$t_{50}$ area (s)",
    "t90_s":            r"$t_{90}$ area (s)",
    "t_maxrate_s":      r"$t_{\mathrm{max\,rate}}$ (s)",
    "initial_rate":     r"initial rate (au·nm/s)",
    "max_growth_rate":  r"max growth rate (au·nm/s)",
    "growth_time_s":    "convergence time (s)",
    "dlambda_redshift": r"$\Delta\lambda$ redshift (nm)",
    "lambda_overshoot": r"$\lambda$ overshoot (nm)",
}
END_DESCRIPTORS = ["lambda_max", "abs_max", "area_final"]
END_LABELS = {"lambda_max": r"$\lambda_{\max}$ (nm)", "abs_max": r"$Ext_{\max}$",
              "area_final": "spectral area"}

CHEM_FEATURES = ["reductant_to_Ag", "NaBH4_to_Ag", "nucleation_vs_growth",
                 "PVP_to_Ag", "acid_to_Ag"]
CHEM_LABELS   = ["reductant/Ag", "NaBH4/Ag", "NaBH4/AscAcid", "PVP/Ag", "acid/Ag"]
FLOW_LABELS   = [f"Q{k+1}\n{eda.REAGENTS[k]}" for k in range(5)]


def savefig(fig, name):
    fig.text(0.995, 0.003, f"source: {NOTEBOOK}", ha="right", va="bottom", fontsize=6, color="0.55")
    for ext in ("png", "pdf"):
        fig.savefig(pc.FIG_DIR / f"{name}.{ext}", dpi=200, bbox_inches="tight")
    print(f"  saved {name}.png/.pdf")


# ── chemistry ratios (mirrors eda_rg024.add_chem_features, self-contained here) ──
def add_chem_features(df):
    mol = df[list(sp.FEATURES)].values * eda.STOCK
    ag = mol[:, 0]
    df = df.copy()
    df["reductant_to_Ag"]      = (mol[:, 3] + mol[:, 4]) / ag
    df["NaBH4_to_Ag"]          = mol[:, 4] / ag
    df["nucleation_vs_growth"] = mol[:, 4] / np.where(mol[:, 3] == 0, np.nan, mol[:, 3])
    df["PVP_to_Ag"]            = mol[:, 1] / ag
    df["acid_to_Ag"]           = mol[:, 2] / ag
    return df


# ── read a run's spectral time series -> per-frame area / abs_max / lambda ───────
def read_series(fp):
    """Return (t, area, absmax, lam) for one H5 run, or None. Uses the SAME dataset keys as
    step36/step39. area(t) = trapezoidal area under the baseline-clipped spectrum; lam(t) =
    wavelength of the frame's maximum."""
    try:
        with h5py.File(fp, "r") as f:
            ak = next((k for k in s36.TS_ABS if k in f), None)
            wk = next((w for w in pc.WL_CANDS if w in f), None)
            mk = next((k for k in s36.TS_AMX if k in f), None)
            tk = next((k for k in s36.TS_TIME if k in f), None)
            if ak is None or wk is None:
                return None
            A = np.asarray(f[ak][()], float); wl = np.asarray(f[wk][()], float)
            amx = np.asarray(f[mk][()], float) if mk else A.max(1)
            if tk is not None:
                ts = [t.decode() if isinstance(t, bytes) else t for t in f[tk][()]]
                try:
                    t = (pd.to_datetime(ts) - pd.to_datetime(ts[0])).total_seconds().values
                except Exception:
                    t = np.arange(len(amx), dtype=float)
            else:
                t = np.arange(len(amx), dtype=float)
    except Exception:
        return None
    area = _trapz(np.clip(A, 0, None), wl, axis=1)
    # lambda(t) from a PROMINENCE-detected plasmon peak (the pipeline's particle detector), NOT a
    # bare argmax: before nucleation / on a broad NIR tail argmax snaps to an edge. A frame's lambda
    # is valid ONLY when its peak clears the same prominence/height bars that define a "particle"
    # (PROM_MIN/ABS_FLOOR); otherwise NaN -> early noise never contaminates the redshift descriptors.
    prom, height, lam = pc.peak_metrics(A, wl)
    lam = np.where((prom >= pc.PROM_MIN) & (height >= pc.ABS_FLOOR), lam, np.nan)
    return np.asarray(t, float), np.asarray(area, float), np.asarray(amx, float), np.asarray(lam, float)


# ── kinetic descriptors from a single (t, g, lam) trajectory ────────────────────
def _smooth(y, win=7, poly=2):
    n = len(y)
    if n < poly + 2:
        return y
    w = min(win, n if n % 2 else n - 1)
    if w <= poly:
        w = poly + 1 + (poly % 2)
    if w > n:
        return y
    return savgol_filter(y, w, poly)


def _cross(t, phi, thr):
    """Earliest time at which progress `phi` first reaches `thr` (linear interpolation between the
    bracketing frames). NaN if never reached."""
    idx = np.argmax(phi >= thr)
    if not (phi >= thr).any():
        return np.nan
    if idx == 0:
        return float(t[0])
    x0, x1 = phi[idx - 1], phi[idx]
    if x1 == x0:
        return float(t[idx])
    frac = (thr - x0) / (x1 - x0)
    return float(t[idx - 1] + frac * (t[idx] - t[idx - 1]))


def kinetic_features(t, g, lam, tail=TAIL):
    """Scalar kinetic descriptors from a growth signal g(t) (spectral area) and lambda(t).
    Baseline = first frame; final plateau = median of the last `tail` frames. Progress
    phi = (g - g_base)/(g_final - g_base). Returns a dict; kinetic fields are NaN when the run
    does not resolve a real growth (too few frames or negligible area span)."""
    nan = dict(area_final=np.nan, induction_s=np.nan, t50_s=np.nan, t90_s=np.nan,
               t_maxrate_s=np.nan, initial_rate=np.nan, max_growth_rate=np.nan,
               dlambda_redshift=np.nan, lambda_overshoot=np.nan, grew=False)
    t = np.asarray(t, float); g = np.asarray(g, float); lam = np.asarray(lam, float)
    n = len(t)
    g_final = float(np.median(g[-tail:])) if n >= tail else float(g[-1])
    out = dict(nan)
    out["area_final"] = g_final
    if n < MIN_FRAMES:
        return out
    g_base = float(g[0]); span = g_final - g_base
    if g_final <= 0 or span / max(abs(g_final), 1e-9) < GROWTH_MIN:
        return out                                  # no genuine growth -> kinetics undefined
    out["grew"] = True
    phi = (g - g_base) / span
    gsm = _smooth(g)
    rate = np.gradient(gsm, t)                       # d(area)/dt, au·nm/s
    out["induction_s"]     = _cross(t, phi, FRAC_IND)
    out["t50_s"]           = _cross(t, phi, FRAC_50)
    out["t90_s"]           = _cross(t, phi, FRAC_90)
    j = int(np.argmax(rate))
    out["t_maxrate_s"]     = float(t[j])
    out["max_growth_rate"] = float(rate[j])
    # lambda-shape descriptors read ONLY over grown frames (phi >= FRAC_IND) that HAVE a real
    # detected peak (finite lam): before nucleation lam is NaN, so it never contaminates the redshift.
    grown = (phi >= FRAC_IND) & np.isfinite(lam)
    if grown.sum() >= 2:
        lam_g = lam[grown]
        lam_final = float(np.median(lam_g[-tail:])) if len(lam_g) >= tail else float(lam_g[-1])
        out["dlambda_redshift"] = lam_final - float(lam_g[0])
        out["lambda_overshoot"] = float(np.max(lam_g)) - lam_final
    # initial rate: linear slope of raw area over the [induction, t50] rising window
    lo = out["induction_s"] if np.isfinite(out["induction_s"]) else float(t[0])
    hi = out["t50_s"] if np.isfinite(out["t50_s"]) else float(t[min(n - 1, n // 2)])
    m = (t >= lo) & (t <= hi)
    if m.sum() >= 2 and np.ptp(t[m]) > 0:
        out["initial_rate"] = float(np.polyfit(t[m], g[m], 1)[0])
    else:
        out["initial_rate"] = out["max_growth_rate"]
    return out


# ── build the descriptor table (cached) ────────────────────────────────────────
def build_table(force=False):
    """One row per particle run: flows, endpoint descriptors, and kinetic descriptors. Convergence
    time is merged from step36's convergence.csv (same plateau criterion). Cached to kinetics.csv."""
    cache = pc.DATA_DIR / "kinetics.csv"
    if cache.exists() and not force:
        d = pd.read_csv(cache)
        print(f"[kinetics] cache -> {len(d)} runs ({int(d.grew.sum())} with resolved growth)")
        return add_chem_features(d)
    df, S, grid = pc.load("clean")
    dfp = df[df.is_particle.values].reset_index(drop=True)
    print(f"[kinetics] extracting kinetic descriptors for {len(dfp)} particle runs (H5 I/O) ...")
    conv = pd.read_csv(pc.DATA_DIR / "convergence.csv")[["filename", "conv_time_s", "conv_status"]]
    rows = []
    for r in dfp.itertuples():
        ser = read_series(r.filepath)
        base = dict(filename=r.filename, session=r.session,
                    Q1=r.Q1, Q2=r.Q2, Q3=r.Q3, Q4=r.Q4, Q5=r.Q5,
                    lambda_max=r.lambda_max, abs_max=r.abs_max, n_spectra=r.n_spectra)
        if ser is None:
            base.update(kinetic_features(np.array([0.0]), np.array([0.0]), np.array([0.0])))
        else:
            base.update(kinetic_features(ser[0], ser[1], ser[3]))     # (t, area, lambda)
        rows.append(base)
    d = pd.DataFrame(rows).merge(conv, on="filename", how="left")
    d["growth_time_s"] = d["conv_time_s"]           # convergence time = the manuscript's growth_time
    d.to_csv(cache, index=False)
    print(f"[kinetics] {len(d)} runs -> cached kinetics.csv "
          f"({int(d.grew.sum())} with resolved growth, "
          f"{int(d.conv_status.eq('ok').sum())} converged)")
    return add_chem_features(d)


# ── correlation machinery ──────────────────────────────────────────────────────
def _pair_stat(x, y, kind="pearson"):
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 4 or x[ok].std() == 0 or y[ok].std() == 0:
        return np.nan
    if kind == "pearson":
        return float(np.corrcoef(x[ok], y[ok])[0, 1])
    from scipy.stats import spearmanr
    return float(spearmanr(x[ok], y[ok]).correlation)


def corr_matrix(rows_df, cols_df, kind="pearson"):
    R = np.full((rows_df.shape[1], cols_df.shape[1]), np.nan)
    for i, rc in enumerate(rows_df.columns):
        for j, cc in enumerate(cols_df.columns):
            R[i, j] = _pair_stat(rows_df[rc].values.astype(float),
                                 cols_df[cc].values.astype(float), kind)
    return R


def _residualize(y, z):
    """Residuals of y after a linear fit on z (both 1-D), NaN-safe on the shared support."""
    y = np.asarray(y, float); z = np.asarray(z, float)
    r = np.full_like(y, np.nan)
    ok = np.isfinite(y) & np.isfinite(z)
    if ok.sum() < 4 or z[ok].std() == 0:
        return r
    b = np.polyfit(z[ok], y[ok], 1)
    r[ok] = y[ok] - np.polyval(b, z[ok])
    return r


def partial_corr_matrix(rows_df, cols_df, control):
    """Pearson correlation between each row (kinetic) and col (flow) after linearly removing
    `control` (lambda_max) from BOTH — the information a flow carries about the trajectory that is
    NOT already visible in the endpoint plasmon position."""
    z = control.values.astype(float)
    R = np.full((rows_df.shape[1], cols_df.shape[1]), np.nan)
    for i, rc in enumerate(rows_df.columns):
        ry = _residualize(rows_df[rc].values.astype(float), z)
        for j, cc in enumerate(cols_df.columns):
            rx = _residualize(cols_df[cc].values.astype(float), z)
            R[i, j] = _pair_stat(ry, rx, "pearson")
    return R


def mi_matrix(kin_df, end_df):
    """Mutual information between each kinetic descriptor (target) and the endpoint descriptors
    (features), normalised per kinetic descriptor by its own entropy proxy MI(kin;kin)->1 scale via
    dividing by the max across endpoints is misleading; we report raw nats (comparable within a row)."""
    from sklearn.feature_selection import mutual_info_regression
    M = np.full((kin_df.shape[1], end_df.shape[1]), np.nan)
    for i, kc in enumerate(kin_df.columns):
        y = kin_df[kc].values.astype(float)
        ok = np.isfinite(y) & np.isfinite(end_df.values.astype(float)).all(1)
        if ok.sum() < 10:
            continue
        M[i] = mutual_info_regression(end_df.values[ok].astype(float), y[ok],
                                      random_state=SEED)
    return M


# ── redundancy model: predict flows from endpoint-only vs endpoint+kinetic ──────
def redundancy_r2(df, kin_cols=KIN_DESCRIPTORS, end_cols=END_DESCRIPTORS,
                  targets=("Q1", "Q2", "Q3", "Q4", "Q5"), k=CV_K):
    """Leave-one-session-out R^2 for predicting each flow from (a) endpoint descriptors only vs
    (b) endpoint + kinetic descriptors. A gain in (b) means the kinetics carry input information the
    endpoint spectrum does not. RandomForest, GroupKFold by session. Rows with any NaN kinetic
    descriptor are dropped (fair, same rows for both feature sets)."""
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import cross_val_predict, GroupKFold
    use = df.dropna(subset=list(kin_cols) + list(end_cols) + list(targets)).reset_index(drop=True)
    groups = use.session.values
    cv = GroupKFold(n_splits=min(k, use.session.nunique()))
    Xe = use[list(end_cols)].values.astype(float)
    Xk = use[list(end_cols) + list(kin_cols)].values.astype(float)

    def _r2(X, y):
        p = cross_val_predict(RandomForestRegressor(300, max_depth=8, random_state=SEED,
                              n_jobs=sp.N_JOBS), X, y, cv=cv, groups=groups)
        return 1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2)

    rows = []
    for q in targets:
        y = use[q].values.astype(float)
        r2e, r2k = _r2(Xe, y), _r2(Xk, y)
        rows.append(dict(flow=q, r2_endpoint=r2e, r2_endpoint_plus_kinetic=r2k, gain=r2k - r2e))
        print(f"  {q}: endpoint R2={r2e:+.3f}  +kinetic R2={r2k:+.3f}  gain={r2k-r2e:+.3f}")
    return pd.DataFrame(rows), len(use)


# ── figures ────────────────────────────────────────────────────────────────────
def fig_heatmap(R, ylabels, xlabels, cmap="RdBu_r", vmin=-1, vmax=1, cbar="Pearson r", fmt="{:.2f}"):
    fig, ax = plt.subplots(figsize=(0.85 * len(xlabels) + 2.4, 0.55 * len(ylabels) + 1.7))
    im = ax.imshow(R, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(xlabels))); ax.set_xticklabels(xlabels, rotation=40, ha="right")
    ax.set_yticks(range(len(ylabels))); ax.set_yticklabels(ylabels)
    for i in range(R.shape[0]):
        for j in range(R.shape[1]):
            if np.isfinite(R[i, j]):
                frac = abs(R[i, j] - vmin) / max(vmax - vmin, 1e-9)
                ax.text(j, i, fmt.format(R[i, j]), ha="center", va="center", fontsize=8,
                        color="white" if (frac > 0.75 or frac < 0.25) else "black")
    cb = fig.colorbar(im, ax=ax, shrink=0.85); cb.set_label(cbar)
    fig.tight_layout(); return fig


def fig_construction(df, examples=None, n=3, seed=SEED):
    """Show HOW the descriptors are built: area(t) for a few runs with induction / t50 / t_maxrate
    marked, plus lambda(t). Picks runs spanning the growth-time range unless `examples` (filenames)
    are given."""
    grew = df[df.grew & df.conv_status.eq("ok")].dropna(subset=["t50_s", "growth_time_s"])
    if examples:
        sel = grew[grew.filename.isin(examples)]
    else:
        q = grew.growth_time_s.quantile([0.15, 0.5, 0.85]).values
        sel = pd.concat([grew.iloc[(grew.growth_time_s - qq).abs().argsort()[:1]] for qq in q])
    fig, axes = plt.subplots(2, len(sel), figsize=(4.2 * len(sel), 6.2), squeeze=False)
    for col, (_, r) in enumerate(sel.iterrows()):
        ser = read_series(_filepath(r.filename))
        if ser is None:
            continue
        t, area, amx, lam = ser
        g_base = area[0]; g_final = np.median(area[-TAIL:]); span = g_final - g_base
        ax = axes[0, col]
        ax.plot(t, area, "-o", ms=3, color="#0072B2", lw=1.3)
        ax.axhline(g_base + FRAC_IND * span, color="0.7", ls=":", lw=.8)
        ax.axhline(g_base + FRAC_50 * span, color="0.7", ls="--", lw=.8)
        for tv, cc, lb in [(r.induction_s, "#E69F00", "induction"),
                           (r.t50_s, "#009E73", r"$t_{50}$"),
                           (r.t_maxrate_s, "#D55E00", r"$t_{\max\,rate}$")]:
            if np.isfinite(tv):
                ax.axvline(tv, color=cc, lw=1.2, alpha=.9, label=lb)
        ax.set_title(f"{r.filename}\n$t_{{conv}}$={r.growth_time_s:.0f}s", fontsize=8)
        ax.set_xlabel("time (s)"); ax.set_ylabel("spectral area (au·nm)")
        if col == 0:
            ax.legend(fontsize=7, loc="lower right")
        ax2 = axes[1, col]
        # plot lambda(t) only over the grown frames actually used for the redshift descriptor
        # (phi>=FRAC_IND & a real detected peak) — matches the descriptor and drops pre-growth noise
        phi = (area - g_base) / span if span else np.zeros_like(area)
        grown = (phi >= FRAC_IND) & np.isfinite(lam)
        ax2.plot(t[grown], lam[grown], "-o", ms=3, color="#7030A0", lw=1.3)
        ax2.set_xlabel("time (s)"); ax2.set_ylabel(r"$\lambda_{\max}(t)$ (nm)")
        ax2.text(0.95, 0.06, f"$\\Delta\\lambda$={r.dlambda_redshift:+.0f} nm",
                 transform=ax2.transAxes, ha="right", fontsize=8, color="0.3")
    fig.suptitle("Kinetic descriptor construction from area(t) and $\\lambda(t)$", y=1.01)
    fig.tight_layout(); return fig


_FILEPATH_MAP = None
def _filepath(name):
    """filepath for a run name, from the clean catalogue (cached lookup)."""
    global _FILEPATH_MAP
    if _FILEPATH_MAP is None:
        df, _, _ = pc.load("clean")
        _FILEPATH_MAP = dict(zip(df.filename, df.filepath))
    return _FILEPATH_MAP[name]


def fig_kinetic_pairs(df, descriptors=KIN_DESCRIPTORS, color_by="lambda_max"):
    """Kinetic descriptors against each other, coloured by endpoint lambda_max — the visual 'are the
    kinetics just a proxy for the final colour?' check."""
    d = df[df.grew].copy()
    n = len(descriptors); labels = [KIN_LABELS[x] for x in descriptors]
    fig, axes = plt.subplots(n, n, figsize=(1.7 * n, 1.7 * n))
    cvals = d[color_by].values
    norm = mpl.colors.Normalize(np.nanmin(cvals), np.nanmax(cvals)); cmap = mpl.cm.viridis
    for i in range(n):
        for j in range(n):
            ax = axes[i, j]
            xi, yi = d[descriptors[j]].values, d[descriptors[i]].values
            ok = np.isfinite(xi) & np.isfinite(yi)
            if i == j:
                ax.hist(xi[np.isfinite(xi)], bins=14, color="#888888"); ax.set_yticks([])
            elif i > j:
                ax.scatter(xi[ok], yi[ok], c=cvals[ok], cmap=cmap, norm=norm, s=13,
                           edgecolor="white", linewidth=0.2)
            else:
                r = _pair_stat(xi, yi)
                if np.isfinite(r):
                    ax.set_facecolor(mpl.cm.RdBu_r((r + 1) / 2))
                    ax.text(0.5, 0.5, f"{r:.2f}", ha="center", va="center", fontsize=9,
                            fontweight="bold", transform=ax.transAxes,
                            color="white" if abs(r) > 0.5 else "black")
                ax.set_xticks([]); ax.set_yticks([])
            if i == n - 1:
                ax.set_xlabel(labels[j], fontsize=7, rotation=40, ha="right")
            elif i != j:
                ax.set_xticklabels([])
            if j == 0:
                ax.set_ylabel(labels[i], fontsize=7)
            elif i != j:
                ax.set_yticklabels([])
            ax.tick_params(labelsize=6)
    fig.tight_layout(); fig.subplots_adjust(right=0.90)
    cax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    fig.colorbar(sm, cax=cax).set_label(END_LABELS.get(color_by, color_by))
    return fig


def fig_redundancy(red_df):
    fig, ax = plt.subplots(figsize=(7, 4.2))
    x = np.arange(len(red_df)); w = 0.38
    ax.bar(x - w / 2, red_df.r2_endpoint, w, label="endpoint only", color="#999999")
    ax.bar(x + w / 2, red_df.r2_endpoint_plus_kinetic, w, label="endpoint + kinetic", color="#0072B2")
    ax.set_xticks(x); ax.set_xticklabels([f"{q}\n{eda.REAGENTS[i]}" for i, q in enumerate(red_df.flow)])
    ax.set_ylabel("leave-session-out $R^2$ (predict flow)")
    ax.axhline(0, color="k", lw=.6)
    ax.set_title("Can we recover the FLOW from the spectrum?  endpoint vs +kinetic")
    for xi, (_, row) in zip(x, red_df.iterrows()):
        ax.text(xi + w / 2, row.r2_endpoint_plus_kinetic + 0.01,
                f"{row.gain:+.2f}", ha="center", fontsize=7, color="#0072B2")
    ax.legend(); fig.tight_layout(); return fig


# ── run ────────────────────────────────────────────────────────────────────────
def main():
    df = build_table()
    grew = df[df.grew].reset_index(drop=True)
    print(f"\n{len(df)} particle runs | {len(grew)} with resolved growth kinetics")
    print("\nKinetic descriptor summary (resolved-growth runs):")
    print(grew[KIN_DESCRIPTORS].describe().T[["mean", "std", "min", "max"]].round(3).to_string())

    xlab_flows = FLOW_LABELS
    ylab_kin = [KIN_LABELS[d] for d in KIN_DESCRIPTORS]

    print("\n[1] Pearson: kinetic descriptors vs raw flows")
    R = corr_matrix(grew[KIN_DESCRIPTORS], grew[list(sp.FEATURES)])
    savefig(fig_heatmap(R, ylab_kin, xlab_flows), "fig42_pearson_flows")

    print("[2] Spearman: kinetic descriptors vs chemistry ratios")
    Rs = corr_matrix(grew[KIN_DESCRIPTORS], grew[CHEM_FEATURES], kind="spearman")
    savefig(fig_heatmap(Rs, ylab_kin, CHEM_LABELS, cbar="Spearman rho"), "fig42_spearman_chem")

    print("[3] partial correlation vs flows, controlling lambda_max")
    Rp = partial_corr_matrix(grew[KIN_DESCRIPTORS], grew[list(sp.FEATURES)], grew["lambda_max"])
    savefig(fig_heatmap(Rp, ylab_kin, xlab_flows,
                        cbar=r"partial Pearson r  ($\lambda_{\max}$ removed)"),
            "fig42_partial_corr_flows")

    print("[4] mutual information: kinetic vs endpoint descriptors")
    M = mi_matrix(grew[KIN_DESCRIPTORS], grew[END_DESCRIPTORS])
    savefig(fig_heatmap(M, ylab_kin, [END_LABELS[e] for e in END_DESCRIPTORS], cmap="viridis",
                        vmin=0, vmax=np.nanmax(M), cbar="mutual information (nats)", fmt="{:.2f}"),
            "fig42_mi_endpoint")

    print("[5] redundancy model: predict flows from endpoint vs endpoint+kinetic")
    red, n_used = redundancy_r2(df)
    red.to_csv(pc.DATA_DIR / "kinetic_redundancy_r2.csv", index=False)
    print(f"  (n={n_used} complete-kinetics runs)")
    savefig(fig_redundancy(red), "fig42_redundancy")

    print("[6] descriptor construction + kinetic pairplot")
    savefig(fig_construction(df), "fig42_construction")
    savefig(fig_kinetic_pairs(df), "fig42_kinetic_pairplot")

    print(f"\nDone -> {pc.FIG_DIR.resolve()}")


if __name__ == "__main__":
    main()


### `ml.step43_endpoint_degeneracy`
*Original analysis code.*

In [ ]:
%%module toofast_analysis.ml.step43_endpoint_degeneracy
# [vendored] verbatim copy of 2FAST_ML/pipeline/step43_endpoint_degeneracy.py (md5 1626c621f69c46d2d14db748b4053636), generated by tools/vendor_ml.py — do not edit by hand.
"""
step43_endpoint_degeneracy.py  —  pipeline step 13  (mirrors 43_endpoint_degeneracy.ipynb)
==========================================================================================
Task A of the manuscript brief — the NECESSITY argument for trajectory observation:

    "Distinct flow conditions can produce near-identical FINAL spectra while following measurably
     different formation trajectories. Any endpoint-only platform would treat these as equivalent."

Method:
  1. peak-normalise every particle final spectrum -> pairwise COSINE-distance matrix (raw spectra AND
     the PCA(4) projection the BO cost actually optimises — `PCASpectrumCost(cosine, k=4)`);
  2. keep candidate pairs below an endpoint-similarity threshold EPS (swept — sensitivity reported);
  3. for each candidate pair compare the step-42 KINETIC descriptors (growth_time, induction,
     max_growth_rate, t50); measure divergence in units of each descriptor's intra-session
     repeatability floor (estimated by frame-bootstrap, a measurement lower bound) AND its robust
     population spread (MAD);
  4. retain endpoint-similar / trajectory-divergent pairs; report INTRA-session pairs separately
     (same reducer age -> divergence is chemistry, not session drift = the strong evidence).

Deliverable: a two-panel figure for the clearest intra-session pair (near-identical finals; divergent
area(t)/lambda(t)) + a summary table of all retained pairs. A NEGATIVE result (no such pairs) is
reported, not discarded — it would mean the endpoint is not degenerate over the sampled domain.

Reuses step-42's kinetics.csv. Runs in the Anaconda 3.11 kernel. Reads the H5 time series only for the
handful of highlighted/bootstrapped runs.
"""
from pathlib import Path
from itertools import combinations
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from . import pipeline_common as pc
from . import step42_kinetic_descriptors as k42

sp = pc.sp
SEED = pc.SEED
NOTEBOOK = "43_endpoint_degeneracy.ipynb"
rng = np.random.default_rng(SEED)

# ── endpoint-similarity thresholds (cosine distance on peak-normalised finals) ──
#   The BO cost is a cosine distance on a PCA(4) projection; "a good match" for the loop is a few
#   ×1e-3. We sweep EPS across that scale and report how many degenerate pairs survive each.
EPS_SWEEP = [0.002, 0.005, 0.010, 0.020, 0.050]
EPS_MAIN  = 0.010            # headline threshold (≈ BO "near-identical endpoint")
N_PC_COST = 4                # PCASpectrumCost k=4 — the space the BO objective lives in
TRAJ_DESCRIPTORS = ["growth_time_s", "induction_s", "max_growth_rate", "t50_s"]
TRAJ_LABELS = {d: k42.KIN_LABELS[d] for d in TRAJ_DESCRIPTORS}
N_BOOT = 200

# ── raw (non-normalised) degeneracy: shape AND amplitude match ────────────────
#   Cosine distance is SCALE-INVARIANT: cosine(S_i, S_j) == cosine(peak-norm(S_i), peak-norm(S_j))
#   exactly, for any positive per-row scaling. So `endpoint_cos` above already measures SHAPE alone,
#   regardless of normalisation — a pair with tiny endpoint_cos can still have wildly different RAW
#   amplitude (as seen in the headline pair: 0.175 vs 0.108 a.u., ~1.6x). "Degenerate in the raw
#   spectrum" therefore needs a SECOND, amplitude-matching criterion on top of the shape one: two
#   recipes are raw-indistinguishable only if BOTH the shape (cosine) AND the endpoint amplitude
#   (abs_max) agree. Amplitude agreement is judged in MAD units of the population's abs_max spread
#   (brief convention: thresholds relative to measured spread, not a fixed constant).
Z_AMP_SWEEP = [0.5, 1.0, 1.5, 2.0, 3.0]
Z_AMP_MAIN  = 1.0            # "amplitude within ~1 typical run-to-run difference" -> counts as a match


def savefig(fig, name):
    fig.text(0.995, 0.003, f"source: {NOTEBOOK}", ha="right", va="bottom", fontsize=6, color="0.55")
    for ext in ("png", "pdf"):
        fig.savefig(pc.FIG_DIR / f"{name}.{ext}", dpi=200, bbox_inches="tight")
    print(f"  saved {name}.png/.pdf")


# ── data: particle finals + their kinetic descriptors, aligned ─────────────────
def load():
    """(df, S, Sn, grid): particle runs with resolved growth kinetics, their RAW (absolute
    absorbance) final spectra `S`, the peak-normalised finals `Sn` (shape only — what the BO cosine
    cost actually sees), and the common wl grid. df carries flows, session, endpoint + kinetic
    descriptors. Both S and Sn are returned so figures can show the degeneracy in shape (normalised —
    the BO's view) AND in absolute amplitude (raw — what the instrument actually recorded)."""
    kin = k42.build_table()                        # cached kinetics.csv (+ chem features)
    dfc, S, grid = pc.load("clean")
    dfc = dfc[dfc.is_particle].reset_index(drop=True)
    # align S rows to kin by filename
    idx = {f: i for i, f in enumerate(dfc.filename)}
    keep = [i for i, f in enumerate(kin.filename) if f in idx]
    kin = kin.iloc[keep].reset_index(drop=True)
    S = S[[idx[f] for f in kin.filename]]
    # only runs with a resolved growth trajectory AND a measured convergence time
    ok = kin.grew.values & np.isfinite(kin.growth_time_s.values)
    kin = kin[ok].reset_index(drop=True); S = S[ok]
    filepath = dict(zip(dfc.filename, dfc.filepath))
    kin["filepath"] = kin.filename.map(filepath)
    Sn, _ = sp.normalise(S, "max")                 # peak-normalised finals (shape only)
    print(f"[degeneracy] {len(kin)} particle runs with resolved kinetics / "
          f"{kin.session.nunique()} sessions")
    return kin, S, Sn, grid


# ── cosine-distance matrices (raw shape + PCA(4) = BO cost space) ───────────────
def cosine_dist(X):
    Xn = X / np.clip(np.linalg.norm(X, axis=1, keepdims=True), 1e-12, None)
    D = 1.0 - Xn @ Xn.T
    np.fill_diagonal(D, 0.0)
    return np.clip(D, 0, 2)


def distance_matrices(Sn):
    """Endpoint cosine distance on (a) the raw peak-normalised spectra and (b) the PCA(N_PC_COST)
    projection that the BO cosine cost actually operates on."""
    D_raw = cosine_dist(Sn)
    mu, comps, _ = sp.fit_pca(Sn, N_PC_COST); scores = (Sn - mu) @ comps.T
    D_pca = cosine_dist(scores)
    return D_raw, D_pca


# ── pair table: endpoint distance + trajectory divergence ──────────────────────
def population_mad(df, cols):
    """Robust population spread (MAD, scaled to sigma) per descriptor — the yardstick for 'a
    typical run-to-run difference'."""
    out = {}
    for c in cols:
        v = df[c].values.astype(float); v = v[np.isfinite(v)]
        out[c] = 1.4826 * np.median(np.abs(v - np.median(v))) if len(v) else np.nan
    return out


def build_pairs(df, D_raw, D_pca, eps=EPS_MAIN, cols=TRAJ_DESCRIPTORS, z_amp_max=Z_AMP_MAIN):
    """Every pair whose ENDPOINT distance (raw cosine) < eps, annotated with each trajectory
    descriptor's absolute difference and its difference in MAD units, whether the pair is
    intra-session, and a summed divergence score (sum of |Δ|/MAD over descriptors). Also annotates
    `d_abs_max`/`z_abs_max` (endpoint amplitude difference, in MAD units of the population's abs_max
    spread) and `raw_degenerate` = shape-similar (already guaranteed by `eps`) AND amplitude-similar
    (z_abs_max < z_amp_max) — the pair is indistinguishable in the RAW, non-normalised spectrum, not
    merely in shape."""
    mad = population_mad(df, cols)
    mad_amp = population_mad(df, ["abs_max"])["abs_max"] if "abs_max" in df.columns else np.nan
    n = len(df); rows = []
    for i, j in combinations(range(n), 2):
        if D_raw[i, j] >= eps:
            continue
        row = dict(run_i=df.filename[i], run_j=df.filename[j],
                   session_i=df.session[i], session_j=df.session[j],
                   intra_session=(df.session[i] == df.session[j]),
                   endpoint_cos=D_raw[i, j], endpoint_cos_pca=D_pca[i, j],
                   lambda_i=df.lambda_max[i], lambda_j=df.lambda_max[j],
                   d_lambda_max=abs(df.lambda_max[i] - df.lambda_max[j]))
        if "abs_max" in df.columns:
            d_amp = abs(df.abs_max[i] - df.abs_max[j])
            z_amp = d_amp / mad_amp if mad_amp and np.isfinite(mad_amp) else np.nan
            row["d_abs_max"] = d_amp; row["z_abs_max"] = z_amp
            row["raw_degenerate"] = bool(np.isfinite(z_amp) and z_amp < z_amp_max)
        score = 0.0
        for c in cols:
            a, b = df[c].values[i], df[c].values[j]
            d = abs(a - b); dn = d / mad[c] if mad[c] and np.isfinite(mad[c]) else np.nan
            row[f"d_{c}"] = d; row[f"z_{c}"] = dn
            if np.isfinite(dn):
                score += dn
        row["divergence_score"] = score
        rows.append(row)
    pairs = pd.DataFrame(rows)
    if len(pairs):
        pairs = pairs.sort_values("divergence_score", ascending=False).reset_index(drop=True)
    return pairs, mad


def sweep_thresholds(df, D_raw, D_pca, eps_list=EPS_SWEEP, cols=TRAJ_DESCRIPTORS,
                     z_divergent=1.0):
    """Sensitivity table: at each endpoint threshold eps, how many endpoint-similar pairs exist, how
    many are trajectory-divergent (growth_time differs by > z_divergent MADs), and how many of those
    are intra-session."""
    rows = []
    for eps in eps_list:
        pairs, _ = build_pairs(df, D_raw, D_pca, eps=eps, cols=cols)
        if len(pairs):
            div = pairs[pairs["z_growth_time_s"] > z_divergent]
            rows.append(dict(eps=eps, n_pairs=len(pairs), n_divergent=len(div),
                             n_divergent_intra=int(div.intra_session.sum())))
        else:
            rows.append(dict(eps=eps, n_pairs=0, n_divergent=0, n_divergent_intra=0))
    return pd.DataFrame(rows)


# ── select several pairs for the gallery: intra AND inter session ──────────────
def select_pairs(pairs, n_each=4, lam_agree=15.0):
    """Up to `n_each` intra-session AND `n_each` inter-session pairs, ranked by divergence score.
    Within each group, pairs whose finals are GENUINELY the same colour (|Δλ_max| < lam_agree) are
    preferred (they isolate the trajectory message from the separate colour-degeneracy of the BO
    cost); pairs failing that are used only to fill out the quota. Returns a DataFrame tagged with a
    `kind` column ('intra'/'inter') and a `clean` flag, ordered intra-then-inter for the gallery."""
    if not len(pairs):
        return pairs.assign(kind=[], clean=[])

    def _pick(sub):
        sub = sub.sort_values("divergence_score", ascending=False)
        clean = sub[sub.d_lambda_max < lam_agree]
        rest = sub[sub.d_lambda_max >= lam_agree]
        return pd.concat([clean, rest]).head(n_each)

    intra = _pick(pairs[pairs.intra_session]).assign(kind="intra")
    inter = _pick(pairs[~pairs.intra_session]).assign(kind="inter")
    sel = pd.concat([intra, inter]).reset_index(drop=True)
    sel["clean"] = sel.d_lambda_max < lam_agree
    return sel


def sweep_amp_thresholds(pairs, z_amp_list=Z_AMP_SWEEP):
    """Sensitivity table for the SECOND (amplitude) criterion, at the fixed shape threshold already
    used to build `pairs`: how many of the shape-similar pairs are ALSO amplitude-similar (raw-
    degenerate) at each z_amp cutoff, and how many of those are intra-session."""
    rows = []
    for z in z_amp_list:
        if not len(pairs) or "z_abs_max" not in pairs.columns:
            rows.append(dict(z_amp_max=z, n_raw_degenerate=0, n_raw_degenerate_intra=0)); continue
        deg = pairs[pairs.z_abs_max < z]
        rows.append(dict(z_amp_max=z, n_raw_degenerate=len(deg),
                         n_raw_degenerate_intra=int(deg.intra_session.sum())))
    return pd.DataFrame(rows)


def select_raw_degenerate_pairs(pairs, n_each=4, z_amp_max=Z_AMP_MAIN, z_amp_tight=0.5, lam_agree=15.0):
    """Pairs degenerate in the RAW (non-normalised) spectrum — shape AND amplitude both match
    (`z_abs_max < z_amp_max`, the reported/honest threshold). Within that qualifying pool, the
    gallery PREFERS a tighter amplitude match (`z_abs_max < z_amp_tight`) so the example figure is
    visually convincing rather than merely population-relative — MAD(abs_max) is large enough here
    that 'within 1 MAD' can still look like a real amplitude gap on the raw panel. Falls back to the
    full qualifying pool (still amplitude-matched at z_amp_max, never unfiltered) if too few tight
    pairs exist in a group. NO fallback to the shape-only pool: if a group is empty even at
    z_amp_max, that is reported honestly rather than padded."""
    if not len(pairs) or "raw_degenerate" not in pairs.columns:
        return pairs.assign(kind=[], clean=[])
    deg = pairs[pairs.z_abs_max < z_amp_max].copy()
    if not len(deg):
        return deg.assign(kind=[], clean=[])

    def _pick(sub):
        sub = sub.sort_values("divergence_score", ascending=False)
        tight = sub[sub.z_abs_max < z_amp_tight]
        rest = sub[sub.z_abs_max >= z_amp_tight]
        return pd.concat([tight, rest]).head(n_each)

    intra = _pick(deg[deg.intra_session]).assign(kind="intra")
    inter = _pick(deg[~deg.intra_session]).assign(kind="inter")
    sel = pd.concat([intra, inter]).reset_index(drop=True)
    sel["clean"] = sel.d_lambda_max < lam_agree     # colour-agreement flag, informational here
    return sel


# ── intra-session repeatability floor via frame-bootstrap ──────────────────────
def bootstrap_repeatability(filepath, cols=("growth_time_s", "induction_s", "max_growth_rate",
                                            "t50_s"), n_boot=N_BOOT):
    """Measurement-repeatability lower bound for one run's kinetic descriptors: resample its spectral
    frames (block-bootstrap in time) and recompute the descriptors, returning the bootstrap std per
    descriptor. This is a floor on intra-session repeatability — a pair whose Δdescriptor dwarfs it
    is divergent for chemical, not measurement, reasons."""
    ser = k42.read_series(filepath)
    if ser is None:
        return {c: np.nan for c in cols}
    t, area, amx, lam = ser
    n = len(t)
    if n < k42.MIN_FRAMES:
        return {c: np.nan for c in cols}
    acc = {c: [] for c in cols}
    for _ in range(n_boot):
        # jitter each frame's area/lambda with a small resampling of neighbouring frames (keeps the
        # time axis, perturbs the signal within its local noise) -> descriptor sampling distribution
        pert = np.clip(np.arange(n) + rng.integers(-1, 2, n), 0, n - 1)
        feat = k42.kinetic_features(t, area[pert], lam[pert])
        for c in cols:
            if c == "growth_time_s":
                continue                       # convergence time comes from step36, not resampled here
            if c in feat and np.isfinite(feat[c]):
                acc[c].append(feat[c])
    return {c: (float(np.std(acc[c])) if len(acc[c]) > 5 else np.nan) for c in cols}


# ── figures ────────────────────────────────────────────────────────────────────
def fig_distance_overview(df, D_raw, pairs):
    """(a) distribution of pairwise endpoint cosine distances with EPS marked; (b) endpoint distance
    vs trajectory divergence for every pair below the largest swept EPS — the degeneracy 'wedge':
    points at LOW endpoint distance but HIGH divergence are the degenerate pairs."""
    iu = np.triu_indices(len(df), 1)
    dist = D_raw[iu]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
    ax[0].hist(dist, bins=60, color="#0072B2"); ax[0].set_yscale("log")
    ax[0].axvline(EPS_MAIN, color="#D55E00", lw=1.4, label=f"EPS={EPS_MAIN}")
    ax[0].set_xlabel("endpoint cosine distance (peak-norm. finals)")
    ax[0].set_ylabel("pair count"); ax[0].set_title("(a) how similar are the finals?")
    ax[0].set_xlim(0, np.percentile(dist, 60)); ax[0].legend(fontsize=8)
    if len(pairs):
        c = np.where(pairs.intra_session, "#D55E00", "#999999")
        ax[1].scatter(pairs.endpoint_cos, pairs.divergence_score, c=c, s=26,
                      edgecolor="white", lw=.3)
        ax[1].set_xlabel(f"endpoint cosine distance (< {EPS_MAIN})")
        ax[1].set_ylabel(r"trajectory divergence  $\sum_d |\Delta d|/\mathrm{MAD}_d$")
        ax[1].set_title("(b) endpoint-similar but trajectory-divergent (orange = intra-session)")
        from matplotlib.lines import Line2D
        ax[1].legend(handles=[Line2D([0], [0], marker="o", ls="", color="#D55E00", label="intra-session"),
                              Line2D([0], [0], marker="o", ls="", color="#999999", label="inter-session")],
                     fontsize=8)
    fig.tight_layout(); return fig


def _pair_indices(df, pair_row):
    i = int(np.where(df.filename == pair_row.run_i)[0][0])
    j = int(np.where(df.filename == pair_row.run_j)[0][0])
    return i, j


def _plot_pair_panels(axp, df, S, Sn, grid, pair_row, legend=False, wl_max=900):
    """Draw the four degeneracy panels for ONE pair onto the given 1x4 array of axes:
    (a) normalised finals — shape only, what the BO cosine cost sees;
    (b) RAW finals — absolute absorbance, what the instrument actually recorded;
    (c) area(t); (d) lambda(t) over grown frames.
    Shared by the single-pair figure and the multi-pair gallery so both stay in sync."""
    i, j = _pair_indices(df, pair_row)
    ser_i = k42.read_series(df.filepath[i]); ser_j = k42.read_series(df.filepath[j])
    ci, cj = "#0072B2", "#D55E00"
    a0, a1, a2, a3 = axp
    li = f"{pair_row.run_i}" + (f" (Q: " + " ".join(f"{df[q][i]:.1f}" for q in sp.FEATURES) + ")" if legend else "")
    lj = f"{pair_row.run_j}" + (f" (Q: " + " ".join(f"{df[q][j]:.1f}" for q in sp.FEATURES) + ")" if legend else "")
    xlim = (grid.min(), min(grid.max(), wl_max))
    # (a) normalised finals
    a0.plot(grid, Sn[i], color=ci, lw=1.6, label=li)
    a0.plot(grid, Sn[j], color=cj, lw=1.6, ls="--", label=lj)
    a0.set_xlim(*xlim)
    if legend:
        a0.legend(fontsize=6, loc="upper right")
    # (b) raw (non-normalised) finals
    a1.plot(grid, S[i], color=ci, lw=1.6)
    a1.plot(grid, S[j], color=cj, lw=1.6, ls="--")
    a1.set_xlim(*xlim)
    if "d_abs_max" in pair_row.index and np.isfinite(pair_row.d_abs_max):
        a1.text(0.02, 0.03, f"Δabs_max={pair_row.d_abs_max:.3f}", transform=a1.transAxes,
                fontsize=6.5, color="0.3")
    # (c) area(t)
    for ser, c, lsty in [(ser_i, ci, "-"), (ser_j, cj, "--")]:
        if ser is not None:
            a2.plot(ser[0], ser[1], marker="o", ms=3, color=c, lw=1.3, ls=lsty)
    # (d) lambda(t) over grown frames
    for ser, c, lsty in [(ser_i, ci, "-"), (ser_j, cj, "--")]:
        if ser is None:
            continue
        t, area, amx, lam = ser
        g_base = area[0]; span = np.median(area[-k42.TAIL:]) - g_base
        phi = (area - g_base) / span if span else np.zeros_like(area)
        gm = (phi >= k42.FRAC_IND) & np.isfinite(lam)
        a3.plot(t[gm], lam[gm], marker="o", ms=3, color=c, lw=1.3, ls=lsty)
    return i, j


def fig_degenerate_pair(df, S, Sn, grid, pair_row, boot_i=None, boot_j=None):
    """The Task-A headline figure for ONE pair: (a) normalised finals (BO's view), (b) RAW finals
    (instrument's view), (c) area(t), (d) lambda(t) — near-identical endpoint, divergent path.
    Bootstrap repeatability (if given) annotates the Δmax-growth-rate significance."""
    fig, ax = plt.subplots(1, 4, figsize=(19, 4.6))
    i, j = _plot_pair_panels(ax, df, S, Sn, grid, pair_row, legend=True)
    ax[0].set_xlabel("wavelength (nm)"); ax[0].set_ylabel("peak-norm. extinction")
    ax[0].set_title(f"(a) FINAL, normalised — cosine dist = {pair_row.endpoint_cos:.4f}")
    ax[1].set_xlabel("wavelength (nm)"); ax[1].set_ylabel("extinction (raw, a.u.)")
    ax[1].set_title(f"(b) FINAL, raw (Δλ_max = {pair_row.d_lambda_max:.1f} nm)")
    ax[2].set_xlabel("time (s)"); ax[2].set_ylabel("spectral area (au·nm)")
    gt = f"$t_{{conv}}$: {pair_row.d_growth_time_s:.0f}s apart " \
         f"({df.growth_time_s[i]:.0f} vs {df.growth_time_s[j]:.0f})"
    ax[2].set_title(f"(c) AREA(t) — {gt}")
    ax[3].set_xlabel("time (s)"); ax[3].set_ylabel(r"$\lambda_{\max}(t)$ (nm)")
    ax[3].set_title(f"(d) $\\lambda(t)$ — max growth rate "
                    f"{df.max_growth_rate[i]:.1f} vs {df.max_growth_rate[j]:.1f} au·nm/s")
    tag = "INTRA-session" if pair_row.intra_session else "inter-session"
    sub = f"{tag} pair"
    if boot_i is not None and np.isfinite(boot_i.get("max_growth_rate", np.nan)):
        sub += (f"   |   repeatability(max rate) ≈ ±{boot_i['max_growth_rate']:.1f} / "
                f"±{boot_j['max_growth_rate']:.1f} au·nm/s (frame-bootstrap)")
    fig.suptitle(f"Endpoint degeneracy — {sub}", y=1.03, fontsize=11)
    fig.tight_layout(); return fig


def fig_degenerate_pairs_grid(df, S, Sn, grid, sel_pairs, title=None):
    """Gallery of SEVERAL degenerate pairs — both INTRA- and INTER-session — one row per pair, four
    columns: (a) normalised finals (the BO's view — near-identical by construction, cosine < EPS),
    (b) the SAME finals raw/non-normalised (the instrument's view — shows the true amplitude, and
    exposes any colour mismatch for the non-'clean' fill-in pairs), (c) area(t), (d) lambda(t). Column
    headers on the top row only; each row is labelled with the pair + intra/inter tag + Δλ_max/Δt_conv
    on its own two left panels so the story is legible without a shared legend. Pass `title` to
    override the default suptitle (e.g. for the raw/amplitude-matched variant of this gallery)."""
    n = len(sel_pairs)
    if n == 0:
        fig, ax = plt.subplots(figsize=(6, 2))
        ax.axis("off")
        ax.text(0.5, 0.5, "no pairs match this criterion", ha="center", va="center", fontsize=11)
        return fig
    fig, axes = plt.subplots(n, 4, figsize=(17, 2.9 * n), squeeze=False)
    col_titles = ["(a) FINAL — normalised (BO's view)", "(b) FINAL — raw (instrument's view)",
                 "(c) AREA(t)", "(d) $\\lambda_{\\max}(t)$"]
    for row, (_, pr) in enumerate(sel_pairs.iterrows()):
        axp = axes[row]
        i, j = _plot_pair_panels(axp, df, S, Sn, grid, pr, legend=(row == 0))
        tag = "INTRA-session" if pr.intra_session else "inter-session"
        clean_tag = "" if pr.get("clean", pr.d_lambda_max < 15.0) else "  [Δλ large]"
        axp[0].set_ylabel(f"{pr.run_i}\nvs {pr.run_j}", fontsize=7)
        axp[0].text(0.02, 0.03, f"[{tag}]{clean_tag}", transform=axp[0].transAxes,
                    fontsize=6.5, color="0.3")
        axp[2].set_title(f"$\\Delta t_{{conv}}$={pr.d_growth_time_s:.0f}s "
                         f"({df.growth_time_s[i]:.0f} vs {df.growth_time_s[j]:.0f})", fontsize=7)
        axp[3].set_title(f"$\\Delta\\lambda_{{\\max}}$={pr.d_lambda_max:.0f}nm  "
                         f"cos={pr.endpoint_cos:.4f}", fontsize=7)
        if row == 0:
            for a, t in zip(axp, col_titles):
                a.set_title(t, fontsize=8)
            axp[2].set_title(col_titles[2] + f"\n$\\Delta t_{{conv}}$={pr.d_growth_time_s:.0f}s "
                             f"({df.growth_time_s[i]:.0f} vs {df.growth_time_s[j]:.0f})", fontsize=7)
            axp[3].set_title(col_titles[3] + f"\n$\\Delta\\lambda_{{\\max}}$={pr.d_lambda_max:.0f}nm  "
                             f"cos={pr.endpoint_cos:.4f}", fontsize=7)
        if row == n - 1:
            axp[0].set_xlabel("wavelength (nm)"); axp[1].set_xlabel("wavelength (nm)")
            axp[2].set_xlabel("time (s)"); axp[3].set_xlabel("time (s)")
        axp[1].set_ylabel("extinction (a.u.)", fontsize=7)
        axp[2].set_ylabel("area (au·nm)", fontsize=7); axp[3].set_ylabel(r"$\lambda_{\max}$ (nm)", fontsize=7)
        for a in axp:
            a.tick_params(labelsize=6)
    default_title = (f"Endpoint-degenerate pairs — {int(sel_pairs.kind.eq('intra').sum())} "
                     f"intra-session + {int(sel_pairs.kind.eq('inter').sum())} inter-session")
    fig.suptitle(title or default_title, y=1.005, fontsize=11)
    fig.tight_layout(); return fig


# ── standalone single-pair figures (large font, presentation size, square) ─────
# ── LEGEND KNOBS for `fig_pair_spectra_solo` — edit here, no need to touch the plotting code ──
#   The legend text is assembled from templates, so wording / units / reagent names / placement are
#   one-line changes. Every knob also has a per-call override (see the function's arguments), so a
#   single figure can deviate without changing the defaults for the whole notebook.
FLOW_UNIT      = "µL/s"                       # the instrument's flow unit (Q_tot = 14 µL/s)
REAGENT        = {"Q1": "AgNO3", "Q2": "PVP", "Q3": "AcOH", "Q4": "Asc", "Q5": "NaBH4"}
FLOW_ITEM_FMT  = "Q{reagent} = {value:.2f}"   # one flow inside the flow line
FLOW_SEP       = ", "                         # separator between flows
FLOW_WRAP      = 3                            # flows per line (0/None = one long line)
SOLO_LABEL_FMT = ("Curve {n}: {name}  —  growth time {growth_time:.0f} s"
                  "\nFlow ({unit}): {flows}")  # {n} {name} {growth_time} {flows} {unit} available
SOLO_LEGEND_KW = dict(loc="upper right", frameon=True, facecolor="white", framealpha=0.92,
                      edgecolor="0.75", labelcolor="black", handlelength=1.6, handletextpad=0.6,
                      borderaxespad=0.5, labelspacing=0.5)
SOLO_HEADROOM  = 0.42                         # top padding (fraction of the curve span) for the legend
                                              # — ignored when an explicit `ymax` is given
# ── same idea for the area-vs-time figure ─────────────────────────────────────
AREA_LABEL_FMT = "Curve {n}: {name}"
AREA_ANNOT_FMT = "growth time {growth_time:.0f} s"   # callout on the emphasised LAST point
AREA_LEGEND_KW = dict(loc="lower right", frameon=True, facecolor="white", framealpha=0.92,
                      edgecolor="0.75", labelcolor="black")


def _flow_text(df, k, reagent=None, item_fmt=None, sep=None, wrap=None):
    """The flow line: each reagent flow named by its CHEMICAL (QAgNO3, QPVP, …) rather than Q1..Q5,
    wrapped onto a new line every `wrap` flows so the legend stays narrow."""
    reagent = REAGENT if reagent is None else reagent
    item_fmt = FLOW_ITEM_FMT if item_fmt is None else item_fmt
    sep = FLOW_SEP if sep is None else sep
    wrap = FLOW_WRAP if wrap is None else wrap
    items = [item_fmt.format(reagent=reagent.get(q, q), flow=q, value=df[q][k]) for q in sp.FEATURES]
    if not wrap:
        return sep.join(items)
    rows = [items[a:a + wrap] for a in range(0, len(items), wrap)]
    return ("\n").join(sep.join(r) + (sep.rstrip() if r is not rows[-1] else "") for r in rows)


def _flow_label(df, k, name, n=1, fmt=None, unit=None, **flow_kw):
    """Legend label: curve number + sample name + its GROWTH (convergence) time — the trajectory
    quantity the identical endpoints hide — then its reagent flows. All in the legend's own (black)
    text colour — the curve colour is carried by the legend's line swatch only."""
    return (SOLO_LABEL_FMT if fmt is None else fmt).format(
        n=n, name=name, growth_time=df.growth_time_s[k], unit=FLOW_UNIT if unit is None else unit,
        flows=_flow_text(df, k, **flow_kw))


def fig_pair_spectra_solo(df, S, Sn, grid, name_i, name_j, use="raw", wl_max=900, fs=18,
                          labels=None, label_fmt=None, unit=None, legend_kw=None, headroom=None,
                          ymin=None, ymax=None):
    """Standalone SQUARE spectra figure for ONE named pair — both finals in SOLID lines (colour is
    the only distinguishing cue), large fonts throughout. COMPACT layout: the axes carry a full BOX
    frame (all four spines) and the legend sits INSIDE it, in headroom opened above the curves, so
    the figure is the box and nothing else. Each entry names the run, its GROWTH TIME (the
    trajectory quantity the degenerate endpoints hide) and its reagent flows (AgNO3 … NaBH4, in
    `FLOW_UNIT`), in black legend text — colour-coding stays in the line swatch.

    `use`: "raw" (S, instrument amplitude, default) or "norm" (Sn, peak-normalised shape — the BO
    cost's view). Legend overrides (defaults = the `SOLO_*` module constants above):
      `labels`     two ready-made legend strings, bypassing the templates entirely (None = keep
                   the generated one for that curve);
      `label_fmt`  a different template ({n}/{name}/{growth_time}/{flows}/{unit});
      `unit`       flow unit shown in the label;
      `legend_kw`  merged over `SOLO_LEGEND_KW` (loc, ncol, fontsize, frameon, …);
      `headroom`   top padding as a fraction of the curve span, to fit a taller/shorter legend;
      `ymin/ymax`  hard y limits (ymax overrides `headroom`)."""
    i = int(np.where(df.filename == name_i)[0][0])
    j = int(np.where(df.filename == name_j)[0][0])
    spec = S if use == "raw" else Sn
    ci, cj = "#0072B2", "#D55E00"
    lab = [_flow_label(df, k, nm, n=c, fmt=label_fmt, unit=unit)
           for c, (k, nm) in enumerate(((i, name_i), (j, name_j)), start=1)]
    if labels is not None:                       # per-call hand-written legend entries
        lab = [new if new is not None else old for new, old in zip(labels, lab)]

    fig, ax = plt.subplots(figsize=(8.5, 8.5))
    ax.plot(grid, spec[i], color=ci, lw=2.8, ls="-", label=lab[0])
    ax.plot(grid, spec[j], color=cj, lw=2.8, ls="-", label=lab[1])
    ax.set_xlim(grid.min(), min(grid.max(), wl_max))
    ax.set_box_aspect(1)
    ax.set_xlabel("Wavelength (nm)", fontsize=fs)
    ax.set_ylabel("Extinction (raw, a.u.)" if use == "raw" else "Extinction (peak-norm.)", fontsize=fs)
    ax.tick_params(labelsize=fs - 2)
    # full box frame — the legend is placed inside it, so the frame is what makes the figure read as
    # one compact block (the house style otherwise drops the top/right spines)
    for sp_ in ax.spines.values():
        sp_.set_visible(True); sp_.set_linewidth(1.2); sp_.set_color("0.15")
    # y range: explicit limits win; otherwise open `headroom` above the curves for the legend
    inwl = (grid >= ax.get_xlim()[0]) & (grid <= ax.get_xlim()[1])
    lo = float(min(spec[i][inwl].min(), spec[j][inwl].min()))
    hi = float(max(spec[i][inwl].max(), spec[j][inwl].max()))
    span = hi - lo
    ax.set_ylim(lo - 0.06 * span if ymin is None else ymin,
                hi + (SOLO_HEADROOM if headroom is None else headroom) * span if ymax is None else ymax)
    ax.legend(**{**SOLO_LEGEND_KW, "fontsize": fs - 9, **(legend_kw or {})})
    fig.tight_layout(); return fig


def fig_pair_area_solo(df, name_i, name_j, fs=18, labels=None, label_fmt=None, legend_kw=None,
                       annot_fmt=None, annotate=True):
    """Standalone SQUARE spectral-area-vs-time figure for ONE named pair, same colour convention as
    `fig_pair_spectra_solo`, large fonts throughout. The time axis starts at 0 and the LAST measured
    point of each trajectory is emphasised (large star) and called out with that run's final GROWTH
    time — the quantity the degenerate endpoints hide. Same legend knobs as the spectra figure
    (`labels`, `label_fmt` with {n}/{name}/{growth_time}, `legend_kw`); `annot_fmt` / `annotate`
    control the end-point callout."""
    i = int(np.where(df.filename == name_i)[0][0])
    j = int(np.where(df.filename == name_j)[0][0])
    ci, cj = "#0072B2", "#D55E00"
    ser_i = k42.read_series(df.filepath[i]); ser_j = k42.read_series(df.filepath[j])
    lab = [(AREA_LABEL_FMT if label_fmt is None else label_fmt).format(
                n=c, name=nm, growth_time=df.growth_time_s[k])
           for c, (k, nm) in enumerate(((i, name_i), (j, name_j)), start=1)]
    if labels is not None:
        lab = [new if new is not None else old for new, old in zip(labels, lab)]

    fig, ax = plt.subplots(figsize=(8.5, 8.5))
    ends = []
    for ser, c, k, lb in [(ser_i, ci, i, lab[0]), (ser_j, cj, j, lab[1])]:
        if ser is None:
            continue
        t, area = ser[0], ser[1]
        ax.plot(t, area, marker="o", ms=7, color=c, lw=2.8, ls="-", label=lb)
        # end of the trajectory: emphasised marker + its final growth (convergence) time
        ax.plot(t[-1], area[-1], marker="*", ms=28, color=c, mec="0.15", mew=1.2, ls="none",
                zorder=5)
        ends.append((float(t[-1]), float(area[-1]), c, k))
    ax.set_box_aspect(1)
    # full box frame (house style otherwise drops the top/right spines)
    for sp_ in ax.spines.values():
        sp_.set_visible(True); sp_.set_linewidth(1.2); sp_.set_color("0.15")
    ax.set_xlim(0, max(e[0] for e in ends) * 1.06 if ends else None)
    ax.set_ylim(top=ax.get_ylim()[1] + 0.10 * np.ptp(ax.get_ylim()))   # room for the end callouts
    if annotate:
        x0, x1 = ax.get_xlim()
        for t_end, a_end, c, k in ends:                # callout leans away from the nearest edge
            right = (t_end - x0) / (x1 - x0) > 0.5
            ax.annotate((AREA_ANNOT_FMT if annot_fmt is None else annot_fmt).format(
                            growth_time=df.growth_time_s[k], t_end=t_end, name=df.filename[k]),
                        xy=(t_end, a_end), xytext=(-16 if right else 16, 14),
                        textcoords="offset points", ha="right" if right else "left", va="bottom",
                        fontsize=fs - 5, color=c, fontweight="bold", zorder=6,
                        bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))
    ax.set_xlabel("Time (s)", fontsize=fs)
    ax.set_ylabel("Spectral Area (au·nm)", fontsize=fs)
    ax.set_title(f"Spectral Area vs. Time — {name_i}  vs  {name_j}", fontsize=fs - 1)
    ax.tick_params(labelsize=fs - 2)
    ax.legend(**{**AREA_LEGEND_KW, "fontsize": fs - 4, **(legend_kw or {})})
    fig.tight_layout(); return fig


# ── extra spotlight examples (variety: colour-degenerate / intra / inter) ──────
PAIR_EXAMPLES = [
    dict(tag="colour_degenerate", name_i="S14-M14_sample_13", name_j="S15-M05_sample_14",
         note="inter-session, cosine-identical shape but Δλ_max≈191nm — the BO cost's colour-blindness"),
    dict(tag="intra_clean", name_i="S15-M08_sample_12", name_j="S15-M08_sample_2",
         note="intra-session, same colour (Δλ_max<4nm) — divergence is chemistry, not session drift"),
    dict(tag="inter_clean", name_i="S14-M14_sample_7", name_j="S26-M01_init_2",
         note="inter-session, raw-degenerate (shape AND amplitude match), Δt_conv≈19s"),
]


# ── run ────────────────────────────────────────────────────────────────────────
def main():
    df, S, Sn, grid = load()
    D_raw, D_pca = distance_matrices(Sn)

    print("\n[1] endpoint-similarity threshold sweep:")
    sweep = sweep_thresholds(df, D_raw, D_pca)
    print(sweep.to_string(index=False))
    sweep.to_csv(pc.DATA_DIR / "degeneracy_threshold_sweep.csv", index=False)

    print(f"\n[2] pairs below EPS={EPS_MAIN}:")
    pairs, mad = build_pairs(df, D_raw, D_pca, eps=EPS_MAIN)
    print(f"  {len(pairs)} endpoint-similar pairs "
          f"({int(pairs.intra_session.sum()) if len(pairs) else 0} intra-session)")
    if not len(pairs):
        print("  NEGATIVE RESULT: no endpoint-similar pairs at this threshold — endpoint not "
              "degenerate over the sampled domain at EPS. Report this (see brief, Task A).")
        return
    pairs.to_csv(pc.DATA_DIR / "degeneracy_pairs.csv", index=False)
    print("\n  MAD (population spread) per trajectory descriptor:")
    for c in TRAJ_DESCRIPTORS:
        print(f"    {c:18s} MAD={mad[c]:.3f}")
    print("\n  top divergent pairs (endpoint-identical, trajectory-different):")
    show = ["run_i", "run_j", "intra_session", "endpoint_cos", "d_growth_time_s",
            "z_growth_time_s", "z_max_growth_rate", "divergence_score"]
    print(pairs[show].head(10).round(3).to_string(index=False))

    # BONUS finding: the BO cosine cost is itself degenerate — near-identical cosine does NOT pin
    # down the final colour. Report how far lambda_max can differ among endpoint-similar pairs.
    print(f"\n[bonus] cosine cost degeneracy: among the {len(pairs)} cosine-similar pairs "
          f"(dist<{EPS_MAIN}), lambda_max differs by up to {pairs.d_lambda_max.max():.0f} nm "
          f"(median {pairs.d_lambda_max.median():.0f} nm) — the BO objective treats them as equal.")

    # headline pair: the most divergent INTRA-session pair whose finals are GENUINELY the same colour
    # (|Δlambda_max| < LAM_AGREE) so all four panels agree. A large-Δlambda pair would be cosine-
    # degenerate but not visually "identical", muddying the trajectory message. Fall back if none.
    LAM_AGREE = 15.0     # nm — finals count as the same colour within inter-session repeatability
    intra = pairs[pairs.intra_session]
    clean_intra = intra[intra.d_lambda_max < LAM_AGREE] if len(intra) else intra
    if len(clean_intra):
        head = clean_intra.iloc[0]
    elif len(intra):
        head = intra.iloc[0]
    else:
        head = pairs.iloc[0]
    print(f"\n[3] headline pair: {head.run_i}  vs  {head.run_j}  "
          f"({'intra' if head.intra_session else 'inter'}-session)")
    print("     frame-bootstrap repeatability floor ...")
    bi = bootstrap_repeatability(head_filepath(df, head.run_i))
    bj = bootstrap_repeatability(head_filepath(df, head.run_j))
    print(f"       run i: {bi}\n       run j: {bj}")

    print("\n[4] pair gallery — several INTRA- and INTER-session pairs, normalised + raw finals")
    sel = select_pairs(pairs, n_each=4, lam_agree=LAM_AGREE)
    print(f"  selected {int(sel.kind.eq('intra').sum())} intra + {int(sel.kind.eq('inter').sum())} "
          f"inter-session pairs ({int(sel.clean.sum())} same-colour / {len(sel)} total)")
    sel.to_csv(pc.DATA_DIR / "degeneracy_pairs_gallery.csv", index=False)

    # [5] RAW (non-normalised) degeneracy: shape-similar pairs that are ALSO amplitude-similar.
    # Cosine distance is scale-invariant, so `endpoint_cos` alone never tests this — it is a genuinely
    # stricter, separate criterion (see build_pairs docstring).
    print(f"\n[5] raw (non-normalised) degeneracy: amplitude-match threshold sweep "
          f"(shape fixed at EPS={EPS_MAIN}):")
    amp_sweep = sweep_amp_thresholds(pairs)
    print(amp_sweep.to_string(index=False))
    amp_sweep.to_csv(pc.DATA_DIR / "degeneracy_amp_threshold_sweep.csv", index=False)

    n_raw_deg = int(pairs.raw_degenerate.sum()) if "raw_degenerate" in pairs.columns else 0
    n_raw_deg_intra = int(pairs.loc[pairs.get("raw_degenerate", False), "intra_session"].sum()) \
        if n_raw_deg else 0
    print(f"\n  {n_raw_deg} pairs are raw-degenerate at Z_AMP_MAIN={Z_AMP_MAIN} "
          f"({n_raw_deg_intra} intra-session, {n_raw_deg - n_raw_deg_intra} inter-session)")
    if n_raw_deg == 0:
        print("  NEGATIVE RESULT: no pair is simultaneously shape- AND amplitude-similar — the "
              "endpoint degeneracy on this dataset is a SHAPE degeneracy only; amplitude still "
              "distinguishes every near-identical-shape pair.")
        sel_raw = pairs.iloc[0:0].assign(kind=[], clean=[])
    else:
        sel_raw = select_raw_degenerate_pairs(pairs, n_each=4, lam_agree=LAM_AGREE)
        print(f"  gallery: {int(sel_raw.kind.eq('intra').sum())} intra + "
              f"{int(sel_raw.kind.eq('inter').sum())} inter-session raw-degenerate pairs")
        sel_raw.to_csv(pc.DATA_DIR / "degeneracy_pairs_raw_gallery.csv", index=False)

    savefig(fig_distance_overview(df, D_raw, pairs), "fig43_distance_overview")
    savefig(fig_degenerate_pair(df, S, Sn, grid, head, bi, bj), "fig43_degenerate_pair")
    savefig(fig_degenerate_pairs_grid(df, S, Sn, grid, sel), "fig43_degenerate_pairs_grid")
    savefig(fig_degenerate_pairs_grid(df, S, Sn, grid, sel_raw,
            title=f"RAW-degenerate pairs (shape AND amplitude match, "
                  f"{int(sel_raw.kind.eq('intra').sum()) if len(sel_raw) else 0} intra + "
                  f"{int(sel_raw.kind.eq('inter').sum()) if len(sel_raw) else 0} inter-session)"),
        "fig43_raw_degenerate_pairs_grid")
    print(f"\nDone -> {pc.FIG_DIR.resolve()}")


def head_filepath(df, name):
    return df.loc[df.filename == name, "filepath"].iloc[0]


if __name__ == "__main__":
    main()


### `ml.step46_colour_map_4d`
*Original analysis code.*

In [ ]:
%%module toofast_analysis.ml.step46_colour_map_4d
# [vendored] verbatim copy of 2FAST_ML/pipeline/step46_colour_map_4d.py (md5 2d4d485701e829c4136baf4c88bbeb19), generated by tools/vendor_ml.py — do not edit by hand.
"""
step46_colour_map_4d.py  —  pipeline step 16 (bonus)  (mirrors 46_colour_map_4d.ipynb)
=====================================================================================
A **4-D perceived-colour map** of the predicted Ag-nanoparticle spectra:

    x = Q1 (AgNO3)   y = Q5 (NaBH4)   3rd axis (slider) = Q3 (acetic acid)   4th axis (slider) = TIME

Each pixel is a full 5-flow recipe: Q1, Q5 and Q3 are set by the two axes + the Q3 slider, and the
REMAINING flow (14 - Q1 - Q3 - Q5) is split between Q2 (PVP) and Q4 (ascorbic acid) in the ratio of
their data medians, so every pixel still sums to Q_TOTAL = 14 µL/s. Pixels where nothing is left for
Q2/Q4 are infeasible and stay white.

The spectrum at a pixel comes from the notebook-40 growth predictor (`step40_growth_prediction`) — the
progress-τ checkpoint GPs on the relative-to-final basis — and is turned into an sRGB colour by the
CIE-colouriser `spectral_pipeline.make_colouriser` (Beer-Lambert transmission through the sample).

TIME AXIS.  The two slider dimensions are genuinely different in kind:
  • `time_mode="seconds"` (DEFAULT, the physical one): the slider is REAL elapsed seconds. Every pixel
    has its own predicted convergence time t_conv(Q) (the step-36/38 RandomForest), so at a fixed
    wall-clock t different pixels sit at different progress τ = clip(t / t_conv, 0, 1). This is what a
    camera pointed at the reactor would see: fast recipes (high NaBH4) reach their final colour while
    slow ones are still pale. Pixels that already converged simply stay at τ = 1.
  • `time_mode="tau"`: the slider is progress τ = t/t_conv, i.e. every pixel is shown at the SAME
    fraction of its own growth (this is what `s40.fig_colormap_time` does). Useful to separate "what
    shape does this recipe pass through" from "how fast does it get there".

AMPLITUDE.  Spectra are relative to each recipe's OWN final peak, so early times are genuinely PALE
(little material yet) — that is the physical colour development and the default (`amplitude="relative"`).
`amplitude="unit_peak"` re-normalises every frame to peak 1, which throws away the growth and shows the
HUE evolution only.

Everything expensive (the GP checkpoint predictions) is done ONCE per Q3 slice; the whole 4-D RGB cube
is then precomputed and cached, so both the ipywidgets sliders and the exported standalone HTML are
instant.

MECHANISTIC PROJECTIONS (§9-§10 of the notebook).  The same machinery is also run on the three
chemically-constrained planes of `step33_spectral_model` §3b — x = the Ag/NaBH4 flow ratio (log),
y = one swept reagent, everything else pinned by chemistry (PVP/Ag ratio, ascorbic acid at the 2 e-
redox stoichiometry, acetic acid fixed) and closed by the 14 uL/s total flow. `build_mech_cube` gives
a (projection x time) colour cube (`fig_mech_time` / `fig_mech_time_row` / `mech_widget` /
`export_mech_html`), and `fig_mech_tconv` draws those same planes coloured by the predicted
CONVERGENCE TIME on the house piecewise scale, with the measured runs overlaid on the same scale.
The recipe closure, the stoichiometry lines and the axis decoration are imported from step33, so a
converged frame here is the same slice as the static nb33 map.

Kernel: Anaconda 3.11.
"""
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from . import pipeline_common as pc
from . import step40_growth_prediction as s40
from . import step33_spectral_model as s33   # mechanistic recipe closure + axis decoration
sp = pc.sp

NOTEBOOK = "46_colour_map_4d.ipynb"
REAGENTS = s40.REAGENTS
Q_TOTAL = float(sp.Q_TOTAL)

# default slider axes — 7 acetic-acid levels, 12 wall-clock times (log-ish, dense where it changes)
N_SLIDER = 7
TIMES_S = np.array([0.0, 1.0, 2.0, 3.0, 5.0, 7.5, 10.0, 15.0, 20.0, 30.0, 45.0, 60.0])
TAUS = np.array([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
# λmax overlay + peak alignment use step40's gated definition (same window and amplitude gate), so the
# contours here and the λ(τ) curve nb40 prints are the same quantity.
LAM_WIN = s40.LMAX_WIN
LAM_H_MIN = s40.LMAX_H_MIN


def savefig(fig, name):
    fig.text(0.995, 0.003, f"source: {NOTEBOOK}", ha="right", va="bottom", fontsize=6, color="0.55")
    for ext in ("png", "pdf"):
        fig.savefig(pc.FIG_DIR / f"{name}.{ext}", dpi=200, bbox_inches="tight")
    print(f"  saved {name}.png/.pdf")


# ══════════════════════════════════════════════════════════════════════════════
# recipe grid: two axes + one slider flow, the other two flows median-filled to sum 14
# ══════════════════════════════════════════════════════════════════════════════
def recipe_grid(Qdata, pair=("Q1", "Q5"), slider_flow="Q3", slider_val=0.0, grid_n=56,
                ax_a=None, ax_b=None):
    """(Qpix, mask, ax_a, ax_b): every feasible pixel's full 5-flow recipe on the `pair` plane at one
    `slider_flow` level. The two remaining flows take the leftover in their data-median ratio."""
    ia, ib = sp.FEATURES.index(pair[0]), sp.FEATURES.index(pair[1])
    isl = sp.FEATURES.index(slider_flow)
    others = [k for k in range(5) if k not in (ia, ib, isl)]
    med = np.median(Qdata, 0)
    prop = med[others] / max(med[others].sum(), 1e-9)
    if ax_a is None:
        lo = pc.TRAIN_Q_FLOOR if pair[0] in ("Q1", "Q5") else 0.0
        ax_a = np.linspace(lo, Qdata[:, ia].max() * 1.02, grid_n)
    if ax_b is None:
        lo = pc.TRAIN_Q_FLOOR if pair[1] in ("Q1", "Q5") else 0.0
        ax_b = np.linspace(lo, Qdata[:, ib].max() * 1.02, grid_n)
    ny, nx = len(ax_b), len(ax_a)
    G = np.zeros((ny, nx, 5)); mask = np.zeros((ny, nx), bool)
    for iy, vb in enumerate(ax_b):
        for ix, va in enumerate(ax_a):
            rem = Q_TOTAL - va - vb - slider_val
            if rem <= 0:                                  # nothing left for Q2/Q4 -> infeasible
                continue
            x = np.zeros(5); x[ia] = va; x[ib] = vb; x[isl] = slider_val; x[others] = rem * prop
            G[iy, ix] = x; mask[iy, ix] = True
    return G[mask], mask, ax_a, ax_b


# ══════════════════════════════════════════════════════════════════════════════
# vectorised λmax + peak-aligned (warped) τ-interpolation for a whole pixel batch
# ══════════════════════════════════════════════════════════════════════════════
def _lam_batch(S, grid, win=LAM_WIN, h_min=LAM_H_MIN):
    """(n,) λmax of each spectrum in `S` (n, n_wl) — the vectorised twin of `s40.GrowthPredictor._lmax`
    (same window, same gates), because a colour map wants tens of thousands of readings at a time.
    NaN when the maximum sits on a window EDGE (no interior peak — a bare tail) or when the frame has
    not yet reached `h_min` of its own final peak (a low-amplitude PCA reconstruction whose tallest
    point wanders into the NIR)."""
    w = (grid >= win[0]) & (grid <= win[1])
    sm = savgol_filter(np.atleast_2d(S)[:, w], 15, 3, axis=1)
    j = np.argmax(sm, axis=1)
    h = sm[np.arange(len(sm)), j]
    ok = (j > 0) & (j < sm.shape[1] - 1) & (h >= h_min)
    return np.where(ok, grid[w][j], np.nan)


def _lam_curve_batch(CK, grid):
    """(n_ck, n_pix) monotone λmax growth per pixel: NaN readings (no interior peak) are back/forward
    filled from the first valid checkpoint, then `maximum.accumulate` enforces the seed→NP red-shift.
    Pixels with NO valid reading at all get NaN (warp then skipped for them)."""
    L = np.array([_lam_batch(CK[k], grid) for k in range(len(CK))])     # (n_ck, n_pix)
    ok = np.isfinite(L)
    any_ok = ok.any(0)
    first = np.argmax(ok, axis=0)                                       # first valid checkpoint
    for k in range(len(L)):                                             # back-fill the leading NaNs
        need = any_ok & (k < first)
        L[k, need] = L[first[need], need]
    for k in range(1, len(L)):                                          # forward-fill interior gaps
        bad = any_ok & ~np.isfinite(L[k])
        L[k, bad] = L[k - 1, bad]
    L[:, ~any_ok] = np.nan
    return np.maximum.accumulate(np.where(np.isfinite(L), L, -np.inf), axis=0) * \
        np.where(any_ok, 1.0, np.nan)


def _shift_batch(S, delta_nm, grid):
    """Translate each spectrum of `S` (n, n_wl) `delta_nm[i]` to the RED, by integer resampling on the
    (near-uniform, 0.57 nm) wavelength grid — vectorised `np.interp(grid - delta, grid, spec)`. The
    ≤9 % non-uniformity of the grid makes the realised shift accurate to a few % of `delta`, which is
    well inside the λmax uncertainty of the model itself."""
    step = float(np.mean(np.diff(grid)))
    k = np.nan_to_num(np.round(np.asarray(delta_nm, float) / step)).astype(int)
    nwl = S.shape[1]
    idx = np.clip(np.arange(nwl)[None, :] - k[:, None], 0, nwl - 1)
    return np.take_along_axis(S, idx, axis=1)


def spectra_at_tau(CK, tau_c, tau_pix, lam_c=None, min_shift=20.0, grid=None):
    """(n_pix, n_wl) relative-to-final spectra with a PER-PIXEL progress `tau_pix` (that is what makes
    a wall-clock time slice possible: at a fixed second, each recipe is at its own τ = t/t_conv).
    `CK` = (n_ck, n_pix, n_wl) checkpoint spectra. With `lam_c` given, adjacent checkpoints are shifted
    to a common λmax before blending (the nb40 peak-aligned interpolation), so the plasmon peak
    TRANSLATES instead of a second peak cross-fading in; pixels that red-shift by < `min_shift` nm (or
    whose λmax could not be read) fall back to plain interpolation."""
    n_ck, n_pix, n_wl = CK.shape
    t = np.clip(np.asarray(tau_pix, float), tau_c[0], tau_c[-1])
    j = np.clip(np.searchsorted(tau_c, t), 1, n_ck - 1)
    a = (t - tau_c[j - 1]) / (tau_c[j] - tau_c[j - 1])
    p = np.arange(n_pix)
    lo = CK[j - 1, p, :]; hi = CK[j, p, :]
    if lam_c is not None:
        span = lam_c[-1] - lam_c[0]
        use = np.isfinite(span) & (span >= min_shift)
        if use.any():
            # λ(τ) as a piecewise-linear interpolation of the monotone checkpoint curve
            lt = lam_c[j - 1, p] + a * (lam_c[j, p] - lam_c[j - 1, p])
            lo = np.where(use[:, None], _shift_batch(lo, lt - lam_c[j - 1, p], grid), lo)
            hi = np.where(use[:, None], _shift_batch(hi, lt - lam_c[j, p], grid), hi)
    return (1 - a)[:, None] * lo + a[:, None] * hi


# ══════════════════════════════════════════════════════════════════════════════
# the 4-D cube
# ══════════════════════════════════════════════════════════════════════════════
def build_cube(model, runs, grid, pair=("Q1", "Q5"), slider_flow="Q3", grid_n=56,
               slider_vals=None, n_slider=N_SLIDER, times=None, time_mode="seconds",
               amplitude="relative", warp=True, verbose=True):
    """Precompute the whole 4-D colour cube.

    Returns a dict with
      `rgb`      (n_slider, n_time, ny, nx, 3) sRGB, white where infeasible
      `lam`      (n_slider, n_time, ny, nx)    predicted λmax (nm), NaN where unreadable/infeasible
      `amp`      (n_slider, n_time, ny, nx)    predicted extinction as a fraction of that pixel's final
      `tconv`    (n_slider, ny, nx)            predicted convergence time (s) per pixel
      `mask`     (n_slider, ny, nx)            feasible pixels
      plus `ax_a`, `ax_b`, `slider_vals`, `times`, `time_mode`, `amplitude`, `pair`, `slider_flow`.
    """
    Qdata = runs[list(sp.FEATURES)].values.astype(float)
    isl = sp.FEATURES.index(slider_flow)
    if slider_vals is None:
        # stay INSIDE the trained Q3 range: clr features make Q3 -> 0 a big extrapolation
        # (log(0.01) vs log(0.14)), and the GP there just falls back to its prior mean — a
        # featureless panel. 2nd..98th percentile of the data keeps every slice interpolation.
        slider_vals = np.round(np.linspace(*np.percentile(Qdata[:, isl], [2, 98]), n_slider), 2)
    slider_vals = np.asarray(slider_vals, float)
    times = np.asarray(TIMES_S if times is None else times, float) if time_mode == "seconds" \
        else np.asarray(TAUS if times is None else times, float)
    colour = sp.make_colouriser(grid)
    tau_c = np.asarray(model.slices, float)

    # one common pair-plane axis for every slider level, so the sliders don't rescale the axes
    _, _, ax_a, ax_b = recipe_grid(Qdata, pair, slider_flow, 0.0, grid_n)
    ny, nx = len(ax_b), len(ax_a)
    ns, nt = len(slider_vals), len(times)
    RGB = np.ones((ns, nt, ny, nx, 3))
    LAM = np.full((ns, nt, ny, nx), np.nan); AMP = np.full((ns, nt, ny, nx), np.nan)
    TC = np.full((ns, ny, nx), np.nan); MASK = np.zeros((ns, ny, nx), bool)

    for si, sv in enumerate(slider_vals):
        Qpix, mask, _, _ = recipe_grid(Qdata, pair, slider_flow, sv, grid_n, ax_a, ax_b)
        MASK[si] = mask
        if not mask.any():
            if verbose:
                print(f"  {slider_flow}={sv:5.2f}  (no feasible pixel)")
            continue
        # the expensive step, ONCE per slider level: every checkpoint for every pixel
        CK = np.array([model.mu + model.ckpt[t].predict(Qpix) @ model.comps for t in model.slices])
        lam_c = _lam_curve_batch(CK, grid) if warp else None
        tconv = s40.predict_conv_time_batch(Qpix)
        TC[si][mask] = tconv
        for ti, tv in enumerate(times):
            tau_pix = np.clip(tv / np.maximum(tconv, 1e-9), 0, 1) if time_mode == "seconds" \
                else np.full(len(Qpix), float(tv))
            S = spectra_at_tau(CK, tau_c, tau_pix, lam_c, grid=grid)
            AMP[si, ti][mask] = S.max(1)
            if amplitude == "unit_peak":
                S = S / np.maximum(S.max(1, keepdims=True), 1e-9)
            RGB[si, ti][mask] = np.array([colour(s) for s in S])
            LAM[si, ti][mask] = _lam_batch(S, grid)
        if verbose:
            print(f"  {slider_flow}={sv:5.2f}  {int(mask.sum()):5d} pixels | "
                  f"t_conv {np.nanmin(tconv):4.1f}-{np.nanmax(tconv):5.1f} s")
    return dict(rgb=RGB, lam=LAM, amp=AMP, tconv=TC, mask=MASK, ax_a=ax_a, ax_b=ax_b,
                slider_vals=slider_vals, times=times, time_mode=time_mode, amplitude=amplitude,
                pair=pair, slider_flow=slider_flow, grid_n=grid_n, warp=warp)


def cube_path(cube):
    tag = f"{cube['pair'][0]}{cube['pair'][1]}_{cube['slider_flow']}_{cube['time_mode']}" \
          f"_{cube['amplitude']}_n{cube['grid_n']}"
    return pc.DATA_DIR / f"colourcube46_{tag}.npz"


def save_cube(cube):
    p = cube_path(cube)
    np.savez_compressed(p, **{k: v for k, v in cube.items()
                              if isinstance(v, (np.ndarray, np.generic))},
                        meta=np.array([cube["time_mode"], cube["amplitude"], cube["pair"][0],
                                       cube["pair"][1], cube["slider_flow"], str(cube["grid_n"]),
                                       str(cube["warp"])]))
    print(f"  cached {p.name}  ({p.stat().st_size/1e6:.1f} MB)")
    return p


def load_cube(pair=("Q1", "Q5"), slider_flow="Q3", time_mode="seconds", amplitude="relative",
              grid_n=56):
    p = cube_path(dict(pair=pair, slider_flow=slider_flow, time_mode=time_mode,
                       amplitude=amplitude, grid_n=grid_n))
    if not p.exists():
        return None
    z = np.load(p, allow_pickle=False)
    m = [str(x) for x in z["meta"]]
    return dict(rgb=z["rgb"], lam=z["lam"], amp=z["amp"], tconv=z["tconv"], mask=z["mask"],
                ax_a=z["ax_a"], ax_b=z["ax_b"], slider_vals=z["slider_vals"], times=z["times"],
                time_mode=m[0], amplitude=m[1], pair=(m[2], m[3]), slider_flow=m[4],
                grid_n=int(m[5]), warp=(m[6] == "True"))


def get_cube(model, runs, grid, force=False, **kw):
    """Cached `build_cube`."""
    if not force:
        c = load_cube(**{k: kw[k] for k in ("pair", "slider_flow", "time_mode", "amplitude",
                                            "grid_n") if k in kw})
        if c is not None:
            print(f"  loaded colour cube from cache {cube_path(c).name}")
            return c
    c = build_cube(model, runs, grid, **kw)
    save_cube(c)
    return c


# ══════════════════════════════════════════════════════════════════════════════
# figures
# ══════════════════════════════════════════════════════════════════════════════
def _time_label(cube, ti):
    return (f"t = {cube['times'][ti]:.3g} s" if cube["time_mode"] == "seconds"
            else f"τ = {cube['times'][ti]:.2f}")


def _panel(ax, cube, si, ti, lam_contours=True, points=None, show_limit=True):
    c = cube; ax_a, ax_b = c["ax_a"], c["ax_b"]
    ax.imshow(c["rgb"][si, ti], origin="lower", aspect="auto",
              extent=[ax_a[0], ax_a[-1], ax_b[0], ax_b[-1]])
    if lam_contours:
        L = c["lam"][si, ti]
        if np.isfinite(L).sum() > 20:
            lo, hi = np.nanpercentile(L, [2, 98])
            if hi - lo > 25:
                cs = ax.contour(ax_a, ax_b, L, levels=np.arange(400, 901, 50), colors="k",
                                linewidths=.5, alpha=.45)
                ax.clabel(cs, inline=True, fontsize=5.5, fmt="%.0f")
    if show_limit:
        rem = Q_TOTAL - c["slider_vals"][si]
        ax.plot([0, rem], [rem, 0], "k--", lw=.9, alpha=.5)
    if points:
        ia = sp.FEATURES.index(c["pair"][0]); ib = sp.FEATURES.index(c["pair"][1])
        for lab, Qp in points.items():
            ax.scatter(Qp[ia], Qp[ib], s=40, facecolor="none", edgecolor="k", lw=1.3)
            ax.annotate(lab, (Qp[ia], Qp[ib]), textcoords="offset points", xytext=(5, 3),
                        fontweight="bold", fontsize=10)
    ax.set_xlim(ax_a[0], ax_a[-1]); ax.set_ylim(ax_b[0], ax_b[-1])


def fig_matrix(cube, slider_idx=None, time_idx=None, lam_contours=True, points=None):
    """The 4-D cube flattened onto paper: rows = the slider flow (acetic acid), columns = time.
    Colour = predicted perceived colour of the growing colloid; thin black contours = predicted λmax."""
    c = cube
    si = list(range(len(c["slider_vals"]))) if slider_idx is None else list(slider_idx)
    ti = list(range(len(c["times"]))) if time_idx is None else list(time_idx)
    fig, axes = plt.subplots(len(si), len(ti), figsize=(1.85 * len(ti), 1.85 * len(si)),
                             squeeze=False)
    for r, s in enumerate(si):
        for k, t in enumerate(ti):
            a = axes[r][k]
            _panel(a, c, s, t, lam_contours, points)
            a.tick_params(labelsize=5.5)
            if r == 0:
                a.set_title(_time_label(c, t), fontsize=8)
            if k == 0:
                a.set_ylabel(f"{c['slider_flow']} = {c['slider_vals'][s]:.2f}\n"
                             f"$Q_{{{c['pair'][1]}}}$ (µL/s)", fontsize=7.5)
            else:
                a.set_yticklabels([])
            if r == len(si) - 1:
                a.set_xlabel(f"$Q_{{{c['pair'][0]}}}$ (µL/s)", fontsize=7.5)
            else:
                a.set_xticklabels([])
    tl = "wall-clock time" if c["time_mode"] == "seconds" else "growth progress τ"
    fig.suptitle(f"4-D colour map — ({c['pair'][0]}, {c['pair'][1]}) plane × "
                 f"{c['slider_flow']} ({REAGENTS[c['slider_flow']]}) × {tl}   |   "
                 f"remaining flow split Q2:Q4 by data median, total 14 µL/s   |   "
                 f"contours = predicted $\\lambda_{{max}}$ (nm)", fontsize=9)
    fig.tight_layout(rect=[0, 0, 1, 0.97]); return fig


def fig_slice(cube, si, ti, points=None):
    """One big panel of the cube — the still that matches a chosen slider position."""
    c = cube
    fig, ax = plt.subplots(figsize=(6.6, 5.4))
    _panel(ax, c, si, ti, True, points)
    ax.set_xlabel(f"$Q_{{{c['pair'][0]}}}$  {REAGENTS[c['pair'][0]]}  (µL/s)", fontsize=12)
    ax.set_ylabel(f"$Q_{{{c['pair'][1]}}}$  {REAGENTS[c['pair'][1]]}  (µL/s)", fontsize=12)
    ax.set_title(f"{c['slider_flow']} ({REAGENTS[c['slider_flow']]}) = "
                 f"{c['slider_vals'][si]:.2f} µL/s   |   {_time_label(c, ti)}", fontsize=12)
    fig.tight_layout(); return fig


def fig_tconv(cube):
    """The convergence-time field behind the wall-clock slider: why a fixed second is a different
    growth fraction from pixel to pixel."""
    c = cube; ns = len(c["slider_vals"])
    fig, axes = plt.subplots(1, ns, figsize=(2.5 * ns, 2.9), squeeze=False); axes = axes[0]
    vmin = np.nanpercentile(c["tconv"], 2); vmax = np.nanpercentile(c["tconv"], 98)
    for a, si in zip(axes, range(ns)):
        im = a.imshow(c["tconv"][si], origin="lower", aspect="auto", cmap="viridis",
                      vmin=vmin, vmax=vmax,
                      extent=[c["ax_a"][0], c["ax_a"][-1], c["ax_b"][0], c["ax_b"][-1]])
        a.set_title(f"{c['slider_flow']} = {c['slider_vals'][si]:.2f}", fontsize=8)
        a.set_xlabel(f"$Q_{{{c['pair'][0]}}}$", fontsize=8); a.tick_params(labelsize=6)
        if si == 0:
            a.set_ylabel(f"$Q_{{{c['pair'][1]}}}$", fontsize=8)
    fig.colorbar(im, ax=axes, label="predicted $t_{conv}$ (s)", fraction=.025)
    fig.suptitle("Predicted convergence time over the same 3-D recipe volume", fontsize=10)
    return fig


# ══════════════════════════════════════════════════════════════════════════════
# interactive: ipywidgets in the notebook, or a standalone self-contained HTML
# ══════════════════════════════════════════════════════════════════════════════
def widget(cube, points=None, figsize=(6.4, 5.2)):
    """Two sliders — the slider flow (acetic acid) and time — over the precomputed cube. Redraw is a
    pure array lookup, so it is instant."""
    import ipywidgets as W
    from IPython.display import display
    c = cube
    s_sl = W.SelectionSlider(options=[(f"{v:.2f}", i) for i, v in enumerate(c["slider_vals"])],
                             value=0, description=f"{c['slider_flow']} ({REAGENTS[c['slider_flow']]})",
                             continuous_update=False, style={"description_width": "150px"},
                             layout=W.Layout(width="620px"))
    tlab = "t (s)" if c["time_mode"] == "seconds" else "progress τ"
    t_sl = W.SelectionSlider(options=[(f"{v:.3g}", i) for i, v in enumerate(c["times"])],
                             value=len(c["times"]) - 1, description=tlab, continuous_update=False,
                             style={"description_width": "150px"}, layout=W.Layout(width="620px"))
    out = W.Output()

    def draw(*_):
        with out:
            out.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=figsize)
            _panel(ax, c, s_sl.value, t_sl.value, True, points)
            ax.set_xlabel(f"$Q_{{{c['pair'][0]}}}$  {REAGENTS[c['pair'][0]]}  (µL/s)")
            ax.set_ylabel(f"$Q_{{{c['pair'][1]}}}$  {REAGENTS[c['pair'][1]]}  (µL/s)")
            L = c["lam"][s_sl.value, t_sl.value]; A = c["amp"][s_sl.value, t_sl.value]
            ax.set_title(f"{c['slider_flow']} = {c['slider_vals'][s_sl.value]:.2f} µL/s   |   "
                         f"{_time_label(c, t_sl.value)}\n"
                         f"$\\lambda_{{max}}$ {np.nanmin(L):.0f}–{np.nanmax(L):.0f} nm   |   "
                         f"growth {100*np.nanmean(A):.0f} % of final (mean)", fontsize=10)
            fig.tight_layout(); plt.show()

    s_sl.observe(draw, "value"); t_sl.observe(draw, "value")
    display(W.VBox([s_sl, t_sl, out])); draw()
    return s_sl, t_sl


def _frames_b64(rgb, upscale=9):
    """(n_slider, n_time) grid of base64 PNGs from an RGB cube — the payload both HTML exporters
    embed, so the viewer needs no server and no Python."""
    import base64, io
    import PIL.Image as Image
    ns, nt, ny, nx, _ = rgb.shape
    out = []
    for si in range(ns):
        row = []
        for ti in range(nt):
            arr = (np.clip(rgb[si, ti], 0, 1) * 255).astype(np.uint8)[::-1]    # origin=lower
            im = Image.fromarray(arr, "RGB").resize((nx * upscale, ny * upscale), Image.BICUBIC)
            buf = io.BytesIO(); im.save(buf, format="PNG")
            row.append("data:image/png;base64," + base64.b64encode(buf.getvalue()).decode())
        out.append(row)
    return out


def export_html(cube, path=None, upscale=9, points=None):
    """Write a SINGLE self-contained HTML file (no server, no Python) with the two sliders. Every
    (slider, time) slice is embedded as a base64 PNG, so it works offline and can be shared as-is."""
    import base64, io, json
    from matplotlib.colors import to_hex
    c = cube
    path = pc.FIG_DIR / (f"fig46_colour_map_4d_{c['slider_flow']}_{c['time_mode']}.html") \
        if path is None else path
    ns, nt = c["rgb"].shape[:2]
    meta = dict(
        frames=_frames_b64(c["rgb"], upscale),
        slider_vals=[float(v) for v in c["slider_vals"]],
        times=[float(v) for v in c["times"]],
        time_mode=c["time_mode"], slider_flow=c["slider_flow"],
        slider_reagent=REAGENTS[c["slider_flow"]],
        xlab=f"{c['pair'][0]} — {REAGENTS[c['pair'][0]]} (µL/s)",
        ylab=f"{c['pair'][1]} — {REAGENTS[c['pair'][1]]} (µL/s)",
        xlim=[float(c["ax_a"][0]), float(c["ax_a"][-1])],
        ylim=[float(c["ax_b"][0]), float(c["ax_b"][-1])],
        lam=[[None if not np.isfinite(np.nanmin(c["lam"][s, t])) else
              [float(np.nanmin(c["lam"][s, t])), float(np.nanmax(c["lam"][s, t]))]
             for t in range(nt)] for s in range(ns)],
        amp=[[float(np.nanmean(c["amp"][s, t])) for t in range(nt)] for s in range(ns)],
        notebook=NOTEBOOK,
    )
    html = _HTML_TEMPLATE.replace("__DATA__", json.dumps(meta))
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"  wrote {path}  ({path.stat().st_size/1e6:.1f} MB)")
    return path


_HTML_TEMPLATE = r"""<!doctype html>
<meta charset="utf-8"><title>4-D nanoparticle colour map</title>
<style>
 :root{--fg:#1b1b1b;--bg:#fff;--mut:#666;--line:#d8d8d8}
 @media (prefers-color-scheme:dark){:root{--fg:#e8e8e8;--bg:#141414;--mut:#9a9a9a;--line:#333}}
 body{margin:0;padding:22px;font:14px/1.5 system-ui,-apple-system,Segoe UI,sans-serif;
      color:var(--fg);background:var(--bg)}
 h1{font-size:17px;margin:0 0 2px} .sub{color:var(--mut);font-size:12.5px;margin:0 0 18px}
 .wrap{max-width:760px}
 .row{display:flex;align-items:stretch;gap:6px}
 .yax{display:flex;flex-direction:column;justify-content:space-between;color:var(--mut);
      font-size:11.5px;font-variant-numeric:tabular-nums}
 .plot{position:relative;border:1px solid var(--line);background:#fff;flex:1;min-width:0}
 .plot img{display:block;width:100%;height:auto;image-rendering:auto}
 .ax{color:var(--mut);font-size:11.5px;display:flex;justify-content:space-between;margin-top:4px}
 .ylab{color:var(--mut);font-size:11.5px;margin-bottom:4px}
 .ctl{margin-top:18px} .ctl label{display:block;font-weight:600;margin-bottom:4px}
 .ctl input[type=range]{width:100%} .val{font-variant-numeric:tabular-nums;color:var(--mut)}
 .stat{margin-top:14px;padding:10px 12px;border:1px solid var(--line);border-radius:6px;
       font-size:12.5px;color:var(--mut)}
 .src{margin-top:20px;font-size:11px;color:var(--mut)}
</style>
<div class="wrap">
<h1>Predicted nanoparticle colour — 4-D map</h1>
<p class="sub">Q1 (AgNO<sub>3</sub>) × Q5 (NaBH<sub>4</sub>) plane, sliced by acetic acid and by time.
Every pixel is a full 5-flow recipe summing to 14 µL/s; white = infeasible (nothing left for PVP /
ascorbic acid).</p>
<div class="ylab" id="ylab"></div>
<div class="row">
  <div class="yax"><span id="y1"></span><span id="y0"></span></div>
  <div class="plot"><img id="img" alt="colour map"></div>
</div>
<div class="ax"><span id="x0"></span><span id="xlab"></span><span id="x1"></span></div>
<div class="ctl">
  <label id="slab"></label>
  <input type="range" id="s" min="0" step="1">
</div>
<div class="ctl">
  <label id="tlab"></label>
  <input type="range" id="t" min="0" step="1">
</div>
<div class="stat" id="stat"></div>
<p class="src">source: <span id="nb"></span></p>
</div>
<script>
const D = __DATA__;
const img=document.getElementById('img'), s=document.getElementById('s'), t=document.getElementById('t');
s.max=D.slider_vals.length-1; t.max=D.times.length-1;
s.value=Math.floor((D.slider_vals.length-1)/2);   // set AFTER max, else the default 50 is clamped to it
t.value=D.times.length-1;
document.getElementById('xlab').textContent=D.xlab;
document.getElementById('ylab').textContent=D.ylab+'  ↑';
document.getElementById('x0').textContent=D.x0s||D.xlim[0].toFixed(1);
document.getElementById('x1').textContent=D.x1s||D.xlim[1].toFixed(1);
document.getElementById('y0').textContent=D.y0s||D.ylim[0].toFixed(1);
document.getElementById('y1').textContent=D.y1s||D.ylim[1].toFixed(1);
document.getElementById('nb').textContent=D.notebook;
function draw(){
  const si=+s.value, ti=+t.value;
  img.src=D.frames[si][ti];
  document.getElementById('slab').innerHTML = D.slider_labels
    ? D.slider_labels[si]
    : D.slider_flow+' — '+D.slider_reagent+': <span class="val">'+D.slider_vals[si].toFixed(2)+' µL/s</span>';
  const tv=D.times[ti];
  document.getElementById('tlab').innerHTML=(D.time_mode==='seconds'
    ? 'elapsed time: <span class="val">'+tv.toFixed(1)+' s</span>'
    : 'growth progress τ: <span class="val">'+tv.toFixed(2)+'</span>');
  const lam=D.lam[si][ti], amp=D.amp[si][ti];
  document.getElementById('stat').innerHTML =
    (lam? 'predicted λ<sub>max</sub> across the panel: <b>'+lam[0].toFixed(0)+'–'+lam[1].toFixed(0)+
          ' nm</b><br>' : '') +
    'mean extinction reached: <b>'+(100*amp).toFixed(0)+' %</b> of each recipe\'s own final';
}
s.oninput=draw; t.oninput=draw; draw();
</script>
"""


# ══════════════════════════════════════════════════════════════════════════════
# PLOTLY HYPERVOLUME — the three reagent axes as a real 3-D VOLUME, time as the animated 4th axis
#   The map above puts Q3 on a slider because a 2-D image only has two axes. Here all three flows are
#   spatial (x = Q1, y = Q5, z = Q3) and rendered as a volume, so the 4-D object is
#   (Q1, Q5, Q3) × time with ONE slider left for time. Three views of the same voxels:
#     • `shells` — `go.Volume` volumetric rendering (nested semi-transparent iso-surfaces) of a
#       MEANINGFUL scalar: λmax (iso-λmax shells) or the growth fraction (the reaction front sweeping
#       through the volume as the clock runs). Its colourscale is built from the REAL perceived colour
#       of the fully-grown spectra in each scalar bin (saturation-boosted), so the shells carry the
#       colloid's own colour.
#     • `slices` — the same `go.Volume` as three filled Q3 planes: a stack of Q1×Q5 maps, the most
#       legible view of the interior.
#     • `true colour` — `go.Scatter3d`, the exact per-voxel perceived colour as a point cloud, no
#       binning, marker size following how far that recipe has actually grown.
#   The infeasible corner (too little left for PVP + ascorbic acid) is a translucent grey wedge.
# ══════════════════════════════════════════════════════════════════════════════
VOL_N = 26                     # voxels per reagent axis (26^3 = 17576, of which ~72 % are feasible)
VOL_TIMES_S = np.array([0.0, 2.0, 4.0, 7.0, 11.0, 16.0, 25.0, 40.0, 60.0])
VOL_TAUS = np.array([0.0, 0.15, 0.3, 0.45, 0.6, 0.75, 0.9, 1.0])


def build_volume(model, runs, grid, axes=("Q1", "Q5", "Q3"), n=VOL_N, times=None,
                 time_mode="seconds", amplitude="relative", warp=True, leftover_min=None,
                 verbose=True):
    """Voxelise the 3-D reagent volume spanned by `axes` and predict every voxel at every time.

    The two flows NOT on an axis (Q2 = PVP, Q4 = ascorbic acid) absorb the leftover
    `14 - sum(axes)` in their data-median ratio. A voxel is kept only if that leftover is at least
    `leftover_min` — default the **2nd percentile of the measured Q2+Q4** (≈1.9 µL/s). Merely requiring
    leftover > 0 would admit recipes with essentially no PVP or ascorbic acid, which is far outside the
    trained simplex (the data's Q2+Q4 spans 1.3-9.0 µL/s) and where the clr-feature GP just returns its
    prior mean.

    Returns a dict of flat per-voxel arrays (`xyz`, `Q`, `tconv`, `keep` mask, plus the full-grid
    `shape`/axis vectors) and per-time (n_time, n_vox) fields `lam`, `amp`, `rgb`."""
    Qdata = runs[list(sp.FEATURES)].values.astype(float)
    ia, ib, ic = [sp.FEATURES.index(a) for a in axes]
    others = [k for k in range(5) if k not in (ia, ib, ic)]
    med = np.median(Qdata, 0)
    prop = med[others] / max(med[others].sum(), 1e-9)
    if leftover_min is None:
        leftover_min = float(np.percentile(Qdata[:, others].sum(1), 2))
    if times is None:
        times = VOL_TIMES_S if time_mode == "seconds" else VOL_TAUS
    times = np.asarray(times, float)

    def axis_vec(k, name):
        lo = pc.TRAIN_Q_FLOOR if name in ("Q1", "Q5") else float(np.percentile(Qdata[:, k], 2))
        return np.linspace(lo, float(np.percentile(Qdata[:, k], 98)), n)
    va, vb, vc = (axis_vec(k, nm) for k, nm in zip((ia, ib, ic), axes))

    # full regular lattice (needed by go.Volume), then the feasibility mask
    A, B, C = np.meshgrid(va, vb, vc, indexing="ij")
    flatA, flatB, flatC = A.ravel(), B.ravel(), C.ravel()
    leftover = Q_TOTAL - flatA - flatB - flatC
    keep = leftover >= leftover_min
    Q = np.zeros((keep.sum(), 5))
    Q[:, ia] = flatA[keep]; Q[:, ib] = flatB[keep]; Q[:, ic] = flatC[keep]
    Q[:, others] = leftover[keep, None] * prop
    if verbose:
        print(f"  lattice {n}^3 = {len(flatA)} voxels, {int(keep.sum())} feasible "
              f"(leftover Q2+Q4 >= {leftover_min:.2f} µL/s)")

    colour = sp.make_colouriser(grid)
    tau_c = np.asarray(model.slices, float)
    # the expensive step, ONCE for the whole volume
    CK = np.array([model.mu + model.ckpt[t].predict(Q) @ model.comps for t in model.slices])
    lam_c = _lam_curve_batch(CK, grid) if warp else None
    tconv = s40.predict_conv_time_batch(Q)
    if verbose:
        print(f"  predicted t_conv {tconv.min():.1f}-{tconv.max():.1f} s | "
              f"{len(times)} time frames")
    LAM = np.empty((len(times), len(Q))); AMP = np.empty_like(LAM)
    RGB = np.empty((len(times), len(Q), 3))
    for ti, tv in enumerate(times):
        tau_pix = np.clip(tv / np.maximum(tconv, 1e-9), 0, 1) if time_mode == "seconds" \
            else np.full(len(Q), float(tv))
        S = spectra_at_tau(CK, tau_c, tau_pix, lam_c, grid=grid)
        AMP[ti] = S.max(1)
        LAM[ti] = _lam_batch(S, grid)
        if amplitude == "unit_peak":
            S = S / np.maximum(S.max(1, keepdims=True), 1e-9)
        RGB[ti] = np.array([colour(s) for s in S])
    return dict(axes=axes, va=va, vb=vb, vc=vc, shape=(n, n, n), keep=keep, Q=Q,
                xyz=(flatA[keep], flatB[keep], flatC[keep]), tconv=tconv, times=times,
                time_mode=time_mode, amplitude=amplitude, lam=LAM, amp=AMP, rgb=RGB,
                leftover_min=leftover_min, n=n)


def volume_path(vol):
    return pc.DATA_DIR / (f"hypervol46_{''.join(vol['axes'])}_{vol['time_mode']}_"
                          f"{vol['amplitude']}_n{vol['n']}.npz")


def get_volume(model, runs, grid, force=False, **kw):
    """Cached `build_volume` (the GP pass over ~6000 voxels is the slow part)."""
    probe = dict(axes=kw.get("axes", ("Q1", "Q5", "Q3")), time_mode=kw.get("time_mode", "seconds"),
                 amplitude=kw.get("amplitude", "relative"), n=kw.get("n", VOL_N))
    p = volume_path(probe)
    if p.exists() and not force:
        z = np.load(p, allow_pickle=False)
        m = [str(x) for x in z["meta"]]
        print(f"  loaded hypervolume from cache {p.name}")
        return dict(axes=(m[0], m[1], m[2]), time_mode=m[3], amplitude=m[4], n=int(m[5]),
                    leftover_min=float(m[6]), va=z["va"], vb=z["vb"], vc=z["vc"],
                    shape=tuple(z["shape"]), keep=z["keep"], Q=z["Q"],
                    xyz=(z["x"], z["y"], z["zc"]), tconv=z["tconv"], times=z["times"],
                    lam=z["lam"], amp=z["amp"], rgb=z["rgb"])
    v = build_volume(model, runs, grid, **kw)
    np.savez_compressed(p, va=v["va"], vb=v["vb"], vc=v["vc"], shape=np.array(v["shape"]),
                        keep=v["keep"], Q=v["Q"], x=v["xyz"][0], y=v["xyz"][1], zc=v["xyz"][2],
                        tconv=v["tconv"], times=v["times"], lam=v["lam"], amp=v["amp"],
                        rgb=v["rgb"],
                        meta=np.array([*v["axes"], v["time_mode"], v["amplitude"], str(v["n"]),
                                       f"{v['leftover_min']:.6f}"]))
    print(f"  cached {p.name}  ({p.stat().st_size/1e6:.1f} MB)")
    return v


def _hex(rgb):
    c = (np.clip(np.asarray(rgb, float), 0, 1) * 255).round().astype(int)
    return ["#%02x%02x%02x" % tuple(v) for v in np.atleast_2d(c)]


def perceived_colorscale(values, rgb, n_bins=24):
    """A Plotly colourscale whose colours are the colloid's OWN perceived colour.

    `values` is the scalar being volume-rendered (λmax, or growth fraction) and `rgb` the matching
    per-voxel perceived colours; each of `n_bins` quantile bins contributes the MEDIAN colour of its
    voxels. So an iso-λmax shell is drawn in the colour a sample with that λmax actually looks like,
    instead of an arbitrary heat map. Returns (colorscale, vmin, vmax)."""
    v = np.asarray(values, float); ok = np.isfinite(v)
    if ok.sum() < n_bins * 2:
        return "Turbo", float(np.nanmin(v)) if ok.any() else 0.0, \
               float(np.nanmax(v)) if ok.any() else 1.0
    vmin, vmax = np.nanpercentile(v[ok], [1, 99])
    edges = np.linspace(vmin, vmax, n_bins + 1)
    idx = np.clip(np.digitize(v, edges) - 1, 0, n_bins - 1)
    cols, pos = [], []
    for b in range(n_bins):
        m = ok & (idx == b)
        if m.sum() >= 3:
            cols.append(np.median(np.asarray(rgb)[m], axis=0))
            pos.append((edges[b] + edges[b + 1]) / 2)
    if len(cols) < 2:
        return "Turbo", float(vmin), float(vmax)
    pos = np.array(pos); cols = np.array(cols)
    t = (pos - vmin) / max(vmax - vmin, 1e-9)
    t[0], t[-1] = 0.0, 1.0                              # anchor the ends of the scale
    return [[float(ti), c] for ti, c in zip(t, _hex(cols))], float(vmin), float(vmax)


# `scale=None` -> build the colourscale from the colloid's REAL perceived colour (`perceived_colorscale`).
# That is the right choice for λmax, where hue and the scalar genuinely track each other. It is the WRONG
# choice for the growth fraction: averaged over the whole volume every amp bin is much the same orange, so
# the scale comes out nearly monochrome and the reaction front is invisible — that field gets Turbo.
FIELDS = {
    "lam": dict(key="lam", label="predicted λmax (nm)", scale=None,
                title="iso-λmax shells — where in the reagent volume the plasmon sits"),
    "amp": dict(key="amp", label="growth (fraction of own final)", scale="Turbo",
                title="the reaction front — how far each recipe has grown"),
}


# ── rendering knobs (2026-09-11 redesign — "make the volume VISIBLE") ─────────────────────────
# The first version drew 17 nested shells at 14 % opacity in a colourscale pooled over every time
# frame: the pale early-growth colours washed the scale out, and 17 translucent layers summed to one
# muddy blob. Fewer, more opaque shells in a SATURATED scale built from the fully-grown colours read
# as distinct iso-λmax surfaces; a stacked-slices view and a bigger point cloud complete it.
VOL_SURFACES   = 9          # iso-shells in the `shells` view
VOL_OPACITY    = 0.30       # per-shell opacity
VOL_SAT_BOOST  = 1.8        # HSV saturation multiplier on the perceived colours of the λ scale
VOL_VAL_BOOST  = 1.15       # HSV value (brightness) multiplier, same scale
VOL_SLICE_PCTS = (10, 40, 70)   # `slices` view: Q3 planes at these percentiles of the z axis (the
                                # top of the axis is mostly inside the infeasible corner)
VOL_POINT_SIZE = 5.0        # `true colour` point cloud marker size (big enough to close the lattice)
VOL_POINT_OPACITY = 0.9
VOL_CAMERA = dict(x=-1.9, y=-1.7, z=1.25)   # from the low-Q1 / low-Q5 side: the plasmon gradient faces
                                            # the viewer and the infeasible corner sits at the back
VOL_WEDGE_COLOR, VOL_WEDGE_OPACITY = "#8a8a8a", 0.16   # the out-of-domain corner


def _boost_rgb(rgb, sat=VOL_SAT_BOOST, val=VOL_VAL_BOOST):
    """Saturate/brighten perceived colours (HSV) — the colloid colours are physically pale (they are
    transmission colours through a thin cell) and the iso-shells need to be told apart at a glance."""
    from matplotlib.colors import rgb_to_hsv, hsv_to_rgb
    hsv = rgb_to_hsv(np.clip(np.asarray(rgb, float), 0, 1))
    hsv[..., 1] = np.clip(hsv[..., 1] * sat, 0, 1)
    hsv[..., 2] = np.clip(hsv[..., 2] * val, 0, 1)
    return hsv_to_rgb(hsv)


def _lattice(vol):
    A, B, C = np.meshgrid(vol["va"], vol["vb"], vol["vc"], indexing="ij")
    return np.c_[A.ravel(), B.ravel(), C.ravel()]


def _extend_field(vol, key, ti, outside="nn", sentinel=None):
    """`vol[key][ti]` on the FULL lattice, as `go.Volume` needs it.

    go.Volume wants a regular lattice, but only the `keep` voxels are feasible recipes. Two ways to
    fill the rest (`outside=`):
      • `"sentinel"` — a value below `isomin`. For iso-SHELLS this is what the first version did and
        it is the wrong choice: every shell then CLOSES along the feasibility boundary, so the
        boundary plane is painted with as many translucent layers as there are levels below the
        local value — the 'pale envelope' that hid the interior. It is the RIGHT choice for the
        filled SLICES, where a white floor on the colourscale turns the corner blank (as the 2-D map
        does) instead of extrapolating colour into it.
      • `"nn"` — NEAREST-NEIGHBOUR extension: each infeasible voxel takes the field of the nearest
        feasible one, so the shells continue smoothly to the lattice faces (open, `caps` off) and
        the boundary is not drawn at all. The extended corner is marked separately by the grey
        wedge (`_domain_wedge`) so it is not mistaken for a prediction.
    NaN is never allowed through — one NaN in `value` makes plotly.js's marching cubes return an
    EMPTY mesh (bare axes, no error). An in-domain NaN (unreadable λmax) takes the nearest readable
    voxel's value."""
    from scipy.spatial import cKDTree
    pts = _lattice(vol); keep = vol["keep"]
    if "_nn_out" not in vol:                                   # cache the index map on the dict
        vol["_nn_out"] = cKDTree(pts[keep]).query(pts[~keep])[1] if (~keep).any() else None
    x = np.asarray(vol[key][ti], float).copy()
    ok = np.isfinite(x)
    if not ok.all():
        _, j = cKDTree(pts[keep][ok]).query(pts[keep][~ok]); x[~ok] = x[ok][j]
    full = np.empty(len(pts)); full[keep] = x
    if (~keep).any():
        full[~keep] = x[vol["_nn_out"]] if outside == "nn" else sentinel
    return full.astype(np.float32)                              # float32 halves the embedded payload


def _named_colorscale(name):
    """A named Plotly colourscale ("Turbo") as the explicit [[t, colour], ...] list."""
    import plotly.colors as pcol
    return [[float(t), c] for t, c in pcol.get_colorscale(name)]


def _domain_wedge(vol):
    """The infeasible corner (Q2+Q4 leftover below `leftover_min`) as a translucent grey convex hull,
    so the reader sees where the trained domain ends. None if every voxel is feasible."""
    import plotly.graph_objects as go
    out = _lattice(vol)[~vol["keep"]]
    if len(out) < 4:
        return None
    return go.Mesh3d(x=out[:, 0], y=out[:, 1], z=out[:, 2], alphahull=0, color=VOL_WEDGE_COLOR,
                     opacity=VOL_WEDGE_OPACITY, flatshading=True, hoverinfo="skip",
                     name=f"outside trained domain (PVP + ascorbic < {vol['leftover_min']:.1f} µL/s)",
                     showlegend=True)


def fig_hypervolume(vol, field="lam", opacity=VOL_OPACITY, surface_count=VOL_SURFACES,
                    point_size=VOL_POINT_SIZE, point_opacity=VOL_POINT_OPACITY, max_points=None,
                    show=("shells", "slices", "points"), points=None, slice_pcts=VOL_SLICE_PCTS,
                    camera=None, wedge=True):
    """The Plotly 4-D figure: a 3-D reagent VOLUME with time on a slider (and a play button).

    Three renderings of the same voxels, switched from the buttons (all animated by the slider):
      `shells`      — `go.Volume` iso-surfaces of `field` ("lam" or "amp"): 9 shells at 30 % opacity
                      in a SATURATED colourscale built from the fully-grown perceived colours.
      `slices`      — the same `go.Volume` drawn as three filled Q3 planes (`slice_pcts`): the most
                      legible view of the interior, a stack of three nb33-style Q1×Q5 maps.
      `true colour` — `go.Scatter3d`, the exact per-voxel perceived colour, marker size ∝ growth.
    The infeasible corner is a grey wedge (`wedge=True`); the four paper recipes `points` are static
    black markers. The colourscale is fixed across time (built once from the LAST frame)."""
    import plotly.graph_objects as go
    v = vol; f = FIELDS[field]; nt = len(v["times"]); last = nt - 1
    x, y, z = v["xyz"]; pts = _lattice(v)
    # colourscale from the FULLY-GROWN frame only: pooling every frame let the pale early colours
    # wash the scale out. Boosted so that neighbouring iso-shells are distinct.
    cs, vmin, vmax = perceived_colorscale(v[f["key"]][last], _boost_rgb(v["rgb"][last]))
    if f["scale"] is not None:
        cs = f["scale"]
    if field == "amp":
        vmin, vmax = 0.0, 1.0

    sub = slice(None)
    if max_points is not None and len(x) > max_points:
        # RANDOM thinning, never `linspace`: the voxels are in lattice order, so evenly-spaced indices
        # sample a sub-lattice and the cloud shows strong moiré striping instead of a solid volume.
        sub = np.sort(np.random.default_rng(pc.SEED).choice(len(x), max_points, replace=False))

    def pt_kw(ti):
        return dict(color=_hex(_boost_rgb(v["rgb"][ti][sub])), opacity=point_opacity,
                    size=(point_size * (0.55 + 0.75 * np.clip(v["amp"][ti][sub], 0, 1.2))
                          ).astype(np.float32))

    def pt_custom(ti):
        return np.stack([v["lam"][ti][sub], v["amp"][ti][sub], v["tconv"][sub]],
                        -1).astype(np.float32)

    hover_xyz = (f"{v['axes'][0]}=%{{x:.2f}}<br>{v['axes'][1]}=%{{y:.2f}}<br>"
                 f"{v['axes'][2]}=%{{z:.2f}}<br>")
    vol_common = dict(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], isomin=vmin, isomax=vmax,
                      cmin=vmin, cmax=vmax, colorscale=cs,
                      caps=dict(x_show=False, y_show=False, z_show=False),
                      hovertemplate=hover_xyz + f"{f['label']}=%{{value:.3g}}<extra></extra>")
    cbar = dict(title=f["label"], thickness=16, len=0.6)
    # slices: the infeasible corner gets a sentinel below `cmin`, which the white floor prepended to
    # the colourscale paints blank (values below cmin clamp to the scale's first colour)
    sentinel = vmin - max(1.0, 0.5 * (vmax - vmin))
    cs_list = cs if isinstance(cs, list) else _named_colorscale(cs)
    cs_white = [[0.0, "#ffffff"], [0.002, "#ffffff"]] + [[max(float(t), 0.0021), c] for t, c in cs_list]
    slice_kw = dict(outside="sentinel", sentinel=sentinel)
    traces, names = [], []
    if "shells" in show:
        traces.append(go.Volume(value=_extend_field(v, f["key"], last), opacity=opacity,
                                surface_count=surface_count, opacityscale="uniform",
                                colorbar=cbar, name="shells", visible=True, **vol_common))
        names.append("shells")
    if "slices" in show:
        locs = [float(np.percentile(v["vc"], p)) for p in slice_pcts]
        traces.append(go.Volume(value=_extend_field(v, f["key"], last, **slice_kw), opacity=1.0,
                                surface=dict(show=False),
                                slices=dict(z=dict(show=True, locations=locs, fill=1.0)),
                                colorbar=cbar, showscale=("shells" not in show),
                                name="slices", visible=("shells" not in show),
                                **{**vol_common, "colorscale": cs_white}))
        names.append("slices")
    if "points" in show:
        traces.append(go.Scatter3d(
            x=x[sub], y=y[sub], z=z[sub], mode="markers",
            marker=dict(**pt_kw(last), line=dict(width=0)),
            name="perceived colour", visible=(len(names) == 0),
            customdata=pt_custom(last),
            hovertemplate=(hover_xyz + "λmax=%{customdata[0]:.0f} nm<br>"
                           "growth=%{customdata[1]:.0%}<br>t_conv=%{customdata[2]:.1f} s"
                           "<extra></extra>")))
        names.append("true colour")
    # static traces (do not move with time): appended AFTER the animated ones and left out of every
    # frame's `traces=` index list — the domain wedge and the four paper recipes A-D
    n_anim = len(traces)
    if wedge:
        w = _domain_wedge(v)
        if w is not None:
            traces.append(w)
    if points:
        ia, ib, ic = (list(sp.FEATURES).index(a) for a in v["axes"])
        labs = list(points); P = np.array([np.asarray(points[l], float) for l in labs])
        traces.append(go.Scatter3d(
            x=P[:, ia], y=P[:, ib], z=P[:, ic], mode="markers+text",
            marker=dict(size=point_size * 3.2, color="black", symbol="circle",
                        line=dict(width=2, color="white")),
            text=[f"<b>{l}</b>" for l in labs], textposition="top center",
            textfont=dict(size=15, color="black"), name="recipes A-D", visible=True,
            customdata=P, hovertemplate=(
                "<b>%{text}</b><br>" + "<br>".join(
                    f"{f}=%{{customdata[{k}]:.2f}}" for k, f in enumerate(sp.FEATURES))
                + "<extra></extra>")))

    fig = go.Figure(data=traces)

    frames = []
    for ti in range(nt):
        fd = []
        for nm in names:
            if nm == "shells":
                fd.append(go.Volume(value=_extend_field(v, f["key"], ti)))
            elif nm == "slices":
                fd.append(go.Volume(value=_extend_field(v, f["key"], ti, **slice_kw)))
            else:
                fd.append(go.Scatter3d(marker=dict(**pt_kw(ti), line=dict(width=0)),
                                       customdata=pt_custom(ti)))
        frames.append(go.Frame(data=fd, traces=list(range(len(fd))), name=_tlabel(v, ti)))
    fig.frames = frames

    unit = "s" if v["time_mode"] == "seconds" else "τ"
    steps = [dict(method="animate", label=_tlabel(v, ti),
                  args=[[_tlabel(v, ti)], dict(mode="immediate", frame=dict(duration=0, redraw=True),
                                               transition=dict(duration=0))])
             for ti in range(nt)]
    buttons = []
    if len(names) > 1:
        # the static traces (wedge, A-D) sit after the animated ones and stay visible under EVERY
        # button; each button shows exactly one animated trace (and hands it the colorbar)
        tail = [True] * (len(traces) - n_anim)
        buttons = [dict(type="buttons", direction="right", x=1.0, y=1.02, xanchor="right",
                        yanchor="bottom", showactive=True, buttons=[
                            dict(label=nm, method="update",
                                 args=[{"visible": [k == i for k in range(n_anim)] + tail,
                                        "showscale": [k == i for k in range(n_anim)]}])
                            for i, nm in enumerate(names)])]
    fig.update_layout(
        title=dict(text=f"<b>{f['title']}</b><br><sub>{v['axes'][0]} {REAGENTS[v['axes'][0]]} × "
                        f"{v['axes'][1]} {REAGENTS[v['axes'][1]]} × {v['axes'][2]} "
                        f"{REAGENTS[v['axes'][2]]}, animated in "
                        f"{'wall-clock time' if v['time_mode']=='seconds' else 'progress τ'}"
                        f" — PVP + ascorbic acid take the remainder to 14 µL/s "
                        f"(≥ {v['leftover_min']:.1f}); source: {NOTEBOOK}</sub>",
                   x=0.5, xanchor="center", y=0.97, yanchor="top"),
        scene=dict(
            xaxis_title=f"{v['axes'][0]} — {REAGENTS[v['axes'][0]]} (µL/s)",
            yaxis_title=f"{v['axes'][1]} — {REAGENTS[v['axes'][1]]} (µL/s)",
            zaxis_title=f"{v['axes'][2]} — {REAGENTS[v['axes'][2]]} (µL/s)",
            aspectmode="cube", camera=dict(eye=camera or VOL_CAMERA),
            domain=dict(x=[0, 1], y=[0, 1])),
        legend=dict(x=0.0, y=0.98, xanchor="left", yanchor="top", bgcolor="rgba(255,255,255,0.7)"),
        updatemenus=buttons + [dict(
            type="buttons", direction="left", x=0.0, y=0.06, xanchor="left", yanchor="top",
            showactive=False, pad=dict(t=0, r=6), buttons=[
                dict(label="▶ play", method="animate",
                     args=[None, dict(mode="immediate", fromcurrent=True,
                                      frame=dict(duration=750, redraw=True),
                                      transition=dict(duration=0))]),
                dict(label="❚❚ pause", method="animate",
                     args=[[None], dict(mode="immediate", frame=dict(duration=0, redraw=False))])])],
        # x=0.28 leaves room for the play/pause buttons AND for the `currentvalue` readout, which plotly
        # draws just above the slider's left end (at x=0.16 it sat behind the pause button).
        sliders=[dict(active=nt - 1, x=0.28, len=0.68, y=0.06, yanchor="top", pad=dict(t=8),
                      currentvalue=dict(prefix="time = ", suffix=f" {unit}", font=dict(size=15),
                                        xanchor="left"),
                      steps=steps)],
        margin=dict(l=0, r=0, t=96, b=8), height=820,
        template="plotly_white")
    return fig


def _tlabel(v, ti):
    return f"{v['times'][ti]:.3g}"


def export_plotly(fig, name):
    """Write a self-contained Plotly HTML (plotly.js inlined — no CDN, works offline)."""
    p = pc.FIG_DIR / f"{name}.html"
    fig.write_html(p, include_plotlyjs="inline", full_html=True, auto_play=False,
                   config=dict(displaylogo=False,
                               modeBarButtonsToRemove=["toImage"], responsive=True))
    print(f"  wrote {p.name}  ({p.stat().st_size/1e6:.1f} MB)")
    return p


# ══════════════════════════════════════════════════════════════════════════════
# MECHANISTIC projections × TIME  —  the nb33 §3b slices, animated
# ══════════════════════════════════════════════════════════════════════════════
# Same three chemically-constrained planes as `step33_spectral_model` §3b (x = the
# Ag/NaBH4 flow ratio on a log axis, y = one swept reagent, everything else pinned by
# chemistry and closed by the 14 µL/s total flow) — but the pixel spectrum now comes from
# the nb40 growth predictor at a chosen TIME instead of the nb33 endpoint model. The
# recipe closure, the redox stoichiometry lines and the axis decoration are imported from
# step33, so the static map and these frames are the same slice, just at different times.
#
# The three projections share ONE common y axis (`MECH_YMAX`) so they can sit in one cube
# and be compared panel to panel; the nb33 statics keep their own per-reagent y range.
# Time semantics are exactly those of the flow cube above: `time_mode="seconds"` gives a
# wall-clock frame (each pixel at its own τ = t/t_conv, so fast recipes finish first),
# `time_mode="tau"` puts every pixel at the same fraction of its own growth.
MECH_N    = 56                       # grid points per axis
MECH_YMAX = 7.6                      # µL/s — common top of the swept-reagent axis
MECH_KEYS = tuple(s33.MECH_ORDER)    # ("acoh", "pvp", "asc")
# convergence-time projections: house piecewise colour scale (most of the bar under
# TCONV_SPLIT seconds, compressed above) + labelled iso-lines in seconds.
TCONV_CMAP       = "viridis"
TCONV_SPLIT      = 20.0                  # s — expanded/compressed boundary of the colour bar
TCONV_SPLIT_FRAC = 0.75                  # fraction of the bar given to [0, TCONV_SPLIT]
TCONV_LEVELS     = (2, 3, 5, 7.5, 10, 15, 20, 30, 45, 60, 90, 120)   # used when contours=True
TCONV_CONTOURS   = False                 # OPT-IN: white second-labelled iso-lines
TCONV_N          = 200                   # the convergence-time maps get their OWN grid: a dense
                                         #   grid costs ~1 s and the field is worth resolving far
                                         #   finer than the spectral colour cube.
PT_OFF_SCALE     = 0.30                  # measured runs NOT on the plotted plane are drawn at this
                                         #   fraction of the on-slice marker area. They stay
                                         #   COLOUR-CODED by t_conv — replacing them with a plain
                                         #   dot (as on the spectral maps) would discard the one
                                         #   quantity these panels exist to show.
# Point / font / legend styling of the mechanistic AXES (ticks, labels, molar axis, stoichiometry
# legend + callouts) is shared with step 33 and exposed there — s33.FS_*, s33.LEGEND_LOC,
# s33.STOICH_LEGEND, s33.PT_ON/OFF, or `s33.mech_style(...)` — scaled here by TC_FS_SCALE.
# What is specific to the t_conv figures lives below; `tconv_style(...)` sets several at once.
TC_FS_SCALE   = 0.85     # multiplies every s33.FS_* on the 3-panel figure (panels are narrower)
TC_FS_SCALE_1 = 1.0      # same, single-panel version
TC_FS_SCALE_V = 0.75     # same, the 3x4 PVP-variant grid (12 small panels: full-size fonts collide)
TC_FS_TITLE   = 8.0      # panel titles (the pinning condition), 3-panel figure
TC_FS_TITLE_1 = 9.0      # title of the single-panel version
TC_TITLE_PAD  = 24       # gap (pt) between the molar top axis and the title (3-panel)
TC_TITLE_PAD_1 = 26      #   single-panel
TC_FS_CBAR    = 8.0      # colour-bar tick labels
TC_FS_CBAR_LABEL = None  # colour-bar label; None -> matplotlib's axes.labelsize
TC_CBAR_LABEL = "predicted convergence time $t_{conv}$ (s)"
TC_CBAR_TICKS = (0, 2, 5, 10, 20, 40, 60, 90, 120, 180)   # those <= vmax are kept
TC_FS_FOOT    = 6.0      # footnote under the figure
TC_SHOW_FOOT  = True     # False -> no footnote (put it in the paper caption)
TC_FS_CLABEL  = 6.2      # labels of the second iso-lines (only if TCONV_CONTOURS / contours=True)
TC_MS         = 30.0     # marker area of the MEASURED runs on the slice, 3-panel figure
TC_MS_1       = 42.0     #   single-panel (off-slice runs are PT_OFF_SCALE x this)

_TC_STYLE_KEYS = ("TC_FS_SCALE", "TC_FS_SCALE_1", "TC_FS_SCALE_V", "TC_FS_TITLE", "TC_FS_TITLE_1", "TC_TITLE_PAD",
                  "TC_TITLE_PAD_1", "TC_FS_CBAR", "TC_FS_CBAR_LABEL", "TC_CBAR_LABEL",
                  "TC_CBAR_TICKS", "TC_FS_FOOT", "TC_SHOW_FOOT", "TC_FS_CLABEL", "TC_MS", "TC_MS_1",
                  "PT_OFF_SCALE", "TCONV_CMAP", "TCONV_SPLIT", "TCONV_SPLIT_FRAC", "TCONV_LEVELS",
                  "TCONV_CONTOURS")

def tconv_style(**kw):
    """Set several t_conv-figure knobs at once: `s46.tconv_style(TC_FS_TITLE=10, TC_MS=40)`.
    Unknown names raise. Axis/legend fonts are step 33's: use `s33.mech_style(...)`. Returns the
    current values of every knob (call with no argument to print them)."""
    for k, v in kw.items():
        if k not in _TC_STYLE_KEYS:
            raise KeyError(f"{k!r} is not a t_conv style knob; choose from {_TC_STYLE_KEYS}")
        globals()[k] = v
    return {k: globals()[k] for k in _TC_STYLE_KEYS}


# ── UNCERTAIN ZONES (hatched) on the mechanistic maps ─────────────────────────────────────
#   U = sigma_epistemic / sigma_noise of the GP (s33.gp_uncertainty_ratio): hatched where U > UNC_THRESH,
#   i.e. where the model's own ignorance exceeds the run-to-run scatter it learnt from the repeats.
#   Colour maps: the GP of the FINAL checkpoint (tau = UNC_CKPT) — the shape the colour converges to.
#   t_conv maps: the step-36 GP on log t_conv.
#   Crossed hatch, black '//' + white '\\', so it reads on pale colloid colours AND on dark viridis.
#   TWO LEVELS (2026-10-05): with a single U > 1 the colour maps were 57-84 % hatched — a 2-D slice of a
#   5-D design mostly runs BETWEEN the 151 runs (median pixel 1.1 NN-spacings from the nearest run) and
#   the spectral GPs have short length scales. So: light hatch = U > 1 (less sure than a repeat of the
#   experiment), dense hatch = U > 1.5 (~>1.5 NN-spacings from any run: genuinely unconstrained).
UNC_SHOW   = False        # OFF since 2026-10-05: the user found the hatching hid the maps ("on voit
                          #   rien"). Kept implemented — set True to audit where the model is unsure.
UNC_LEVELS = (1.0, 1.5)   # U thresholds, light -> dense
UNC_HATCHES = (("/", "\\"), ("///", "\\\\\\"))   # (black, white) pattern per level
UNC_CKPT   = 1.0          # growth checkpoint whose GP defines the colour-map uncertainty
UNC_LINE   = True         # dashed boundary at the FIRST level
UNC_LABELS = ("model less certain than a repeat of the experiment",
              "model unconstrained (far from any run)")


def mech_unc_grid(proj_key, rr, yy, which="colour", model=None):
    """U on the (rr, yy) grid of one projection (NaN outside the feasible domain)."""
    G, ok = s33.mech_recipes(s33.MECH_PROJ[proj_key], rr, yy)
    U = np.full(ok.shape, np.nan)
    if ok.any():
        if which == "colour":
            U[ok] = s33.spectral_uncertainty_ratio(model.ckpt[UNC_CKPT], G[ok])
        else:
            U[ok] = s33.gp_uncertainty_ratio(s40._conv_model()["reg"], G[ok])
    return U


def add_mech_uncertainty(cube, model):
    """Attach `cube["unc"]` (n_proj, n, n): the colour-map uncertainty ratio U of every projection.
    Cheap (a GP std on ~3 000 pixels per plane) and recomputed rather than cached."""
    rr, yy = np.asarray(cube["rr"]), np.asarray(cube["yy"])
    cube["unc"] = np.array([mech_unc_grid(k, rr, yy, "colour", model) for k in cube["keys"]])
    for k, U in zip(cube["keys"], cube["unc"]):
        print(f"  {k:12s} U median {np.nanmedian(U):.2f}  max {np.nanmax(U):.2f}  -> hatched " +
              "  ".join(f"U>{t:g}: {100 * np.nanmean(U > t):4.1f} %" for t in UNC_LEVELS))
    return cube


def draw_unc(ax, lx, yy, U, fs=1.0, levels=None):
    """Crossed hatch per uncertainty level (light above levels[0], dense above levels[1], ...) +
    a dashed boundary at the first level."""
    levels = tuple(UNC_LEVELS if levels is None else levels)
    if U is None or not np.isfinite(U).any() or np.nanmax(U) <= levels[0]:
        return
    Uf = np.nan_to_num(U, nan=-1.0)
    hi = np.nanmax(U) * 1.01 + 1e-9
    edges = [l for l in levels if l < hi] + [hi]
    for j in range(len(edges) - 1):
        kb, kw = UNC_HATCHES[min(j, len(UNC_HATCHES) - 1)]
        for col, pat in (("k", kb), ("w", kw)):
            with plt.rc_context({"hatch.color": col, "hatch.linewidth": 0.6 * fs}):
                ax.contourf(lx, yy, Uf, levels=[edges[j], edges[j + 1]], colors="none",
                            hatches=[pat], zorder=3.5)
    if UNC_LINE:
        ax.contour(lx, yy, Uf, levels=[levels[0]], colors="k", linewidths=0.9 * fs,
                   linestyles=[(0, (3, 2))], zorder=3.6)


def unc_handles():
    """Legend proxies, one per hatch level."""
    from matplotlib.patches import Patch
    out = []
    for j, t in enumerate(UNC_LEVELS):
        with plt.rc_context({"hatch.color": "k"}):
            out.append(Patch(facecolor="white", edgecolor="k",
                             hatch=UNC_HATCHES[min(j, len(UNC_HATCHES) - 1)][0] * 2, lw=0.8,
                             label=f"{UNC_LABELS[min(j, len(UNC_LABELS) - 1)]}  "
                                   f"($\\sigma_{{model}}/\\sigma_{{noise}} > {t:g}$)"))
    return out


def _unc_fig_legend(fig, y=0.0, room=0.0):
    """Figure-level legend entry for the hatch; `room` (figure fraction) is added to the bottom
    margin first so the box does not sit on the x labels of a tight_layout'ed figure."""
    if UNC_SHOW:
        if room:
            fig.subplots_adjust(bottom=fig.subplotpars.bottom + room)
        h = unc_handles()
        fig.legend(handles=h, ncol=len(h), loc="lower center", bbox_to_anchor=(0.5, y),
                   fontsize=7.5, frameon=True, framealpha=.95, edgecolor="0.5")


# ── ANISOTROPY THRESHOLD on the acetic-acid planes ────────────────────────────────────────────
#   Silver spheres resonate at ~400-450 nm (yellow); anisotropic growth (plates/discs) red-shifts the
#   plasmon well beyond. On the AcOH planes the bottom band is yellow (λmax ≈ 408 nm) at every Ag/NaBH4
#   ratio: too little acid for anisotropic growth. The line is the predicted FINAL-state λmax = ANISO_LAM
#   iso-level, taken as the FIRST crossing going UP each column (so it is the minimum [AcOH] needed at
#   that Ag/NaBH4 ratio; columns that never red-shift — low Ag/NaBH4 — have no threshold and no line).
#   Drawn only on final-state frames (tau cube, τ = 1): at intermediate times everything is still yellow.
ANISO_LINE  = False       # OFF since 2026-10-08 (user: "vire la ligne de seuil d'AcOH"); True to redraw
ANISO_LAM   = 450.0       # nm: sphere / anisotropic boundary
ANISO_LABEL = "anisotropic growth threshold ($\\lambda_{max}$ = %g nm)"


def aniso_threshold(L, yy, thr=None):
    """y (flow units) per column where the final λmax first reaches `thr` going up from the bottom,
    linearly interpolated between grid rows; NaN where it never does."""
    thr = ANISO_LAM if thr is None else thr
    out = np.full(L.shape[1], np.nan)
    for j in range(L.shape[1]):
        prev = None
        for i in range(L.shape[0]):
            v = L[i, j]
            if not np.isfinite(v):
                continue
            if v >= thr:
                if prev is not None:
                    yp, vp = prev
                    out[j] = yp + (thr - vp) / max(v - vp, 1e-9) * (yy[i] - yp)
                break
            prev = (yy[i], v)
    return out


def draw_aniso(ax, cube, si, ti, fs=1.0, label=True):
    """The threshold line on an AcOH-plane panel showing the FINAL state; no-op otherwise."""
    proj = s33.MECH_PROJ[cube["keys"][si]]
    final = cube["time_mode"] == "tau" and np.isclose(float(cube["times"][ti]), 1.0)
    if not (ANISO_LINE and final and proj["axis"] == "Q3"):
        return None
    lx = np.log10(np.asarray(cube["rr"]))
    yc = aniso_threshold(cube["lam"][si, ti], np.asarray(cube["yy"]))
    if np.isfinite(yc).sum() < 3:
        return None
    ax.plot(lx, yc, color="k", lw=3.6 * fs, alpha=.35, zorder=4.4)
    ax.plot(lx, yc, color="w", lw=2.0 * fs, ls=(0, (5, 2.5)), zorder=4.5,
            label=ANISO_LABEL % ANISO_LAM if label else None)
    return yc


def mech_pixels(key, n=MECH_N, ymax=MECH_YMAX):
    """(Qpix, mask, rr, yy) for one mechanistic projection — step33's chemical closure on a
    (log ratio × swept flow) grid. Infeasible pixels (a flow ≤ 0, or Q1/Q5 under the training
    floor) are dropped from `Qpix` and False in `mask`."""
    proj = s33.MECH_PROJ[key]
    rr = np.geomspace(*s33.RATIO_RANGE, n)
    yy = np.linspace(0.0, ymax, n)
    G, ok = s33.mech_recipes(proj, rr, yy)
    return G[ok], ok, rr, yy


def build_mech_cube(model, runs, grid, keys=MECH_KEYS, n=MECH_N, ymax=MECH_YMAX, times=None,
                    time_mode="seconds", amplitude="relative", warp=True, verbose=True):
    """Precompute the (projection × time) colour cube of the mechanistic slices.

    Returns `rgb` (n_proj, n_time, n, n, 3), `lam`, `amp`, `tconv`, `mask`, `ood` (distance to the
    training recipes, in units of their own median nearest-neighbour spacing), plus `rr`, `yy`,
    `keys`, `times`, `time_mode`, `amplitude`."""
    Qdata = runs[list(sp.FEATURES)].values.astype(float)
    cache = s33.domain_scale(Qdata)
    times = np.asarray(TIMES_S if times is None else times, float) if time_mode == "seconds" \
        else np.asarray(TAUS if times is None else times, float)
    colour = sp.make_colouriser(grid)
    tau_c = np.asarray(model.slices, float)
    ns, nt = len(keys), len(times)
    RGB = np.ones((ns, nt, n, n, 3))
    LAM = np.full((ns, nt, n, n), np.nan); AMP = np.full((ns, nt, n, n), np.nan)
    TC = np.full((ns, n, n), np.nan); OOD = np.full((ns, n, n), np.nan)
    MASK = np.zeros((ns, n, n), bool)
    rr = yy = None
    for si, key in enumerate(keys):
        Qpix, mask, rr, yy = mech_pixels(key, n, ymax)
        MASK[si] = mask
        OOD[si][mask] = s33.domain_distance(Qpix, Qdata, cache)
        # the expensive step, ONCE per projection: every checkpoint for every pixel
        CK = np.array([model.mu + model.ckpt[t].predict(Qpix) @ model.comps for t in model.slices])
        lam_c = _lam_curve_batch(CK, grid) if warp else None
        tconv = s40.predict_conv_time_batch(Qpix)
        TC[si][mask] = tconv
        for ti, tv in enumerate(times):
            tau_pix = np.clip(tv / np.maximum(tconv, 1e-9), 0, 1) if time_mode == "seconds" \
                else np.full(len(Qpix), float(tv))
            S = spectra_at_tau(CK, tau_c, tau_pix, lam_c, grid=grid)
            AMP[si, ti][mask] = S.max(1)
            if amplitude == "unit_peak":
                S = S / np.maximum(S.max(1, keepdims=True), 1e-9)
            RGB[si, ti][mask] = np.array([colour(s) for s in S])
            LAM[si, ti][mask] = _lam_batch(S, grid)
        if verbose:
            print(f"  {key:5s} {int(mask.sum()):5d} pixels | t_conv {np.nanmin(tconv):4.1f}-"
                  f"{np.nanmax(tconv):5.1f} s | distance to data med {np.nanmedian(OOD[si]):.2f} "
                  f"max {np.nanmax(OOD[si]):.2f}")
    return dict(rgb=RGB, lam=LAM, amp=AMP, tconv=TC, mask=MASK, ood=OOD, rr=rr, yy=yy,
                keys=list(keys), times=times, time_mode=time_mode, amplitude=amplitude,
                n=n, ymax=ymax, warp=warp)


def mech_cube_path(cube):
    return pc.DATA_DIR / (f"mechcube46_{'-'.join(cube['keys'])}_{cube['time_mode']}"
                          f"_{cube['amplitude']}_n{cube['n']}.npz")


def save_mech_cube(cube):
    p = mech_cube_path(cube)
    np.savez_compressed(p, rgb=cube["rgb"], lam=cube["lam"], amp=cube["amp"], tconv=cube["tconv"],
                        mask=cube["mask"], ood=cube["ood"], rr=cube["rr"], yy=cube["yy"],
                        times=cube["times"],
                        meta=np.array(cube["keys"] + [cube["time_mode"], cube["amplitude"],
                                                      str(cube["n"]), str(cube["ymax"]),
                                                      str(cube["warp"])]))
    print(f"  cached {p.name} ({p.stat().st_size/1e6:.1f} MB)")


def load_mech_cube(keys=MECH_KEYS, time_mode="seconds", amplitude="relative", n=MECH_N):
    p = mech_cube_path(dict(keys=list(keys), time_mode=time_mode, amplitude=amplitude, n=n))
    if not p.exists():
        return None
    z = np.load(p, allow_pickle=False)
    m = [str(x) for x in z["meta"]]
    nk = len(m) - 5
    return dict(rgb=z["rgb"], lam=z["lam"], amp=z["amp"], tconv=z["tconv"], mask=z["mask"],
                ood=z["ood"], rr=z["rr"], yy=z["yy"], times=z["times"], keys=m[:nk],
                time_mode=m[nk], amplitude=m[nk + 1], n=int(m[nk + 2]), ymax=float(m[nk + 3]),
                warp=(m[nk + 4] == "True"))


def get_mech_cube(model, runs, grid, force=False, **kw):
    """Cached `build_mech_cube`."""
    if not force:
        c = load_mech_cube(**{k: kw[k] for k in ("keys", "time_mode", "amplitude", "n") if k in kw})
        if c is not None:
            print(f"  loaded mechanistic cube from cache {mech_cube_path(c).name}")
            return c
    c = build_mech_cube(model, runs, grid, **kw)
    save_mech_cube(c)
    return c


# ── drawing ───────────────────────────────────────────────────────────────────
def _mech_time_label(cube, ti):
    if cube["time_mode"] == "seconds":
        return f"t = {cube['times'][ti]:.3g} s"
    v = float(cube["times"][ti])
    return "final state (τ = 1)" if np.isclose(v, 1.0) else f"τ = {v:.2f}"


def mech_panel(ax, cube, si, ti, Qdata=None, cache=None, lam_contours=None, clabels=True,
               veil=None, fs=1.0, points=True, stoich=True, legend=True, molar=True, mark=None):
    """One (projection, time) frame with the same annotations as the static nb33 map. The
    lambda_max iso-lines are OPT-IN and follow `s33.MECH_LAM` unless `lam_contours` is given."""
    lam_contours = s33.MECH_LAM if lam_contours is None else lam_contours
    veil = s33.MECH_VEIL if veil is None else veil   # extrapolation veil: OFF unless s33.MECH_VEIL
    c = cube; proj = s33.MECH_PROJ[c["keys"][si]]
    rr = np.asarray(c["rr"]); yy = np.asarray(c["yy"])
    lx = np.log10(rr); ext = [lx[0], lx[-1], yy[0], yy[-1]]
    ax.imshow(c["rgb"][si, ti], origin="lower", aspect="auto", extent=ext)
    if lam_contours:
        L = c["lam"][si, ti]
        if np.isfinite(L).sum() > 20 and np.nanmax(L) - np.nanmin(L) > 25:
            cs = ax.contour(lx, yy, L, levels=s33.MECH_LEVELS, colors="k",
                            linewidths=.7 * fs, alpha=.55)
            if clabels:
                import matplotlib.patheffects as pe
                for t in ax.clabel(cs, fmt="%.0f nm", fontsize=6.2 * fs, inline=True):
                    t.set_path_effects([pe.withStroke(linewidth=1.8, foreground="w")])
    if veil:
        d = c["ood"][si]
        v = np.zeros(d.shape + (4,)); v[..., :3] = 0.62
        v[..., 3] = np.where(np.nan_to_num(d, nan=0.0) > s33.OOD_MAX, 0.40, 0.0)
        ax.imshow(v, origin="lower", aspect="auto", extent=ext)
    if UNC_SHOW and "unc" in c:                    # see add_mech_uncertainty
        draw_unc(ax, lx, yy, c["unc"][si], fs)
    yc = draw_aniso(ax, c, si, ti, fs)
    s33.mech_decorate(ax, proj, rr, yy, Qdata, cache, stoich=stoich, points=points,
                      fs=fs, legend=legend, molar=molar, mark=mark)
    if yc is not None and legend:                  # the AcOH planes have no stoich legend of their own
        ax.legend(fontsize=s33.FS_LEGEND * fs, loc="upper right", **s33.LEGEND_KW).set_zorder(6)


def fig_mech_time(cube, time_idx=None, Qdata=None, cache=None):
    """The whole cube on paper: one ROW per mechanistic projection, one COLUMN per time."""
    c = cube
    ti = list(range(len(c["times"]))) if time_idx is None else list(time_idx)
    ns = len(c["keys"])
    fig, axes = plt.subplots(ns, len(ti), figsize=(2.35 * len(ti), 2.5 * ns), squeeze=False)
    for r in range(ns):
        for k, t in enumerate(ti):
            a = axes[r][k]
            mech_panel(a, c, r, t, Qdata, cache, clabels=False, fs=0.62,
                       points=False, legend=False, molar=(r == 0))
            if r == 0:
                a.set_title(_mech_time_label(c, t), fontsize=8, pad=18)
            if k:
                a.set_yticklabels([]); a.set_ylabel("")
            if k == len(ti) - 1:                   # what is pinned on this row (right margin)
                a.text(1.04, 0.5, s33.MECH_PROJ[c["keys"][r]]["cond"], transform=a.transAxes,
                       fontsize=6.6, rotation=270, va="center", ha="left")
            if r < ns - 1:
                a.set_xlabel("")
    tl = "wall-clock time" if c["time_mode"] == "seconds" else "growth progress τ"
    xdesc = ("$[AgNO_3]/[NaBH_4]$ molar ratio (log), y = reactor concentration (mM)"
             if s33.MECH_UNITS == "conc" else "$Q_{AgNO_3}/Q_{NaBH_4}$ (log)")
    fig.suptitle(f"Mechanistic projections × {tl} — x = {xdesc}, other reagents pinned by "
                 f"chemistry, total 14 µL/s"
                 + ("   |   contours = predicted $\\lambda_{max}$" if s33.MECH_LAM else ""),
                 fontsize=9)
    fig.tight_layout(rect=[0, 0.035 if "unc" in c and UNC_SHOW else 0, 1, 0.965])
    if "unc" in c:
        _unc_fig_legend(fig)
    return fig


def fig_mech_time_row(cube, si, time_idx=None, Qdata=None, cache=None):
    """One projection across time, bigger — the figure for a single panel of nb33 §3b."""
    c = cube
    ti = list(range(len(c["times"]))) if time_idx is None else list(time_idx)
    fig, axes = plt.subplots(1, len(ti), figsize=(3.1 * len(ti), 3.4), squeeze=False)
    for k, t in enumerate(ti):
        a = axes[0][k]
        mech_panel(a, c, si, t, Qdata, cache, clabels=False, fs=0.7, points=False, legend=False)
        a.set_title(_mech_time_label(c, t), fontsize=9, pad=18)
        if k:
            a.set_yticklabels([]); a.set_ylabel("")
    fig.suptitle(s33.MECH_PROJ[c["keys"][si]]["cond"], fontsize=9)
    fig.tight_layout(rect=[0, 0.07 if "unc" in c and UNC_SHOW else 0, 1, 0.94])
    if "unc" in c:
        _unc_fig_legend(fig)
    return fig


def fig_mech_frame(cube, si, ti, Qdata=None, cache=None, mark=None):
    """A single frame, full size (all annotations on)."""
    c = cube
    fig, ax = plt.subplots(figsize=(6.8, 5.8))
    mech_panel(ax, c, si, ti, Qdata, cache, mark=mark)
    L = c["lam"][si, ti]; A = c["amp"][si, ti]
    ax.set_title(f"{s33.MECH_PROJ[c['keys'][si]]['cond']}   |   {_mech_time_label(c, ti)}\n"
                 f"$\\lambda_{{max}}$ {np.nanmin(L):.0f}–{np.nanmax(L):.0f} nm   |   "
                 f"growth {100*np.nanmean(A):.0f} % of final (mean)",
                 fontsize=TC_FS_TITLE_1, pad=TC_TITLE_PAD_1)   # same knob as the t_conv single maps
    fig.tight_layout()
    if "unc" in c:
        _unc_fig_legend(fig, room=0.05)
    return fig


def mech_tconv_grid(key, n=None, ymax=MECH_YMAX, Qdata=None, cache=None):
    """(T, ood, rr, yy) — the predicted convergence-time field of one mechanistic projection on its
    OWN grid, independent of the spectral colour cube. Cheap (a RandomForest, no GP), so `n` can be
    several times the cube's resolution. `ood` is the distance to the training recipes (NaN, and T
    NaN, outside the feasible/trained region)."""
    n = TCONV_N if n is None else n
    Qpix, mask, rr, yy = mech_pixels(key, n, ymax)
    T = np.full((n, n), np.nan); T[mask] = s40.predict_conv_time_batch(Qpix)
    O = np.full((n, n), np.nan)
    if Qdata is not None:
        O[mask] = s33.domain_distance(Qpix, Qdata, cache)
    return T, O, rr, yy


def _tconv_panel(ax, key, T, O, rr, yy, norm, cmap, contours, levels, vmax, fs,
                 Qdata=None, cache=None, measured=None, stoich=True, legend=True, ms=30,
                 mark=None):
    """Shared body of the two convergence-time figures."""
    proj = s33.MECH_PROJ[key]
    lx = np.log10(rr); ext = [lx[0], lx[-1], yy[0], yy[-1]]
    im = ax.imshow(T, origin="lower", aspect="auto", cmap=cmap, norm=norm, extent=ext,
                   interpolation="nearest")
    if contours and np.isfinite(T).sum() > 20:
        cs = ax.contour(lx, yy, T, levels=[v for v in levels if v < vmax], colors="w",
                        linewidths=.7 * fs, alpha=.75)
        ax.clabel(cs, fmt="%g s", fontsize=TC_FS_CLABEL * fs, inline=True)
    if s33.MECH_VEIL:                                       # same extrapolation veil as the colour maps
        v = np.zeros(O.shape + (4,)); v[..., :3] = 1.0      #   (OFF by default, like nb33)
        v[..., 3] = np.where(np.nan_to_num(O, nan=0.0) > s33.OOD_MAX, 0.35, 0.0)
        ax.imshow(v, origin="lower", aspect="auto", extent=ext, interpolation="nearest")
    if UNC_SHOW:                                            # GP on log t_conv: sigma_epi / sigma_noise
        draw_unc(ax, lx, yy, mech_unc_grid(key, rr, yy, "tconv"), fs)
    s33.mech_decorate(ax, proj, rr, yy, Qdata, cache, stoich=stoich, points=False, fs=fs,
                      legend=legend and bool(proj.get("stoich")), mark=mark)
    if measured is not None:                                # the real runs, on the SAME colour scale
        r_d = measured["Q1"].values / measured["Q5"].values
        y_d = measured[proj["axis"]].values
        inx = (r_d >= rr[0]) & (r_d <= rr[-1]) & (y_d <= yy[-1])
        # Split by whether the run really lies ON this plane, like the colour maps do — but keep
        # BOTH classes colour-coded by measured t_conv (a plain black dot would throw away the
        # very quantity this figure is about). Off-slice runs are shrunk and thinned instead.
        near = s33.slice_distance(proj, measured[list(sp.FEATURES)].values.astype(float),
                                  cache) < s33.POINT_TOL if cache is not None else \
               np.ones(len(measured), bool)
        for m, size, lw, alpha in ((inx & ~near, ms * PT_OFF_SCALE, .5, .75),
                                   (inx & near,  ms,               .8, 1.0)):
            if m.any():
                ax.scatter(np.log10(r_d[m]), y_d[m], c=measured["t_meas"].values[m], cmap=cmap,
                           norm=norm, s=size * fs ** 2, edgecolor="w", lw=lw * fs,
                           alpha=alpha, zorder=5)
    return im


def fig_mech_tconv(cube, Qdata=None, cache=None, measured=None, cmap=TCONV_CMAP, vmax=None,
                   levels=TCONV_LEVELS, stoich=True, contours=None, n=None, mark=None):
    """The **convergence time** on the three mechanistic planes: same chemically-constrained slices
    as the colour maps (x = the Ag/NaBH4 flow ratio, y = one swept reagent, everything else pinned by
    chemistry), coloured by the predicted t_conv of the step-36/38 flow model instead of by the
    colloid's colour. This is the field behind the wall-clock slider — why a fixed second is a
    different growth fraction from pixel to pixel — and a design chart in its own right: it says how
    long a recipe needs before it stops changing.

    RESOLUTION: the field is recomputed here on its own `n × n` grid (`TCONV_N`, default 200 — about
    4× the spectral cube), because the t_conv model is a RandomForest and a dense grid is cheap.
    `cube` is used only for the projection list and the y range.

    The colour scale is the house piecewise norm (`eda_rg024.conv_norm`): most of the bar spent on
    0-`TCONV_SPLIT` s, compressed above, because that is where the chemistry actually varies. The
    second-labelled white iso-lines are OPT-IN (`contours=True`, or set `TCONV_CONTOURS`). Pass
    `measured` (`mech_conv_points(runs)`) to overlay the real runs coloured by their MEASURED t_conv
    on the SAME scale — a point that matches its background is a point the model got right."""
    from . import eda_rg024 as eda
    c = cube; ns = len(c["keys"])
    contours = TCONV_CONTOURS if contours is None else contours
    fields = [mech_tconv_grid(k, n, c["ymax"], Qdata, cache) for k in c["keys"]]
    vmax = float(np.nanpercentile([f[0] for f in fields], 98)) if vmax is None else float(vmax)
    norm = eda.conv_norm(0.0, vmax, TCONV_SPLIT, TCONV_SPLIT_FRAC)
    fig, axes = plt.subplots(1, ns, figsize=(4.6 * ns, 4.4), squeeze=False); axes = axes[0]
    if UNC_SHOW:                    # room for the hatch legend BEFORE the colour bar steals space
        fig.subplots_adjust(bottom=fig.subplotpars.bottom + 0.07)
    for si, ax in enumerate(axes):
        T, O, rr, yy = fields[si]
        im = _tconv_panel(ax, c["keys"][si], T, O, rr, yy, norm, cmap, contours, levels, vmax,
                          TC_FS_SCALE, Qdata, cache, measured, stoich, ms=TC_MS, mark=mark)
        ax.set_title(s33.MECH_PROJ[c["keys"][si]]["cond"], fontsize=TC_FS_TITLE, pad=TC_TITLE_PAD)
    cb = fig.colorbar(im, ax=axes, fraction=.024, pad=.02,
                      ticks=[t for t in TC_CBAR_TICKS if t <= vmax])
    _cbar_style(cb)
    if TC_SHOW_FOOT:
        fig.text(0.005, 0.004, f"colour scale expanded below {TCONV_SPLIT:g} s   ·   "
                 f"{fields[0][0].shape[0]}² grid"
                 + ("   ·   white veil = recipe far from any training run" if s33.MECH_VEIL else "")
                 + ("   ·   circles = MEASURED $t_{conv}$ on the same scale — large = run ON this "
                    "slice, small = same $(r, y)$ but a different balance on the pinned reagents"
                    if measured is not None else ""),
                 fontsize=TC_FS_FOOT, color="0.4", ha="left", va="bottom")
    _unc_fig_legend(fig, y=0.0)
    return fig


def _cbar_style(cb):
    """Colour-bar label + tick fonts from the TC_* knobs (shared by the two t_conv figures)."""
    cb.set_label(TC_CBAR_LABEL, fontsize=TC_FS_CBAR_LABEL or plt.rcParams["axes.labelsize"])
    cb.ax.tick_params(labelsize=TC_FS_CBAR)


def fig_mech_tconv_single(cube, si, Qdata=None, cache=None, measured=None, cmap=TCONV_CMAP,
                          vmax=None, levels=TCONV_LEVELS, contours=None, n=None, mark=None):
    """One convergence-time projection, full page width (for the paper). Same fine grid and the same
    opt-in iso-lines as `fig_mech_tconv`."""
    from . import eda_rg024 as eda
    c = cube
    contours = TCONV_CONTOURS if contours is None else contours
    if vmax is None:            # keep the colour scale identical across the three projections
        vmax = float(np.nanpercentile([mech_tconv_grid(k, n, c["ymax"])[0] for k in c["keys"]], 98))
    norm = eda.conv_norm(0.0, vmax, TCONV_SPLIT, TCONV_SPLIT_FRAC)
    T, O, rr, yy = mech_tconv_grid(c["keys"][si], n, c["ymax"], Qdata, cache)
    fig, ax = plt.subplots(figsize=(6.8, 5.8))
    im = _tconv_panel(ax, c["keys"][si], T, O, rr, yy, norm, cmap, contours, levels, vmax,
                      TC_FS_SCALE_1, Qdata, cache, measured, stoich=True, ms=TC_MS_1, mark=mark)
    # FIXED layout: tight_layout + a stolen colour-bar axis + the molar twin axis put the bar on top
    # of the map once a bottom band is reserved for the hatch legend.
    bot = (0.20 if TC_SHOW_FOOT else 0.17) if UNC_SHOW else 0.11
    fig.subplots_adjust(left=0.11, right=0.83, bottom=bot, top=0.86)
    cax = fig.add_axes([0.855, bot, 0.025, 0.86 - bot])
    cb = fig.colorbar(im, cax=cax, ticks=[t for t in TC_CBAR_TICKS if t <= vmax])
    _cbar_style(cb)
    ax.set_title(s33.MECH_PROJ[c["keys"][si]]["cond"], fontsize=TC_FS_TITLE_1, pad=TC_TITLE_PAD_1)
    if TC_SHOW_FOOT:
        fig.text(0.005, 0.004, f"colour scale expanded below {TCONV_SPLIT:g} s   ·   {T.shape[0]}² grid"
                 + ("   ·   white veil = extrapolation" if s33.MECH_VEIL else "")
                 + ("   ·   circles = MEASURED $t_{conv}$ — large = run ON this slice, small = same "
                    "$(r, y)$ but a different balance on the pinned reagents"
                    if measured is not None else ""),
                 fontsize=TC_FS_FOOT, color="0.4", ha="left", va="bottom")
    _unc_fig_legend(fig, y=0.035 if TC_SHOW_FOOT else 0.0)
    return fig


# ── re-pinned PVP/Ag variants: the same acoh / asc planes at other capping ratios ──
PVP_VARIANTS   = (0.5, 1.0, 2.0)          # Q_PVP/Q_Ag FLOW ratios (molar = x 1.2/1.8 = x 0.667)
VARIANT_BASES  = ("acoh", "asc")          # the projections that PIN PVP/Ag (the "pvp" one sweeps it)

def pvp_variant_keys(ratios=PVP_VARIANTS, bases=VARIANT_BASES):
    """Register the variants in s33.MECH_PROJ and return their keys, row-major (ratio, base).
    The ratio equal to s33.PVP_TO_AG maps back onto the default key, so its cube is shared."""
    keys = []
    for r in ratios:
        for b in bases:
            keys.append(b if np.isclose(r, s33.PVP_TO_AG) else s33.mech_variant(b, pvp_to_ag=r))
    return keys


def fig_mech_variants(cube, ratios=PVP_VARIANTS, bases=VARIANT_BASES, ti=-1, Qdata=None,
                      cache=None, measured=None, cmap=TCONV_CMAP, vmax=None, n=None, mark=None):
    """Rows = PVP/Ag ratio, columns = [colour of each base projection | t_conv of each base].
    Colour = frame `ti` of `cube` (default: the last wall-clock frame); t_conv on its own dense grid
    with ONE colour scale for every panel, measured runs overlaid as in `fig_mech_tconv` (with the
    on-/off-slice split computed against EACH row's own pinning, so "on-slice" really means it)."""
    from . import eda_rg024 as eda
    c = cube; nb = len(bases)
    keys = [b if np.isclose(r, s33.PVP_TO_AG) else f"{b}_pvp{r:g}" for r in ratios for b in bases]
    missing = [k for k in keys if k not in c["keys"]]
    if missing:
        raise KeyError(f"cube lacks {missing}: build it with keys=s46.pvp_variant_keys()")
    fields = {k: mech_tconv_grid(k, n, c["ymax"], Qdata, cache) for k in keys}
    if vmax is None:
        vmax = float(np.nanpercentile([f[0] for f in fields.values()], 98))
    norm = eda.conv_norm(0.0, vmax, TCONV_SPLIT, TCONV_SPLIT_FRAC)
    fig, axes = plt.subplots(len(ratios), 2 * nb, figsize=(4.1 * 2 * nb, 3.9 * len(ratios)),
                             squeeze=False)
    fs = TC_FS_SCALE_V

    def short(k):            # the row label already says PVP/Ag: keep only the other condition
        return ",   ".join(p for p in s33.MECH_PROJ[k]["cond"].split(",   ") if "PVP" not in p)

    for ri, r in enumerate(ratios):
        for bi, b in enumerate(bases):
            k = keys[ri * nb + bi]; si = c["keys"].index(k)
            a = axes[ri][bi]                                   # colour
            mech_panel(a, c, si, ti, Qdata, cache, clabels=False, fs=fs, points=Qdata is not None,
                       legend=(ri == 0), molar=True, veil=False, mark=mark)
            a.set_title(short(k), fontsize=TC_FS_TITLE * fs, pad=TC_TITLE_PAD * fs)
            a = axes[ri][nb + bi]                              # convergence time
            T, O, rr, yy = fields[k]
            im = _tconv_panel(a, k, T, O, rr, yy, norm, cmap, False, TCONV_LEVELS, vmax, fs,
                              Qdata, cache, measured, stoich=True, legend=False, ms=TC_MS,
                              mark=mark)
            a.set_title(short(k), fontsize=TC_FS_TITLE * fs, pad=TC_TITLE_PAD * fs)
        axes[ri][0].annotate(
            f"$Q_{{PVP}}/Q_{{Ag}} = {r:g}$\n($n_{{PVP}}/n_{{Ag}} = {s33.molar_per_ag('Q2', r):.2g}$)",
            (-0.36, 0.5), xycoords="axes fraction", rotation=90, ha="center", va="center",
            fontsize=(TC_FS_TITLE + 1) * fs, fontweight="bold")
    tlab = _mech_time_label(c, ti if ti >= 0 else len(c["times"]) + ti)
    fig.tight_layout(rect=[0.02, 0.035 if UNC_SHOW else 0.02, 0.93, 0.965])
    _unc_fig_legend(fig, y=0.0)
    for j0, txt in ((0, f"predicted colour  ({tlab})"), (nb, "predicted convergence time")):
        x0 = axes[0][j0].get_position().x0; x1 = axes[0][j0 + nb - 1].get_position().x1
        fig.text((x0 + x1) / 2, 0.992, txt, ha="center", va="top", fontsize=(TC_FS_TITLE + 2) * fs,
                 fontweight="bold")
    cax = fig.add_axes([0.945, 0.15, 0.012, 0.7])
    cb = fig.colorbar(im, cax=cax, ticks=[t for t in TC_CBAR_TICKS if t <= vmax])
    _cbar_style(cb)
    return fig


def abcd_mark(runs, abcd):
    """`mark` argument for the mechanistic figures: [(row index in `runs`, label), ...] from a
    {label: run name} dict (the nb46 §1 `ABCD`). Rows index `Qdata = runs[FEATURES].values`."""
    pos = {r: i for i, r in enumerate(runs.run.values)}
    missing = [r for r in abcd.values() if r not in pos]
    if missing:
        print(f"  WARNING: not in runs, not marked: {missing}")
    return [(pos[r], lab) for lab, r in abcd.items() if r in pos]


def mech_conv_points(runs):
    """`runs` + their MEASURED convergence time (s), ready for the `measured=` overlay above.
    Runs whose t_conv could not be measured are dropped."""
    t = s40.conv_time_table(runs)[["run", "t_meas"]]
    return runs.merge(t, on="run", how="inner")


def mech_widget(cube, Qdata=None, cache=None, figsize=(6.6, 5.4)):
    """Projection selector + time slider over the precomputed mechanistic cube (instant redraw)."""
    import ipywidgets as W
    from IPython.display import display
    c = cube
    p_sl = W.Dropdown(options=[(f"{s40.REAGENTS[s33.MECH_PROJ[k]['axis']]}  ({k})", i)
                               for i, k in enumerate(c["keys"])],
                      value=0, description="projection", style={"description_width": "110px"},
                      layout=W.Layout(width="420px"))
    tlab = "t (s)" if c["time_mode"] == "seconds" else "progress τ"
    t_sl = W.SelectionSlider(options=[(f"{v:.3g}", i) for i, v in enumerate(c["times"])],
                             value=len(c["times"]) - 1, description=tlab, continuous_update=False,
                             style={"description_width": "110px"}, layout=W.Layout(width="620px"))
    out = W.Output()

    def draw(*_):
        with out:
            out.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=figsize)
            mech_panel(ax, c, p_sl.value, t_sl.value, Qdata, cache)
            L = c["lam"][p_sl.value, t_sl.value]; A = c["amp"][p_sl.value, t_sl.value]
            ax.set_title(f"{s33.MECH_PROJ[c['keys'][p_sl.value]]['cond']}   |   "
                         f"{_mech_time_label(c, t_sl.value)}\n"
                         f"$\\lambda_{{max}}$ {np.nanmin(L):.0f}–{np.nanmax(L):.0f} nm   |   "
                         f"growth {100*np.nanmean(A):.0f} % of final (mean)", fontsize=9, pad=26)
            fig.tight_layout(); plt.show()

    p_sl.observe(draw, "value"); t_sl.observe(draw, "value")
    display(W.VBox([p_sl, t_sl, out])); draw()
    return p_sl, t_sl


def export_mech_html(cube, path=None, upscale=9):
    """The mechanistic cube as ONE self-contained HTML file (projection + time sliders), same
    viewer as the flow cube — the x axis is the log flow ratio, so its end labels are strings."""
    import json
    c = cube
    path = pc.FIG_DIR / f"fig46_mech_projections_{c['time_mode']}.html" if path is None else path
    meta = dict(
        frames=_frames_b64(c["rgb"], upscale),
        slider_vals=[float(i) for i in range(len(c["keys"]))],
        slider_labels=[s33.MECH_PROJ[k]["cond"].replace("$", "") for k in c["keys"]],
        times=[float(v) for v in c["times"]],
        time_mode=c["time_mode"], slider_flow="projection", slider_reagent="",
        xlab="Q_AgNO3 / Q_NaBH4  (flow ratio, log)  —  ×50 = molar ratio",
        ylab="swept reagent flow (µL/s)  —  [C] (mM) = stock × Q / 14",
        xlim=[float(np.log10(c["rr"][0])), float(np.log10(c["rr"][-1]))],
        ylim=[float(c["yy"][0]), float(c["yy"][-1])],
        x0s=f"{c['rr'][0]:g}", x1s=f"{c['rr'][-1]:g}",
        lam=[[None if not np.isfinite(np.nanmin(c["lam"][s, t])) else
              [float(np.nanmin(c["lam"][s, t])), float(np.nanmax(c["lam"][s, t]))]
             for t in range(len(c["times"]))] for s in range(len(c["keys"]))],
        amp=[[float(np.nanmean(c["amp"][s, t])) for t in range(len(c["times"]))]
             for s in range(len(c["keys"]))],
        notebook=NOTEBOOK,
    )
    with open(path, "w", encoding="utf-8") as f:
        f.write(_HTML_TEMPLATE.replace("__DATA__", json.dumps(meta)))
    print(f"  wrote {path}  ({path.stat().st_size/1e6:.1f} MB)")
    return path


def main():
    runs, F, grid = s40.load()
    model = s40.fit_deploy(runs, F)

    print(f"{len(runs)} runs | Plotly HYPERVOLUME (Q1 × Q5 × Q3) × time ...")
    vol = get_volume(model, runs, grid, axes=("Q1", "Q5", "Q3"), n=VOL_N, time_mode="seconds",
                     amplitude="relative", warp=True)
    export_plotly(fig_hypervolume(vol, field="lam"), "fig46_hypervolume_lambda")
    export_plotly(fig_hypervolume(vol, field="amp"), "fig46_hypervolume_growth")

    print(f"\nstatic 2-D colour map (Q1×Q5 plane, Q3 + time on sliders) ...")
    cube = get_cube(model, runs, grid, pair=("Q1", "Q5"), slider_flow="Q3", grid_n=56,
                    time_mode="seconds", amplitude="relative", warp=True)
    savefig(fig_matrix(cube, time_idx=[0, 2, 4, 6, 8, 11]), "fig46_matrix")
    savefig(fig_slice(cube, len(cube["slider_vals"]) // 2, len(cube["times"]) - 1), "fig46_slice")
    savefig(fig_tconv(cube), "fig46_tconv")
    export_html(cube)

    print(f"\nMECHANISTIC projections x time (nb33 §3b slices, animated) ...")
    Qdata = runs[list(sp.FEATURES)].values.astype(float)
    cache = s33.domain_scale(Qdata)
    mech = get_mech_cube(model, runs, grid, time_mode="seconds", amplitude="relative", warp=True)
    savefig(fig_mech_time(mech, time_idx=[0, 2, 4, 6, 8, 11], Qdata=Qdata, cache=cache),
            "fig46_mech_time")
    for si, k in enumerate(mech["keys"]):
        savefig(fig_mech_time_row(mech, si, [0, 2, 4, 6, 8, 11], Qdata, cache),
                f"fig46_mech_time_{k}")
    savefig(fig_mech_frame(mech, len(mech["keys"]) - 1, len(mech["times"]) - 1, Qdata, cache),
            "fig46_mech_frame")
    meas = mech_conv_points(runs)
    savefig(fig_mech_tconv(mech, Qdata, cache, meas), "fig46_mech_tconv")
    for si, k in enumerate(mech["keys"]):
        savefig(fig_mech_tconv_single(mech, si, Qdata, cache, meas), f"fig46_mech_tconv_{k}")
    export_mech_html(mech)
    print("Done. In the notebook: `s46.fig_hypervolume(vol).show()` / `s46.mech_widget(mech)`.")


if __name__ == "__main__":
    main()


### `ml/run`

In [ ]:
"""Driver of the ML chain of the paper — runs, in order, exactly the cells of the original notebooks 30, 31b, 33, 36
(as a SCRIPT: the notebook does not save the deployed t_conv model), 40, 43 and 46 that the paper figures and numbers
depend on, with the vendored original code. Exploratory cells (benchmarks not used downstream,
widgets, HTML/3-D exports, ablations) are not run.

Data products  -> data/processed/ml/   (the original caches: delete the folder for a rebuild from raw/)
Figures        -> figures/ml/          (original file names, e.g. fig40_recipes4.png = Fig 5 A-D)

Usage:  python -m toofast_analysis.ml.run            (≈ 30 min; every step reuses the caches already present)
"""
import numpy as np
import pandas as pd

from toofast_analysis.ml import pipeline_common as pc

sp = pc.sp

# Fig 5 A-D (nb40 §4, nb46 §1)
ABCD = {"A": "S25-M02_init_4", "B": "S28-M02_init_4", "C": "S14-M14_sample_6", "D": "S14-M14_sample_1"}
SI_SEED = 4                                           # nb40 §5: random draw of the 9 S24 recipes
# S10 pair (decision 2026-10-22): closest final shape among the growth-divergent pairs of the aligned
# spectra (cosine 0.0014, Ext_max 0.65 / 0.61, growth 12 / 31 s). nb43 §7 had S14-M14_sample_13 / S15-M05_sample_14,
# chosen on misaligned spectra (true cosine 0.045).
S10_PAIR = ("S14-M14_sample_10", "S15-M05_sample_13")


def step30():
    """nb30: selection (189) and clean set, nothing dropped from `selection`."""
    df, S, grid = pc.build_selection(force_rescan=False)
    df["is_outlier"] = pc.flag_outliers(df, S)
    keep = (~df.neg_drop) & (~df.is_outlier)
    clean = df[keep].reset_index(drop=True); Sc = S[keep.values]
    pc.save(df, S, grid, "selection")
    pc.save(clean, Sc, grid, "clean")
    return dict(n_selection=len(df), n_clean=len(clean))


def step31b():
    """nb31b: hard-cut cascade -> hardcut (151), the spectral-model training set."""
    from toofast_analysis.ml import step31b_hardcut_pipeline as s31b
    df, S, grid = pc.load("selection")
    st, summary = s31b.run_cascade(df, S)
    fin, Sfin = st["final"]
    pc.save(fin, Sfin, grid, tag="hardcut")
    summary.to_csv(pc.DATA_DIR / "hardcut_cascade.csv", index=False)
    cols = (["filename", "session"] + pc.FEATURES +
            ["lambda_max", "spec_std_grid", "peak_prom", "peak_height", "is_flat", "neg_fraction"])
    for key, name in [("cut_out", "hardcut_removed_flowcut"), ("neg_out", "hardcut_negatives"),
                      ("flat_out", "hardcut_remaining_flats"), ("out_out", "hardcut_outliers")]:
        st[key][0][cols].to_csv(pc.DATA_DIR / f"{name}.csv", index=False)
    return summary


def step33():
    """nb33: deployed spectral model (pinned FORCE_MODEL, the benchmark only chose among names) + shape metrics."""
    import joblib
    from toofast_analysis.ml import step33_spectral_model as s33
    df, S, wl = s33.load()
    Q = df[sp.FEATURES].values.astype(float); groups = df.session.values
    Sn, _ = sp.normalise(S, s33.NORMALIZE)
    mu, comps, evr = sp.fit_pca(Sn, s33.N_PC); scores = (Sn - mu) @ comps.T
    if s33.FORCE_MODEL is None:
        raise RuntimeError("step33.FORCE_MODEL is None: the deployed model would depend on the benchmark argmin")
    f_, m_ = s33.FORCE_MODEL
    feat = sp.FEATURE_SETS[f_]; fac = sp.MODELS[m_]
    Sp = sp.reconstruct(mu, comps, sp.cv_scores(Q, scores, feat, fac, groups, s33.CV_K))
    rep = s33.shape_report(Sn, Sp, Q, wl)
    model = sp.SpectralModel(s33.N_PC, feat, fac).fit(Q, scores)
    np.savez(pc.DATA_DIR / "spectral_pca_basis.npz", wavelengths=wl, mean=mu, components=comps,
             evr=evr, normalize=s33.NORMALIZE)
    joblib.dump(dict(model=model, feature=f_, mean=mu, components=comps,
                     wavelengths=wl, normalize=s33.NORMALIZE, n_pc=s33.N_PC),
                pc.DATA_DIR / "spectral_model.pkl")
    return dict(sam_median=float(np.median(rep["sam"])), base_median=float(np.median(rep["base"])),
                floor=float(rep["floor"]), n_pairs=int(rep["n_pairs"]),
                frac_below_15=float((rep["sam"] < 15).mean()))


def step36():
    """step36 SCRIPT: benchmark + explicit deployment of the t_conv model (data/processed/ml/convergence_model.pkl)."""
    from toofast_analysis.ml import step36_convergence_model as s36
    s36.main()


def pearson(A, B):
    """Pearson correlation between matching spectra (last axis = wavelength)."""
    A = A - A.mean(-1, keepdims=True)
    B = B - B.mean(-1, keepdims=True)
    return (A * B).sum(-1) / np.maximum(np.linalg.norm(A, axis=-1) * np.linalg.norm(B, axis=-1), 1e-12)


def kinetic_pearson_loso(s40, runs, F, grid):
    """Spectral agreement quoted in the text (decision 2026-10-22): Pearson r between measured and predicted spectra
    at each τ, TRUE leave-one-session-out (13 folds, as t_conv), same 151 runs and repeat pairs as kinetic_report.
    The no-model baseline is the mean trajectory of the TRAINING runs of each fold (kinetic_report averages all runs,
    the held-out one included). The model is step40's own, through kinetic_report with k = number of sessions."""
    from sklearn.model_selection import GroupKFold
    k = runs.session.nunique()
    kin = s40.kinetic_report(runs, F, grid, k=k)
    meas, pred, taus = kin["meas"], kin["pred"], kin["taus"]
    base = np.full_like(meas, np.nan)
    for tr_i, te_i in GroupKFold(n_splits=k).split(runs, groups=runs.session.values):   # = kinetic_report's folds
        base[te_i] = meas[tr_i].mean(0)
    Q = runs[list(sp.FEATURES)].values.astype(float)
    dQ = np.sqrt((((Q[:, None, :] - Q[None, :, :]) / Q.std(0)) ** 2).sum(-1))
    np.fill_diagonal(dQ, np.inf)
    i, j = np.where(np.triu(dQ < s40.KIN_REP_DQ, 1))                      # = kinetic_repeat_floor's pairs
    return dict(taus=taus, n_folds=k, r_model=pearson(meas, pred), r_base=pearson(meas, base),
                r_repeat=pearson(meas[i], meas[j]), sam_model=kin["sam"], sam_base=s40._sam(meas, base),
                sam_repeat=s40._sam(meas[i], meas[j]))


def step40():
    """nb40: Fig 5 A-D (fig40_recipes4), S24 (fig40_recipes9_SI), S23 parity (fig40_conv_time), kinetic metrics."""
    from toofast_analysis.ml import step40_growth_prediction as s40
    runs, F, grid = s40.load()
    s40.fit_deploy(runs, F)
    ct, name = s40.conv_time_table_loso()
    st = s40.conv_time_stats(ct)
    s40.savefig(s40.fig_conv_time(ct, st, protocol="leave-session-out (one fold per session)"), "fig40_conv_time")
    s40.savefig(s40.fig_conv_time_log(ct), "fig40_conv_time_log")
    recipes = list(ABCD.values())
    fig4 = s40.fig_known_recipes(runs, F, grid, recipes, ncol=2)
    s40.savefig(fig4, "fig40_recipes4")
    fig4.recipe_metrics.to_csv(pc.FIG_DIR / "fig40_recipes4_metrics.csv", index=False)
    si = s40.sample_recipes(runs, n=9, seed=SI_SEED, exclude=recipes)
    fig9 = s40.fig_recipe_grid(runs, F, grid, si, ncol=3)
    s40.savefig(fig9, "fig40_recipes9_SI")
    fig9.recipe_metrics.to_csv(pc.FIG_DIR / "fig40_recipes9_SI_metrics.csv", index=False)
    kin = s40.kinetic_report(runs, F, grid, k=5)
    s40.savefig(s40.fig_kinetic_metrics(kin), "fig40_kinetic_metrics")
    return dict(conv_reg=name, log_r2=float(ct.attrs["log_r2"]), conv_stats=st, si_recipes=si, kin=kin,
                kin_r=kinetic_pearson_loso(s40, runs, F, grid),
                recipes4=fig4.recipe_metrics, recipes9=fig9.recipe_metrics)


def d43_load(d43):
    """`d43.load()` with the final spectra aligned to their runs. The vendored load() drops the non-particle rows of
    the clean table but not of its spectra, so from the first dropped row on every run got another run's final
    spectrum (found 2026-10-22). Here it is given the particle runs only, with the spectra filtered the
    same way: its own particle filter then drops nothing."""
    load = pc.load

    def particles(name):
        df, S, grid = load(name)
        keep = df.is_particle.values
        return df[keep].reset_index(drop=True), S[keep], grid

    pc.load = particles
    try:
        return d43.load()
    finally:
        pc.load = load


def step43():
    """nb43 §7: S10 (fig43_pair_solo_spectra / _area), with the notebook's legend knobs."""
    from toofast_analysis.ml import step43_endpoint_degeneracy as d43
    df, S, Sn, grid = d43_load(d43)
    i, j = (int(np.flatnonzero(df.filename == n)[0]) for n in S10_PAIR)
    print(f"S10 pair: endpoint cosine distance {1 - Sn[i] @ Sn[j] / np.linalg.norm(Sn[i]) / np.linalg.norm(Sn[j]):.4f}")
    d43.REAGENT = {"Q1": r"AgNO_3", "Q2": r"PVP", "Q3": r"AcOH", "Q4": r"AscAcid", "Q5": r"NaBH_4"}
    d43.FLOW_ITEM_FMT = r"$Q_{{\mathrm{{{reagent}}}}}$ = {value:.2f}"
    d43.FLOW_UNIT = "µL/s"
    d43.FLOW_WRAP = 3
    d43.AREA_ANNOT_FMT = "final time {t_end:.0f} s"     # the star marks the last spectrum, not the growth time
    legend = dict(label_fmt="Sample {n}\nFlow ({unit}): {flows}",
                  legend_kw=dict(loc="lower right", fontsize=14, ncol=1, frameon=True,
                                 facecolor="white", framealpha=0.9, edgecolor="0.75"))
    area_legend = dict(label_fmt="Sample {n}",
                       legend_kw=dict(loc="lower right", fontsize=16, ncol=1, frameon=True,
                                      facecolor="white", framealpha=0.92, edgecolor="0.75"))
    a, b = S10_PAIR
    d43.savefig(d43.fig_pair_spectra_solo(df, S, Sn, grid, a, b, ymax=0.75, **legend), "fig43_pair_solo_spectra")
    d43.savefig(d43.fig_pair_area_solo(df, a, b, **area_legend), "fig43_pair_solo_area")
    return df.set_index("filename").loc[list(S10_PAIR)]


def step46():
    """nb46 §9-§13: mechanistic colour / t_conv maps at tau = 1 (Fig 5 top = fig46_single_*_acoh_pvp1_conc,
    S23bis = the pvp / asc planes). Cubes are recomputed with the CURRENT growth model (decision (a), 2026-10-12)."""
    from toofast_analysis.ml import step33_spectral_model as s33
    from toofast_analysis.ml import step40_growth_prediction as s40
    from toofast_analysis.ml import step46_colour_map_4d as s46
    runs, F, grid = s40.load()
    model = s40.fit_deploy(runs, F)
    Qdata = runs[list(sp.FEATURES)].values.astype(float)
    cache = s33.domain_scale(Qdata)
    mech = s46.get_mech_cube(model, runs, grid, time_mode="seconds", amplitude="relative", warp=True)
    s46.add_mech_uncertainty(mech, model)
    mech_tau = s46.get_mech_cube(model, runs, grid, time_mode="tau", amplitude="relative", warp=True)
    # style cell of nb46 §11 (shared step-33 axes + t_conv specifics)
    s33.mech_style(
        FS_TICK=13.0, FS_LABEL=13, FS_LEGEND=8.4, FS_ANNOT=8.3, FS_MOLAR=5.0, LEGEND_LOC="upper left",
        STOICH_LEGEND={"asc2": r"ascorbic stoich. ($2e^-$): $Q_{Asc}=0.5\,Q_{Ag}$",
                       "asc1": r"if $1e^-$ per ascorbic: $Q_{Asc}=Q_{Ag}$",
                       "total": r"full $e^-$ balance incl. NaBH$_4$"},
        ANNOT_ABOVE="excess reductant\n(full Ag reduction)", ANNOT_BELOW="sub-stoichiometric\n(Ag$^+$ left over)")
    s46.tconv_style(
        TC_FS_SCALE=1, TC_FS_SCALE_1=1.0, TC_FS_TITLE=11.5, TC_FS_TITLE_1=12, TC_TITLE_PAD=24, TC_TITLE_PAD_1=26,
        TC_FS_CBAR=13.0, TC_FS_CBAR_LABEL=14.0, TC_CBAR_LABEL="predicted convergence time $t_{conv}$ (s)",
        TC_CBAR_TICKS=(0, 2, 5, 10, 20, 40, 60, 90, 120, 180), TC_FS_FOOT=2.0, TC_SHOW_FOOT=False,
        TC_FS_CLABEL=6.2, TC_MS=30.0, TC_MS_1=42.0, PT_OFF_SCALE=0.30, TCONV_CMAP="viridis",
        TCONV_SPLIT=20.0, TCONV_SPLIT_FRAC=0.75, TCONV_CONTOURS=False)
    meas = s46.mech_conv_points(runs)
    vkeys = s46.pvp_variant_keys()
    mech_pvp = s46.get_mech_cube(model, runs, grid, keys=tuple(vkeys), time_mode="seconds",
                                 amplitude="relative", warp=True)
    mech_pvp_tau = s46.get_mech_cube(model, runs, grid, keys=tuple(vkeys), time_mode="tau",
                                     amplitude="relative", warp=True)
    s46.add_mech_uncertainty(mech_pvp_tau, model)
    s33.MECH_UNITS = "conc"
    s33.MARK_SHAPE = "circle"
    mark = s46.abcd_mark(runs, ABCD)
    vmax_single = float(np.nanpercentile(
        [s46.mech_tconv_grid(k)[0] for k in dict.fromkeys(mech["keys"] + mech_pvp["keys"])], 98))
    done = []
    for cube in (mech_tau, mech_pvp_tau):
        ti = int(np.argmin(np.abs(np.asarray(cube["times"]) - 1.0)))
        for si, k in enumerate(cube["keys"]):
            if k in done:
                continue
            done.append(k)
            tag = s33.mech_file_tag(k)
            s46.savefig(s46.fig_mech_frame(cube, si, ti, Qdata, cache, mark=mark), f"fig46_single_colour_{tag}_conc")
            s46.savefig(s46.fig_mech_tconv_single(cube, si, Qdata, cache, meas, vmax=vmax_single, mark=mark),
                        f"fig46_single_tconv_{tag}_conc")
    return dict(vmax_single=vmax_single, keys=done)


STEPS = [step30, step31b, step33, step36, step40, step43, step46]
SUMMARY = pc.DATA_DIR / "ml_summary.json"


def summarise_pearson(kr):
    """Medians per τ of the LOSO Pearson metrics (+ the spectral angles on the same predictions, for the record)."""
    med = lambda x: [float(np.median(x[:, i])) for i in range(len(kr["taus"]))]
    return dict(taus=[float(t) for t in kr["taus"]], n_folds=int(kr["n_folds"]), n_pairs=int(len(kr["r_repeat"])),
                r_model_median=med(kr["r_model"]), r_base_median=med(kr["r_base"]),
                r_repeat_median=med(kr["r_repeat"]),
                frac_r_above_0p95=[float((kr["r_model"][:, i] > 0.95).mean()) for i in range(len(kr["taus"]))],
                frac_r_above_repeat=[float((kr["r_model"][:, i] >= np.median(kr["r_repeat"][:, i])).mean())
                                     for i in range(len(kr["taus"]))],
                sam_model_median=med(kr["sam_model"]), sam_base_median=med(kr["sam_base"]),
                sam_repeat_median=med(kr["sam_repeat"]),
                frac_sam_below_15=[float((kr["sam_model"][:, i] < 15).mean()) for i in range(len(kr["taus"]))])


def summarise(out):
    """The numbers quoted in the paper, from the step outputs -> data/processed/ml/ml_summary.json (read by the tests)."""
    kin, st = out["step40"]["kin"], out["step40"]["conv_stats"]
    taus = [float(t) for t in kin["taus"]]
    pair = out["step43"]
    return dict(
        ml_dataset=dict(n_selection=out["step30"]["n_selection"], n_clean=out["step30"]["n_clean"],
                        cascade=out["step31b"].to_dict(orient="records")),
        spectral_model_final=out["step33"],
        tconv=dict(regressor=out["step40"]["conv_reg"], log_r2=out["step40"]["log_r2"],
                   **{k: (float(v) if isinstance(v, (float, np.floating)) else int(v)) for k, v in st.items()
                      if isinstance(v, (int, float, np.integer, np.floating))}),
        kinetic=dict(taus=taus, n_pairs=int(kin["n_pairs"]),
                     sam_median=[float(np.median(kin["sam"][:, i])) for i in range(len(taus))],
                     base_median=[float(np.median(kin["base"][:, i])) for i in range(len(taus))],
                     floor=[float(f) for f in kin["floor"]],
                     frac_below_15=[float((kin["sam"][:, i] < 15).mean()) for i in range(len(taus))],
                     sam_median_all=float(np.median(kin["sam"]))),
        kinetic_pearson_loso=summarise_pearson(out["step40"]["kin_r"]),
        si_recipes=list(out["step40"]["si_recipes"]),
        s10_pair={k: dict(growth_time_s=float(pair.loc[k, "growth_time_s"])) for k in pair.index},
        fig46=out["step46"])


def main():
    import json
    import matplotlib
    matplotlib.use("Agg")
    out = {}
    for f in STEPS:
        print(f"\n===== {f.__name__} =====", flush=True)
        out[f.__name__] = f()
    SUMMARY.write_text(json.dumps(summarise(out), indent=1), encoding="utf-8")
    print(f"summary -> {SUMMARY}")
    return out


## Figure

In [ ]:
import json
from IPython.display import Image, display
import pandas as pd

out = main()
summary = json.loads(SUMMARY.read_text(encoding='utf-8'))
FIG = FIGURES / 'ml'


## Dataset and t_conv model (S23)

In [ ]:
print(pd.DataFrame(summary['ml_dataset']['cascade']).to_string(index=False))
t = summary['tconv']
print(f"t_conv ({t['regressor']}, leave-one-session-out): log-R2 {t['log_r2']:.3f} | MAE {t['MAE']:.2f} s | medAE {t['medAE']:.2f} s")
display(Image(str(FIG / 'fig40_conv_time.png')))


## Fig 5 bottom — recipes A–D (leave-one-session-out predictions)

In [ ]:
print(pd.read_csv(FIG / 'fig40_recipes4_metrics.csv').round(2).to_string(index=False))
display(Image(str(FIG / 'fig40_recipes4.png')))


## Spectral angles along the growth (5 grouped folds by session)

⚑ The text quotes 12.3° (τ = 1), 13.4° (τ = 0.2), 66 % below 15°; the code gives the values below.

In [ ]:
k = summary['kinetic']
print(pd.DataFrame({'tau': k['taus'], 'model median (deg)': k['sam_median'], 'no-model (deg)': k['base_median'],
                    'floor (deg)': k['floor'], '% < 15 deg': [100 * f for f in k['frac_below_15']]}).round(2).to_string(index=False))
display(Image(str(FIG / 'fig40_kinetic_metrics.png')))


## Fig 5 top — colour and t_conv maps (AcOH plane, PVP/Ag = 1, τ = 1)

In [ ]:
for n in ('fig46_single_colour_acoh_pvp1_conc', 'fig46_single_tconv_acoh_pvp1_conc'):
    display(Image(str(FIG / f'{n}.png')))


## S23bis — the PVP and ascorbic-acid planes

In [ ]:
for n in ('pvp', 'asc_pvp1'):
    for kind in ('colour', 'tconv'):
        display(Image(str(FIG / f'fig46_single_{kind}_{n}_conc.png')))


## S24 — 9 randomly drawn recipes (seed 4)

In [ ]:
print(summary['si_recipes'])
display(Image(str(FIG / 'fig40_recipes9_SI.png')))


## S10 — endpoint degeneracy: same final spectrum, different growth

In [ ]:
print(summary['s10_pair'])
for n in ('fig43_pair_solo_spectra', 'fig43_pair_solo_area'):
    display(Image(str(FIG / f'{n}.png')))
